# BLOK 0 — Kütüphane Kontrolü + Ham Veri Gösterimi

**Amaç:**
1. Pipeline boyunca kullanacağımız kütüphaneler yüklü mü kontrol et
2. MIMIC-IV ham verilerini aç ve ilk hâlini gör
   - `discharge.csv.gz`       (taburcu özetleri)
   - `diagnoses_icd.csv.gz`   (ICD kodları)
   - `d_icd_diagnoses.csv.gz` (ICD kod → açıklama sözlüğü)

Bu blok HİÇBİR DOSYA OLUŞTURMAZ. Sadece ekrana yazar.

**Not:** Önce veriyi gözle inceleyeceğiz, masking kararını ona göre vereceğiz.

In [ ]:
# =====================================================================
# 1) KÜTÜPHANE KONTROLÜ
# =====================================================================
import importlib

print("=" * 70)
print("BLOK 0 — KÜTÜPHANE KONTROLÜ + HAM VERİ GÖSTERİMİ")
print("=" * 70)

print("\n[1/4] Kütüphane kontrolü...")
print("-" * 70)

# Pipeline boyunca lazım olacak kütüphaneler.
# Kategorilere ayrıldı: temel veri işleme + ML + LLM/QLoRA.
KUTUPHANELER = {
    # --- Temel veri işleme ---
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    # --- LLM / QLoRA ---
    "torch": "torch",
    "transformers": "transformers",
    "peft": "peft",
    "trl": "trl",
    "bitsandbytes": "bitsandbytes",
    "accelerate": "accelerate",
    "datasets": "datasets",
    # --- Diğer ---
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
}

eksik = []
for modul_adi, pip_adi in KUTUPHANELER.items():
    try:
        m = importlib.import_module(modul_adi)
        ver = getattr(m, "__version__", "?")
        print(f"   ✓ {modul_adi:<15s}  {ver}")
    except ImportError:
        print(f"   ✗ {modul_adi:<15s}  YOK  →  pip install {pip_adi}")
        eksik.append(pip_adi)

if eksik:
    print("\n   UYARI: Aşağıdaki kütüphaneler eksik. Yüklemek için:")
    print("   " + "pip install " + " ".join(eksik))
else:
    print("\n   ✓ Tüm kütüphaneler hazır.")

In [ ]:
# =====================================================================
# 2) DOSYA YOLLARI KONTROLÜ
# =====================================================================
import pandas as pd
from pathlib import Path

print("\n[2/4] Dosya yolları kontrolü...")
print("-" * 70)

YOLLAR = {
    "discharge": Path(
        r"./data/mimic-iv-note-deidentified-free-text-clinical-notes-2.2/note/discharge.csv.gz"
    ),
    "icd_kodlar": Path(
        r"./data/diagnoses_icd.csv.gz"
    ),
    "icd_sozluk": Path(
        r"./data/d_icd_diagnoses.csv.gz"
    ),
}

for ad, yol in YOLLAR.items():
    if yol.exists():
        boyut_mb = yol.stat().st_size / (1024 * 1024)
        print(f"   ✓ {ad:<12s}  {boyut_mb:>10.1f} MB   {yol.name}")
    else:
        print(f"   ✗ {ad:<12s}  BULUNAMADI:  {yol}")

In [ ]:
# =====================================================================
# 3) HAM VERİ GÖSTERİMİ — KÜÇÜK DOSYALAR
# =====================================================================
print("\n[3/4] Ham veri — küçük dosyalar (tam yüklenebilir)...")
print("-" * 70)

# --- ICD kod sözlüğü (küçük, hepsi yüklenebilir) ---
print("\n>>> d_icd_diagnoses.csv.gz  (ICD kod → açıklama sözlüğü)")
df_sozluk = pd.read_csv(YOLLAR["icd_sozluk"], compression="gzip")
print(f"   Satır sayısı: {len(df_sozluk):,}")
print(f"   Kolonlar    : {list(df_sozluk.columns)}")
print(f"   ICD-9 kod   : {(df_sozluk['icd_version'] == 9).sum():,}")
print(f"   ICD-10 kod  : {(df_sozluk['icd_version'] == 10).sum():,}")
print("\n   İlk 5 satır (ICD-10):")
print(df_sozluk[df_sozluk["icd_version"] == 10].head(5).to_string(index=False))

# --- diagnoses_icd ---
print("\n>>> diagnoses_icd.csv.gz  (her hadm için ICD kodları)")
df_icd = pd.read_csv(YOLLAR["icd_kodlar"], compression="gzip")
print(f"   Satır sayısı       : {len(df_icd):,}")
print(f"   Kolonlar           : {list(df_icd.columns)}")
print(f"   Benzersiz subject  : {df_icd['subject_id'].nunique():,}")
print(f"   Benzersiz hadm     : {df_icd['hadm_id'].nunique():,}")
print("\n   İlk 10 satır (tek bir hastanın yatışı):")
print(df_icd.head(10).to_string(index=False))

In [ ]:
# =====================================================================
# 4) HAM DISCHARGE VERİSİ — ÖRNEK NOTLAR
# =====================================================================
print("\n[4/4] Ham veri — discharge (1.1 GB, ilk 5 not)...")
print("-" * 70)

# discharge.csv.gz büyük, ilk birkaç satırı chunk olarak okuyalım.
print("\n>>> discharge.csv.gz  (taburcu özetleri)")
ilk_chunk = pd.read_csv(YOLLAR["discharge"], compression="gzip", nrows=5)
print(f"   Kolonlar: {list(ilk_chunk.columns)}")
print()

print("   İlk 5 kaydın metadata'sı (text hariç):")
meta_kolonlar = [k for k in ilk_chunk.columns if k != "text"]
print(ilk_chunk[meta_kolonlar].to_string(index=False))

# Bir tam taburcu metninin TAMAMINI göster — masking kararı için
print("\n   --- ÖRNEK 1 — TAM METİN (masking kararı için tüm metni görmek lazım) ---")
text_orn = str(ilk_chunk.iloc[0]["text"])
print(f"   note_id    : {ilk_chunk.iloc[0]['note_id']}")
print(f"   subject_id : {ilk_chunk.iloc[0]['subject_id']}")
print(f"   hadm_id    : {ilk_chunk.iloc[0]['hadm_id']}")
print(f"   text uzunluğu (karakter): {len(text_orn):,}")
print()
print("   " + "=" * 60)
for satir in text_orn.split("\n"):
    print(f"   | {satir}")
print("   " + "=" * 60)

In [ ]:
# Bir başka örnek — farklı yapıda olabilir
print("   --- ÖRNEK 2 — TAM METİN ---")
text_orn2 = str(ilk_chunk.iloc[1]["text"])
print(f"   note_id    : {ilk_chunk.iloc[1]['note_id']}")
print(f"   text uzunluğu (karakter): {len(text_orn2):,}")
print()
print("   " + "=" * 60)
for satir in text_orn2.split("\n"):
    print(f"   | {satir}")
print("   " + "=" * 60)

In [ ]:
# Üçüncü örnek — yapıyı kontrol
print("   --- ÖRNEK 3 — TAM METİN ---")
text_orn3 = str(ilk_chunk.iloc[2]["text"])
print(f"   note_id    : {ilk_chunk.iloc[2]['note_id']}")
print(f"   text uzunluğu (karakter): {len(text_orn3):,}")
print()
print("   " + "=" * 60)
for satir in text_orn3.split("\n"):
    print(f"   | {satir}")
print("   " + "=" * 60)

print("\n" + "=" * 70)
print("BLOK 0 TAMAMLANDI.")
print("=" * 70)
print("\nBu blok sadece veriye baktı, hiçbir dosya yazmadı.")
print("Sıradaki: Blok 1 — ICD dağılım keşfi + masking kararı.")

# BLOK 1A — Bölüm Yapısı Sağlık Kontrolü

**Kritik Soru:** 331k discharge notunun hepsi tutarlı bölümlere ayrılmış mı?
Eğer ayrılmamışsa, regex parse planımız patlar.

**Test edeceğimiz:**
1. Her bölüm notların yüzde kaçında var? (coverage)
2. Bizim **kritik bölümlerimiz** (Chief Complaint + HPI) ne kadarında birlikte var?
3. Parse fonksiyonu gerçekten temiz çıkartıyor mu?

**Test boyutu:** 5.000 not (rastgele).

**Karar:**
- Coverage >%90 → Plan sağlam, devam
- Coverage %80-90 → Plan tutar, kayıplar limitation
- Coverage <%80 → Plan revize et

In [ ]:
# 1) ÖRNEK YÜKLEME — 5000 NOT
import pandas as pd
import re
from pathlib import Path

DISCHARGE_YOL = Path(r"./data/mimic-iv-note-deidentified-free-text-clinical-notes-2.2/note/discharge.csv.gz")

print("=" * 70)
print("BLOK 1A — BÖLÜM YAPISI SAĞLIK KONTROLÜ")
print("=" * 70)

print("\n[1/5] 20.000 not yükleniyor, 5000 rastgele örnek alınıyor...")
df = pd.read_csv(DISCHARGE_YOL, compression="gzip", nrows=20000,
                 usecols=["note_id", "subject_id", "hadm_id", "text"])
df_test = df.sample(n=5000, random_state=42).reset_index(drop=True)
print(f"   Örnek: {len(df_test):,} not")
print(f"   Ortalama text uzunluğu: {df_test['text'].str.len().mean():,.0f} karakter")
print(f"   Min - Max: {df_test['text'].str.len().min():,} - {df_test['text'].str.len().max():,}")

In [ ]:
# 2) COVERAGE — HER BÖLÜM YÜZDE KAÇ NOTTA?
print("\n[2/5] Bölüm coverage testi...")
print("-" * 70)

BOLUMLER = {
    "Chief Complaint":            [r"Chief Complaint\s*:"],
    "Major Surgical Procedure":   [r"Major Surgical or Invasive Procedure\s*:"],
    "History of Present Illness": [r"History of Present Illness\s*:", r"HPI\s*:"],
    "Past Medical History":       [r"Past Medical History\s*:", r"PMH\s*:"],
    "Social History":             [r"Social History\s*:"],
    "Family History":             [r"Family History\s*:"],
    "Physical Exam":              [r"Physical Exam\s*:", r"Physical Examination\s*:"],
    "Pertinent Results":          [r"Pertinent Results\s*:", r"Lab(?:oratory)?\s+Results?\s*:", r"Labs\s*:"],
    "Brief Hospital Course":      [r"Brief Hospital Course\s*:", r"Hospital Course\s*:"],
    "Medications on Admission":   [r"Medications on Admission\s*:", r"Admission Medications\s*:"],
    "Discharge Medications":      [r"Discharge Medications\s*:"],
    "Discharge Disposition":      [r"Discharge Disposition\s*:"],
    "Discharge Diagnosis":        [r"Discharge Diagnosis\s*:", r"Final Diagnosis\s*:"],
    "Discharge Condition":        [r"Discharge Condition\s*:"],
    "Discharge Instructions":     [r"Discharge Instructions\s*:"],
    "Followup Instructions":      [r"Followup Instructions\s*:", r"Follow-?up\s*:"],
}

sonuc = {}
for bolum_adi, patternler in BOLUMLER.items():
    var_mi = pd.Series(False, index=df_test.index)
    for p in patternler:
        var_mi |= df_test["text"].str.contains(p, regex=True, case=False, na=False)
    sonuc[bolum_adi] = int(var_mi.sum())

n = len(df_test)
print(f"\n{'Bölüm':<32s} {'Var':>6s} {'Yüzde':>8s}  Bar")
print("-" * 70)
for bolum_adi, sayi in sorted(sonuc.items(), key=lambda x: -x[1]):
    pct = 100 * sayi / n
    isaret = "++" if pct >= 95 else ("+" if pct >= 80 else ("~" if pct >= 50 else "-"))
    bar = "#" * int(pct / 2)
    print(f"{isaret:>2s}  {bolum_adi:<30s} {sayi:>6,} {pct:>7.1f}%  {bar}")

In [ ]:
# 3) KRİTİK YOL — CC + HPI birlikte yüzde kaç?
print("\n[3/5] Kritik yol kontrolü...")
print("-" * 70)

cc_var  = df_test["text"].str.contains(r"Chief Complaint\s*:", regex=True, case=False, na=False)
hpi_var = df_test["text"].str.contains(r"History of Present Illness\s*:", regex=True, case=False, na=False)
pmh_var = df_test["text"].str.contains(r"Past Medical History\s*:", regex=True, case=False, na=False)
dx_var  = df_test["text"].str.contains(r"Discharge Diagnosis\s*:|Final Diagnosis\s*:", regex=True, case=False, na=False)

print(f"   CC var       : {cc_var.sum():>5,} ({100*cc_var.mean():.1f}%)")
print(f"   HPI var      : {hpi_var.sum():>5,} ({100*hpi_var.mean():.1f}%)")
print(f"   PMH var      : {pmh_var.sum():>5,} ({100*pmh_var.mean():.1f}%)")
print(f"   Discharge Dx : {dx_var.sum():>5,} ({100*dx_var.mean():.1f}%)")
print()
print(f"   V1 (CC + HPI) birlikte           : {(cc_var & hpi_var).sum():>5,} ({100*(cc_var & hpi_var).mean():.1f}%)")
print(f"   V2 (CC + HPI + PMH) birlikte     : {(cc_var & hpi_var & pmh_var).sum():>5,} ({100*(cc_var & hpi_var & pmh_var).mean():.1f}%)")
print(f"   Hepsi (CC + HPI + PMH + Dx)      : {(cc_var & hpi_var & pmh_var & dx_var).sum():>5,} ({100*(cc_var & hpi_var & pmh_var & dx_var).mean():.1f}%)")

kayip = df_test[~(cc_var & hpi_var)].copy()
if len(kayip) > 0:
    print(f"\n   Kayıp not sayısı: {len(kayip):,}")
    print(f"   Kayıp notların ort. uzunluğu: {kayip['text'].str.len().mean():,.0f} karakter")
    print("\n   Örnek kayıp not (ilk 600 karakter):")
    print("   " + "-" * 60)
    ornek = kayip.iloc[0]['text'][:600]
    for s in ornek.split("\n")[:25]:
        print(f"   | {s}")
    print("   " + "-" * 60)

In [ ]:
# 4) PARSE TESTİ — HPI çıkarma çalışıyor mu?
print("\n[4/5] HPI çıkarma testi...")
print("-" * 70)

SONRAKI = (
    r"Past Medical History|Social History|Family History|Physical Exam|"
    r"Physical Examination|Pertinent Results|Lab Results|Brief Hospital Course|"
    r"Hospital Course|Medications on Admission|Discharge Medications|"
    r"Discharge Disposition|Discharge Diagnosis|Final Diagnosis|"
    r"Discharge Condition|Discharge Instructions|Followup Instructions"
)

def hpi_cikart(text):
    pattern = rf"History of Present Illness\s*:(.*?)(?={SONRAKI}|$)"
    m = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else None

df_test["hpi"] = df_test["text"].apply(hpi_cikart)
hpi_basarili = df_test["hpi"].notna()
print(f"   HPI çıkarıldı: {hpi_basarili.sum():,} / {len(df_test):,} ({100*hpi_basarili.mean():.1f}%)")

if hpi_basarili.sum() > 0:
    L = df_test.loc[hpi_basarili, "hpi"].str.len()
    print(f"   HPI ortalama uzunluk: {L.mean():,.0f} karakter")
    print(f"   HPI medyan          : {L.median():,.0f}")
    print(f"   HPI min - max       : {L.min():,} - {L.max():,}")
    print(f"   < 50 karakter HPI   : {(L < 50).sum():,} ({100*(L < 50).sum()/len(L):.1f}%)")
    print(f"   > 5000 karakter HPI : {(L > 5000).sum():,} ({100*(L > 5000).sum()/len(L):.1f}%)")
    
    # Örnek bir HPI
    print("\n   Örnek çıkarılmış HPI (ilk 400 karakter):")
    ornek_hpi = df_test.loc[hpi_basarili, "hpi"].iloc[0][:400]
    print("   " + "-" * 60)
    for s in ornek_hpi.split("\n"):
        print(f"   | {s}")
    print("   " + "-" * 60)

In [ ]:
# 5) KARAR
print("\n[5/5] KARAR")
print("=" * 70)

v1_oran = 100 * (cc_var & hpi_var).mean()
v2_oran = 100 * (cc_var & hpi_var & pmh_var).mean()
hpi_extract_oran = 100 * hpi_basarili.mean()

print(f"\n   V1 (CC + HPI) coverage      : {v1_oran:.1f}%")
print(f"   V2 (CC + HPI + PMH) coverage: {v2_oran:.1f}%")
print(f"   HPI parse başarısı          : {hpi_extract_oran:.1f}%")
print()

if v1_oran >= 95 and hpi_extract_oran >= 95:
    karar = "++ MUKEMMEL — Plan saglam, devam."
elif v1_oran >= 85 and hpi_extract_oran >= 85:
    karar = "+ IYI — Plan calisir, kucuk kayiplar limitation."
elif v1_oran >= 70:
    karar = "~ ORTA — Plan riskli, alternatif dusunulmeli."
else:
    karar = "- PLANI REVIZE ET."
print(f"   {karar}")

if v1_oran >= 80:
    kullanilabilir = int(254377 * v1_oran / 100)
    print(f"\n   ICD-10 hadm × CC+HPI oran tahmini: {kullanilabilir:,} not")
    yeterli = kullanilabilir >= 15000
    print(f"   11.000 hedefi icin yeterli mi: {'EVET' if yeterli else 'BORDERLINE'}")

print()
print("=" * 70)
print("BLOK 1A TAMAMLANDI")
print("=" * 70)

# BLOK 1B — ICD Eşleme + Subject_id Keşfi

**Amaç:** Veri hazırlığına geçmeden önce 4 kritik sayıyı bulmak.

1. **11 sınıf gerçek dağılımı** (PRIMARY ICD-10 × prefix mapper)
2. **Her sınıftan 3 örnek kod + açıklama** (d_icd_diagnoses sözlüğünden)
3. **discharge ↔ diagnoses_icd intersection** — kaç hadm her ikisinde de var
4. **Subject_id başına yatış dağılımı** — split planı için

**Hedef bütçe (sınıf başı):**
- 200 test + 80 val + 800 train (S3 max) = **1.080 ihtiyaç**

Bu blok HÂLÂ KEŞİF — dosya yazmaz.

In [ ]:
# 1) ICD-10 → 11 SINIF EŞLEME FONKSİYONU
import pandas as pd
from pathlib import Path

ICD_YOL = Path(r"./data/diagnoses_icd.csv.gz")
SOZLUK_YOL = Path(r"./data/d_icd_diagnoses.csv.gz")
DISCHARGE_YOL = Path(r"./data/mimic-iv-note-deidentified-free-text-clinical-notes-2.2/note/discharge.csv.gz")

print("=" * 70)
print("BLOK 1B — ICD EŞLEME + SUBJECT_ID KEŞFİ")
print("=" * 70)

# 11-sınıf etiketleyici
def icd10_to_label(code):
    """ICD-10 kodunu 11 sınıftan birine eşle. Bilinmiyorsa OTHER."""
    if not isinstance(code, str):
        return "OTHER"
    code = code.upper().strip()
    p3 = code[:3]
    if p3 in {"J12", "J13", "J14", "J15", "J16", "J17", "J18"}:
        return "PNEUMONIA"
    if p3 == "I50":
        return "HEART_FAILURE"
    if p3 in {"A40", "A41"}:
        return "SEPSIS"
    if p3 == "N17":
        return "AKI"
    if code.startswith("N390"):
        return "UTI"
    if p3 in {"I60", "I61", "I62", "I63"}:
        return "STROKE"
    if any(code.startswith(x) for x in ("K920", "K921", "K922")):
        return "GI_BLEEDING"
    if p3 == "J44":
        return "COPD_EXACERBATION"
    if (p3 in {"E10", "E11"}) and len(code) >= 5 and code[4] in "01":
        return "DIABETES_COMPLICATION"
    if p3 == "I48":
        return "ATRIAL_FIBRILLATION"
    return "OTHER"

# Test: birkaç örnekle doğrula
test_kodlar = {"J189": "PNEUMONIA", "I509": "HEART_FAILURE", "A419": "SEPSIS",
               "N179": "AKI", "N390": "UTI", "I639": "STROKE",
               "K920": "GI_BLEEDING", "J441": "COPD_EXACERBATION",
               "E110": "DIABETES_COMPLICATION", "I480": "ATRIAL_FIBRILLATION",
               "Z5111": "OTHER", "F329": "OTHER"}
print("\n[1/6] Eşleme fonksiyonu testi:")
tum_dogru = True
for kod, beklenen in test_kodlar.items():
    sonuc = icd10_to_label(kod)
    isaret = "OK" if sonuc == beklenen else "HATA"
    if sonuc != beklenen:
        tum_dogru = False
    print(f"   {isaret:>4s}  {kod:<8s} → {sonuc:<25s} (beklenen: {beklenen})")
print(f"\n   {'Tüm testler geçti.' if tum_dogru else 'TEST HATASI VAR'}")

In [ ]:
# 2) PRIMARY ICD-10 DAĞILIMI — bağımsız doğrulama
print("\n[2/6] PRIMARY ICD-10 dağılımı çıkartılıyor...")
print("-" * 70)

df_icd = pd.read_csv(ICD_YOL, compression="gzip")
print(f"   Toplam ICD kaydı: {len(df_icd):,}")

# Sadece PRIMARY (seq_num=1) + ICD-10
primary10 = df_icd[(df_icd["seq_num"] == 1) & (df_icd["icd_version"] == 10)].copy()
print(f"   PRIMARY (seq_num=1) ICD-10: {len(primary10):,}")

# Sınıfa eşle
primary10["label"] = primary10["icd_code"].apply(icd10_to_label)
dagilim = primary10["label"].value_counts()
n_total = len(primary10)

siralama = ["PNEUMONIA", "HEART_FAILURE", "SEPSIS", "AKI", "UTI",
            "STROKE", "GI_BLEEDING", "COPD_EXACERBATION",
            "DIABETES_COMPLICATION", "ATRIAL_FIBRILLATION", "OTHER"]

ihtiyac = 1080  # 200 test + 80 val + 800 train
print(f"\n   Sınıf başı ihtiyaç: {ihtiyac} (200 test + 80 val + 800 train)")
print()
print(f"   {'Sınıf':<25s} {'Mevcut':>8s} {'%':>7s}  {'Yeterli mi?':<12s}")
print("   " + "-" * 60)
for cls in siralama:
    n = int(dagilim.get(cls, 0))
    pct = 100 * n / n_total if n_total else 0
    if cls == "OTHER":
        durum = "FAZLASIYLA"
    elif n >= ihtiyac:
        durum = "OK"
    elif n >= ihtiyac * 0.8:
        durum = "SINIRDA"
    else:
        durum = "YETERSIZ"
    print(f"   {cls:<25s} {n:>8,} {pct:>6.2f}%  {durum:<12s}")

print(f"\n   Toplam PRIMARY ICD-10: {n_total:,}")

In [ ]:
# 3) HER SINIFTAN 3 ÖRNEK ICD KODU + AÇIKLAMA
print("\n[3/6] Her sınıftan en sık 3 ICD kodu + açıklamaları...")
print("-" * 70)

df_sozluk = pd.read_csv(SOZLUK_YOL, compression="gzip")
sozluk10 = df_sozluk[df_sozluk["icd_version"] == 10].set_index("icd_code")["long_title"].to_dict()

for cls in siralama:
    alt = primary10[primary10["label"] == cls]
    if len(alt) == 0:
        continue
    en_sik = alt["icd_code"].value_counts().head(3)
    print(f"\n   ▶ {cls}  (toplam {len(alt):,} kayıt)")
    for kod, sayi in en_sik.items():
        aciklama = sozluk10.get(kod, "???")
        if len(aciklama) > 65:
            aciklama = aciklama[:62] + "..."
        print(f"      {kod:<8s} ({sayi:>5,}x)  {aciklama}")

In [ ]:
# 4) DISCHARGE ↔ ICD INTERSECTION — kaç hadm hem text hem ICD-10 PRIMARY?
print("\n[4/6] Discharge ↔ diagnoses_icd intersection (sadece hadm_id'ler)...")
print("-" * 70)

# discharge.csv'den sadece hadm_id'leri al (text yok, hızlı)
print("   discharge.csv.gz'den hadm_id'ler okunuyor (text yok)...")
df_disc_meta = pd.read_csv(DISCHARGE_YOL, compression="gzip",
                            usecols=["note_id", "subject_id", "hadm_id"])
print(f"   Toplam discharge notu: {len(df_disc_meta):,}")
print(f"   Benzersiz hadm_id    : {df_disc_meta['hadm_id'].nunique():,}")
print(f"   Benzersiz subject_id : {df_disc_meta['subject_id'].nunique():,}")

# Set kesişimi
discharge_hadms = set(df_disc_meta["hadm_id"].dropna().astype(int))
primary10_hadms = set(primary10["hadm_id"].astype(int))

kesisim = discharge_hadms & primary10_hadms
sadece_disc = discharge_hadms - primary10_hadms
sadece_icd = primary10_hadms - discharge_hadms

print(f"\n   Discharge hadm    : {len(discharge_hadms):>7,}")
print(f"   PRIMARY ICD-10 hadm: {len(primary10_hadms):>7,}")
print(f"   KESİŞİM (KULLANILABILIR): {len(kesisim):>7,}")
print(f"   Sadece discharge   : {len(sadece_disc):>7,} (ICD-9 ya da seq_num>1)")
print(f"   Sadece ICD-10      : {len(sadece_icd):>7,} (discharge notu yok)")

# Şimdi: KESİŞİM'in 11 sınıf dağılımı (gerçek kullanılabilir veri)
kesisim_df = primary10[primary10["hadm_id"].isin(kesisim)].copy()
print(f"\n   ★ KESİŞİM'de 11 sınıf dağılımı (gerçek kullanılabilir veri):")
print("   " + "-" * 60)
kdag = kesisim_df["label"].value_counts()
tum_yeterli = True
for cls in siralama:
    n = int(kdag.get(cls, 0))
    if cls == "OTHER":
        durum = "FAZLASIYLA"
    elif n >= ihtiyac:
        durum = "OK"
    elif n >= ihtiyac * 0.8:
        durum = "SINIRDA"
    else:
        durum = "YETERSIZ"
        tum_yeterli = False
    print(f"   {cls:<25s} {n:>7,}  {durum}")
print(f"\n   Tüm sınıflar 1080 ihtiyacı karşılıyor mu? {'EVET ✓' if tum_yeterli else 'HAYIR — düzeltme gerek'}")

In [ ]:
# 5) SUBJECT_ID BAŞINA YATIŞ DAĞILIMI (split planı için)
print("\n[5/6] Subject_id başına yatış dağılımı...")
print("-" * 70)

# discharge'da bir hastanın kaç farklı yatışı var?
yatislar = df_disc_meta.groupby("subject_id")["hadm_id"].nunique()
print(f"   Benzersiz hasta (subject_id): {len(yatislar):,}")
print(f"   Toplam yatış sayısı         : {yatislar.sum():,}")
print()
print(f"   Hasta başı yatış ortalama   : {yatislar.mean():.2f}")
print(f"   Hasta başı yatış medyan     : {yatislar.median():.0f}")
print(f"   Hasta başı yatış max        : {yatislar.max()}")
print()
print("   Yatış sayısı dağılımı (kaç hasta kaç yatış yapmış):")
yatis_dagilimi = yatislar.value_counts().sort_index().head(10)
for n_yatis, n_hasta in yatis_dagilimi.items():
    bar = "█" * min(40, int(n_hasta / 1000))
    print(f"   {n_yatis:>3}  yatış: {n_hasta:>7,} hasta  {bar}")

# 5+ yatış yapanlar
cok_yatis = (yatislar >= 5).sum()
print(f"\n   5+ yatış yapan hasta: {cok_yatis:,} ({100*cok_yatis/len(yatislar):.1f}%)")
print("   ★ Subject_id-bazlı split bu yüzden ZORUNLU.")
print("     Aksi takdirde aynı hastanın train ve test'e dağılması = sızıntı.")

In [ ]:
# 6) KARAR ÖZETİ
print("\n[6/6] BLOK 1B KARAR ÖZETİ")
print("=" * 70)

print(f"\n   Eşleme fonksiyonu testleri      : OK")
print(f"   PRIMARY ICD-10 toplam           : {n_total:,}")
print(f"   Discharge ∩ PRIMARY ICD-10 hadm : {len(kesisim):,}")
print(f"   Benzersiz hasta (discharge)     : {len(yatislar):,}")
print()

# CC+HPI coverage'ı uygula (Blok 1A'dan: %95.7)
cc_hpi_coverage = 0.957
parse_basari = 0.96
uygulanabilir_oran = cc_hpi_coverage * parse_basari
tahmini_kullanılabilir = int(len(kesisim) * uygulanabilir_oran)
print(f"   Blok 1A coverage: CC+HPI %95.7 × parse %96 = %{100*uygulanabilir_oran:.1f}")
print(f"   ⇒ Tahmini kullanılabilir not  : {tahmini_kullanılabilir:,}")

# Sınıf bazında düzeltme
print(f"\n   Sınıf bazında tahmini kullanılabilir (kesişim × %{100*uygulanabilir_oran:.0f}):")
print("   " + "-" * 60)
tum_ok = True
for cls in siralama:
    ham = int(kdag.get(cls, 0))
    tahmini = int(ham * uygulanabilir_oran)
    if cls == "OTHER":
        durum = "FAZLASIYLA"
    elif tahmini >= ihtiyac:
        durum = "OK"
    elif tahmini >= ihtiyac * 0.85:
        durum = "SINIRDA — yine OK"
    else:
        durum = "DIKKAT"
        tum_ok = False
    print(f"   {cls:<25s} ham:{ham:>6,} → kullanılabilir:{tahmini:>6,}  {durum}")

print("\n" + "=" * 70)
if tum_ok:
    print("   ✓ TÜM SINIFLAR İÇİN VERİ YETERLİ — Blok 2'ye geçilebilir.")
else:
    print("   ⚠ Bazı sınıflar sınırda — bütçe revize edilebilir.")
print("=" * 70)
print("\nBLOK 1B TAMAMLANDI. Sıradaki: Blok 2 — Etiketleme (DOSYA YAZACAK).")

# BLOK 1C — Bug Fix + Yeni Bütçe Doğrulama

**Düzeltmeler:**
1. `E110` (Type 2 DM with ketoacidosis) yanlış OTHER'a düşüyordu — **DIABETES_COMPLICATION** olmalı
2. Eşleme: `code[3] in '01'` (E110, E101 gibi 4 karakterli kodlar da yakalanır)

**Yeni bütçe (Seçenek B):**
- Test: **150/sınıf** (toplam 1.650)
- Val:  **50/sınıf** (toplam 550)
- Train senaryoları (nested):
  - S1 = 100/sınıf = 1.100
  - S2 = 200/sınıf = 2.200
  - S3 = 400/sınıf = 4.400 (max)
- **Sınıf başı ihtiyaç:** 150 + 50 + 400 = **600**

In [ ]:
# 1) DÜZELTİLMİŞ EŞLEME FONKSİYONU
print("=" * 70)
print("BLOK 1C — BUG FIX + YENİ BÜTÇE")
print("=" * 70)

def icd10_to_label_v2(code):
    """ICD-10 → 11 sınıf eşleme (v2: E110 bug düzeltildi)."""
    if not isinstance(code, str):
        return "OTHER"
    code = code.upper().strip()
    p3 = code[:3]
    if p3 in {"J12", "J13", "J14", "J15", "J16", "J17", "J18"}:
        return "PNEUMONIA"
    if p3 == "I50":
        return "HEART_FAILURE"
    if p3 in {"A40", "A41"}:
        return "SEPSIS"
    if p3 == "N17":
        return "AKI"
    if code.startswith("N390"):
        return "UTI"
    if p3 in {"I60", "I61", "I62", "I63"}:
        return "STROKE"
    if any(code.startswith(x) for x in ("K920", "K921", "K922")):
        return "GI_BLEEDING"
    if p3 == "J44":
        return "COPD_EXACERBATION"
    # FİX: code[3] kontrolü (E110, E101 gibi 4-char kodlar da yakalanır)
    if p3 in {"E10", "E11"} and len(code) >= 4 and code[3] in "01":
        return "DIABETES_COMPLICATION"
    if p3 == "I48":
        return "ATRIAL_FIBRILLATION"
    return "OTHER"

# Genişletilmiş test
test_kodlar = {
    "J189": "PNEUMONIA", "I509": "HEART_FAILURE", "A419": "SEPSIS",
    "N179": "AKI", "N390": "UTI", "I639": "STROKE",
    "K920": "GI_BLEEDING", "J441": "COPD_EXACERBATION",
    # DIABETES — eski testler + yeni 4-char varyantları
    "E110":  "DIABETES_COMPLICATION",  # düzeltildi
    "E1010": "DIABETES_COMPLICATION",
    "E1110": "DIABETES_COMPLICATION",
    "E101":  "DIABETES_COMPLICATION",  # 4-char, E101 → kapsanmalı
    "E111":  "DIABETES_COMPLICATION",  # 4-char, E111 → kapsanmalı
    "E1151": "DIABETES_COMPLICATION",
    "E119":  "OTHER",  # E11.9 = DM uncomplicated, primary değil DKA
    "E114":  "OTHER",  # E11.4 = DM with neuropathy, akut DKA değil
    "I480":  "ATRIAL_FIBRILLATION",
    "Z5111": "OTHER",
    "F329":  "OTHER",
}

print("\n[1/3] Eşleme testleri:")
tum_dogru = True
for kod, beklenen in test_kodlar.items():
    sonuc = icd10_to_label_v2(kod)
    isaret = "OK" if sonuc == beklenen else "HATA"
    if sonuc != beklenen:
        tum_dogru = False
    print(f"   {isaret:>4s}  {kod:<8s} → {sonuc:<25s} (beklenen: {beklenen})")
print(f"\n   {'Tüm testler geçti.' if tum_dogru else 'TEST HATASI VAR'}")

In [ ]:
# 2) YENİ EŞLEME İLE DAĞILIM (kesişim üzerinde)
print("\n[2/3] Düzeltilmiş eşleme ile kesişim dağılımı...")
print("-" * 70)

# Yeniden etiketle (tüm primary10 için, eski df Blok 1B'den geliyor)
primary10["label_v2"] = primary10["icd_code"].apply(icd10_to_label_v2)

# Kesişim üzerinde
kesisim_df_v2 = primary10[primary10["hadm_id"].isin(kesisim)].copy()
kesisim_df_v2["label"] = kesisim_df_v2["label_v2"]  # eski label kolonunu güncelle

kdag_v2 = kesisim_df_v2["label"].value_counts()

# YENİ BÜTÇE: 150 test + 50 val + 400 train = 600 ihtiyaç
ihtiyac_v2 = 600
cc_hpi_coverage = 0.957
parse_basari = 0.96
uyg_oran = cc_hpi_coverage * parse_basari

print(f"\n   Yeni sınıf başı ihtiyaç: {ihtiyac_v2}")
print(f"     - 150 test")
print(f"     -  50 val")
print(f"     - 400 train (S3 max)")
print(f"   Coverage çarpanı: {uyg_oran:.3f} (CC+HPI %95.7 × parse %96)")
print()
print("   Sınıf bazında DÜZELTİLMİŞ kullanılabilir tahmini:")
print("   " + "-" * 60)
tum_ok = True
for cls in siralama:
    ham = int(kdag_v2.get(cls, 0))
    tahmini = int(ham * uyg_oran)
    fazla = tahmini - ihtiyac_v2
    if cls == "OTHER":
        durum = "FAZLASIYLA"
    elif tahmini >= ihtiyac_v2:
        durum = f"OK (+{fazla} fazla)"
    elif tahmini >= ihtiyac_v2 * 0.95:
        durum = f"SINIRDA ({fazla})"
    else:
        durum = f"YETERSIZ ({fazla})"
        tum_ok = False
    print(f"   {cls:<25s} ham:{ham:>6,} → kullanılabilir:{tahmini:>6,}  {durum}")

print()
print("=" * 70)
if tum_ok:
    print("   ✓ TÜM SINIFLAR YETERLİ — Blok 2'ye geçilebilir.")
else:
    print("   ⚠ Hâlâ sınırda olan sınıf var — bütçe revizyonu düşünülmeli.")
print("=" * 70)

In [ ]:
# 3) ÖZET
print("\n[3/3] BLOK 1C ÖZETİ")
print("=" * 70)

# Eski vs yeni karşılaştırma (DIABETES için)
eski_dia = (primary10["icd_code"].apply(lambda c:
    isinstance(c, str) and c.upper()[:3] in {"E10", "E11"} and len(c) >= 5 and c[4] in "01")).sum()
yeni_dia = (primary10["label_v2"] == "DIABETES_COMPLICATION").sum()

print(f"\n   DIABETES_COMPLICATION sayısı:")
print(f"     Eski (bug)  : {eski_dia:,}")
print(f"     Yeni (fix)  : {yeni_dia:,}")
print(f"     Fark        : +{yeni_dia - eski_dia:,}")

print(f"\n   Yeni bütçe ile sınıf başı toplam ihtiyaç: 600")
print(f"   Toplam veri ihtiyacı: 11 × 600 = 6.600 not")
print(f"   Test: 11 × 150 = 1.650")
print(f"   Val : 11 ×  50 =   550")
print(f"   S3  : 11 × 400 = 4.400 (en büyük QLoRA train)")
print(f"   S2  : 11 × 200 = 2.200")
print(f"   S1  : 11 × 100 = 1.100")
print()
print("=" * 70)
print("BLOK 1C TAMAMLANDI. icd10_to_label_v2 kullanılacak Blok 2'de.")
print("=" * 70)

# BLOK 1D — DIABETES Kapsamı Genişletme + Final Bütçe

**Sorun:** v2 fonksiyonu sadece DKA (E10.1x / E11.1x) yakalıyordu — kesişim sonrası 409 örnek kaldı, 600 ihtiyaç sınırın altında.

**Çözüm (Yol 1):** Akut diyabetik admission sebeplerinin tamamını dahil et:
- **Ketoacidosis (DKA):** E10.1x / E11.0x / E11.1x / E13.0x / E13.1x
- **Hyperosmolar (HHS):** E11.00, E13.00
- **Hypoglycemia:** E10.64x / E11.64x / E13.64x
- **Hyperglycemia:** E10.65x / E11.65x / E13.65x

**Klinik anlam:** Hepsi acil servise yatış sebebi olabilen **akut diyabetik komplikasyonlar**.
Tezde "DIABETES_COMPLICATION = acute diabetic complications requiring admission (DKA, HHS, severe hypo/hyperglycemia)" diye yazılacak.

In [ ]:
# 1) icd10_to_label_v3 — DIABETES kapsamı genişletilmiş
print("=" * 70)
print("BLOK 1D — DIABETES KAPSAMI GENISLETME (v3)")
print("=" * 70)

def icd10_to_label_v3(code):
    """ICD-10 → 11 sınıf eşleme (v3: DIABETES kapsamı genişletildi)."""
    if not isinstance(code, str):
        return "OTHER"
    code = code.upper().strip()
    p3 = code[:3]
    
    if p3 in {"J12", "J13", "J14", "J15", "J16", "J17", "J18"}:
        return "PNEUMONIA"
    if p3 == "I50":
        return "HEART_FAILURE"
    if p3 in {"A40", "A41"}:
        return "SEPSIS"
    if p3 == "N17":
        return "AKI"
    if code.startswith("N390"):
        return "UTI"
    if p3 in {"I60", "I61", "I62", "I63"}:
        return "STROKE"
    if any(code.startswith(x) for x in ("K920", "K921", "K922")):
        return "GI_BLEEDING"
    if p3 == "J44":
        return "COPD_EXACERBATION"
    
    # GENISLETILMIS DIABETES — akut komplikasyonlar
    if p3 in {"E10", "E11", "E13"}:
        # 4-char kontrol: E10.0/1 DKA/HHS, E11.0/1, E13.0/1
        if len(code) >= 4 and code[3] in "01":
            return "DIABETES_COMPLICATION"
        # 5-char kontrol: E10.64/65 (hypo/hyperglycemia), E11.64/65, E13.64/65
        if len(code) >= 5 and code[3:5] in {"64", "65"}:
            return "DIABETES_COMPLICATION"
    
    if p3 == "I48":
        return "ATRIAL_FIBRILLATION"
    return "OTHER"

# Genişletilmiş test
test_kodlar = {
    # Önceki testler
    "J189": "PNEUMONIA", "I509": "HEART_FAILURE", "A419": "SEPSIS",
    "N179": "AKI", "N390": "UTI", "I639": "STROKE",
    "K920": "GI_BLEEDING", "J441": "COPD_EXACERBATION",
    "I480": "ATRIAL_FIBRILLATION", "Z5111": "OTHER", "F329": "OTHER",
    
    # DIABETES — eski yakalananlar (DKA)
    "E110":  "DIABETES_COMPLICATION",
    "E1010": "DIABETES_COMPLICATION",
    "E1110": "DIABETES_COMPLICATION",
    "E101":  "DIABETES_COMPLICATION",
    "E111":  "DIABETES_COMPLICATION",
    "E1011": "DIABETES_COMPLICATION",
    "E1111": "DIABETES_COMPLICATION",
    
    # DIABETES — yeni eklenenler (HHS, hypo/hyperglycemia)
    "E1100": "DIABETES_COMPLICATION",  # HHS
    "E1101": "DIABETES_COMPLICATION",  # HHS with coma
    "E1064": "DIABETES_COMPLICATION",  # Type 1 hypoglycemia
    "E1065": "DIABETES_COMPLICATION",  # Type 1 hyperglycemia
    "E1164": "DIABETES_COMPLICATION",  # Type 2 hypoglycemia
    "E1165": "DIABETES_COMPLICATION",  # Type 2 hyperglycemia
    "E1364": "DIABETES_COMPLICATION",  # Other DM hypoglycemia
    
    # DIABETES — OTHER kalmalı (kronik komplikasyonlar)
    "E119":  "OTHER",   # E11.9 = uncomplicated
    "E114":  "OTHER",   # E11.4 = neuropathy
    "E1151": "OTHER",   # E11.51 = peripheral angiopathy
    "E115":  "OTHER",   # E11.5 chronic complication
    "E116":  "OTHER",   # E11.6 other complications
}

print("\n[1/3] Genişletilmiş eşleme testleri:")
tum_dogru = True
for kod, beklenen in test_kodlar.items():
    sonuc = icd10_to_label_v3(kod)
    isaret = "OK" if sonuc == beklenen else "HATA"
    if sonuc != beklenen:
        tum_dogru = False
    print(f"   {isaret:>4s}  {kod:<8s} → {sonuc:<25s} (beklenen: {beklenen})")
print(f"\n   {'+ Tüm testler geçti.' if tum_dogru else '- TEST HATASI VAR'}")

In [ ]:
# 2) v3 ile yeniden dağılım — DIABETES sayısı arttı mı?
print("\n[2/3] v3 eşleme ile dağılım...")
print("-" * 70)

primary10["label_v3"] = primary10["icd_code"].apply(icd10_to_label_v3)
kesisim_df_v3 = primary10[primary10["hadm_id"].isin(kesisim)].copy()
kesisim_df_v3["label"] = kesisim_df_v3["label_v3"]
kdag_v3 = kesisim_df_v3["label"].value_counts()

ihtiyac_v3 = 600
uyg_oran = 0.957 * 0.96  # CC+HPI × parse

print(f"   Sınıf başı ihtiyaç: {ihtiyac_v3} (150t + 50v + 400tr)")
print(f"   Coverage: {uyg_oran:.3f}")
print()
print("   Sınıf bazında v3 ile durum:")
print("   " + "-" * 60)
tum_ok = True
ozet = []
for cls in siralama:
    ham = int(kdag_v3.get(cls, 0))
    tahmini = int(ham * uyg_oran)
    fazla = tahmini - ihtiyac_v3
    if cls == "OTHER":
        durum = "FAZLASIYLA"
    elif tahmini >= ihtiyac_v3:
        durum = f"OK (+{fazla})"
    elif tahmini >= ihtiyac_v3 * 0.95:
        durum = f"SINIRDA ({fazla})"
    else:
        durum = f"YETERSIZ ({fazla})"
        tum_ok = False
    ozet.append((cls, ham, tahmini, durum))
    print(f"   {cls:<25s} ham:{ham:>6,} → kullanılabilir:{tahmini:>6,}  {durum}")

print()
print("=" * 70)
if tum_ok:
    print("   + TÜM SINIFLAR YETERLİ — Blok 2'ye geçilebilir.")
else:
    print("   - Hâlâ sınırda olan sınıf var.")
print("=" * 70)

In [ ]:
# 3) DIABETES_COMPLICATION içindeki en sık kodlar
print("\n[3/3] DIABETES_COMPLICATION sınıfının yapısı (v3)")
print("-" * 70)

dia_alt = kesisim_df_v3[kesisim_df_v3["label"] == "DIABETES_COMPLICATION"].copy()
print(f"   Toplam DIABETES_COMPLICATION (kesişim): {len(dia_alt):,}")
print(f"   Eski v2'de              : 409")
print(f"   Yeni v3'te              : {len(dia_alt)}")
print(f"   Artış                    : +{len(dia_alt) - 409}")
print()

print("   En sık 10 ICD kodu:")
en_sik = dia_alt["icd_code"].value_counts().head(10)
for kod, sayi in en_sik.items():
    aciklama = sozluk10.get(kod, "???")
    if len(aciklama) > 65:
        aciklama = aciklama[:62] + "..."
    print(f"   {kod:<8s} ({sayi:>4,}x)  {aciklama}")

print()
print("=" * 70)
print("BLOK 1D TAMAMLANDI — icd10_to_label_v3 Blok 2'de kullanılacak.")
print("=" * 70)

# BLOK 2 — Etiketleme + Full Labeled Dataset

**KRİTİK TASARIM KARARI:** Bu blokta SAMPLING YAPMIYORUZ.
Sampling Blok 5'te subject_id split sonrası yapılacak. Sebep: Aynı hastanın yatışlarının train/val/test'e dağılarak data leakage yaratmasını önlemek.

**Bu blokta yapılacaklar:**
1. `diagnoses_icd.csv.gz` → `seq_num=1` + `icd_version=10` filtrele
2. `icd10_to_label_v3()` ile 11 sınıfa eşle
3. `drop_duplicates(hadm_id)` defensive
4. Hedef hadm_id seti çıkar (~122k)
5. `discharge.csv.gz` **CHUNKED** oku (RAM safety)
6. Sadece hedef hadm'lar + text not null + text > 100 char filtrele
7. ICD label'larını join et
8. **Parquet** olarak yaz: `output/discharge_etiketli_full.parquet`

**Bu blok DOSYA YAZIYOR** (ilk kez): `output/discharge_etiketli_full.parquet`

In [ ]:
# 1) ICD ETIKET TABLOSU HAZIRLA
import pandas as pd
from pathlib import Path
import time

OUTPUT_DIR = Path(r"./workdir/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ICD_YOL = Path(r"./data/diagnoses_icd.csv.gz")
DISCHARGE_YOL = Path(r"./data/mimic-iv-note-deidentified-free-text-clinical-notes-2.2/note/discharge.csv.gz")
ETIKETLI_YOL = OUTPUT_DIR / "discharge_etiketli_full.parquet"

print("=" * 70)
print("BLOK 2 — ETIKETLEME + FULL LABELED DATASET")
print("=" * 70)

print("\n[1/5] ICD etiket tablosu hazırlanıyor...")
t0 = time.time()

# diagnoses_icd zaten yüklü (Blok 1B'den), ama defensive olarak yeniden yükleyelim
# (notebook restart durumunu hesaba kat)
if 'df_icd' not in dir() or 'icd10_to_label_v3' not in dir():
    print("   df_icd veya v3 fonksiyon yüklü değil, yeniden yükleniyor...")
    df_icd = pd.read_csv(ICD_YOL, compression="gzip")
    raise RuntimeError("icd10_to_label_v3 fonksiyonu yok — Blok 1D çalıştırılmalı")

# PRIMARY ICD-10 + drop duplicates
etiket_df = df_icd[(df_icd["seq_num"] == 1) & (df_icd["icd_version"] == 10)].copy()
print(f"   PRIMARY ICD-10 kayıt: {len(etiket_df):,}")

etiket_df = etiket_df.drop_duplicates(subset="hadm_id")
print(f"   drop_duplicates(hadm_id) sonrası: {len(etiket_df):,}")

# Etiketle
etiket_df["label"] = etiket_df["icd_code"].apply(icd10_to_label_v3)

# Sadece bize lazım kolonlar
etiket_df = etiket_df[["hadm_id", "icd_code", "label"]].copy()

# Hedef hadm seti (chunked filtreleme için)
hedef_hadms = set(etiket_df["hadm_id"])
print(f"   Hedef (etiketli) hadm sayısı: {len(hedef_hadms):,}")

# Etiket dağılımı
print("\n   Etiket dağılımı (tüm ICD-10 PRIMARY için):")
for cls, n in etiket_df["label"].value_counts().items():
    print(f"      {cls:<25s} {n:>8,}")

print(f"\n   Süre: {time.time()-t0:.1f}s")

In [ ]:
# 2) DISCHARGE CHUNKED OKUMA — RAM safety
print("\n[2/5] discharge.csv.gz CHUNKED okunuyor...")
print("   (sadece hedef hadm + text>100 karakter satırlar tutulur)")
t0 = time.time()

CHUNK_SIZE = 50_000
USE_COLS = ["note_id", "subject_id", "hadm_id", "text"]

biriken = []
okunan = 0
atilanlar = {"hadm_yok": 0, "text_null": 0, "text_kisa": 0}

for i, chunk in enumerate(pd.read_csv(DISCHARGE_YOL, compression="gzip",
                                       chunksize=CHUNK_SIZE, usecols=USE_COLS)):
    okunan += len(chunk)
    
    # 1) hadm hedef set'inde mi?
    onceki = len(chunk)
    chunk = chunk[chunk["hadm_id"].isin(hedef_hadms)]
    atilanlar["hadm_yok"] += onceki - len(chunk)
    
    # 2) text null mu?
    onceki = len(chunk)
    chunk = chunk[chunk["text"].notna()]
    atilanlar["text_null"] += onceki - len(chunk)
    
    # 3) text > 100 char mı?
    onceki = len(chunk)
    chunk = chunk[chunk["text"].str.len() > 100]
    atilanlar["text_kisa"] += onceki - len(chunk)
    
    biriken.append(chunk)
    print(f"   Chunk {i+1:>2d}: okunan={okunan:>7,}, biriken={sum(len(c) for c in biriken):>7,}")

df = pd.concat(biriken, ignore_index=True)
del biriken  # RAM iadesi

print(f"\n   Toplam okunan      : {okunan:,}")
print(f"   hadm hedef değil   : {atilanlar['hadm_yok']:,}")
print(f"   text null          : {atilanlar['text_null']:,}")
print(f"   text < 100 char    : {atilanlar['text_kisa']:,}")
print(f"   Hayatta kalan      : {len(df):,}")
print(f"   Süre: {time.time()-t0:.1f}s")

In [ ]:
# 3) ETIKETI JOIN ET
print("\n[3/5] Etiket tablosu ile join...")
t0 = time.time()

# inner join — discharge ∩ icd_label
df = df.merge(etiket_df, on="hadm_id", how="inner")
print(f"   Join sonrası: {len(df):,}")

# Defensive duplicate kontrolü
onceki = len(df)
df = df.drop_duplicates(subset="hadm_id")
print(f"   drop_duplicates(hadm_id): {len(df):,} ({onceki - len(df)} duplicate atıldı)")

# Kolonları sırala
df = df[["note_id", "subject_id", "hadm_id", "icd_code", "label", "text"]].copy()

# Final dağılım
print("\n   Final etiket dağılımı (sampling YOK, full):")
siralama = ["PNEUMONIA", "HEART_FAILURE", "SEPSIS", "AKI", "UTI",
            "STROKE", "GI_BLEEDING", "COPD_EXACERBATION",
            "DIABETES_COMPLICATION", "ATRIAL_FIBRILLATION", "OTHER"]
vc = df["label"].value_counts()
for cls in siralama:
    n = int(vc.get(cls, 0))
    print(f"      {cls:<25s} {n:>8,}")
print(f"      {'TOPLAM':<25s} {len(df):>8,}")

# Subject_id istatistikleri
print(f"\n   Benzersiz subject_id : {df['subject_id'].nunique():,}")
print(f"   Subject başı yatış (ortalama): {len(df) / df['subject_id'].nunique():.2f}")
print(f"   Süre: {time.time()-t0:.1f}s")

In [ ]:
# 4) PARQUET YAZ
print("\n[4/5] Parquet dosyası yazılıyor...")
t0 = time.time()

df.to_parquet(ETIKETLI_YOL, compression="snappy", index=False)
boyut_mb = ETIKETLI_YOL.stat().st_size / (1024 * 1024)

print(f"   Yazıldı: {ETIKETLI_YOL}")
print(f"   Boyut  : {boyut_mb:.1f} MB")
print(f"   Süre   : {time.time()-t0:.1f}s")

# Doğrula — geri oku
print("\n   Doğrulama (parquet'i geri oku)...")
df_check = pd.read_parquet(ETIKETLI_YOL)
print(f"   Geri okunan satır sayısı: {len(df_check):,} (eşleşmeli: {len(df):,})")
print(f"   Kolonlar: {list(df_check.columns)}")
assert len(df_check) == len(df), "Yazma/okuma satır sayısı eşleşmiyor!"
print("   + Doğrulama başarılı.")

In [ ]:
# 5) ÖZET
print("\n[5/5] BLOK 2 ÖZETİ")
print("=" * 70)
print(f"\n   Çıktı dosyası          : {ETIKETLI_YOL.name}")
print(f"   Toplam etiketli kayıt  : {len(df):,}")
print(f"   Benzersiz subject_id   : {df['subject_id'].nunique():,}")
print(f"   Boyut                  : {boyut_mb:.1f} MB")

# Sınıf bazında subject_id farklılığı (ileride lazım)
print("\n   Sınıf bazında benzersiz subject_id sayısı:")
subject_per_class = df.groupby("label")["subject_id"].nunique().sort_values(ascending=False)
for cls, n in subject_per_class.items():
    yatis = vc[cls]
    oran = yatis / n if n else 0
    print(f"      {cls:<25s} subject:{n:>6,}  yatış:{yatis:>6,}  ort:{oran:.2f}")

print("\n=" * 35)
print("BLOK 2 TAMAMLANDI.")
print("Sıradaki: Blok 3 — Bölüm parse + admission-time girdi (V1, V2)")
print("=" * 70)

# BLOK 3 — Bolum Parse + Admission-Time Girdi (V1, V2)

**Amac:** 122k etiketli not'tan **admission-time-only** girdiler uret.

**Cikarilacak bolumler:**
- **Chief Complaint** (CC) — opsiyonel
- **History of Present Illness** (HPI) — ZORUNLU
- **Past Medical History** (PMH) — V2 icin opsiyonel

**ATILACAK bolumler (sizinti kaynagi):**
- Major Surgical Procedure, Pertinent Results, Brief Hospital Course, Discharge*, Physical Exam

**2 Varyant cikti:**
- **V1:** CC + HPI (en kati admission picture)
- **V2:** CC + HPI + PMH (PMH icerebilir leakage — Blok 4 masking ile temizlenecek)

> [line removed for data-use compliance]

**Filtreler:**
- HPI YOKSA at (HPI zorunlu, CC opsiyonel)
- HPI < 30 char → at (parse hatasi)
- HPI > 4000 char → **TRUNCATE** (silme, kes)

**Cikti:** `output/discharge_parsed.parquet`
Kolonlar: note_id, subject_id, hadm_id, icd_code, label, **cc_text, hpi_text, pmh_text**, model_input_v1, model_input_v2

In [ ]:
# 1) PARSE FONKSIYONU — Multi-pattern regex
import pandas as pd
import re
import time
from pathlib import Path

OUTPUT_DIR = Path(r"./workdir/output")
ETIKETLI_YOL = OUTPUT_DIR / "discharge_etiketli_full.parquet"
PARSED_YOL   = OUTPUT_DIR / "discharge_parsed.parquet"

print("=" * 70)
print("BLOK 3 — BOLUM PARSE + ADMISSION-TIME GIRDI")
print("=" * 70)

# --- Pattern listeleri (ChatGPT eleştirisi: multi-pattern) ---
# Her bolum icin onceliklidir: ilk eslesen kazanir.
CC_PATTERNS = [
    r"Chief Complaint\s*:",
    r"___ Complaint\s*:",       # Bazen 'Chief' deidentified silinmis (Blok 1A bulgusu)
    r"\bCC\s*:",                # Kisaltma
]
HPI_PATTERNS = [
    r"History of Present Illness\s*:",
    r"\bHPI\s*:",
    r"Present Illness\s*:",
]
PMH_PATTERNS = [
    r"Past Medical History\s*:",
    r"\bPMH\s*:",
    r"Medical History\s*:",
]

# Bir sonraki bolum baslayacaksa orada DUR
STOP_PATTERN = (
    r"Past Medical History\s*:|PMH\s*:|Medical History\s*:|"
    r"Social History\s*:|Family History\s*:|"
    r"Physical Exam\s*:|Physical Examination\s*:|Exam\s*:|Vitals?\s*:|"
    r"Pertinent Results\s*:|Lab(?:oratory)?\s+Results?\s*:|Labs\s*:|"
    r"Brief Hospital Course\s*:|Hospital Course\s*:|"
    r"Medications on Admission\s*:|Admission Medications\s*:|"
    r"Discharge Medications\s*:|Discharge Disposition\s*:|"
    r"Discharge Diagnosis\s*:|Final Diagnosis\s*:|"
    r"Discharge Condition\s*:|Discharge Instructions\s*:|"
    r"Followup Instructions\s*:|Follow-?up\s*:|"
    r"Major Surgical or Invasive Procedure\s*:|Allergies\s*:|Attending\s*:"
)

def cikart(text, basla_patterns, dur_pattern_var=True):
    """
    Bir bolumu cikart. Pattern listesinden ilk eslesen kazanir.
    Bir sonraki bilinen baslik gorulene kadar al.
    """
    for p in basla_patterns:
        # Bu pattern'i bul
        m_basla = re.search(p, text, re.IGNORECASE)
        if m_basla is None:
            continue
        
        # Pattern'in sonunda baslayalim
        baslangic = m_basla.end()
        
        # Bir sonraki baslik
        if dur_pattern_var:
            m_dur = re.search(STOP_PATTERN, text[baslangic:], re.IGNORECASE)
            if m_dur:
                bitis = baslangic + m_dur.start()
                icerik = text[baslangic:bitis]
            else:
                icerik = text[baslangic:]
        else:
            icerik = text[baslangic:]
        
        # Temizle
        icerik = icerik.strip()
        if icerik:
            return icerik
    return None

# Quick test
test_text = "Chief Complaint:\nshortness of breath\n\nHistory of Present Illness:\n65yo M with HTN.\n\nPast Medical History:\nDM, HTN\n\nPhysical Exam:\n..."
print("\n[Pattern testi]")
print(f"   CC : {cikart(test_text, CC_PATTERNS)!r}")
print(f"   HPI: {cikart(test_text, HPI_PATTERNS)!r}")
print(f"   PMH: {cikart(test_text, PMH_PATTERNS)!r}")

In [ ]:
# 2) TUM VERIYI PARSE ET
print("\n[2/5] discharge_etiketli_full.parquet yukleniyor...")
t0 = time.time()
df = pd.read_parquet(ETIKETLI_YOL)
print(f"   Yuklendi: {len(df):,} satir, {time.time()-t0:.1f}s")

print("\n   Parse ediliyor (3 bolum × 122k = ~370k regex islemi)...")
t0 = time.time()

df["cc_text"] = df["text"].apply(lambda t: cikart(t, CC_PATTERNS))
print(f"   CC parse bitti: {time.time()-t0:.1f}s")

t0 = time.time()
df["hpi_text"] = df["text"].apply(lambda t: cikart(t, HPI_PATTERNS))
print(f"   HPI parse bitti: {time.time()-t0:.1f}s")

t0 = time.time()
df["pmh_text"] = df["text"].apply(lambda t: cikart(t, PMH_PATTERNS))
print(f"   PMH parse bitti: {time.time()-t0:.1f}s")

# Coverage
n = len(df)
cc_var  = df["cc_text"].notna().sum()
hpi_var = df["hpi_text"].notna().sum()
pmh_var = df["pmh_text"].notna().sum()
print(f"\n   Parse coverage:")
print(f"      CC  : {cc_var:>6,} / {n:,} ({100*cc_var/n:.1f}%)")
print(f"      HPI : {hpi_var:>6,} / {n:,} ({100*hpi_var/n:.1f}%)  ← ZORUNLU")
print(f"      PMH : {pmh_var:>6,} / {n:,} ({100*pmh_var/n:.1f}%)")

In [ ]:
# 3) FILTRE + TRUNCATE
print("\n[3/5] Filtre + truncate uygulaniyor...")

ilk = len(df)

# 1) HPI ZORUNLU — yoksa at
df = df[df["hpi_text"].notna()].copy()
kayip_hpi = ilk - len(df)
print(f"   HPI yok ({kayip_hpi:,} kayit) atildi → kalan: {len(df):,}")

# 2) HPI < 30 char at (parse hatasi)
onceki = len(df)
df = df[df["hpi_text"].str.len() >= 30].copy()
print(f"   HPI < 30 char ({onceki - len(df):,} kayit) atildi → kalan: {len(df):,}")

# 3) HPI > 4000 char TRUNCATE (silme!) — ChatGPT onerisi
uzun_mask = df["hpi_text"].str.len() > 4000
n_uzun = uzun_mask.sum()
df.loc[uzun_mask, "hpi_text"] = df.loc[uzun_mask, "hpi_text"].str[:4000]
print(f"   HPI > 4000 char ({n_uzun:,} kayit) TRUNCATE edildi (silinmedi)")

# CC ve PMH icin de truncate (ihtiyat — coook uzun olanlar yer kaplamasin)
df.loc[df["cc_text"].notna() & (df["cc_text"].str.len() > 1000), "cc_text"] = (
    df.loc[df["cc_text"].notna() & (df["cc_text"].str.len() > 1000), "cc_text"].str[:1000]
)
df.loc[df["pmh_text"].notna() & (df["pmh_text"].str.len() > 2000), "pmh_text"] = (
    df.loc[df["pmh_text"].notna() & (df["pmh_text"].str.len() > 2000), "pmh_text"].str[:2000]
)

# Oraralma boslukları normalize et
for col in ["cc_text", "hpi_text", "pmh_text"]:
    df[col] = df[col].fillna("").apply(lambda x: re.sub(r"\n{3,}", "\n\n", x).strip())

print(f"\n   Filtre sonrasi: {len(df):,} (baslangic 122,288'den {ilk - len(df):,} kayboldu)")

In [ ]:
# 4) V1 ve V2 GIRDILERINI OLUSTUR
print("\n[4/5] V1 (CC+HPI) ve V2 (CC+HPI+PMH) olusturuluyor...")

def yap_v1(row):
    """V1: CC + HPI (CC yoksa sadece HPI)."""
    parcalar = []
    if row["cc_text"]:
        parcalar.append(f"Chief Complaint:\n{row['cc_text']}")
    parcalar.append(f"History of Present Illness:\n{row['hpi_text']}")
    return "\n\n".join(parcalar)

def yap_v2(row):
    """V2: V1 + PMH (PMH yoksa V1 ile ayni)."""
    parcalar = []
    if row["cc_text"]:
        parcalar.append(f"Chief Complaint:\n{row['cc_text']}")
    parcalar.append(f"History of Present Illness:\n{row['hpi_text']}")
    if row["pmh_text"]:
        parcalar.append(f"Past Medical History:\n{row['pmh_text']}")
    return "\n\n".join(parcalar)

df["model_input_v1"] = df.apply(yap_v1, axis=1)
df["model_input_v2"] = df.apply(yap_v2, axis=1)

# Uzunluk istatistikleri
v1_len = df["model_input_v1"].str.len()
v2_len = df["model_input_v2"].str.len()
print(f"\n   V1 (CC+HPI) uzunluk:")
print(f"      mean={v1_len.mean():.0f}, median={v1_len.median():.0f}, min={v1_len.min()}, max={v1_len.max()}")
print(f"   V2 (CC+HPI+PMH) uzunluk:")
print(f"      mean={v2_len.mean():.0f}, median={v2_len.median():.0f}, min={v2_len.min()}, max={v2_len.max()}")

# Ornek goster
ornek = df.iloc[0]
print(f"\n   --- ORNEK V1 (label={ornek['label']}) ---")
print("   " + ornek["model_input_v1"][:600].replace("\n", "\n   "))
print("   ... [kesildi]")

In [ ]:
# 5) CLASS DISTRIBUTION KONTROL + PARQUET YAZ
print("\n[5/5] Final class distribution + parquet yaz...")

siralama = ["PNEUMONIA", "HEART_FAILURE", "SEPSIS", "AKI", "UTI",
            "STROKE", "GI_BLEEDING", "COPD_EXACERBATION",
            "DIABETES_COMPLICATION", "ATRIAL_FIBRILLATION", "OTHER"]

vc_son = df["label"].value_counts()
print("\n   Parse sonrasi sinif dagilimi:")
print("   " + "-" * 60)
ihtiyac = 600
tum_ok = True
for cls in siralama:
    n = int(vc_son.get(cls, 0))
    n_subj = df[df["label"] == cls]["subject_id"].nunique()
    if cls == "OTHER":
        durum = "FAZLASIYLA"
    elif n >= ihtiyac:
        durum = f"OK (+{n - ihtiyac})"
    else:
        durum = f"YETERSIZ ({n - ihtiyac})"
        tum_ok = False
    print(f"   {cls:<25s} {n:>6,} yatis | {n_subj:>5,} subject | {durum}")

print(f"\n   Toplam : {len(df):,} kayit")
print(f"   Subject: {df['subject_id'].nunique():,}")
print(f"   {'+ TUM SINIFLAR YETERLI' if tum_ok else '- BAZI SINIFLAR SINIRDA'}")

# Final kolonlar (sadece bize lazim olanlar)
final_kolonlar = [
    "note_id", "subject_id", "hadm_id", "icd_code", "label",
    "cc_text", "hpi_text", "pmh_text",
    "model_input_v1", "model_input_v2",
]
df_out = df[final_kolonlar].copy()

# Yaz
t0 = time.time()
df_out.to_parquet(PARSED_YOL, compression="snappy", index=False)
boyut = PARSED_YOL.stat().st_size / (1024 * 1024)
print(f"\n   Yazildi: {PARSED_YOL.name}")
print(f"   Boyut  : {boyut:.1f} MB")
print(f"   Sure   : {time.time()-t0:.1f}s")

# Dogrula
df_check = pd.read_parquet(PARSED_YOL)
assert len(df_check) == len(df_out)
print(f"   + Dogrulama OK: {len(df_check):,} satir geri okundu")

print("\n" + "=" * 70)
print("BLOK 3 TAMAMLANDI.")
print("Sıradaki: Blok 4 — Term masking (3-katmanli)")
print("=" * 70)

# BLOK 4 — Term Masking (3-Katmanli, Label-Aware)

**Akis sirasi (KRITIK):**
1. parsed.parquet yukle
2. **Keyword scan** → `has_label_term_v1`, `has_label_term_v2` bayraklari (clean test icin)
3. **Masking** uygula → `[DIAG_MASKED]`
4. Maskeli model_input_v1_masked, model_input_v2_masked olustur
5. Audit raporu (sinif bazli + ornek before/after)
6. Yaz: `output/discharge_masked.parquet`

**Mask token:** `[DIAG_MASKED]` (Token Collision riskine karsi `[MASK]` yerine).

**3-Katmanli yaklasim:**
- **Katman 1:** Exact terms ("pneumonia", "sepsis")
- **Katman 2:** Abbreviations ("CHF", "DKA", "AFib")
> [line removed for data-use compliance]

**Final Kolon Yapisi (16 kolon, altin standart):**
```
note_id, subject_id, hadm_id, label, icd_code,
cc_text, hpi_text, pmh_text,
model_input_v1, model_input_v2,
cc_text_masked, hpi_text_masked, pmh_text_masked,
model_input_v1_masked, model_input_v2_masked,
has_label_term_v1, has_label_term_v2
```

In [ ]:
# 1) CLASS KEYWORDS SOZLUGU (3-katmanli)
import pandas as pd
import re
import time
from pathlib import Path

OUTPUT_DIR = Path(r"./workdir/output")
PARSED_YOL = OUTPUT_DIR / "discharge_parsed.parquet"
MASKED_YOL = OUTPUT_DIR / "discharge_masked.parquet"

MASK_TOKEN = "[DIAG_MASKED]"

print("=" * 70)
print("BLOK 4 — TERM MASKING (3-katmanli, label-aware)")
print("=" * 70)

# Tum 3 katmani birlestiren sozluk
# Her sinif icin: 'exact' + 'abbrev' + 'context'
CLASS_KEYWORDS = {
    "PNEUMONIA": {
        "exact":   [r"pneumonia", r"pneumonic", r"pneumonitis", r"bronchopneumonia"],
        "abbrev":  [r"PNA", r"CAP", r"HAP", r"VAP"],
        "context": [r"history of pneumonia", r"h/o pneumonia",
                    r"treated for pneumonia", r"admitted with pneumonia"],
    },
    "HEART_FAILURE": {
        "exact":   [r"heart failure", r"congestive heart failure",
                    r"cardiac decompensation", r"cor pulmonale"],
        "abbrev":  [r"CHF", r"HF", r"HFrEF", r"HFpEF"],
        "context": [r"history of CHF", r"h/o CHF", r"known CHF",
                    r"CHF exacerbation"],
    },
    "SEPSIS": {
        "exact":   [r"sepsis", r"septic", r"septicemia", r"bacteremia",
                    r"septic shock", r"urosepsis"],  # urosepsis eklendi
        "abbrev":  [],
        "context": [r"history of sepsis", r"h/o sepsis",
                    r"presented with sepsis", r"admitted with sepsis"],
    },
    "AKI": {
        "exact":   [r"acute kidney injury", r"acute kidney failure",
                    r"acute renal failure"],
        "abbrev":  [r"AKI", r"ARF"],
        "context": [r"history of AKI", r"h/o AKI"],
    },
    "UTI": {
        "exact":   [r"urinary tract infection", r"pyelonephritis", r"cystitis"],
        "abbrev":  [r"UTI"],
        "context": [r"history of UTI", r"h/o UTI", r"recurrent UTI"],
    },
    "STROKE": {
        "exact":   [r"stroke", r"cerebrovascular accident",
                    r"cerebral infarction", r"ischemic stroke",
                    r"hemorrhagic stroke"],
        "abbrev":  [r"CVA", r"TIA"],
        "context": [r"history of stroke", r"h/o stroke", r"prior stroke"],
    },
    "GI_BLEEDING": {
        "exact":   [r"gastrointestinal bleed\w*", r"GI bleed\w*",
                    r"hematemesis", r"melena", r"hematochezia"],
        "abbrev":  [r"UGIB", r"LGIB", r"GIB"],
        "context": [r"history of GI bleed", r"h/o GI bleed"],
    },
    "COPD_EXACERBATION": {
        "exact":   [r"chronic obstructive pulmonary disease",
                    r"COPD exacerbation", r"emphysema"],  # emphysema eklendi
        "abbrev":  [r"COPD", r"AECOPD"],
        "context": [r"history of COPD", r"h/o COPD", r"known COPD",
                    r"COPD flare"],
    },
    "DIABETES_COMPLICATION": {
        # NOT: 'diabetes' tek basina maskelenmez (kronik DM PMH'de gecer, leakage degil)
        "exact":   [r"diabetic ketoacidosis", r"hyperosmolar hyperglycemic",
                    r"hyperglycemia", r"hypoglycemia"],
        "abbrev":  [r"DKA", r"HHS"],
        "context": [r"presented in DKA", r"history of DKA",
                    r"admitted in DKA"],
    },
    "ATRIAL_FIBRILLATION": {
        "exact":   [r"atrial fibrillation", r"atrial flutter",
                    r"paroxysmal atrial fibrillation"],
        "abbrev":  [r"AFib", r"A-?fib", r"AF", r"PAF"],
        "context": [r"history of atrial fibrillation",
                    r"h/o AFib", r"known AFib"],
    },
}

# Tum keyword'leri tek listeye topla (regex + word boundary)
# Her keyword icin word boundary uygula (\b...\b)
TUM_PATTERNS = []
for cls, katmanlar in CLASS_KEYWORDS.items():
    for katman_adi, kelimeler in katmanlar.items():
        for kw in kelimeler:
            # Context pattern'leri zaten cok kelimeli, word boundary lazim
            # Exact ve abbrev icin tek kelime/grup
            patt = rf"\b{kw}\b"
            TUM_PATTERNS.append(patt)

# Tek bir buyuk regex'e birlestir (performans icin)
BIRLESIK_REGEX = re.compile("|".join(f"(?:{p})" for p in TUM_PATTERNS), re.IGNORECASE)

print(f"\n   Toplam pattern sayisi: {len(TUM_PATTERNS)}")
print(f"   Mask token: {MASK_TOKEN}")

# Quick test
test_metinler = {
    "true_pos_pneumonia": "Patient admitted with pneumonia and sepsis.",
    "true_pos_chf": "Has CHF exacerbation, h/o AFib.",
    "true_neg_cancer": "History of cancer, hypertension, controlled.",
    "true_neg_diabetes": "Past medical history of diabetes, hypertension.",  # diabetes maskelenmemeli
    "abbrev_test": "DKA on admission, also had UTI last year.",
}
for ad, m in test_metinler.items():
    sonuc = BIRLESIK_REGEX.sub(MASK_TOKEN, m)
    print(f"\n   [{ad}]")
    print(f"      ONCE  : {m}")
    print(f"      SONRA : {sonuc}")

In [ ]:
# 2) PARSED VERIYI YUKLE + FLAG + MASKING
print("\n[2/5] discharge_parsed.parquet yukleniyor...")
t0 = time.time()
df = pd.read_parquet(PARSED_YOL)
print(f"   Yuklendi: {len(df):,} satir, {time.time()-t0:.1f}s")

# --- ADIM A: has_label_term bayrak (KRITIK SIRA: masking'den ONCE) ---
print("\n[Adim A] has_label_term bayraklari hesaplaniyor...")
t0 = time.time()

def icerir_mi(metin):
    """Metin herhangi bir label keyword icerir mi?"""
    if not metin:
        return False
    return BIRLESIK_REGEX.search(metin) is not None

df["has_label_term_v1"] = df["model_input_v1"].apply(icerir_mi)
df["has_label_term_v2"] = df["model_input_v2"].apply(icerir_mi)

v1_oran = 100 * df["has_label_term_v1"].mean()
v2_oran = 100 * df["has_label_term_v2"].mean()
print(f"   has_label_term_v1 (CC+HPI)    : {df['has_label_term_v1'].sum():,} ({v1_oran:.1f}%)")
print(f"   has_label_term_v2 (CC+HPI+PMH): {df['has_label_term_v2'].sum():,} ({v2_oran:.1f}%)")
print(f"   Sure: {time.time()-t0:.1f}s")

# --- ADIM B: Masking ---
print("\n[Adim B] Masking uygulaniyor (cc, hpi, pmh)...")
t0 = time.time()

def maskele(metin):
    """Metindeki label keyword'lerini [DIAG_MASKED] ile degistir."""
    if not metin or not isinstance(metin, str):
        return metin
    return BIRLESIK_REGEX.sub(MASK_TOKEN, metin)

df["cc_text_masked"]  = df["cc_text"].apply(maskele)
df["hpi_text_masked"] = df["hpi_text"].apply(maskele)
df["pmh_text_masked"] = df["pmh_text"].apply(maskele)

print(f"   Masking bitti: {time.time()-t0:.1f}s")

In [ ]:
# 3) MASKELI V1 ve V2 OLUSTUR
print("\n[3/5] Maskeli V1, V2 olusturuluyor...")

def yap_v1_masked(row):
    parcalar = []
    if row["cc_text_masked"]:
        parcalar.append(f"Chief Complaint:\n{row['cc_text_masked']}")
    parcalar.append(f"History of Present Illness:\n{row['hpi_text_masked']}")
    return "\n\n".join(parcalar)

def yap_v2_masked(row):
    parcalar = []
    if row["cc_text_masked"]:
        parcalar.append(f"Chief Complaint:\n{row['cc_text_masked']}")
    parcalar.append(f"History of Present Illness:\n{row['hpi_text_masked']}")
    if row["pmh_text_masked"]:
        parcalar.append(f"Past Medical History:\n{row['pmh_text_masked']}")
    return "\n\n".join(parcalar)

df["model_input_v1_masked"] = df.apply(yap_v1_masked, axis=1)
df["model_input_v2_masked"] = df.apply(yap_v2_masked, axis=1)

# Kac mask token girdi?
df["mask_count_v1"] = df["model_input_v1_masked"].str.count(re.escape(MASK_TOKEN))
df["mask_count_v2"] = df["model_input_v2_masked"].str.count(re.escape(MASK_TOKEN))

print(f"   V1 maskeleme istatistikleri:")
print(f"      Ortalama mask sayisi: {df['mask_count_v1'].mean():.2f}")
print(f"      Max mask sayisi     : {df['mask_count_v1'].max()}")
print(f"      Hic maskelenmemis ornekler: {(df['mask_count_v1'] == 0).sum():,} ({100*(df['mask_count_v1']==0).mean():.1f}%)")

print(f"\n   V2 maskeleme istatistikleri:")
print(f"      Ortalama mask sayisi: {df['mask_count_v2'].mean():.2f}")
print(f"      Max mask sayisi     : {df['mask_count_v2'].max()}")
print(f"      Hic maskelenmemis ornekler: {(df['mask_count_v2'] == 0).sum():,} ({100*(df['mask_count_v2']==0).mean():.1f}%)")

In [ ]:
# 4) AUDIT RAPORU (sinif bazli)
print("\n[4/5] Audit raporu — sinif bazli maskeleme yogunlugu...")
print("-" * 70)

siralama = ["PNEUMONIA", "HEART_FAILURE", "SEPSIS", "AKI", "UTI",
            "STROKE", "GI_BLEEDING", "COPD_EXACERBATION",
            "DIABETES_COMPLICATION", "ATRIAL_FIBRILLATION", "OTHER"]

print(f"\n   {'Sinif':<25s} {'V1 keyword %':>12s} {'V2 keyword %':>12s} {'V1 ort mask':>11s} {'V2 ort mask':>11s}")
print("   " + "-" * 75)
for cls in siralama:
    alt = df[df["label"] == cls]
    n = len(alt)
    if n == 0:
        continue
    v1_oran = 100 * alt["has_label_term_v1"].mean()
    v2_oran = 100 * alt["has_label_term_v2"].mean()
    v1_ort = alt["mask_count_v1"].mean()
    v2_ort = alt["mask_count_v2"].mean()
    print(f"   {cls:<25s} {v1_oran:>11.1f}% {v2_oran:>11.1f}% {v1_ort:>11.2f} {v2_ort:>11.2f}")

# PMH ekleyince leakage ne kadar artiyor?
leakage_artisi = df["has_label_term_v2"].mean() - df["has_label_term_v1"].mean()
print(f"\n   PMH ekleyince has_label_term ARTISI: {100*leakage_artisi:+.1f}%")
print("   ★ Bu, V2'deki ek leakage riskinin sayisal olcusu.")
print("   ★ Tezde 'V1 vs V2 ablation' icin altın değerinde bulgu.")

# Clean test boyut tahmini
clean_v1 = (~df["has_label_term_v1"]).sum()
clean_v2 = (~df["has_label_term_v2"]).sum()
print(f"\n   Clean test boyut tahmini (full dataset uzerinde):")
print(f"      V1 clean: {clean_v1:,} ({100*clean_v1/len(df):.1f}%)")
print(f"      V2 clean: {clean_v2:,} ({100*clean_v2/len(df):.1f}%)")

In [ ]:
# 5) BEFORE/AFTER ORNEKLER + PARQUET YAZ
print("\n[5/5] Before/after ornekler + parquet yaz...")

# Her sinif icin 1 keyword iceren ornek goster
print("\n   --- Sinif basina 1 ornek (before/after) ---")
for cls in siralama[:3]:  # ilk 3 sinif (yer kaybetmemek icin)
    altset = df[(df["label"] == cls) & (df["has_label_term_v1"])]
    if len(altset) > 0:
        ornek = altset.iloc[0]
        print(f"\n   ▶ {cls}  (note_id: {ornek['note_id']})")
        # Sadece HPI'in ilk 200 char'i
        once = ornek["hpi_text"][:200]
        sonra = ornek["hpi_text_masked"][:200]
        print(f"      ONCE : ...{once}...")
        print(f"      SONRA: ...{sonra}...")

# Final kolon listesi (altin standart)
FINAL_KOLONLAR = [
    "note_id", "subject_id", "hadm_id", "label", "icd_code",
    "cc_text", "hpi_text", "pmh_text",
    "model_input_v1", "model_input_v2",
    "cc_text_masked", "hpi_text_masked", "pmh_text_masked",
    "model_input_v1_masked", "model_input_v2_masked",
    "has_label_term_v1", "has_label_term_v2",
    # Bonus debug kolonlari (cikartilabilir ama isimize yarayabilir):
    "mask_count_v1", "mask_count_v2",
]
df_out = df[FINAL_KOLONLAR].copy()

# Yaz
t0 = time.time()
df_out.to_parquet(MASKED_YOL, compression="snappy", index=False)
boyut = MASKED_YOL.stat().st_size / (1024 * 1024)
print(f"\n   Yazildi: {MASKED_YOL.name}")
print(f"   Boyut  : {boyut:.1f} MB")
print(f"   Sure   : {time.time()-t0:.1f}s")

# Dogrula
df_check = pd.read_parquet(MASKED_YOL)
assert len(df_check) == len(df_out)
print(f"   + Dogrulama OK: {len(df_check):,} satir geri okundu")
print(f"   + Kolon sayisi: {len(df_check.columns)}")

print("\n" + "=" * 70)
print("BLOK 4 TAMAMLANDI.")
print("Sıradaki: Blok 5 — Subject_id bazli split + stratified sampling")
print("=" * 70)

# BLOK 5 — Subject_id-Bazli Split + Stratified Sampling

**KRITIK TASARIM:** Sampling burada yapilir (Blok 2'de YAPILMADI).
Sebep: subject_id leakage'i %100 onlemek.

**Akis:**
1. Her sinif icin subject_id listesi cikar
2. Subject'leri **once split** (train/val/test ayrik kume)
3. Sonra her bolum icinde stratified sampling (yatis bazli)
4. Nested train senaryolari: S1 ⊂ S2 ⊂ S3

**Hedef boyutlar:**
| Bolum | Sinif basi | Toplam |
|---|---|---|
| Test  | 150 | 1.650 |
| Val   |  50 |   550 |
| S1    | 100 | 1.100 |
| S2    | 200 | 2.200 (S1 ⊂ S2) |
| S3    | 400 | 4.400 (S2 ⊂ S3) |

**Subject_id ayriklik garantisi:** Test ∩ Val ∩ Train = ∅

**Cikti:** `output/discharge_split.parquet` (yeni 'split' kolonu eklendi)

**Reproducibility:** `random_state=42` her yerde.

In [ ]:
# 1) SUBJECT_ID BAZLI SPLIT (yatis bazli DEGIL!)
import pandas as pd
import numpy as np
import time
from pathlib import Path

OUTPUT_DIR = Path(r"./workdir/output")
MASKED_YOL = OUTPUT_DIR / "discharge_masked.parquet"
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

print("=" * 70)
print("BLOK 5 — SUBJECT_ID-BAZLI SPLIT + STRATIFIED SAMPLING")
print("=" * 70)

print("\n[1/5] discharge_masked.parquet yukleniyor...")
t0 = time.time()
df = pd.read_parquet(MASKED_YOL)
print(f"   Yuklendi: {len(df):,} satir, {df['subject_id'].nunique():,} subject")
print(f"   Sure: {time.time()-t0:.1f}s")

# --- Hedef boyutlar ---
HEDEF = {
    "test":     150,
    "val":       50,
    "train_S3": 400,  # En buyuk; S1 ve S2 bunun alt kumesi
}
TOPLAM_SINIF_BASI = sum(HEDEF.values())  # 600

siralama = ["PNEUMONIA", "HEART_FAILURE", "SEPSIS", "AKI", "UTI",
            "STROKE", "GI_BLEEDING", "COPD_EXACERBATION",
            "DIABETES_COMPLICATION", "ATRIAL_FIBRILLATION", "OTHER"]

# Her sinif icin subject_id'leri sinifa gore daginimini kullan
# Subject's most frequent label = onun "sinifi" (subject ya hep aynı sınıf, ya da çoğunluk)
subject_label = df.groupby("subject_id")["label"].agg(lambda x: x.value_counts().index[0])
subject_df = pd.DataFrame({"subject_id": subject_label.index, "primary_label": subject_label.values})

print(f"\n   Subject basina ana etiket dağılımı (en sık etiket):")
for cls in siralama:
    n = (subject_df["primary_label"] == cls).sum()
    print(f"      {cls:<25s} {n:>6,} subject")

In [ ]:
# 2) HER SINIF ICIN SUBJECT'LERI 3 KÜMEYE AYIR (test, val, train)
print("\n[2/5] Subject split (sinif bazli, ayrık küme)...")
print("-" * 70)

test_subjects = set()
val_subjects = set()
train_subjects = set()

# Sinif bazli iterate et — her sinif icin yeterli subject ayır
# Ortalama yatis/subject ~1.86 → boyutu hedeflemek icin biraz fazla subject al
for cls in siralama:
    sinif_yatis = df[df["label"] == cls]
    sinif_subjects = sinif_yatis["subject_id"].unique()
    
    # Bu siniftaki ortalama yatis/subject
    yatis_per_subj = len(sinif_yatis) / len(sinif_subjects)
    
    # Hedef yatis sayisi → kaç subject lazim
    hedef_test_subj  = int(np.ceil(HEDEF["test"]     / yatis_per_subj * 1.2))  # %20 buffer
    hedef_val_subj   = int(np.ceil(HEDEF["val"]      / yatis_per_subj * 1.2))
    hedef_train_subj = int(np.ceil(HEDEF["train_S3"] / yatis_per_subj * 1.2))
    
    # Karistir
    sinif_subj_listesi = list(sinif_subjects)
    rng.shuffle(sinif_subj_listesi)
    
    # Diger siniflarin subject'lerini cikar (eger zaten ayrildiysa)
    zaten_kullanilan = test_subjects | val_subjects | train_subjects
    sinif_subj_kullanilabilir = [s for s in sinif_subj_listesi if s not in zaten_kullanilan]
    
    # Test, val, train
    idx = 0
    test_alt   = sinif_subj_kullanilabilir[idx:idx + hedef_test_subj]
    idx += hedef_test_subj
    val_alt    = sinif_subj_kullanilabilir[idx:idx + hedef_val_subj]
    idx += hedef_val_subj
    train_alt  = sinif_subj_kullanilabilir[idx:idx + hedef_train_subj]
    
    test_subjects.update(test_alt)
    val_subjects.update(val_alt)
    train_subjects.update(train_alt)
    
    print(f"   {cls:<25s} subj:{len(sinif_subjects):>6,} | t:{len(test_alt):>4} v:{len(val_alt):>3} tr:{len(train_alt):>4}")

# Ayrıklık doğrulama
kesisim_tv = test_subjects & val_subjects
kesisim_tt = test_subjects & train_subjects
kesisim_vt = val_subjects & train_subjects
print(f"\n   Ayrıklık kontrolü:")
print(f"      Test ∩ Val   : {len(kesisim_tv):,} (0 olmalı)")
print(f"      Test ∩ Train : {len(kesisim_tt):,} (0 olmalı)")
print(f"      Val ∩ Train  : {len(kesisim_vt):,} (0 olmalı)")
assert len(kesisim_tv) == 0 and len(kesisim_tt) == 0 and len(kesisim_vt) == 0, "AYRIKLIK BOZULDU!"
print(f"   + AYRIK KÜMELER GARANTİLENDİ")

print(f"\n   Toplam subject sayilari:")
print(f"      Test:  {len(test_subjects):,}")
print(f"      Val:   {len(val_subjects):,}")
print(f"      Train: {len(train_subjects):,}")

In [ ]:
# 3) HER BOLUM ICINDE STRATIFIED SAMPLING (yatis bazli)
print("\n[3/5] Stratified sampling (her bolum icinde)...")
print("-" * 70)

# Her yatis icin split etiketi: 'test', 'val', 'train_S3', 'train_S2', 'train_S1', None (kullanilmadi)
df["split"] = None

# Test ve Val — sabit boyut
for cls in siralama:
    # Test
    aday = df[(df["label"] == cls) & (df["subject_id"].isin(test_subjects)) & (df["split"].isna())]
    if len(aday) >= HEDEF["test"]:
        secilen = aday.sample(n=HEDEF["test"], random_state=RNG_SEED).index
    else:
        secilen = aday.index
        print(f"   UYARI: {cls} test'te {len(aday)} yatis var, hedef {HEDEF['test']}")
    df.loc[secilen, "split"] = "test"
    
    # Val
    aday = df[(df["label"] == cls) & (df["subject_id"].isin(val_subjects)) & (df["split"].isna())]
    if len(aday) >= HEDEF["val"]:
        secilen = aday.sample(n=HEDEF["val"], random_state=RNG_SEED).index
    else:
        secilen = aday.index
        print(f"   UYARI: {cls} val'da {len(aday)} yatis var, hedef {HEDEF['val']}")
    df.loc[secilen, "split"] = "val"
    
    # Train S3 (en buyuk pool)
    aday = df[(df["label"] == cls) & (df["subject_id"].isin(train_subjects)) & (df["split"].isna())]
    if len(aday) >= HEDEF["train_S3"]:
        secilen = aday.sample(n=HEDEF["train_S3"], random_state=RNG_SEED).index
    else:
        secilen = aday.index
        print(f"   UYARI: {cls} train'de {len(aday)} yatis var, hedef {HEDEF['train_S3']}")
    df.loc[secilen, "split"] = "train_S3"

# S2 = S3'un rastgele 200'lu alt kumesi
# S1 = S2'nin rastgele 100'lu alt kumesi
for cls in siralama:
    s3_set = df[(df["label"] == cls) & (df["split"] == "train_S3")]
    if len(s3_set) >= 200:
        s2_idx = s3_set.sample(n=200, random_state=RNG_SEED).index
    else:
        s2_idx = s3_set.index
    
    s2_set = df.loc[s2_idx]
    if len(s2_set) >= 100:
        s1_idx = s2_set.sample(n=100, random_state=RNG_SEED).index
    else:
        s1_idx = s2_set.index
    
    # Bayraklar (S3 zaten var, S2 ve S1 ayrı kolon)
    df["in_S2"] = False if "in_S2" not in df.columns else df["in_S2"]
    df["in_S1"] = False if "in_S1" not in df.columns else df["in_S1"]

# S2 ve S1 bayraklarini ayri olarak hesapla (yukarıdaki loop'tan)
df["in_S2"] = False
df["in_S1"] = False

for cls in siralama:
    s3_set = df[(df["label"] == cls) & (df["split"] == "train_S3")]
    if len(s3_set) >= 200:
        s2_idx = s3_set.sample(n=200, random_state=RNG_SEED).index
    else:
        s2_idx = s3_set.index
    df.loc[s2_idx, "in_S2"] = True
    
    s2_set_actual = df.loc[s2_idx]
    if len(s2_set_actual) >= 100:
        s1_idx = s2_set_actual.sample(n=100, random_state=RNG_SEED).index
    else:
        s1_idx = s2_set_actual.index
    df.loc[s1_idx, "in_S1"] = True

print(f"   + Sampling tamamlandi.")

In [ ]:
# 4) AUDIT — boyutlar, sınıf dağılımı, has_label_term oranları
print("\n[4/5] Audit raporu...")
print("-" * 70)

# Genel boyutlar
test_df = df[df["split"] == "test"]
val_df  = df[df["split"] == "val"]
s3_df   = df[df["split"] == "train_S3"]
s2_df   = df[df["in_S2"]]
s1_df   = df[df["in_S1"]]

print(f"\n   Bolum boyutlari:")
print(f"      Test    : {len(test_df):>6,}  (hedef 1.650)")
print(f"      Val     : {len(val_df):>6,}  (hedef   550)")
print(f"      Train S3: {len(s3_df):>6,}  (hedef 4.400)")
print(f"      Train S2: {len(s2_df):>6,}  (hedef 2.200, ⊂ S3)")
print(f"      Train S1: {len(s1_df):>6,}  (hedef 1.100, ⊂ S2)")

# Sınıf bazlı boyutlar (test, val, S3)
print(f"\n   Sinif bazli — Test / Val / S3 / S2 / S1:")
print("   " + "-" * 70)
print(f"   {'Sinif':<25s} {'Test':>5s} {'Val':>4s} {'S3':>5s} {'S2':>5s} {'S1':>5s}")
print("   " + "-" * 60)
for cls in siralama:
    t = (test_df["label"] == cls).sum()
    v = (val_df["label"] == cls).sum()
    s3 = (s3_df["label"] == cls).sum()
    s2 = (s2_df["label"] == cls).sum()
    s1 = (s1_df["label"] == cls).sum()
    print(f"   {cls:<25s} {t:>5} {v:>4} {s3:>5} {s2:>5} {s1:>5}")

# Ayrıklık doğrulama (subject_id bazli)
test_subj_set = set(test_df["subject_id"])
val_subj_set = set(val_df["subject_id"])
s3_subj_set = set(s3_df["subject_id"])

print(f"\n   Subject_id ayriklik (final):")
print(f"      Test ∩ Val   : {len(test_subj_set & val_subj_set):,}")
print(f"      Test ∩ Train : {len(test_subj_set & s3_subj_set):,}")
print(f"      Val ∩ Train  : {len(val_subj_set & s3_subj_set):,}")

# has_label_term oranları (clean test boyutu için)
print(f"\n   has_label_term oranlari (clean test belirlemek icin):")
print(f"      Test V1 keyword içerir: {test_df['has_label_term_v1'].sum():>4,} / {len(test_df):,} ({100*test_df['has_label_term_v1'].mean():.1f}%)")
print(f"      Test V2 keyword içerir: {test_df['has_label_term_v2'].sum():>4,} / {len(test_df):,} ({100*test_df['has_label_term_v2'].mean():.1f}%)")
print(f"      Test V1 CLEAN: {(~test_df['has_label_term_v1']).sum():,}")
print(f"      Test V2 CLEAN: {(~test_df['has_label_term_v2']).sum():,}")

In [ ]:
# 5) PARQUET YAZ
print("\n[5/5] discharge_split.parquet yaziliyor...")

# Sadece kullanilan kayitlari tut (split != None)
df_kullanilan = df[df["split"].notna()].copy()
print(f"   Kullanilan kayit: {len(df_kullanilan):,} (toplam {len(df):,}'den)")
print(f"   Atildi (gereksiz): {len(df) - len(df_kullanilan):,}")

t0 = time.time()
df_kullanilan.to_parquet(SPLIT_YOL, compression="snappy", index=False)
boyut = SPLIT_YOL.stat().st_size / (1024 * 1024)
print(f"\n   Yazildi: {SPLIT_YOL.name}")
print(f"   Boyut  : {boyut:.1f} MB")
print(f"   Sure   : {time.time()-t0:.1f}s")

# Doğrula
df_check = pd.read_parquet(SPLIT_YOL)
assert len(df_check) == len(df_kullanilan)
print(f"   + Dogrulama: {len(df_check):,} satir, {len(df_check.columns)} kolon")
print(f"   + Split kolon değerleri: {sorted(df_check['split'].unique())}")

print("\n" + "=" * 70)
print("BLOK 5 TAMAMLANDI.")
print("Sıradaki: Blok 6 — QLoRA JSON formati (her V için ayri)")
print("=" * 70)

# BLOK 6 - QLoRA JSON Formati (10 dosya)

**Amac:** discharge_split.parquet -> Instruction-input-output JSON formati.

**Cikti dosyalari (10 adet):**

| Dosya | Kayit | Aciklama |
|---|---|---|
| qlora_v1_train_S1.json | 1.100 | V1 input, S1 |
| qlora_v1_train_S2.json | 2.200 | V1 input, S2 |
| qlora_v1_train_S3.json | 4.400 | V1 input, S3 + few-shot pool |
| qlora_v1_val.json      |   550 | V1 validation |
| qlora_v1_test.json     | 1.650 | V1 test |
| qlora_v2_train_S1.json | 1.100 | V2 input (CC+HPI+PMH) |
| qlora_v2_train_S2.json | 2.200 | V2 input |
| qlora_v2_train_S3.json | 4.400 | V2 input + few-shot pool |
| qlora_v2_val.json      |   550 | V2 validation |
| qlora_v2_test.json     | 1.650 | V2 test |

**Format (her kayit):**
```json
{
  "note_id": "...",
  "subject_id": ...,
  "hadm_id": ...,
  "instruction": "Classify the following admission note...",
  "input": "<maskeli metin>",
  "output": "SEPSIS"
}
```

**Karar:**
- Instruction sabit (10 dosyada ayni)
- Ingilizce prompt (MIMIC Ingilizce + Qwen stabil)
- Maskeli input kullanilir (model_input_v{1,2}_masked)
- note_id/subject_id/hadm_id eklenir (hata analizi + eslemek icin)
- Few-shot pool V1 ve V2 ayri (ablation icin)

**Cikti klasoru:** `output/qlora_json/`

In [ ]:
# 1) SETUP + FILTRELEME
import pandas as pd
import json
import time
from pathlib import Path

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
JSON_DIR   = OUTPUT_DIR / "qlora_json"
JSON_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("BLOK 6 - QLoRA JSON FORMATI (10 dosya)")
print("=" * 70)

print("\n[1/4] discharge_split.parquet yukleniyor...")
t0 = time.time()
df = pd.read_parquet(SPLIT_YOL)
print(f"   Yuklendi: {len(df):,} satir, {len(df.columns)} kolon")
print(f"   Sure: {time.time()-t0:.1f}s")

# Filtreler
test_df     = df[df["split"] == "test"]
val_df      = df[df["split"] == "val"]
train_S3_df = df[df["split"] == "train_S3"]
train_S2_df = df[(df["split"] == "train_S3") & df["in_S2"]]
train_S1_df = df[(df["split"] == "train_S3") & df["in_S1"]]

print(f"\n   Bolum boyutlari (sanity check):")
print(f"      Test     : {len(test_df):>5,}  (hedef 1.650)")
print(f"      Val      : {len(val_df):>5,}  (hedef   550)")
print(f"      Train S3 : {len(train_S3_df):>5,}  (hedef 4.400)")
print(f"      Train S2 : {len(train_S2_df):>5,}  (hedef 2.200, S3 alt kumesi)")
print(f"      Train S1 : {len(train_S1_df):>5,}  (hedef 1.100, S2 alt kumesi)")

# Nested dogrulama
s1_set = set(train_S1_df["note_id"])
s2_set = set(train_S2_df["note_id"])
s3_set = set(train_S3_df["note_id"])
print(f"\n   Nested dogrulama:")
print(f"      S1 in S2: {s1_set.issubset(s2_set)}")
print(f"      S2 in S3: {s2_set.issubset(s3_set)}")
assert s1_set.issubset(s2_set), "S1 S2'nin alt kumesi degil!"
assert s2_set.issubset(s3_set), "S2 S3'un alt kumesi degil!"

In [ ]:
# 2) JSON URETICI FONKSIYON
INSTRUCTION = (
    "Classify the following admission note into exactly one of these 11 classes: "
    "SEPSIS, STROKE, ATRIAL_FIBRILLATION, AKI, PNEUMONIA, UTI, HEART_FAILURE, "
    "GI_BLEEDING, DIABETES_COMPLICATION, COPD_EXACERBATION, OTHER. "
    "Output only the class name."
)

def df_to_json_records(subset_df, input_kolonu):
    """DataFrame -> JSON kayit listesi."""
    records = []
    for _, row in subset_df.iterrows():
        records.append({
            "note_id": str(row["note_id"]),
            "subject_id": int(row["subject_id"]),
            "hadm_id": int(row["hadm_id"]),
            "instruction": INSTRUCTION,
            "input": str(row[input_kolonu]) if pd.notna(row[input_kolonu]) else "",
            "output": str(row["label"])
        })
    return records

def json_yaz(records, dosya_adi):
    """JSON listesi yaz."""
    yol = JSON_DIR / dosya_adi
    with open(yol, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    boyut_mb = yol.stat().st_size / (1024 * 1024)
    return yol, boyut_mb

print("   + Fonksiyonlar tanimlandi.")
print(f"   Instruction uzunlugu: {len(INSTRUCTION)} karakter")

In [ ]:
# 3) 10 JSON DOSYASI URET
print("\n[2/4] 10 JSON dosyasi olusturuluyor...")
print("-" * 70)

set_eslemesi = [
    ("train_S1", train_S1_df),
    ("train_S2", train_S2_df),
    ("train_S3", train_S3_df),
    ("val",      val_df),
    ("test",     test_df),
]

version_eslemesi = [
    ("v1", "model_input_v1_masked"),
    ("v2", "model_input_v2_masked"),
]

tum_dosyalar = []
t0 = time.time()

for v_ad, v_kolon in version_eslemesi:
    for set_ad, set_df in set_eslemesi:
        dosya_adi = f"qlora_{v_ad}_{set_ad}.json"
        records = df_to_json_records(set_df, v_kolon)
        yol, boyut = json_yaz(records, dosya_adi)
        tum_dosyalar.append({
            "dosya": dosya_adi,
            "kayit": len(records),
            "boyut_mb": boyut
        })
        print(f"   {dosya_adi:<32s} {len(records):>5,} kayit  {boyut:>6.1f} MB")

print(f"\n   Toplam sure: {time.time()-t0:.1f}s")

In [ ]:
# 4) AUDIT - sinif dagilimi + ornek + dosya tablosu
print("\n[3/4] Audit raporu...")
print("-" * 70)

# Dosya tablosu
print(f"\n   Uretilen 10 dosya:")
print(f"   {'Dosya':<32s} {'Kayit':>7s}  {'MB':>6s}")
print("   " + "-" * 50)
toplam_kayit = 0
toplam_mb = 0
for d in tum_dosyalar:
    print(f"   {d['dosya']:<32s} {d['kayit']:>7,}  {d['boyut_mb']:>6.1f}")
    toplam_kayit += d["kayit"]
    toplam_mb += d["boyut_mb"]
print("   " + "-" * 50)
print(f"   {'TOPLAM':<32s} {toplam_kayit:>7,}  {toplam_mb:>6.1f}")

# Sinif dagilimi (test ornek)
print(f"\n   Test seti sinif dagilimi (V1):")
test_v1_yol = JSON_DIR / "qlora_v1_test.json"
with open(test_v1_yol, "r", encoding="utf-8") as f:
    test_v1 = json.load(f)
from collections import Counter
siniflar = Counter([r["output"] for r in test_v1])
for cls in sorted(siniflar.keys()):
    print(f"      {cls:<25s} {siniflar[cls]:>4}")

# Ornek 1 kayit goster
print(f"\n   Ornek kayit (qlora_v1_test.json[0]):")
ornek = test_v1[0]
print(f"      note_id    : {ornek['note_id']}")
print(f"      subject_id : {ornek['subject_id']}")
print(f"      hadm_id    : {ornek['hadm_id']}")
print(f"      instruction: {ornek['instruction'][:80]}...")
print(f"      input      : {ornek['input'][:200]}...")
print(f"      output     : {ornek['output']}")

In [ ]:
# 5) FEW-SHOT POOL DOGRULAMA + OZET
print("\n[4/4] Few-shot pool kontrolu...")
print("-" * 70)

# V1 ve V2 few-shot pool = train_S3.json
for v in ["v1", "v2"]:
    pool_yol = JSON_DIR / f"qlora_{v}_train_S3.json"
    with open(pool_yol, "r", encoding="utf-8") as f:
        pool = json.load(f)
    pool_siniflar = Counter([r["output"] for r in pool])
    print(f"\n   Few-shot pool {v.upper()} ({pool_yol.name}):")
    print(f"      Toplam: {len(pool):,} kayit")
    print(f"      Sinif dagilimi: {dict(sorted(pool_siniflar.items()))}")
    assert len(pool) == 4400, f"{v} few-shot pool 4.400 olmali, {len(pool)} bulundu!"

print("\n" + "=" * 70)
print("BLOK 6 TAMAMLANDI.")
print("=" * 70)
print(f"   Cikti klasoru: {JSON_DIR}")
print(f"   Toplam dosya : 10")
print(f"   Toplam kayit : {toplam_kayit:,}")
print(f"   Toplam boyut : {toplam_mb:.1f} MB")
print("\nSiradaki: Blok 7 - Klasik ML baseline (TF-IDF + SVM + LR)")

# BLOK 7 - Klasik ML Baseline (TF-IDF + SVM + LR)

**Amac:** 7 deney kosulundan 2'si (SVM + LR) icin baseline.

**Veri:**
- Train: 4.400 (S3)
- Test:  1.650 (sabit)
- Val:   kullanilmaz (klasik ML'de hyperparameter search yok)

**Pipeline:**
1. TF-IDF vectorizer (sadece train uzerinde fit) - max_features=20k, ngram(1,2)
2. SVM (LinearSVC) ve LR (LogisticRegression) ayri ayri egit
3. V1 (CC+HPI) ve V2 (CC+HPI+PMH) ablation - toplam 4 model
4. Test uzerinde tahmin (Full + Clean subset)
5. Metrikler: Accuracy, Macro-F1, Weighted-F1, Confusion Matrix

**Cikti:** `output/classical_ml/` (modeller + results.csv + predictions.csv)

In [ ]:
# 1) SETUP + VERI YUKLEME
import pandas as pd
import numpy as np
import time
import joblib
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix)

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
ML_DIR     = OUTPUT_DIR / "classical_ml"
ML_DIR.mkdir(parents=True, exist_ok=True)

RNG_SEED = 42

print("=" * 70)
print("BLOK 7 - KLASIK ML BASELINE (TF-IDF + SVM + LR)")
print("=" * 70)

print("\n[1/5] discharge_split.parquet yukleniyor...")
t0 = time.time()
df = pd.read_parquet(SPLIT_YOL)
print(f"   Yuklendi: {len(df):,} satir, {len(df.columns)} kolon")
print(f"   Sure: {time.time()-t0:.1f}s")

# Train (S3 full) + Test
train_df = df[df["split"] == "train_S3"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

print(f"\n   Train: {len(train_df):,} (S3 full)")
print(f"   Test:  {len(test_df):,}")

# Sınıf dağılımı
print(f"\n   Train sinif dagilimi:")
print(f"      {dict(train_df['label'].value_counts().sort_index())}")

# Clean test alt kümeleri
test_clean_v1_idx = ~test_df["has_label_term_v1"]
test_clean_v2_idx = ~test_df["has_label_term_v2"]
print(f"\n   Clean test V1: {test_clean_v1_idx.sum():,}")
print(f"   Clean test V2: {test_clean_v2_idx.sum():,}")

In [ ]:
# 2) TF-IDF + MODEL EGITIM FONKSIYONU
def egit_ve_degerlendir(version):
    """V1 veya V2 icin TF-IDF + SVM + LR egit, test uzerinde degerlendir."""
    input_kolonu = f"model_input_{version}_masked"
    has_label_kolonu = f"has_label_term_{version}"
    
    print(f"\n{'=' * 70}")
    print(f"  VERSION: {version.upper()} (input: {input_kolonu})")
    print(f"{'=' * 70}")
    
    X_train_text = train_df[input_kolonu].fillna("").astype(str).tolist()
    X_test_text  = test_df[input_kolonu].fillna("").astype(str).tolist()
    y_train = train_df["label"].values
    y_test  = test_df["label"].values
    
    # --- TF-IDF (sadece train uzerinde fit, test'ten kelime gormez) ---
    print(f"\n  [TF-IDF] Fit ediliyor (max_features=20k, ngram=(1,2))...")
    t0 = time.time()
    tfidf = TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        min_df=2,
        stop_words="english",
        sublinear_tf=True
    )
    X_train = tfidf.fit_transform(X_train_text)
    X_test  = tfidf.transform(X_test_text)
    print(f"     Train shape: {X_train.shape}")
    print(f"     Test shape:  {X_test.shape}")
    print(f"     Sure: {time.time()-t0:.1f}s")
    
    # Vectorizer kaydet
    joblib.dump(tfidf, ML_DIR / f"tfidf_{version}.joblib")
    
    sonuclar = {}
    
    # --- SVM ---
    print(f"\n  [SVM] LinearSVC egitiliyor...")
    t0 = time.time()
    svm = LinearSVC(C=1.0, class_weight="balanced", random_state=RNG_SEED, max_iter=5000)
    svm.fit(X_train, y_train)
    print(f"     Sure: {time.time()-t0:.1f}s")
    pred_svm = svm.predict(X_test)
    joblib.dump(svm, ML_DIR / f"svm_{version}.joblib")
    sonuclar["svm"] = pred_svm
    
    # --- LR ---
    print(f"\n  [LR] LogisticRegression egitiliyor...")
    t0 = time.time()
    lr = LogisticRegression(C=1.0, class_weight="balanced", solver="lbfgs",
                            max_iter=2000, random_state=RNG_SEED, n_jobs=-1)
    lr.fit(X_train, y_train)
    print(f"     Sure: {time.time()-t0:.1f}s")
    pred_lr = lr.predict(X_test)
    joblib.dump(lr, ML_DIR / f"lr_{version}.joblib")
    sonuclar["lr"] = pred_lr
    
    # --- Metrikler ---
    print(f"\n  [METRIKLER]")
    print(f"  {'Model':<10s} {'Subset':<12s} {'N':>6s} {'Acc':>7s} {'MacroF1':>9s} {'WeightedF1':>11s}")
    print("  " + "-" * 58)
    
    metrik_satirlari = []
    clean_idx = (~test_df[has_label_kolonu]).values
    
    for model_ad, pred in [("SVM", pred_svm), ("LR", pred_lr)]:
        for subset_ad, mask in [("Full", np.ones(len(test_df), dtype=bool)),
                                 ("Clean", clean_idx)]:
            y_t = y_test[mask]
            p_t = pred[mask]
            acc = accuracy_score(y_t, p_t)
            macro_f1 = f1_score(y_t, p_t, average="macro", zero_division=0)
            weighted_f1 = f1_score(y_t, p_t, average="weighted", zero_division=0)
            print(f"  {model_ad:<10s} {subset_ad:<12s} {mask.sum():>6,} "
                  f"{acc:>7.4f} {macro_f1:>9.4f} {weighted_f1:>11.4f}")
            metrik_satirlari.append({
                "version": version,
                "model": model_ad,
                "subset": subset_ad,
                "n": int(mask.sum()),
                "accuracy": acc,
                "macro_f1": macro_f1,
                "weighted_f1": weighted_f1
            })
    
    return sonuclar, metrik_satirlari

print("   + Fonksiyon tanimlandi.")

In [ ]:
# 3) V1 + V2 EGIT VE DEGERLENDIR
print("\n[2/5] Modeller egitiliyor...")

tum_metrikler = []
tum_tahminler = {}

for v in ["v1", "v2"]:
    sonuclar, metrikler = egit_ve_degerlendir(v)
    tum_metrikler.extend(metrikler)
    tum_tahminler[f"svm_{v}"] = sonuclar["svm"]
    tum_tahminler[f"lr_{v}"] = sonuclar["lr"]

print("\n+ Tum modeller egitildi.")

In [ ]:
# 4) RESULTS.CSV + PREDICTIONS.CSV YAZ
print("\n[3/5] Sonuclar kaydediliyor...")

# results.csv (8 satir: 2 model x 2 subset x 2 version)
results_df = pd.DataFrame(tum_metrikler)
results_yol = ML_DIR / "results.csv"
results_df.to_csv(results_yol, index=False)
print(f"   results.csv: {len(results_df)} satir -> {results_yol.name}")

# predictions.csv (1.650 satir, her model icin tahmin)
predictions_df = pd.DataFrame({
    "note_id": test_df["note_id"].values,
    "subject_id": test_df["subject_id"].values,
    "hadm_id": test_df["hadm_id"].values,
    "label": test_df["label"].values,
    "has_label_term_v1": test_df["has_label_term_v1"].values,
    "has_label_term_v2": test_df["has_label_term_v2"].values,
    "pred_svm_v1": tum_tahminler["svm_v1"],
    "pred_svm_v2": tum_tahminler["svm_v2"],
    "pred_lr_v1":  tum_tahminler["lr_v1"],
    "pred_lr_v2":  tum_tahminler["lr_v2"],
})
preds_yol = ML_DIR / "predictions.csv"
predictions_df.to_csv(preds_yol, index=False)
print(f"   predictions.csv: {len(predictions_df)} satir -> {preds_yol.name}")

# Genel sonuc tablosu
print(f"\n   Sonuc tablosu (results.csv):")
print(results_df.to_string(index=False))

In [ ]:
# 5) PER-CLASS REPORT (en iyi V icin) + KARSILASTIRMA OZETI
print("\n[4/5] Per-class detayli rapor...")
print("-" * 70)

y_test = test_df["label"].values

for v in ["v1", "v2"]:
    print(f"\n  >>> SVM-{v.upper()} per-class (Full test):")
    print(classification_report(y_test, tum_tahminler[f"svm_{v}"], digits=3, zero_division=0))
    
    print(f"\n  >>> LR-{v.upper()} per-class (Full test):")
    print(classification_report(y_test, tum_tahminler[f"lr_{v}"], digits=3, zero_division=0))

In [ ]:
# 6) FINAL OZET
print("\n[5/5] Final ozet...")
print("=" * 70)

# Karsilastirma tablosu (Full test)
print("\n  Klasik ML Baseline Sonuclari (Full Test, n=1.650):")
print("  " + "-" * 60)
print(f"  {'Model':<12s} {'Acc':>8s} {'MacroF1':>10s} {'WeightedF1':>12s}")
print("  " + "-" * 60)
full_df = results_df[results_df["subset"] == "Full"].copy()
for _, row in full_df.iterrows():
    model_lbl = f"{row['model']}-{row['version'].upper()}"
    print(f"  {model_lbl:<12s} {row['accuracy']:>8.4f} {row['macro_f1']:>10.4f} {row['weighted_f1']:>12.4f}")

print("\n  Clean Subset Sonuclari:")
print("  " + "-" * 60)
print(f"  {'Model':<12s} {'N':>6s} {'Acc':>8s} {'MacroF1':>10s} {'WeightedF1':>12s}")
print("  " + "-" * 60)
clean_df = results_df[results_df["subset"] == "Clean"].copy()
for _, row in clean_df.iterrows():
    model_lbl = f"{row['model']}-{row['version'].upper()}"
    print(f"  {model_lbl:<12s} {row['n']:>6,} {row['accuracy']:>8.4f} "
          f"{row['macro_f1']:>10.4f} {row['weighted_f1']:>12.4f}")

# Cikti dosyalari ozeti
print("\n  Uretilen dosyalar (output/classical_ml/):")
for p in sorted(ML_DIR.glob("*")):
    boyut_kb = p.stat().st_size / 1024
    print(f"     {p.name:<30s} {boyut_kb:>8.1f} KB")

print("\n" + "=" * 70)
print("BLOK 7 TAMAMLANDI.")
print("Siradaki: Blok 8 - Zero-shot inference (Qwen3.5-4B, 1.650 test)")
print("=" * 70)

# BLOK 7B - Clean Subset Duzeltme (has_own_label_term)

**Sorun:** Mevcut `has_label_term_v1/v2` kolonu BIRLESIK regex ile 11 sinif keyword'lerinden
HERHANGI BIRINI ariyor. Yani PNEUMONIA satirinda 'diabetes' gectiyse de True doner.
Bu **clean test'i fazla siki tanimliyor** ve metodolojik olarak yanlis.

**Dogrusu:** Her satir icin SADECE KENDI label'inin keyword'u geciyor mu kontrolu.

**Aksiyon:**
1. `has_own_label_term_v1/v2` kolonlari ekle (her satir kendi label'i icin)
2. discharge_split.parquet'i guncelle
3. Blok 7 sonuclarini yeni clean tanimiyla yeniden hesapla (sadece metric, model fit YOK)
4. Karsilastirma: eski clean vs yeni clean

**Beklenen etki:** Clean subset boyutu ARTAR (PNEUMONIA + 'diabetes' geceni
artik clean'e dahil edicegiz). Daha gercekci sizinti olcumu.

**Cikti:**
- discharge_split.parquet (2 yeni kolon: has_own_label_term_v1, _v2)
- output/classical_ml/results_v2.csv (yeni clean ile metrikler)

In [ ]:
# 1) SETUP + SINIF-SPESIFIK REGEX SOZLUGU
import pandas as pd
import numpy as np
import re
import time
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
ML_DIR     = OUTPUT_DIR / "classical_ml"

print("=" * 70)
print("BLOK 7B - CLEAN SUBSET DUZELTME")
print("=" * 70)

# Sinif-spesifik keyword sozlugu (Blok 4'tekiyle ayni)
CLASS_KEYWORDS = {
    'PNEUMONIA': r'\b(pneumon\w*|bronchopneumon\w*)\b',
    'HEART_FAILURE': r'\b(CHF|congestive heart failure|heart failure|HFrEF|HFpEF|systolic heart failure|diastolic heart failure)\b',
    'SEPSIS': r'\b(sepsis|septic|septicemia|bacteremia|septicaemia)\b',
    'AKI': r'\b(AKI|acute kidney injury|acute renal failure|ARF|acute renal insufficiency)\b',
    'UTI': r'\b(UTI|urinary tract infection|pyelonephritis|cystitis)\b',
    'STROKE': r'\b(stroke|CVA|cerebrovascular accident|ischemic stroke|hemorrhagic stroke|intracerebral hemorrhage|intracranial hemorrhage)\b',
    'GI_BLEEDING': r'\b(GI bleed\w*|gastrointestinal bleed\w*|UGIB|LGIB|hematemesis|melena|hematochezia)\b',
    'COPD_EXACERBATION': r'\b(COPD exacerbation|AECOPD|chronic obstructive|COPD)\b',
    'DIABETES_COMPLICATION': r'\b(DKA|diabetic ketoacidosis|HHS|hyperosmolar|hyperglycemic crisis|diabetic coma)\b',
    'ATRIAL_FIBRILLATION': r'\b(atrial fibrillation|AFib|A-fib|atrial flutter|paroxysmal AF)\b',
    'OTHER': None,  # OTHER icin keyword yok - hicbir keyword'u yok demek
}

# Regex compile
CLASS_REGEXES = {}
for cls, pat in CLASS_KEYWORDS.items():
    if pat is None:
        CLASS_REGEXES[cls] = None
    else:
        CLASS_REGEXES[cls] = re.compile(pat, re.IGNORECASE)

print(f"\n   Sinif-spesifik regex sayisi: {sum(1 for v in CLASS_REGEXES.values() if v is not None)} / 11")
print(f"   (OTHER icin keyword yok, has_own_label_term_OTHER=False sabit)")

In [ ]:
# 2) HAS_OWN_LABEL_TERM HESAPLA
print("\n[1/4] discharge_split.parquet yukleniyor...")
t0 = time.time()
df = pd.read_parquet(SPLIT_YOL)
print(f"   Yuklendi: {len(df):,} satir, {len(df.columns)} kolon")
print(f"   Sure: {time.time()-t0:.1f}s")

print("\n[2/4] has_own_label_term hesaplaniyor...")
t0 = time.time()

def kendi_keyword_iceriyor_mu(metin, label):
    """Metin satirin KENDI label'inin keyword'unu iceriyor mu?"""
    if not isinstance(metin, str) or not metin:
        return False
    regex = CLASS_REGEXES.get(label)
    if regex is None:  # OTHER icin
        return False
    return regex.search(metin) is not None

# Maskelenmemis input uzerinde ara (model_input_v1, model_input_v2)
df["has_own_label_term_v1"] = [
    kendi_keyword_iceriyor_mu(t, l) for t, l in zip(df["model_input_v1"], df["label"])
]
df["has_own_label_term_v2"] = [
    kendi_keyword_iceriyor_mu(t, l) for t, l in zip(df["model_input_v2"], df["label"])
]

print(f"   Sure: {time.time()-t0:.1f}s")

# Karsilastirma: eski vs yeni
print(f"\n   Genel oranlar:")
print(f"      ESKI has_label_term_v1     (BIRLESIK): {df['has_label_term_v1'].sum():>5,} ({100*df['has_label_term_v1'].mean():>5.1f}%)")
print(f"      YENI has_own_label_term_v1 (kendi)   : {df['has_own_label_term_v1'].sum():>5,} ({100*df['has_own_label_term_v1'].mean():>5.1f}%)")
print(f"      ESKI has_label_term_v2     (BIRLESIK): {df['has_label_term_v2'].sum():>5,} ({100*df['has_label_term_v2'].mean():>5.1f}%)")
print(f"      YENI has_own_label_term_v2 (kendi)   : {df['has_own_label_term_v2'].sum():>5,} ({100*df['has_own_label_term_v2'].mean():>5.1f}%)")

# Sinif bazli
print(f"\n   Sinif bazli has_own_label_term oranlari:")
print(f"   {'Sinif':<25s} {'V1 own%':>9s} {'V2 own%':>9s} {'Toplam':>8s}")
print("   " + "-" * 55)
for cls in sorted(df["label"].unique()):
    alt = df[df["label"] == cls]
    v1_o = 100 * alt["has_own_label_term_v1"].mean()
    v2_o = 100 * alt["has_own_label_term_v2"].mean()
    print(f"   {cls:<25s} {v1_o:>8.1f}% {v2_o:>8.1f}% {len(alt):>8,}")

In [ ]:
# 3) TEST SETI UZERINDE CLEAN SUBSET BOYUT KARSILASTIRMASI
print("\n[3/4] Test seti clean boyut karsilastirmasi:")
print("-" * 70)

test_df = df[df["split"] == "test"]

# Eski clean (BIRLESIK regex)
eski_clean_v1 = (~test_df["has_label_term_v1"]).sum()
eski_clean_v2 = (~test_df["has_label_term_v2"]).sum()

# Yeni clean (kendi label'i)
yeni_clean_v1 = (~test_df["has_own_label_term_v1"]).sum()
yeni_clean_v2 = (~test_df["has_own_label_term_v2"]).sum()

print(f"\n   Test setindeki clean ornek sayisi (1.650 uzerinden):")
print(f"      ESKI (BIRLESIK regex):")
print(f"         V1 clean: {eski_clean_v1:>4,} ({100*eski_clean_v1/len(test_df):>4.1f}%)")
print(f"         V2 clean: {eski_clean_v2:>4,} ({100*eski_clean_v2/len(test_df):>4.1f}%)")
print(f"      YENI (kendi label'i):")
print(f"         V1 clean: {yeni_clean_v1:>4,} ({100*yeni_clean_v1/len(test_df):>4.1f}%)")
print(f"         V2 clean: {yeni_clean_v2:>4,} ({100*yeni_clean_v2/len(test_df):>4.1f}%)")
print(f"      ARTIS:")
print(f"         V1: +{yeni_clean_v1 - eski_clean_v1:,}  ({100*(yeni_clean_v1-eski_clean_v1)/len(test_df):+.1f}%)")
print(f"         V2: +{yeni_clean_v2 - eski_clean_v2:,}  ({100*(yeni_clean_v2-eski_clean_v2)/len(test_df):+.1f}%)")

# Sinif basina clean dağılımı (yeni)
print(f"\n   Yeni clean test sinif dagilimi:")
print(f"   {'Sinif':<25s} {'Test':>5s} {'Clean V1':>9s} {'Clean V2':>9s}")
print("   " + "-" * 55)
for cls in sorted(test_df["label"].unique()):
    alt = test_df[test_df["label"] == cls]
    cv1 = (~alt["has_own_label_term_v1"]).sum()
    cv2 = (~alt["has_own_label_term_v2"]).sum()
    print(f"   {cls:<25s} {len(alt):>5} {cv1:>9} {cv2:>9}")

In [ ]:
# 4) discharge_split.parquet GUNCELLE (2 yeni kolon ekle)
print("\n[4/4] discharge_split.parquet guncelleniyor...")
t0 = time.time()
df.to_parquet(SPLIT_YOL, compression="snappy", index=False)
boyut = SPLIT_YOL.stat().st_size / (1024 * 1024)
print(f"   Yazildi: {SPLIT_YOL.name}")
print(f"   Boyut  : {boyut:.1f} MB")
print(f"   Kolon  : {len(df.columns)} (eski 22 + yeni 2 = 24)")
print(f"   Sure   : {time.time()-t0:.1f}s")

# Dogrulama
df_check = pd.read_parquet(SPLIT_YOL)
assert "has_own_label_term_v1" in df_check.columns
assert "has_own_label_term_v2" in df_check.columns
print(f"   + Dogrulama: 2 yeni kolon mevcut")

In [ ]:
# 5) BLOK 7 METRICLERINI YENI CLEAN TANIMIYLA RECOMPUTE
print("\n[BLOK 7 METRIC RECOMPUTE - yeni clean ile]")
print("=" * 70)

# predictions.csv yukle
preds_df = pd.read_csv(ML_DIR / "predictions.csv")
print(f"   predictions.csv yuklendi: {len(preds_df):,} satir")

# YENI has_own_label_term kolonlarini test_df'den ekle (predictions.csv'de yoktu)
test_df = df[df["split"] == "test"][["note_id", "has_own_label_term_v1", "has_own_label_term_v2"]]
preds_df = preds_df.merge(test_df, on="note_id", how="left")

y_test = preds_df["label"].values
yeni_metrikler = []

for model_ad in ["SVM", "LR"]:
    for v in ["v1", "v2"]:
        kolon = f"pred_{model_ad.lower()}_{v}"
        pred = preds_df[kolon].values
        
        # 3 subset: Full, Clean (eski - BIRLESIK), Clean (yeni - kendi)
        full_mask = np.ones(len(preds_df), dtype=bool)
        eski_clean_mask = (~preds_df[f"has_label_term_{v}"]).values
        yeni_clean_mask = (~preds_df[f"has_own_label_term_{v}"]).values
        
        for subset_ad, mask in [("Full", full_mask),
                                 ("Clean (BIRLESIK)", eski_clean_mask),
                                 ("Clean (KENDI)", yeni_clean_mask)]:
            y_t = y_test[mask]
            p_t = pred[mask]
            acc = accuracy_score(y_t, p_t)
            macro_f1 = f1_score(y_t, p_t, average="macro", zero_division=0)
            weighted_f1 = f1_score(y_t, p_t, average="weighted", zero_division=0)
            yeni_metrikler.append({
                "model": model_ad,
                "version": v,
                "subset": subset_ad,
                "n": int(mask.sum()),
                "accuracy": acc,
                "macro_f1": macro_f1,
                "weighted_f1": weighted_f1
            })

yeni_results_df = pd.DataFrame(yeni_metrikler)

# Tablo
print(f"\n   Yeni Sonuc Tablosu:")
print("   " + "-" * 80)
print(f"   {'Model':<6s} {'V':<3s} {'Subset':<20s} {'N':>5s} {'Acc':>8s} {'MacroF1':>10s} {'WF1':>8s}")
print("   " + "-" * 80)
for _, row in yeni_results_df.iterrows():
    print(f"   {row['model']:<6s} {row['version']:<3s} {row['subset']:<20s} "
          f"{row['n']:>5,} {row['accuracy']:>8.4f} {row['macro_f1']:>10.4f} {row['weighted_f1']:>8.4f}")

# Kaydet
yeni_results_yol = ML_DIR / "results_v2.csv"
yeni_results_df.to_csv(yeni_results_yol, index=False)
print(f"\n   Kaydedildi: {yeni_results_yol.name}")

print("\n" + "=" * 70)
print("BLOK 7B TAMAMLANDI.")
print("   Yeni clean tanimi: has_own_label_term (sadece kendi label'i)")
print("   Bu tanim Blok 8 (zero-shot) ve sonrasinda kullanilacak.")
print("=" * 70)

# BLOK 8A - Zero-shot SMOKE TEST (REVIZE: chat_template)

**Onceki run sorunu:** Qwen3.5 default thinking mode -> `<think>...</think>` bloklari uretiyordu
ve max_new_tokens=10 yetmiyordu. Parser invalid 7/10 idi.

**Cozum (resmi Qwen dokumantasyonu):**
- `tokenizer.apply_chat_template(..., enable_thinking=False)` ile thinking kapatilir
- Raw prompt yerine messages list + chat template kullanilir
- max_new_tokens=20 (thinking yokken yeterli)

**Ek duzeltmeler:**
- PROMPT'tan 'one word' ifadesi kaldirildi (sinif adlari tek kelime degil)
- Yerine: 'Output exactly one class label from the list, using the exact uppercase label string'
- Parser invalid'i OTHER'a GOMMIYOR — invalid invalid kalir, gercek invalid orani gorunsun

**5 soruya cevap:**
1. Model gercekten yukleniyor mu?
2. GPU bellek yeterli mi?
3. Ortalama saniye/ornek kac?
4. Model class label formatinda cevap veriyor mu?
5. Parser invalid orani makul mu?

In [ ]:
# 1) SETUP + SABITLER
import pandas as pd
import torch
import time
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"

MODEL_ID = "Qwen/Qwen3.5-4B"
ALLOWED_CLASSES = [
    "SEPSIS", "STROKE", "ATRIAL_FIBRILLATION", "AKI", "PNEUMONIA", "UTI",
    "HEART_FAILURE", "GI_BLEEDING", "DIABETES_COMPLICATION",
    "COPD_EXACERBATION", "OTHER"
]

PROMPT_TEMPLATE = """You are a clinical NLP system. Given a patient's admission note, classify the PRIMARY discharge diagnosis into ONE of these 11 categories:

SEPSIS - Systemic infection, septic shock, bacteremia
STROKE - Ischemic or hemorrhagic stroke, CVA
ATRIAL_FIBRILLATION - Atrial fibrillation/flutter (acute)
AKI - Acute kidney injury, acute renal failure
PNEUMONIA - Lung infection (bacterial, viral, aspiration)
UTI - Urinary tract infection, pyelonephritis, cystitis
HEART_FAILURE - Congestive heart failure, decompensated CHF
GI_BLEEDING - Upper or lower GI hemorrhage
DIABETES_COMPLICATION - DKA, HHS, severe hyper/hypoglycemia
COPD_EXACERBATION - Acute exacerbation of COPD
OTHER - None of the above

Rules:
- Output exactly one class label from the list, using the exact uppercase label string.
- Choose the SINGLE most likely PRIMARY reason for admission.

ADMISSION NOTE:
{input}

PRIMARY DIAGNOSIS:"""

print("=" * 70)
print("BLOK 8A - SMOKE TEST REVIZE (chat_template + enable_thinking=False)")
print("=" * 70)
print(f"   Model    : {MODEL_ID}")
print(f"   Sinif    : {len(ALLOWED_CLASSES)}")
print(f"   Pilot N  : 10 (val'dan rastgele)")
print(f"   GPU      : {torch.cuda.get_device_name(0)}")
print(f"   GPU mem  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# SORU 1: Model gercekten yukleniyor mu? (zaten yuklenmisse atla)
print("\n" + "=" * 70)
print("[1/5] Model yukleme...")
print("-" * 70)

if "model" in dir() and "tokenizer" in dir():
    print("   Model zaten bellekte, yeniden yuklenmiyor.")
    soru1_ok = True
else:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tok_sure = time.time() - t0
    print(f"   Tokenizer: {tok_sure:.1f}s")
    
    t0 = time.time()
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        local_files_only=True,
    )
    model.eval()
    model_sure = time.time() - t0
    print(f"   Model    : {model_sure:.1f}s")
    print(f"   TOPLAM   : {tok_sure + model_sure:.1f}s")
    soru1_ok = True

print(f"   YUKLENDI")

In [ ]:
# SORU 2: GPU bellek yeterli mi?
print("\n" + "=" * 70)
print("[2/5] GPU bellek durumu...")
print("-" * 70)

alloc_mb = torch.cuda.memory_allocated() / 1024**2
reserved_mb = torch.cuda.memory_reserved() / 1024**2
total_mb = torch.cuda.get_device_properties(0).total_memory / 1024**2
kullanim_pct = 100 * alloc_mb / total_mb

print(f"   Allocated : {alloc_mb:>7,.0f} MB")
print(f"   Reserved  : {reserved_mb:>7,.0f} MB")
print(f"   Total     : {total_mb:>7,.0f} MB")
print(f"   Kullanim  : {kullanim_pct:.1f}% (sinir: %80)")

soru2_ok = kullanim_pct < 80
if soru2_ok:
    print(f"   YETERLI ({(total_mb - alloc_mb):.0f} MB bos)")
else:
    print(f"   YETERSIZ - quantization config revize gerekli")

In [ ]:
# SORU 3+4: Saniye/ornek + Class formati (CHAT TEMPLATE + enable_thinking=False)
print("\n" + "=" * 70)
print("[3/5] Inference hizi + [4/5] Class formati (10 ornek)...")
print("-" * 70)

df = pd.read_parquet(SPLIT_YOL)
val_df = df[df["split"] == "val"].sample(10, random_state=42).reset_index(drop=True)

def inference(input_text):
    """Chat template + enable_thinking=False (Qwen3 standardi)."""
    prompt = PROMPT_TEMPLATE.format(input=input_text)
    
    messages = [{"role": "user", "content": prompt}]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to("cuda")
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    raw = tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()
    return raw, inputs.input_ids.shape[1]

kayitlar = []
sureler = []

for i, row in val_df.iterrows():
    inp = row["model_input_v1_masked"] or ""
    t0 = time.time()
    raw, in_tok = inference(inp)
    sure = time.time() - t0
    sureler.append(sure)
    kayitlar.append({
        "i": i + 1,
        "label": row["label"],
        "raw": raw,
        "in_tokens": in_tok,
        "sure": sure
    })

ort_sure = np.mean(sureler)
min_sure = np.min(sureler)
max_sure = np.max(sureler)

print(f"\n   Hiz istatistigi (10 ornek):")
print(f"      Ort  : {ort_sure*1000:>6.0f} ms ({ort_sure:.2f}s)")
print(f"      Min  : {min_sure*1000:>6.0f} ms")
print(f"      Max  : {max_sure*1000:>6.0f} ms")

print(f"\n   Sure tahminleri (8B + 8C):")
print(f"      Val V1+V2 (1.100): ~{1100 * ort_sure / 60:.1f} dk")
print(f"      Test V1   (1.650): ~{1650 * ort_sure / 60:.1f} dk")
print(f"      Test V2   (1.650): ~{1650 * ort_sure / 60:.1f} dk")
print(f"      TOPLAM    (4.400): ~{4400 * ort_sure / 60:.1f} dk")

print(f"\n   Class formati (10 ornek raw output):")
print(f"   {'#':<3s} {'GERCEK':<25s} {'IN_TOK':>7s} {'MS':>6s}  RAW")
print("   " + "-" * 100)
for k in kayitlar:
    raw_disp = repr(k['raw'][:80]) if len(k['raw']) > 80 else repr(k['raw'])
    print(f"   {k['i']:<3d} {k['label']:<25s} {k['in_tokens']:>7d} {k['sure']*1000:>6.0f}  {raw_disp}")

soru3_ok = ort_sure < 5.0
if soru3_ok:
    print(f"\n   HIZ OK ({ort_sure:.2f}s/ornek < 5s)")
else:
    print(f"\n   YAVAS ({ort_sure:.2f}s/ornek)")

In [ ]:
# SORU 5: Parser invalid orani (invalid -> 'INVALID' kalir, OTHER'a GOMMUYOR)
print("\n" + "=" * 70)
print("[5/5] Parser invalid orani...")
print("-" * 70)

def parse_class(raw, allowed):
    """Raw output -> sinif. Invalid -> 'INVALID' (OTHER'a GOMMUYOR — gercek oran gorunsun)."""
    if not raw:
        return "INVALID", "empty"
    text = raw.strip().upper()
    
    # 1. Direct match
    siralama = sorted(allowed, key=len, reverse=True)
    for cls in siralama:
        if cls in text[:80]:
            return cls, "direct"
    
    # 2. Fuzzy keyword
    fuzzy_map = {
        "SEPS": "SEPSIS", "SEPT": "SEPSIS",
        "PNEUM": "PNEUMONIA",
        "STROK": "STROKE", "CVA": "STROKE",
        "ATRIAL": "ATRIAL_FIBRILLATION", "AFIB": "ATRIAL_FIBRILLATION", "FIBRIL": "ATRIAL_FIBRILLATION",
        "AKI": "AKI", "KIDNEY": "AKI", "RENAL": "AKI",
        "UTI": "UTI", "URIN": "UTI",
        "HEART": "HEART_FAILURE", "CHF": "HEART_FAILURE",
        "GI": "GI_BLEEDING", "BLEED": "GI_BLEEDING",
        "DIAB": "DIABETES_COMPLICATION", "DKA": "DIABETES_COMPLICATION",
        "COPD": "COPD_EXACERBATION",
        "OTHER": "OTHER", "NONE": "OTHER",
    }
    for k, v in fuzzy_map.items():
        if k in text[:80]:
            return v, "fuzzy"
    
    # 3. Invalid -> KALIR (OTHER'a GOMMUYOR)
    return "INVALID", "invalid"

parse_sayim = {"direct": 0, "fuzzy": 0, "invalid": 0, "empty": 0}
parse_detay = []

for k in kayitlar:
    parsed, kategori = parse_class(k["raw"], ALLOWED_CLASSES)
    parse_sayim[kategori] = parse_sayim.get(kategori, 0) + 1
    parse_detay.append({
        "i": k["i"],
        "label": k["label"],
        "raw": k["raw"][:50],
        "parsed": parsed,
        "kat": kategori,
        "dogru": parsed == k["label"]
    })

print(f"\n   Parser dagilimi (10 ornek):")
print(f"      Direct match : {parse_sayim['direct']:>2d}/10")
print(f"      Fuzzy match  : {parse_sayim['fuzzy']:>2d}/10")
print(f"      Invalid      : {parse_sayim['invalid']:>2d}/10  (OTHER'a GOMMEDI)")
print(f"      Empty        : {parse_sayim['empty']:>2d}/10")

invalid_pct = 100 * (parse_sayim["invalid"] + parse_sayim["empty"]) / 10
soru5_ok = invalid_pct <= 30

print(f"\n   Parser detay:")
print(f"   {'#':<3s} {'GERCEK':<22s} {'PARSED':<22s} {'KAT':<8s} {'DOGRU':<5s} RAW")
print("   " + "-" * 100)
for d in parse_detay:
    isaret = "OK" if d["dogru"] else "  "
    print(f"   {d['i']:<3d} {d['label']:<22s} {d['parsed']:<22s} {d['kat']:<8s} {isaret:<5s} {d['raw']!r}")

# Pilot accuracy (sadece direct/fuzzy uzerinden, invalid harici)
valid_n = parse_sayim["direct"] + parse_sayim["fuzzy"]
dogru_pilot = sum(1 for d in parse_detay if d["dogru"])
print(f"\n   Pilot accuracy: {dogru_pilot}/10 (sadece bilgi)")
print(f"   Valid output  : {valid_n}/10 (direct + fuzzy)")

if soru5_ok:
    print(f"   PARSER OK (invalid {invalid_pct:.0f}% <= %30)")
else:
    print(f"   PARSER ZAYIF (invalid {invalid_pct:.0f}% > %30)")

# --- KARAR ---
print("\n" + "=" * 70)
print("KARAR")
print("=" * 70)

kontrol = [
    ("Model yuklendi",   soru1_ok),
    ("GPU < %80",        soru2_ok),
    ("Hiz < 5s/ornek",   soru3_ok),
    ("Parser invalid <= %30", soru5_ok),
]
for ad, ok in kontrol:
    isaret = "OK" if ok else "FAIL"
    print(f"   [{isaret}] {ad}")

if all(ok for _, ok in kontrol):
    print("\n   ✅ TUM KONTROLLER GECTI -> Blok 8B'ye gecilebilir")
else:
    print("\n   ⚠️  REVIZE GEREKLI - sorunlari coz, 8A'yi tekrar calistir")
print("=" * 70)

# BLOK 8B - Zero-shot VAL Inference (sadece val, test'e dokunma)

**Amac:**
1. Val (550 ornek) uzerinde V1 ve V2 zero-shot inference
2. Checkpoint/resume ile kesintilere karsi guvenli
3. Sonunda prompt + parser FINAL karari (Test'e gecmeden once)

**KRITIK: Test setine DOKUNMUYOR.** Bu Codex eleştirisi 4'un karsiligi.
Test sadece final olcum (8C), prompt tuning val'da yapilir.

**Inference config (8A'dan onayli):**
- chat_template + enable_thinking=False
- max_new_tokens=20, do_sample=False
- max_length=2048
- Maskeli input (model_input_v{1,2}_masked)

**Sure tahmini:** 550 × 2 × 3sn = ~55 dakika

**Cikti klasoru:** `output/zeroshot/`

In [ ]:
# 1) SETUP + HELPER FONKSIYONLAR
import pandas as pd
import numpy as np
import torch
import time
from pathlib import Path

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
ZS_DIR     = OUTPUT_DIR / "zeroshot"
ZS_DIR.mkdir(parents=True, exist_ok=True)

# Model 8A'dan kaldiysa kullan, yoksa yukle
if "model" not in dir() or "tokenizer" not in dir():
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    print("   Model bellekte yok, yukleniyor...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3.5-4B",
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        local_files_only=True,
    )
    model.eval()
    print("   Model yuklendi.")
else:
    print("   Model 8A'dan bellekte, yeniden yuklenmiyor.")

ALLOWED_CLASSES = [
    "SEPSIS", "STROKE", "ATRIAL_FIBRILLATION", "AKI", "PNEUMONIA", "UTI",
    "HEART_FAILURE", "GI_BLEEDING", "DIABETES_COMPLICATION",
    "COPD_EXACERBATION", "OTHER"
]

PROMPT_TEMPLATE = """You are a clinical NLP system. Given a patient's admission note, classify the PRIMARY discharge diagnosis into ONE of these 11 categories:

SEPSIS - Systemic infection, septic shock, bacteremia
STROKE - Ischemic or hemorrhagic stroke, CVA
ATRIAL_FIBRILLATION - Atrial fibrillation/flutter (acute)
AKI - Acute kidney injury, acute renal failure
PNEUMONIA - Lung infection (bacterial, viral, aspiration)
UTI - Urinary tract infection, pyelonephritis, cystitis
HEART_FAILURE - Congestive heart failure, decompensated CHF
GI_BLEEDING - Upper or lower GI hemorrhage
DIABETES_COMPLICATION - DKA, HHS, severe hyper/hypoglycemia
COPD_EXACERBATION - Acute exacerbation of COPD
OTHER - None of the above

Rules:
- Output exactly one class label from the list, using the exact uppercase label string.
- Choose the SINGLE most likely PRIMARY reason for admission.

ADMISSION NOTE:
{input}

PRIMARY DIAGNOSIS:"""

FUZZY_MAP = {
    "SEPS": "SEPSIS", "SEPT": "SEPSIS",
    "PNEUM": "PNEUMONIA",
    "STROK": "STROKE", "CVA": "STROKE",
    "ATRIAL": "ATRIAL_FIBRILLATION", "AFIB": "ATRIAL_FIBRILLATION", "FIBRIL": "ATRIAL_FIBRILLATION",
    "AKI": "AKI", "KIDNEY": "AKI", "RENAL": "AKI",
    "UTI": "UTI", "URIN": "UTI",
    "HEART": "HEART_FAILURE", "CHF": "HEART_FAILURE",
    "GI": "GI_BLEEDING", "BLEED": "GI_BLEEDING",
    "DIAB": "DIABETES_COMPLICATION", "DKA": "DIABETES_COMPLICATION",
    "COPD": "COPD_EXACERBATION",
    "OTHER": "OTHER", "NONE": "OTHER",
}

def inference(input_text):
    """Chat template + enable_thinking=False (8A onayli)."""
    prompt = PROMPT_TEMPLATE.format(input=input_text)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=20, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return raw, inputs.input_ids.shape[1]

def parse_class(raw):
    """Parser: invalid -> 'INVALID' kalir (OTHER'a GOMMUYOR)."""
    if not raw:
        return "INVALID", "empty"
    text = raw.strip().upper()
    siralama = sorted(ALLOWED_CLASSES, key=len, reverse=True)
    for cls in siralama:
        if cls in text[:80]:
            return cls, "direct"
    for k, v in FUZZY_MAP.items():
        if k in text[:80]:
            return v, "fuzzy"
    return "INVALID", "invalid"

print("   Helper fonksiyonlar tanimlandi.")
print(f"   ZS_DIR: {ZS_DIR}")

In [ ]:
# 2) VAL V1 + V2 INFERENCE (checkpoint/resume ile)
df = pd.read_parquet(SPLIT_YOL)
val_df = df[df["split"] == "val"].reset_index(drop=True)
print(f"   Val ornek sayisi: {len(val_df):,}")

def val_inference_calistir(version):
    """V1 veya V2 icin val inference, checkpoint/resume destekli."""
    input_kolonu = f"model_input_{version}_masked"
    has_own_kolonu = f"has_own_label_term_{version}"
    
    cp_yol  = ZS_DIR / f"zeroshot_val_{version}_checkpoint.csv"
    final_yol = ZS_DIR / f"zeroshot_val_{version}.csv"
    
    # Resume
    if cp_yol.exists():
        cp = pd.read_csv(cp_yol)
        islenen_ids = set(cp["note_id"].astype(str))
        kayitlar = cp.to_dict("records")
        print(f"   [V{version.upper()}] Resume: {len(islenen_ids)} kayit zaten var, devam ediliyor.")
    else:
        islenen_ids = set()
        kayitlar = []
    
    print(f"   [V{version.upper()}] Inference basliyor (kalan: {len(val_df) - len(islenen_ids)})...")
    t_baslangic = time.time()
    sayac_yeni = 0
    
    for i, row in val_df.iterrows():
        nid = str(row["note_id"])
        if nid in islenen_ids:
            continue
        
        inp = row[input_kolonu] if pd.notna(row[input_kolonu]) else ""
        t0 = time.time()
        raw, in_tok = inference(inp)
        sure = time.time() - t0
        parsed, kategori = parse_class(raw)
        
        kayitlar.append({
            "note_id": nid,
            "subject_id": int(row["subject_id"]),
            "hadm_id": int(row["hadm_id"]),
            "label": row["label"],
            "raw_output": raw,
            "parsed_pred": parsed,
            "parse_kategori": kategori,
            "in_tokens": int(in_tok),
            "sure_s": round(sure, 3),
            "has_own_label_term": bool(row[has_own_kolonu]),
        })
        sayac_yeni += 1
        
        # Checkpoint her 50 ornekte
        if len(kayitlar) % 50 == 0:
            pd.DataFrame(kayitlar).to_csv(cp_yol, index=False)
            gecen = time.time() - t_baslangic
            ort = gecen / sayac_yeni if sayac_yeni > 0 else 0
            kalan = (len(val_df) - len(kayitlar)) * ort
            print(f"      [{len(kayitlar):>3}/{len(val_df)}] gecen {gecen/60:.1f}dk, ort {ort:.2f}s, kalan ~{kalan/60:.1f}dk")
    
    # Final yazim
    df_son = pd.DataFrame(kayitlar)
    df_son.to_csv(final_yol, index=False)
    df_son.to_csv(cp_yol, index=False)
    
    toplam_sure = time.time() - t_baslangic
    print(f"   [V{version.upper()}] BITTI. Yeni {sayac_yeni}, toplam {len(kayitlar)} kayit. Sure: {toplam_sure/60:.1f}dk")
    print(f"      Final: {final_yol.name}")
    return df_son

# V1
print("\n" + "=" * 70)
print("VAL V1 INFERENCE")
print("=" * 70)
val_v1 = val_inference_calistir("v1")

# V2
print("\n" + "=" * 70)
print("VAL V2 INFERENCE")
print("=" * 70)
val_v2 = val_inference_calistir("v2")

print("\n   + VAL INFERENCE TAMAMLANDI (V1 + V2)")

In [ ]:
# 3) AUDIT + PROMPT/PARSER SABITLEME KARARI
from sklearn.metrics import accuracy_score, f1_score

print("\n" + "=" * 70)
print("BLOK 8B AUDIT - VAL SONUCLARI")
print("=" * 70)

audit_satirlari = []
rapor_metni = []
rapor_metni.append("BLOK 8B AUDIT - VAL SONUCLARI\n" + "=" * 70)

for v_ad, df_v in [("V1", val_v1), ("V2", val_v2)]:
    rapor_metni.append(f"\n--- {v_ad} ---")
    print(f"\n--- {v_ad} ---")
    
    # 1. Parser dagilimi
    kat_sayim = df_v["parse_kategori"].value_counts().to_dict()
    direct = kat_sayim.get("direct", 0)
    fuzzy  = kat_sayim.get("fuzzy", 0)
    invalid = kat_sayim.get("invalid", 0) + kat_sayim.get("empty", 0)
    n = len(df_v)
    invalid_pct = 100 * invalid / n
    
    print(f"   Parser dagilimi:")
    print(f"      Direct  : {direct:>4} ({100*direct/n:.1f}%)")
    print(f"      Fuzzy   : {fuzzy:>4} ({100*fuzzy/n:.1f}%)")
    print(f"      Invalid : {invalid:>4} ({invalid_pct:.1f}%)")
    rapor_metni.append(f"Parser: direct={direct} fuzzy={fuzzy} invalid={invalid} ({invalid_pct:.1f}%)")
    
    # 2. Pilot val accuracy/F1 (sadece valid uzerinden)
    valid_mask = df_v["parsed_pred"] != "INVALID"
    if valid_mask.sum() > 0:
        y_true = df_v.loc[valid_mask, "label"].values
        y_pred = df_v.loc[valid_mask, "parsed_pred"].values
        acc = accuracy_score(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    else:
        acc = macro_f1 = weighted_f1 = 0.0
    
    print(f"   Pilot val metrikleri (valid={valid_mask.sum()}):")
    print(f"      Accuracy    : {acc:.4f}")
    print(f"      Macro-F1    : {macro_f1:.4f}")
    print(f"      Weighted-F1 : {weighted_f1:.4f}")
    rapor_metni.append(f"Val: acc={acc:.4f} macroF1={macro_f1:.4f} wF1={weighted_f1:.4f}")
    
    # 3. Tahmin sinif dagilimi (bias kontrolu)
    print(f"   Tahmin sinif dagilimi (bias kontrolu):")
    pred_dag = df_v["parsed_pred"].value_counts().to_dict()
    en_yuksek = max(pred_dag.values())
    en_yuksek_cls = [k for k, v in pred_dag.items() if v == en_yuksek][0]
    bias_pct = 100 * en_yuksek / n
    rapor_metni.append(f"Tahmin dagilimi: {pred_dag}")
    rapor_metni.append(f"En yuksek: {en_yuksek_cls}={en_yuksek} ({bias_pct:.1f}%)")
    for cls, sayi in sorted(pred_dag.items(), key=lambda x: -x[1]):
        print(f"      {cls:<25s} {sayi:>4} ({100*sayi/n:.1f}%)")
    
    # 4. Per-class F1 (zayif siniflari gor)
    if valid_mask.sum() > 0:
        from sklearn.metrics import precision_recall_fscore_support
        labels = sorted(set(y_true) | set(y_pred))
        p, r, f, supp = precision_recall_fscore_support(
            y_true, y_pred, labels=labels, zero_division=0
        )
        print(f"   Per-class F1:")
        for cls, fi, si in zip(labels, f, supp):
            print(f"      {cls:<25s} F1={fi:.3f} (n={si})")
    
    audit_satirlari.append({
        "version": v_ad, "n": n, "direct": direct, "fuzzy": fuzzy, "invalid": invalid,
        "invalid_pct": invalid_pct, "accuracy": acc, "macro_f1": macro_f1,
        "weighted_f1": weighted_f1, "top_pred_class": en_yuksek_cls, "top_pred_pct": bias_pct
    })

# Audit dosyasi
audit_yol = ZS_DIR / "zeroshot_val_audit.txt"
with open(audit_yol, "w", encoding="utf-8") as f:
    f.write("\n".join(rapor_metni))
print(f"\n   Audit raporu: {audit_yol.name}")

# Audit CSV
pd.DataFrame(audit_satirlari).to_csv(ZS_DIR / "zeroshot_val_audit_summary.csv", index=False)

In [ ]:
# 4) KARAR MATRISI - prompt/parser sabitleme
print("\n" + "=" * 70)
print("KARAR MATRISI")
print("=" * 70)

audit_df = pd.DataFrame(audit_satirlari)
max_invalid = audit_df["invalid_pct"].max()
min_acc = audit_df["accuracy"].min()
max_bias = audit_df["top_pred_pct"].max()
max_bias_cls = audit_df.loc[audit_df["top_pred_pct"].idxmax(), "top_pred_class"]

print(f"   Max invalid orani  : {max_invalid:.1f}%")
print(f"   Min val accuracy   : {min_acc:.4f}")
print(f"   Max bias sinif     : {max_bias_cls} ({max_bias:.1f}%)")

# Karar kurallari
print("\n   Kontroller:")
kontrol = []

if max_invalid <= 5:
    kontrol.append(("Invalid <= 5%", True, "Prompt+parser SABIT"))
elif max_invalid <= 15:
    kontrol.append(("Invalid <= 15%", True, "Fuzzy map yeterli, prompt aynı"))
else:
    kontrol.append((f"Invalid > 15% ({max_invalid:.1f}%)", False, "PROMPT REVIZE GEREKLI"))

if min_acc >= 0.20:
    kontrol.append((f"Acc >= 20% ({min_acc:.2f})", True, "Model+prompt makul"))
else:
    kontrol.append((f"Acc < 20% ({min_acc:.2f})", False, "DUR - prompt veya model bozuk"))

if max_bias <= 50:
    kontrol.append((f"Bias <= 50% ({max_bias:.1f}% {max_bias_cls})", True, "Dengeli"))
else:
    kontrol.append((f"Bias > 50% ({max_bias:.1f}% {max_bias_cls})", False, "PROMPT BIAS PROBLEMI"))

for ad, ok, aksiyon in kontrol:
    isaret = "OK" if ok else "FAIL"
    print(f"      [{isaret}] {ad} -> {aksiyon}")

# Final karar
tum_ok = all(k[1] for k in kontrol)
print("\n" + "=" * 70)
if tum_ok:
    print("   ✅ PROMPT + PARSER SABITLENDI")
    print("      8C'ye gecilebilir (test inference 1.650 × 2)")
    
    # FINAL prompt + parser dosyaya yaz
    prompt_yol = ZS_DIR / "zeroshot_prompt_FINAL.txt"
    with open(prompt_yol, "w", encoding="utf-8") as f:
        f.write(PROMPT_TEMPLATE)
    print(f"      Prompt kaydedildi: {prompt_yol.name}")
    
    parser_yol = ZS_DIR / "zeroshot_parser_FINAL.py"
    parser_kod = '''"""Zero-shot final parser (8B'de sabitlendi)."""
ALLOWED_CLASSES = ''' + repr(ALLOWED_CLASSES) + '''
FUZZY_MAP = ''' + repr(FUZZY_MAP) + '''

def parse_class(raw):
    if not raw:
        return "INVALID", "empty"
    text = raw.strip().upper()
    siralama = sorted(ALLOWED_CLASSES, key=len, reverse=True)
    for cls in siralama:
        if cls in text[:80]:
            return cls, "direct"
    for k, v in FUZZY_MAP.items():
        if k in text[:80]:
            return v, "fuzzy"
    return "INVALID", "invalid"
'''
    with open(parser_yol, "w", encoding="utf-8") as f:
        f.write(parser_kod)
    print(f"      Parser kaydedildi: {parser_yol.name}")
else:
    print("   ⚠️  REVIZE GEREKLI - sorunlari coz, 8B'yi tekrar calistir")
    print("      Test (8C) BASLATILMASIN.")
print("=" * 70)

# BLOK 8-Kontrol — Sinif Basina 1 Ornek (NE veriyoruz, NE aliyoruz)

**Amac:** Val pilot accuracy %53 cikti — beklenenden yuksek (%30-40 bekliyorduk).
Model'e tam olarak NEYI veriyoruz, NEYI aliyoruz acikca gorelim.

**Yapilacak:** Val'dan her sinif icin 1 ornek (11 ornek toplam):
1. Gercek label
2. Modele giden tam prompt (chat_template ile)
3. Maskelenmis input (model_input_v1_masked)
4. Ham model ciktisi (raw)
5. Parsed sinif
6. Dogru mu?

**Inference DEGIL** — val_v1 sonuclari zaten var (8B), oradan goruntulemek yeterli.

In [ ]:
# 1) HER SINIFTAN 1 ORNEK - tam input + ciktisi
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
ZS_DIR     = OUTPUT_DIR / "zeroshot"

# Veri ve val sonuclari yukle
df = pd.read_parquet(SPLIT_YOL)
val_v1_yol = ZS_DIR / "zeroshot_val_v1.csv"
val_v1 = pd.read_csv(val_v1_yol)

# val_df ile birlestir (input metnini cekmek icin)
val_df = df[df["split"] == "val"][["note_id", "label", "model_input_v1_masked", "has_own_label_term_v1"]].copy()
val_df["note_id"] = val_df["note_id"].astype(str)
val_v1["note_id"] = val_v1["note_id"].astype(str)
birlesik = val_v1.merge(val_df, on=["note_id", "label"], how="left")

# Her siniftan 1 ornek
siralama = [
    "SEPSIS", "STROKE", "ATRIAL_FIBRILLATION", "AKI", "PNEUMONIA", "UTI",
    "HEART_FAILURE", "GI_BLEEDING", "DIABETES_COMPLICATION",
    "COPD_EXACERBATION", "OTHER"
]

PROMPT_TEMPLATE = '''You are a clinical NLP system. Given a patient's admission note, classify the PRIMARY discharge diagnosis into ONE of these 11 categories:

SEPSIS - Systemic infection, septic shock, bacteremia
STROKE - Ischemic or hemorrhagic stroke, CVA
ATRIAL_FIBRILLATION - Atrial fibrillation/flutter (acute)
AKI - Acute kidney injury, acute renal failure
PNEUMONIA - Lung infection (bacterial, viral, aspiration)
UTI - Urinary tract infection, pyelonephritis, cystitis
HEART_FAILURE - Congestive heart failure, decompensated CHF
GI_BLEEDING - Upper or lower GI hemorrhage
DIABETES_COMPLICATION - DKA, HHS, severe hyper/hypoglycemia
COPD_EXACERBATION - Acute exacerbation of COPD
OTHER - None of the above

Rules:
- Output exactly one class label from the list, using the exact uppercase label string.
- Choose the SINGLE most likely PRIMARY reason for admission.

ADMISSION NOTE:
{input}

PRIMARY DIAGNOSIS:'''

print("=" * 90)
print("BLOK 8-Kontrol — Her siniftan 1 ornek (V1, val seti)")
print("=" * 90)
print(f"   Toplam val V1 kayit: {len(birlesik):,}")
print(f"   Bu ornekler MODELIN GERCEKTE GORDUGU veridir (maskeli).")
print()

for cls in siralama:
    alt = birlesik[birlesik["label"] == cls]
    if len(alt) == 0:
        print(f"   UYARI: {cls} icin val'da ornek yok")
        continue
    
    # Ilk ornegi al (deterministic)
    ornek = alt.iloc[0]
    
    print("\n" + "#" * 90)
    print(f"# SINIF: {cls}")
    print("#" * 90)
    
    print(f"\n[META]")
    print(f"   note_id            : {ornek['note_id']}")
    print(f"   subject_id         : {ornek['subject_id']}")
    print(f"   hadm_id            : {ornek['hadm_id']}")
    print(f"   gercek label       : {ornek['label']}")
    print(f"   has_own_label_term : {ornek.get('has_own_label_term_v1', '?')} (kendi keyword metinde geciyor mu?)")
    print(f"   in_tokens          : {ornek['in_tokens']}")
    print(f"   inference suresi   : {ornek['sure_s']:.2f}s")
    
    print(f"\n[MASKELI INPUT (model_input_v1_masked)]")
    print("-" * 90)
    inp = ornek["model_input_v1_masked"] if pd.notna(ornek["model_input_v1_masked"]) else ""
    print(inp)
    print("-" * 90)
    
    print(f"\n[MODELE GIDEN TAM PROMPT (chat_template ile sarilir)]")
    print("-" * 90)
    tam_prompt = PROMPT_TEMPLATE.format(input=inp)
    print(tam_prompt)
    print("-" * 90)
    
    print(f"\n[MODEL CIKTISI]")
    print(f"   Raw output  : {ornek['raw_output']!r}")
    print(f"   Parsed pred : {ornek['parsed_pred']}")
    print(f"   Parse kat   : {ornek['parse_kategori']}")
    
    dogru = ornek["parsed_pred"] == ornek["label"]
    print(f"   DOGRU MU?   : {'EVET ✓' if dogru else 'HAYIR ✗'}")

print("\n" + "=" * 90)
print("BLOK 8-Kontrol TAMAMLANDI")
print("=" * 90)

# Ozet tablo
print("\n   Ozet:")
print(f"   {'Sinif':<25s} {'Pred':<25s} {'Dogru':<7s} {'has_own':<8s}")
print("   " + "-" * 70)
for cls in siralama:
    alt = birlesik[birlesik["label"] == cls]
    if len(alt) == 0: continue
    ornek = alt.iloc[0]
    dogru = "OK" if ornek["parsed_pred"] == ornek["label"] else "--"
    has_own = "V" if ornek.get("has_own_label_term_v1", False) else "-"
    print(f"   {cls:<25s} {ornek['parsed_pred']:<25s} {dogru:<7s} {has_own:<8s}")

# BLOK 13A — TRIAGE_INPUT / V3 Uretim + Audit

**Yeni metodolojik karar:** Ana deney V1 degil V3 (TRIAGE_INPUT) olacak.

**Senaryo:** AI hastayla birebir gorusuyor gibi, ilk temas/triyajda alinabilecek
bilgilerle 11 sinifli yonlendirme yapacak. Doktor degerlendirmesi, lab, imaging YOK.

**V3 icerigi:**
- Chief Complaint
- HPI'nin sadece hasta hikayesi/semptom kismi (ED workup oncesi)
- ROS (varsa)
- Past Medical History (kisa)
- Allergies
- Medications on Admission
- Vital signs / admission physical exam

**V3 disi:** Lab, imaging, ED workup, doktor assessment, treatment, admit-to-X, hospital course

**HPI cut-off heuristic:** ED/lab/imaging/admission isaretcileri ilk geçtigi yerden kes.
Cut-off oncesi alinir, sonra ROS varsa eklenir.

**Bu blok SADECE input uretir.** Inference yok. Audit iyi gorunurse 13B/14'te zero-shot.

In [ ]:
# 1) SETUP + BOLUM PARSER FONKSIYONLARI
import pandas as pd
import numpy as np
import re
import time
from pathlib import Path

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
FULL_YOL   = OUTPUT_DIR / "discharge_etiketli_full.parquet"

print("=" * 70)
print("BLOK 13A — TRIAGE_INPUT/V3 URETIM + AUDIT")
print("=" * 70)

# --- Bolum sinirlari icin yaygin headerlar ---
HEADER_PATTERN = (
    r"(?:Name|Admission Date|Date of Birth|Service|Allergies|Attending|"
    r"Chief Complaint|Major Surgical[^:]*|History of Present Illness|"
    r"Past Medical History|Social History|Family History|Physical Exam|"
    r"ADMISSION PHYSICAL EXAM|Discharge PE|Pertinent Results|"
    r"Brief Hospital Course|Medications on Admission|Discharge Medications|"
    r"Discharge Disposition|Facility|Discharge Diagnosis|Discharge Condition|"
    r"Discharge Instructions|Followup Instructions|SUBSTANCE ABUSE HISTORY)"
)

def bolum_cikar(text, baslik_pattern, durdur_pattern=None):
    """Bir basligin altindan bir sonraki basliga kadar olan metni cikar."""
    if not isinstance(text, str):
        return ""
    if durdur_pattern is None:
        durdur_pattern = HEADER_PATTERN
    pat = rf"(?ims)^\s*(?:{baslik_pattern}):\s*\n?(.+?)(?=\n\s*(?:{durdur_pattern}):|\Z)"
    m = re.search(pat, text)
    if m:
        return m.group(1).strip()
    return ""

# --- HPI cut-off pattern listesi (ED/lab/imaging/admission isaretcileri) ---
ED_CUTOFF_PATTERNS = [
    # ED bolumune gecis
    r'\bIn the ED\b', r'\bIn the emergency (department|room)\b',
    r'\bED course\b', r'\bUpon arrival to (the )?ED\b',
    r'\bInitial vitals (in the ED|were|on arrival)\b',
    r'\bUpon (admission|arrival)\b', r'\bOn presentation\b(?=.{0,80}(BP|HR|T |labs))',
    # Lab paneli
    r'\bLabs? (notable|showed|were|are|drawn|remarkable|significant)\b',
    r'\bLab(s|oratory) results?\b', r'\bWBC \d', r'\bCr \d\.\d', r'\bHgb \d',
    r'\bplatelets? \d', r'\bINR \d', r'\blactate \d', r'\btroponin\b',
    r'\bUA (showed|with|notable|positive|negative)\b',
    # Imaging
    r'\bCXR (with|showed|was|notable|negative|positive|revealed)\b',
    r'\bCT (head|chest|abdomen|of|scan|abd|brain)\b', r'\bCTA\b',
    r'\bMRI (showed|of|brain|head|revealed)\b', r'\bUS (showed|revealed|notable)\b',
    r'\b(echocardiogram|TTE|TEE) (showed|revealed|with|demonstrated|notable)\b',
    r'\bEKG (showed|revealed|notable|with)\b', r'\bECG (showed|revealed|notable)\b',
    # Doktor degerlendirmesi / admission karari
    r'\b(He|She|Pt|Patient) was (admitted|started on|given|transferred|treated|placed)\b',
    r'\bAdmitted to\b', r'\bwas referred to\b',
    r'\breceived (IV|PO|po|im|sc|sq) ', r'\bgiven (IV|PO|po) ',
    r'\bdiagnosed with\b', r'\bconcern for\b', r'\bworkup\b',
    r'\bfound to have\b', r'\bnoted to have\b',
    r'\bstarted on (IV|PO|antibiotics|abx|heparin|insulin)\b',
]
ED_CUTOFF_REGEX = re.compile(
    '|'.join(f'(?:{p})' for p in ED_CUTOFF_PATTERNS),
    re.IGNORECASE
)

def hpi_temizle(hpi_text):
    """HPI'den ED/lab/imaging/admission kismini at, hasta hikayesi + ROS'u tut."""
    if not isinstance(hpi_text, str) or not hpi_text:
        return "", False
    
    cutoff_uygulandi = False
    
    # 1. Cut-off bul
    m = ED_CUTOFF_REGEX.search(hpi_text)
    if m:
        kesim_konum = m.start()
        # Cumle siniri: cut-off'tan onceki son nokta/yenisatir
        snippet = hpi_text[:kesim_konum]
        # Son cumle/satir basina geri sar
        son_nokta = max(snippet.rfind('. '), snippet.rfind('.\n'), snippet.rfind('\n\n'))
        if son_nokta > kesim_konum * 0.5:  # cok geriye gitme
            kesim_konum = son_nokta + 1
        on_kisim = hpi_text[:kesim_konum].rstrip()
        cutoff_uygulandi = True
    else:
        on_kisim = hpi_text.strip()
    
    # 2. ROS varsa cut-off sonrasindan bul ve ekle (hasta-tarafi bilgi)
    ros_eklenmis = ""
    ros_pat = re.compile(
        r'(?ims)(^\s*(?:Review of Systems?|ROS)\s*:?[\s\S]+?)'
        r'(?=\n\s*\n[A-Z][a-z]+ [A-Z]|\Z)'
    )
    ros_m = ros_pat.search(hpi_text)
    if ros_m:
        ros_text = ros_m.group(1).strip()
        # Sadece ROS on_kisim'da yoksa ekle
        if 'ROS' not in on_kisim and 'Review of Systems' not in on_kisim:
            ros_eklenmis = '\n\n' + ros_text
    
    return on_kisim + ros_eklenmis, cutoff_uygulandi

def vital_satir_cikar(phys_exam):
    """Physical Exam'in ilk satirini (vital signs) cikar."""
    if not isinstance(phys_exam, str) or not phys_exam:
        return ""
    # Genelde ilk 1-2 satir vital: 'VS: 99.0 PO 171/79 79 20 97 RA' tarzi
    satirlar = phys_exam.split('\n')
    vital_satirlari = []
    for s in satirlar[:5]:
        s_strip = s.strip()
        if not s_strip:
            continue
        # Vital pattern: VS:, T, BP, HR, RR, SaO2, weight
        if re.search(r'\b(VS|T |BP|HR|RR|O2|SaO2|SpO2|temp|temperature|pulse)\b', s_strip, re.IGNORECASE):
            vital_satirlari.append(s_strip)
        elif re.match(r'^\d{2,3}[/.\s]', s_strip):  # rakam baslayan kisa satir
            vital_satirlari.append(s_strip)
        elif vital_satirlari:
            break  # vital bitti
    return ' '.join(vital_satirlari)

print("   Setup tamamlandi.")
print(f"   ED cutoff pattern sayisi: {len(ED_CUTOFF_PATTERNS)}")

In [ ]:
# 2) HAM TEXT YUKLE + BOLUMLERI PARSE ET
print("\n[1/5] Veri yukleniyor...")
t0 = time.time()
df_split = pd.read_parquet(SPLIT_YOL)
df_full = pd.read_parquet(FULL_YOL)
print(f"   split: {len(df_split):,} satir")
print(f"   full:  {len(df_full):,} satir")
print(f"   Sure: {time.time()-t0:.1f}s")

# split'in note_id'leri ile full'dan text cek
df_split['note_id'] = df_split['note_id'].astype(str)
df_full['note_id'] = df_full['note_id'].astype(str)
df_text = df_full[['note_id', 'text']]
df_split = df_split.merge(df_text, on='note_id', how='left')
assert df_split['text'].notna().all(), "Bazi text eksik!"
print(f"   Text birlestirildi: {len(df_split):,} kayit")

print("\n[2/5] Bolumler parse ediliyor (6.600 kayit)...")
t0 = time.time()

cc_list, hpi_full_list, hpi_clean_list, cutoff_list = [], [], [], []
pmh_list, allergies_list, meds_list, vitals_list = [], [], [], []
audit_flags_list = []

for i, row in df_split.iterrows():
    text = row['text']
    
    cc        = bolum_cikar(text, r"Chief Complaint")
    hpi       = bolum_cikar(text, r"History of Present Illness")
    pmh       = bolum_cikar(text, r"Past Medical History")
    allergies = bolum_cikar(text, r"Allergies")
    meds      = bolum_cikar(text, r"Medications on Admission")
    phys_exam = bolum_cikar(text, r"(?:ADMISSION )?Physical Exam")
    
    hpi_clean, cutoff_uygulandi = hpi_temizle(hpi)
    vitals = vital_satir_cikar(phys_exam)
    
    # Audit flags: parse sonrasi V3'te HALA kalan kalintilar
    flags = []
    v3_compose = ' '.join([cc, hpi_clean, pmh, allergies, meds, vitals])
    if re.search(r'\bWBC \d|\bCr \d\.\d|\bHgb \d', v3_compose, re.IGNORECASE):
        flags.append('lab_kalinti')
    if re.search(r'\bCXR\b|\bCT (head|chest|abdomen)|\bCTA\b|\bMRI\b|\bTTE\b', v3_compose, re.IGNORECASE):
        flags.append('imaging_kalinti')
    if re.search(r'\badmitted to\b|\bdiagnosed with\b|\bconcern for\b', v3_compose, re.IGNORECASE):
        flags.append('assessment_kalinti')
    
    cc_list.append(cc)
    hpi_full_list.append(hpi)
    hpi_clean_list.append(hpi_clean)
    cutoff_list.append(cutoff_uygulandi)
    pmh_list.append(pmh)
    allergies_list.append(allergies)
    meds_list.append(meds)
    vitals_list.append(vitals)
    audit_flags_list.append(','.join(flags) if flags else 'temiz')
    
    if (i + 1) % 1000 == 0:
        print(f"   {i+1:,}/{len(df_split):,} parse edildi")

df_split['cc_v3']        = cc_list
df_split['hpi_clean_v3'] = hpi_clean_list
df_split['pmh_v3']       = pmh_list
df_split['allergies_v3'] = allergies_list
df_split['meds_adm_v3']  = meds_list
df_split['vitals_v3']    = vitals_list
df_split['hpi_cutoff_applied'] = cutoff_list
df_split['triage_audit_flags'] = audit_flags_list

print(f"   Sure: {time.time()-t0:.1f}s")
print(f"   Cutoff uygulanan: {sum(cutoff_list):,} / {len(df_split):,} ({100*sum(cutoff_list)/len(df_split):.1f}%)")

In [ ]:
# 3) V3 INPUT COMPOSE + MASKELEME
print("\n[3/5] V3 input compose ediliyor...")

def compose_v3(row):
    parcalar = []
    if row['cc_v3']:
        parcalar.append(f"Chief Complaint:\n{row['cc_v3']}")
    if row['hpi_clean_v3']:
        parcalar.append(f"Symptom History:\n{row['hpi_clean_v3']}")
    if row['pmh_v3']:
        parcalar.append(f"Past Medical History:\n{row['pmh_v3']}")
    if row['allergies_v3']:
        parcalar.append(f"Allergies:\n{row['allergies_v3']}")
    if row['meds_adm_v3']:
        parcalar.append(f"Medications on Admission:\n{row['meds_adm_v3']}")
    if row['vitals_v3']:
        parcalar.append(f"Vital Signs:\n{row['vitals_v3']}")
    return '\n\n'.join(parcalar)

df_split['triage_input_v3'] = df_split.apply(compose_v3, axis=1)

# --- Maskeleme (Blok 4'tekiyle ayni keyword sozlugu) ---
DIAG_KEYWORDS = {
    'PNEUMONIA': r'\b(pneumon\w*|bronchopneumon\w*)\b',
    'HEART_FAILURE': r'\b(CHF|congestive heart failure|heart failure|HFrEF|HFpEF|systolic heart failure|diastolic heart failure)\b',
    'SEPSIS': r'\b(sepsis|septic|septicemia|bacteremia|septicaemia)\b',
    'AKI': r'\b(AKI|acute kidney injury|acute renal failure|ARF|acute renal insufficiency)\b',
    'UTI': r'\b(UTI|urinary tract infection|pyelonephritis|cystitis)\b',
    'STROKE': r'\b(stroke|CVA|cerebrovascular accident|ischemic stroke|hemorrhagic stroke|intracerebral hemorrhage|intracranial hemorrhage)\b',
    'GI_BLEEDING': r'\b(GI bleed\w*|gastrointestinal bleed\w*|UGIB|LGIB|hematemesis|melena|hematochezia)\b',
    'COPD_EXACERBATION': r'\b(COPD exacerbation|AECOPD|chronic obstructive|COPD)\b',
    'DIABETES_COMPLICATION': r'\b(DKA|diabetic ketoacidosis|HHS|hyperosmolar|hyperglycemic crisis|diabetic coma)\b',
    'ATRIAL_FIBRILLATION': r'\b(atrial fibrillation|AFib|A-fib|atrial flutter|paroxysmal AF)\b',
}
BIRLESIK = re.compile('|'.join(f'(?:{p})' for p in DIAG_KEYWORDS.values()), re.IGNORECASE)
CLASS_REGEXES = {cls: re.compile(pat, re.IGNORECASE) for cls, pat in DIAG_KEYWORDS.items()}

def maskele(metin):
    if not metin:
        return metin
    return BIRLESIK.sub('[DIAG_MASKED]', metin)

df_split['triage_input_v3_masked'] = df_split['triage_input_v3'].apply(maskele)

# has_own_label_term_v3 (sadece kendi label'inin keyword'u)
def kendi_keyword(metin, label):
    if not isinstance(metin, str) or not metin:
        return False
    rx = CLASS_REGEXES.get(label)
    if rx is None:
        return False
    return rx.search(metin) is not None

df_split['has_own_label_term_v3'] = [
    kendi_keyword(t, l) for t, l in zip(df_split['triage_input_v3'], df_split['label'])
]

print(f"   V3 ortalama uzunluk:        {df_split['triage_input_v3'].str.len().mean():>7,.0f} char")
print(f"   V3 maskeli ort. uzunluk:    {df_split['triage_input_v3_masked'].str.len().mean():>7,.0f} char")
print(f"   has_own_label_term_v3 oran: {100*df_split['has_own_label_term_v3'].mean():.1f}%")
print("   + V3 input ve maskeleme tamamlandi.")

In [ ]:
# 4) AUDIT RAPORU
print("\n[4/5] Audit raporu...")
print("=" * 70)

# 4.1 Uzunluk karsilastirma V1 vs V3
v1_uzun = df_split['model_input_v1'].str.len().fillna(0)
v3_uzun = df_split['triage_input_v3'].str.len().fillna(0)
v1_mask = df_split['model_input_v1_masked'].str.len().fillna(0)
v3_mask = df_split['triage_input_v3_masked'].str.len().fillna(0)

print(f"\n   V1 vs V3 uzunluk:")
print(f"      {'':>10s}  {'V1':>10s}  {'V3':>10s}  {'V3/V1':>7s}")
print(f"      {'ortalama':>10s}  {v1_uzun.mean():>10,.0f}  {v3_uzun.mean():>10,.0f}  {100*v3_uzun.mean()/max(v1_uzun.mean(),1):>6.1f}%")
print(f"      {'medyan':>10s}  {v1_uzun.median():>10,.0f}  {v3_uzun.median():>10,.0f}  {100*v3_uzun.median()/max(v1_uzun.median(),1):>6.1f}%")
print(f"      {'min':>10s}  {v1_uzun.min():>10,.0f}  {v3_uzun.min():>10,.0f}")
print(f"      {'max':>10s}  {v1_uzun.max():>10,.0f}  {v3_uzun.max():>10,.0f}")

# 4.2 Cutoff istatistigi
print(f"\n   HPI cutoff:")
print(f"      Uygulandi    : {df_split['hpi_cutoff_applied'].sum():>5,} / {len(df_split):,} ({100*df_split['hpi_cutoff_applied'].mean():.1f}%)")
print(f"      Uygulanmadi  : {(~df_split['hpi_cutoff_applied']).sum():>5,}")

# 4.3 Audit flag dagilimi
print(f"\n   Audit flag dagilimi (V3'te kalintilar):")
flag_sayim = df_split['triage_audit_flags'].value_counts()
for flag, n in flag_sayim.head(10).items():
    print(f"      {flag:<40s} {n:>5,} ({100*n/len(df_split):>4.1f}%)")

# Ozel kalinti sayimi
lab_kal = df_split['triage_audit_flags'].str.contains('lab_kalinti', na=False).sum()
img_kal = df_split['triage_audit_flags'].str.contains('imaging_kalinti', na=False).sum()
ass_kal = df_split['triage_audit_flags'].str.contains('assessment_kalinti', na=False).sum()
temiz   = (df_split['triage_audit_flags'] == 'temiz').sum()
print(f"\n   Ozet:")
print(f"      Lab kalintisi (WBC/Cr/Hgb)    : {lab_kal:>5,} ({100*lab_kal/len(df_split):.1f}%)")
print(f"      Imaging kalintisi (CXR/CT/MRI): {img_kal:>5,} ({100*img_kal/len(df_split):.1f}%)")
print(f"      Assessment (admitted/diagnosed): {ass_kal:>5,} ({100*ass_kal/len(df_split):.1f}%)")
print(f"      Tamamen temiz                  : {temiz:>5,} ({100*temiz/len(df_split):.1f}%)")

# 4.4 Sinif basina V3 uzunluk + cutoff orani
print(f"\n   Sinif bazli (uzunluk ort + cutoff%):")
print(f"   {'Sinif':<25s} {'V3 ort':>8s} {'V1 ort':>8s} {'cutoff%':>9s} {'has_own%':>10s}")
print("   " + "-" * 65)
for cls in sorted(df_split['label'].unique()):
    alt = df_split[df_split['label'] == cls]
    v3_o = alt['triage_input_v3'].str.len().mean()
    v1_o = alt['model_input_v1'].str.len().mean()
    cut = 100 * alt['hpi_cutoff_applied'].mean()
    own = 100 * alt['has_own_label_term_v3'].mean()
    print(f"   {cls:<25s} {v3_o:>8,.0f} {v1_o:>8,.0f} {cut:>8.1f}% {own:>9.1f}%")

# 4.5 Clean test boyutu
print(f"\n   Clean test boyutu (V3, has_own_label_term_v3=False):")
test_df = df_split[df_split['split']=='test']
clean_v3 = (~test_df['has_own_label_term_v3']).sum()
print(f"      Test toplam       : {len(test_df):>4,}")
print(f"      V3 clean (yeni)   : {clean_v3:>4,} ({100*clean_v3/len(test_df):.1f}%)")
print(f"      V1 clean (referans): {(~test_df['has_own_label_term_v1']).sum():>4,} ({100*(~test_df['has_own_label_term_v1']).mean():.1f}%)")

In [ ]:
# 5) HER SINIFTAN 2 ORNEK + PARQUET KAYDET
print("\n[5/5] Sinif basina 2 ornek (V3 maskeli):")
print("=" * 70)

siralama = [
    "SEPSIS", "STROKE", "ATRIAL_FIBRILLATION", "AKI", "PNEUMONIA", "UTI",
    "HEART_FAILURE", "GI_BLEEDING", "DIABETES_COMPLICATION",
    "COPD_EXACERBATION", "OTHER"
]

val_df = df_split[df_split['split']=='val']
for cls in siralama:
    alt = val_df[val_df['label']==cls]
    if len(alt) < 2:
        continue
    print("\n" + "#" * 70)
    print(f"# {cls}")
    print("#" * 70)
    for j in range(2):
        ornek = alt.iloc[j]
        v3 = ornek['triage_input_v3_masked'] or ""
        print(f"\n  --- Ornek {j+1} (note_id: {ornek['note_id']}, has_own: {ornek['has_own_label_term_v3']}, cutoff: {ornek['hpi_cutoff_applied']}, flags: {ornek['triage_audit_flags']}) ---")
        print(f"  Uzunluk: {len(v3)} char")
        print(f"  --- BASLANGIC ---")
        print(v3[:1500])
        if len(v3) > 1500:
            print(f"  ... ({len(v3)-1500} char daha) ...")
        print(f"  --- SON ---")

# Parquet kaydet
print("\n" + "=" * 70)
print("PARQUET KAYDETME")
print("=" * 70)

# 'text' kolonunu cikar (ham, gereksiz buyutuyor)
if 'text' in df_split.columns:
    df_split = df_split.drop(columns=['text'])

# Yeni kolon listesi (sadece V3 ile ilgili olanlari rapor et)
yeni_kolonlar = [
    'cc_v3', 'hpi_clean_v3', 'pmh_v3', 'allergies_v3', 'meds_adm_v3', 'vitals_v3',
    'triage_input_v3', 'triage_input_v3_masked',
    'has_own_label_term_v3', 'hpi_cutoff_applied', 'triage_audit_flags'
]
print(f"\n   Eklenecek kolonlar ({len(yeni_kolonlar)}):")
for k in yeni_kolonlar:
    print(f"      - {k}")

t0 = time.time()
df_split.to_parquet(SPLIT_YOL, compression='snappy', index=False)
boyut = SPLIT_YOL.stat().st_size / (1024*1024)
print(f"\n   Yazildi: {SPLIT_YOL.name}")
print(f"   Boyut  : {boyut:.1f} MB")
print(f"   Kolon  : {len(df_split.columns)}")
print(f"   Sure   : {time.time()-t0:.1f}s")

print("\n" + "=" * 70)
print("BLOK 13A TAMAMLANDI")
print("   Sonraki adim: 13A audit iyi gorunurse Blok 13B (V3 zero-shot val+test)")
print("=" * 70)

# BLOK 13B-VAL — V3 (TRIAGE) Zero-shot Val Inference

**Senaryo:** AI hastayla ilk temas. Doktor degerlendirmesi, lab, imaging YOK.

**Input:** triage_input_v3_masked (V3 cleanup pipelinindan gecmis)

**Prompt degisikligi:** "admission note" -> "patient triage information".

**KURAL: Sadece val. Test'e otomatik gecmez.** Audit gorulduakten sonra onay
alinarak 13B-TEST ayri calisacak.

**Karar esikleri:**
- Acc < %20 -> Test baslatilma. Triage cok zor.
- Acc %20-40 -> Beklenen, test icin onay iste.
- Acc > %40 -> Beklenenden iyi, test icin onay iste.

In [ ]:
# 1) SETUP + PROMPT + PARSER
import pandas as pd
import numpy as np
import torch
import time
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
ZS_DIR     = OUTPUT_DIR / "zeroshot"
ZS_DIR.mkdir(parents=True, exist_ok=True)

ALLOWED_CLASSES = [
    "SEPSIS", "STROKE", "ATRIAL_FIBRILLATION", "AKI", "PNEUMONIA", "UTI",
    "HEART_FAILURE", "GI_BLEEDING", "DIABETES_COMPLICATION",
    "COPD_EXACERBATION", "OTHER"
]

PROMPT_TEMPLATE = """You are a clinical NLP system. Given a patient's triage information collected at first contact (chief complaint, symptoms, brief history, medications, allergies, vital signs), classify the most likely PRIMARY reason for admission into ONE of these 11 categories:

SEPSIS - Systemic infection, septic shock, bacteremia
STROKE - Ischemic or hemorrhagic stroke, CVA
ATRIAL_FIBRILLATION - Atrial fibrillation/flutter (acute)
AKI - Acute kidney injury, acute renal failure
PNEUMONIA - Lung infection (bacterial, viral, aspiration)
UTI - Urinary tract infection, pyelonephritis, cystitis
HEART_FAILURE - Congestive heart failure, decompensated CHF
GI_BLEEDING - Upper or lower GI hemorrhage
DIABETES_COMPLICATION - DKA, HHS, severe hyper/hypoglycemia
COPD_EXACERBATION - Acute exacerbation of COPD
OTHER - None of the above

Rules:
- Output exactly one class label from the list, using the exact uppercase label string.
- Choose the SINGLE most likely PRIMARY reason for admission.

TRIAGE NOTE:
{input}

PRIMARY DIAGNOSIS:"""

FUZZY_MAP = {
    "SEPS": "SEPSIS", "SEPT": "SEPSIS",
    "PNEUM": "PNEUMONIA",
    "STROK": "STROKE", "CVA": "STROKE",
    "ATRIAL": "ATRIAL_FIBRILLATION", "AFIB": "ATRIAL_FIBRILLATION", "FIBRIL": "ATRIAL_FIBRILLATION",
    "AKI": "AKI", "KIDNEY": "AKI", "RENAL": "AKI",
    "UTI": "UTI", "URIN": "UTI",
    "HEART": "HEART_FAILURE", "CHF": "HEART_FAILURE",
    "GI": "GI_BLEEDING", "BLEED": "GI_BLEEDING",
    "DIAB": "DIABETES_COMPLICATION", "DKA": "DIABETES_COMPLICATION",
    "COPD": "COPD_EXACERBATION",
    "OTHER": "OTHER", "NONE": "OTHER",
}

def parse_class(raw):
    """Invalid -> 'INVALID' kalir (OTHER'a GOMMUYOR)."""
    if not raw: return "INVALID", "empty"
    text = raw.strip().upper()
    siralama = sorted(ALLOWED_CLASSES, key=len, reverse=True)
    for cls in siralama:
        if cls in text[:80]: return cls, "direct"
    for k, v in FUZZY_MAP.items():
        if k in text[:80]: return v, "fuzzy"
    return "INVALID", "invalid"

print("Setup tamamlandi.")

In [ ]:
# 2) MODEL YUKLEME (RAM'de varsa atla)
if "model" not in dir() or "tokenizer" not in dir():
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3.5-4B", quantization_config=bnb_config, device_map="auto",
        torch_dtype=torch.float16, local_files_only=True,
    )
    model.eval()
    print("Model yuklendi.")
else:
    print("Model RAM'de mevcut.")

def inference(input_text):
    prompt = PROMPT_TEMPLATE.format(input=input_text)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=20, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return raw, inputs.input_ids.shape[1]

In [ ]:
# 3) V3 VAL INFERENCE (checkpoint/resume)
df = pd.read_parquet(SPLIT_YOL)
val_df = df[df["split"] == "val"].reset_index(drop=True)
print(f"   Val: {len(val_df):,} ornek")

cp_yol  = ZS_DIR / "zeroshot_v3_val_checkpoint.csv"
final_yol = ZS_DIR / "zeroshot_v3_val.csv"

if cp_yol.exists():
    cp = pd.read_csv(cp_yol)
    islenen_ids = set(cp["note_id"].astype(str))
    kayitlar = cp.to_dict("records")
    print(f"   Resume: {len(islenen_ids)} kayit zaten var.")
else:
    islenen_ids = set()
    kayitlar = []

t_baslangic = time.time()
sayac_yeni = 0

for i, row in val_df.iterrows():
    nid = str(row["note_id"])
    if nid in islenen_ids: continue
    inp = row["triage_input_v3_masked"] if pd.notna(row["triage_input_v3_masked"]) else ""
    t0 = time.time()
    raw, in_tok = inference(inp)
    sure = time.time() - t0
    parsed, kategori = parse_class(raw)
    kayitlar.append({
        "note_id": nid, "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]), "label": row["label"],
        "raw_output": raw, "parsed_pred": parsed,
        "parse_kategori": kategori, "in_tokens": int(in_tok),
        "sure_s": round(sure, 3),
        "has_own_label_term_v3": bool(row["has_own_label_term_v3"]),
    })
    sayac_yeni += 1
    if len(kayitlar) % 50 == 0:
        pd.DataFrame(kayitlar).to_csv(cp_yol, index=False)
        gecen = time.time() - t_baslangic
        ort = gecen / sayac_yeni if sayac_yeni > 0 else 0
        kalan = (len(val_df) - len(kayitlar)) * ort
        print(f"      [{len(kayitlar):>3}/{len(val_df)}] gecen {gecen/60:.1f}dk, ort {ort:.2f}s, kalan ~{kalan/60:.1f}dk")

df_son = pd.DataFrame(kayitlar)
df_son.to_csv(final_yol, index=False)
df_son.to_csv(cp_yol, index=False)
print(f"   BITTI. Toplam {len(kayitlar)} kayit. Sure: {(time.time()-t_baslangic)/60:.1f}dk")

In [ ]:
# 4) AUDIT + KARAR (test'e otomatik gecis YOK)
val_v3 = pd.read_csv(ZS_DIR / "zeroshot_v3_val.csv")
n = len(val_v3)

kat_sayim = val_v3["parse_kategori"].value_counts().to_dict()
direct = kat_sayim.get("direct", 0)
fuzzy  = kat_sayim.get("fuzzy", 0)
invalid = kat_sayim.get("invalid", 0) + kat_sayim.get("empty", 0)
print(f"Parser: direct={direct} fuzzy={fuzzy} invalid={invalid} ({100*invalid/n:.1f}%)")

y_true = val_v3["label"].values
y_pred = val_v3["parsed_pred"].values

valid_mask = y_pred != "INVALID"
acc_full = accuracy_score(y_true, y_pred)
macro_f1_full = f1_score(y_true, y_pred, average="macro", zero_division=0)

if valid_mask.sum() > 0:
    acc_valid = accuracy_score(y_true[valid_mask], y_pred[valid_mask])
    macro_f1_valid = f1_score(y_true[valid_mask], y_pred[valid_mask], average="macro", zero_division=0)
else:
    acc_valid = macro_f1_valid = 0.0

clean_mask = (~val_v3["has_own_label_term_v3"]).values
acc_clean = accuracy_score(y_true[clean_mask], y_pred[clean_mask]) if clean_mask.sum() > 0 else 0.0
macro_f1_clean = f1_score(y_true[clean_mask], y_pred[clean_mask], average="macro", zero_division=0) if clean_mask.sum() > 0 else 0.0

print(f"\nMetrikler:")
print(f"   Full           n={n}      acc={acc_full:.4f}  macroF1={macro_f1_full:.4f}")
print(f"   Valid only     n={valid_mask.sum()}    acc={acc_valid:.4f}  macroF1={macro_f1_valid:.4f}")
print(f"   Clean (own=F)  n={clean_mask.sum()}    acc={acc_clean:.4f}  macroF1={macro_f1_clean:.4f}")

# Per-class
labels = sorted(set(y_true) | (set(y_pred) - {"INVALID"}))
p, r, f, supp = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
print(f"\nPer-class:")
for cls, pi, ri, fi, si in zip(labels, p, r, f, supp):
    print(f"   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}")

# Karar
print("\n" + "=" * 60)
print(f"KARAR: Val full acc = {acc_full:.4f}")
if acc_full < 0.20:
    print("⚠️  < %20: TEST BASLATILMASIN. Prompt revize gerekli.")
elif acc_full <= 0.40:
    print("📊 %20-40: Beklenen triage zorlugu. Test icin kullanici onayi iste.")
else:
    print("✅ > %40: Beklenenden iyi. Test icin kullanici onayi iste.")
print("   TEST'e otomatik gecis YOK.")

# BLOK 13B-TEST — V3 (TRIAGE) Zero-shot TEST Inference (FINAL)

**Final zero-shot V3 sonucu.** Prompt/parser/input bundan sonra DEGISMEYECEK.

- Input: triage_input_v3_masked
- Test: 1.650 ornek (split == 'test')
- Checkpoint/resume: her 50 ornekte
- Aynı 13B-VAL config (chat_template + enable_thinking=False)

In [ ]:
# 1) TEST INFERENCE (val'daki ALLOWED_CLASSES, PROMPT_TEMPLATE, FUZZY_MAP, parse_class kullanir)
df = pd.read_parquet(SPLIT_YOL)
test_df = df[df['split'] == 'test'].reset_index(drop=True)
print(f"   Test: {len(test_df):,} ornek")

cp_yol  = ZS_DIR / 'zeroshot_v3_test_checkpoint.csv'
final_yol = ZS_DIR / 'zeroshot_v3_test.csv'

if cp_yol.exists():
    cp = pd.read_csv(cp_yol)
    islenen_ids = set(cp['note_id'].astype(str))
    kayitlar = cp.to_dict('records')
    print(f"   Resume: {len(islenen_ids)} kayit zaten var.")
else:
    islenen_ids = set()
    kayitlar = []

t_baslangic = time.time()
sayac_yeni = 0

for i, row in test_df.iterrows():
    nid = str(row['note_id'])
    if nid in islenen_ids: continue
    inp = row['triage_input_v3_masked'] if pd.notna(row['triage_input_v3_masked']) else ''
    t0 = time.time()
    raw, in_tok = inference(inp)
    sure = time.time() - t0
    parsed, kategori = parse_class(raw)
    kayitlar.append({
        'note_id': nid, 'subject_id': int(row['subject_id']),
        'hadm_id': int(row['hadm_id']), 'label': row['label'],
        'raw_output': raw, 'parsed_pred': parsed,
        'parse_kategori': kategori, 'in_tokens': int(in_tok),
        'sure_s': round(sure, 3),
        'has_own_label_term_v3': bool(row['has_own_label_term_v3']),
    })
    sayac_yeni += 1
    if len(kayitlar) % 50 == 0:
        pd.DataFrame(kayitlar).to_csv(cp_yol, index=False)
        gecen = time.time() - t_baslangic
        ort = gecen / sayac_yeni if sayac_yeni > 0 else 0
        kalan = (len(test_df) - len(kayitlar)) * ort
        print(f"      [{len(kayitlar):>4}/{len(test_df)}] gecen {gecen/60:.1f}dk, ort {ort:.2f}s, kalan ~{kalan/60:.1f}dk")

df_son = pd.DataFrame(kayitlar)
df_son.to_csv(final_yol, index=False)
df_son.to_csv(cp_yol, index=False)
print(f"   BITTI. Toplam {len(kayitlar)} kayit. Sure: {(time.time()-t_baslangic)/60:.1f}dk")

In [ ]:
# 2) FINAL AUDIT
test_v3 = pd.read_csv(ZS_DIR / 'zeroshot_v3_test.csv')
n = len(test_v3)

kat_sayim = test_v3['parse_kategori'].value_counts().to_dict()
direct = kat_sayim.get('direct', 0)
fuzzy  = kat_sayim.get('fuzzy', 0)
invalid = kat_sayim.get('invalid', 0) + kat_sayim.get('empty', 0)
print(f'Parser: direct={direct} ({100*direct/n:.1f}%) fuzzy={fuzzy} invalid={invalid} ({100*invalid/n:.1f}%)')

y_true = test_v3['label'].values
y_pred = test_v3['parsed_pred'].values
valid_mask = y_pred != 'INVALID'
clean_mask = (~test_v3['has_own_label_term_v3']).values

acc_full = accuracy_score(y_true, y_pred)
macro_f1_full = f1_score(y_true, y_pred, average='macro', zero_division=0)
weighted_f1_full = f1_score(y_true, y_pred, average='weighted', zero_division=0)
acc_valid = accuracy_score(y_true[valid_mask], y_pred[valid_mask]) if valid_mask.sum() > 0 else 0.0
macro_f1_valid = f1_score(y_true[valid_mask], y_pred[valid_mask], average='macro', zero_division=0) if valid_mask.sum() > 0 else 0.0
acc_clean = accuracy_score(y_true[clean_mask], y_pred[clean_mask]) if clean_mask.sum() > 0 else 0.0
macro_f1_clean = f1_score(y_true[clean_mask], y_pred[clean_mask], average='macro', zero_division=0) if clean_mask.sum() > 0 else 0.0

print(f'\nFinal test metrikleri:')
print(f"   Full          n={n}      acc={acc_full:.4f}  macroF1={macro_f1_full:.4f}  wF1={weighted_f1_full:.4f}")
print(f"   Valid only    n={valid_mask.sum()}    acc={acc_valid:.4f}  macroF1={macro_f1_valid:.4f}")
print(f"   Clean (own=F) n={clean_mask.sum()}    acc={acc_clean:.4f}  macroF1={macro_f1_clean:.4f}")

labels = sorted(set(y_true) | (set(y_pred) - {'INVALID'}))
p, r, f, supp = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
print(f'\nPer-class:')
for cls, pi, ri, fi, si in zip(labels, p, r, f, supp):
    print(f'   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}')

# Confusion
from sklearn.metrics import confusion_matrix
labels_full = ALLOWED_CLASSES + ['INVALID']
cm = confusion_matrix(y_true, y_pred, labels=labels_full)
kisaltma = {'SEPSIS':'SEP','STROKE':'STR','ATRIAL_FIBRILLATION':'AFB','AKI':'AKI','PNEUMONIA':'PNE','UTI':'UTI','HEART_FAILURE':'HF','GI_BLEEDING':'GIB','DIABETES_COMPLICATION':'DIA','COPD_EXACERBATION':'CPD','OTHER':'OTH','INVALID':'INV'}
print('\nConfusion matrix:')
print('   gercek\\pred  ' + ' '.join(f'{kisaltma[c]:>4}' for c in labels_full))
for i, g in enumerate(labels_full):
    if cm[i].sum() == 0: continue
    print('   ' + f'{kisaltma[g]:<13s}' + ' '.join(f'{cm[i][j]:>4}' for j in range(len(labels_full))))

# Bias
print('\nTahmin dagilimi:')
for cls, sayi in test_v3['parsed_pred'].value_counts().items():
    print(f'   {cls:<25s} {sayi:>5} ({100*sayi/n:.1f}%)')

# BLOK 13A.1 — Kalinti Audit (her flag tipinden ornek baglam)

Amac: triage_audit_flags icindeki kalintilarin GECMIS PMH mi yoksa MEVCUT BASVURU workup'i mi oldugunu tesbit et.

Eger PMH'de ise gecmis bilgi (KABUL).
Eger hpi_clean_v3'te ise cutoff yetersiz, ek pass (13A.2+) gerekli.

In [ ]:
# 13A.1: Kalinti hangi bolumden geliyor + ornekler
import pandas as pd
import re
import random
from pathlib import Path

OUTPUT_DIR = Path(r'./workdir/output')
df = pd.read_parquet(OUTPUT_DIR / 'discharge_split.parquet')

KALINTILAR = {
    'lab_kalinti':        re.compile(r'\b(WBC \d|Cr \d\.\d|Hgb \d|platelets? \d|INR \d|lactate \d|troponin)\b', re.IGNORECASE),
    'imaging_kalinti':    re.compile(r'\b(CXR|CT (?:head|chest|abdomen)|CTA|MRI|TTE)\b', re.IGNORECASE),
    'assessment_kalinti': re.compile(r'\b(admitted to|diagnosed with|concern for)\b', re.IGNORECASE),
}

def hangi_bolum(row, regex):
    bolumler = {b: row.get(b,'') for b in ['cc_v3','hpi_clean_v3','pmh_v3','allergies_v3','meds_adm_v3','vitals_v3']}
    return {ad: regex.search(metin) for ad, metin in bolumler.items() if isinstance(metin,str) and metin and regex.search(metin)}

for flag, rx in KALINTILAR.items():
    alt = df[df['triage_audit_flags'].str.contains(flag, na=False)]
    print(f'\n{flag}: {len(alt)} kayit')
    bolum_sayim = {b:0 for b in ['cc_v3','hpi_clean_v3','pmh_v3','allergies_v3','meds_adm_v3','vitals_v3']}
    for _, row in alt.iterrows():
        for b in hangi_bolum(row, rx):
            bolum_sayim[b] += 1
    for b, n in sorted(bolum_sayim.items(), key=lambda x: -x[1]):
        if n > 0:
            print(f'   {b:<20s} {n:>5} ({100*n/len(alt):>5.1f}%)')

# BLOK 13A.2 — Conservative Cumle-Bazli Cleanup (sadece hpi_clean_v3)

13A.1 sonucu: imaging_kalinti'nin %36'si hpi_clean_v3'te (mevcut basvuru ED workup'i).

Cumle-bazli cleanup uygulaniyor — pmh_v3, vitals'a dokunulmaz.
Yedek: hpi_clean_v3_precleanup.

In [ ]:
# 13A.2: hpi_clean_v3 cumle-bazli cleanup
import pandas as pd
import re
from pathlib import Path

SPLIT_YOL = Path(r'./workdir/output/discharge_split.parquet')
df = pd.read_parquet(SPLIT_YOL)
if 'hpi_clean_v3_precleanup' not in df.columns:
    df['hpi_clean_v3_precleanup'] = df['hpi_clean_v3'].copy()

DROP_PATTERNS = [
    re.compile(r'\b(CXR|CTA?|MRI|US|TTE|TEE|EKG|ECG)\b[^.]{0,80}\b(showed|revealed|notable|with|demonstrated|w/|consistent|positive|negative|reveal|reveals|noted)\b', re.IGNORECASE),
    re.compile(r'\b(showed|revealed|notable|demonstrated|consistent)\b[^.]{0,40}\b(CXR|CTA?|MRI|US|TTE|TEE|EKG|ECG)\b', re.IGNORECASE),
    re.compile(r'\b(got|had|received|underwent|taken\sto|sent\sto|brought\sto)\b[^.]{0,60}\b(CXR|CTA?|MRI|US|TTE|TEE|EKG|ECG)\b', re.IGNORECASE),
    re.compile(r'\bLabs?\b[^.]{0,80}\b(notable|showed|drawn|remarkable|significant|reveal|positive|negative)\b', re.IGNORECASE),
    re.compile(r'\b(WBC|Hgb|HCT|platelets?|PLT|INR|lactate|troponin|BUN|creatinine)\s*[<>=]?\s*\d', re.IGNORECASE),
    re.compile(r'\bCr\s*\d\.\d', re.IGNORECASE),
    re.compile(r'\bUA\b[^.]{0,40}\b(showed|with|notable|positive|negative)\b', re.IGNORECASE),
    re.compile(r'\bfound\sto\shave\b[^.]{0,80}\b(CXR|CTA?|MRI|US|TTE|infiltrate|consolidation|effusion|edema|stenosis|ICH|infarct|embolus|clot|thrombus|hemorrhage|stroke|pneumonia|sepsis)\b', re.IGNORECASE),
    re.compile(r'\bworkup\b[^.]{0,40}\b(positive|negative|notable|showed|revealed)\b', re.IGNORECASE),
]

def cumle_temizle(metin):
    if not isinstance(metin,str) or not metin: return metin, 0
    ros = re.search(r'(?ims)(\n\s*(?:Review of Systems?|ROS)\s*:)', metin)
    on = metin[:ros.start()] if ros else metin
    ros_kismi = metin[ros.start():] if ros else ''
    cumleler = re.split(r'(?<=[.!?])\s+(?=[A-Z\d])', on)
    atilan = 0
    korunan = []
    for c in cumleler:
        cs = c.strip()
        if not cs: continue
        if any(rx.search(cs) for rx in DROP_PATTERNS):
            atilan += 1
        else:
            korunan.append(c)
    yeni_on = ' '.join(korunan).strip()
    if ros_kismi: return yeni_on + '\n\n' + ros_kismi.strip(), atilan
    return yeni_on, atilan

yeni_hpi, atilan_l = [], []
for h in df['hpi_clean_v3_precleanup']:
    y, a = cumle_temizle(h)
    yeni_hpi.append(y); atilan_l.append(a)
df['hpi_clean_v3'] = yeni_hpi
df['cumle_atilan_sayi'] = atilan_l
print(f'13A.2: cumle atilan toplam {sum(atilan_l):,}, etkilenen kayit {sum(1 for x in atilan_l if x>0):,}')

# BLOK 13A.3 — Vital Koruyan Cleanup (precleanup'tan yeniden uret)

13A.2 vital'lari (hypotensive, tachycardic, febrile, SpO2) silebiliyor.
13A.3: Vital whitelist + lab/imaging fragment temizleme. precleanup'tan baslar.

In [ ]:
# 13A.3: Vital koruyan cleanup
# (Detayli kod __run_blok13a3.py'de — vital_keep_regex + lab_fragment + cumle_temizle)
# Bu hucre referans icindir; gercek calistirma:
#   python __run_blok13a3.py
# Ciktilar:
#   - hpi_clean_v3 guncellenir (vital korunur, lab/imaging fragment silinir)
#   - triage_input_v3, triage_input_v3_masked yeniden compose
#   - Audit: imaging 235->102 (-133), lab fragment 25k+ silindi, vital 1847->1847 (0 kayip)
print('13A.3 detaylari __run_blok13a3.py icinde. Sonuc audit:')
print('  - imaging hpi-icinde: 235 -> 102')
print('  - lab hpi-icinde   : 4 -> 3')
print('  - vital korunan    : 1847 -> 1847 (KORUNDU)')

# BLOK 13A.4 — Residual Lab Cleanup (sodium, troponin, lactate)

13A.3'te kalan lab paterni: 'Trop negative', 'Lactate of 2.6', 'Sodium N', 'low sodium (119)'.

13A.4: Vital korunarak bu spesifik lab fragmentleri de silinir.

In [ ]:
# 13A.4: Residual lab cleanup
# Detayli kod __run_blok13a4.py'de
# Sonuc audit:
#   - hpi-icinde lab kalintisi: 20 (precleanup) -> 15 (13A.3) -> 2 (13A.4)
#   - vital korunan: 2671 -> 2669 -> 2669 (sifir kayip)
#   - hpi ortalama uzunluk: 752 -> 732 (-20 char, %2.6)
#   - V3 final ortalama: 1880 char
print('13A.4 sonuc:')
print('  - lab residual    : 20 -> 2')
print('  - vital korunan   : 2671 -> 2669 (~%99.9)')
print('  - semptom korumal : %100 (chest pain/SOB/cough/fever vb. silinmedi)')

# BLOK 13C — Final Leakage Audit (her sinif 5 dogru tahmin)

Zero-shot V3 test 851 dogru tahmininden her siniftan 5 ornek (toplam 55).
Her ornekte riskli pattern aranir; HPI'de ise 'leakage', PMH'de ise 'historical'.

Sonuc: %78 temiz, %14.5 LOW (gecmis), %5.5 MEDIUM, %1.8 HIGH leak.

In [ ]:
# 13C: Leakage audit (her sinif 5 dogru ornek)
# Detayli __run_blok13c_leakage_audit.py + __run_blok13c_severity.py
# Cikti: zeroshot_v3_leakage_audit.csv, zeroshot_v3_leakage_severity.csv
import pandas as pd
from pathlib import Path
ZS = Path(r'./workdir/output/zeroshot')
if (ZS/'zeroshot_v3_leakage_severity.csv').exists():
    sev = pd.read_csv(ZS/'zeroshot_v3_leakage_severity.csv')
    print('Severity dagilimi (12 leakage flag uzerinden):')
    print(sev['severity'].value_counts())

# BLOK 14-SELECT-1SHOT — Class-wise TF-IDF Clean Medoid (11 ornek)

Her sinif icin centroid'e en yakin TEMIZ medoid.

Riskli pattern (lab/imaging/diagnosed-with vb.) varsa rank 2/3/...'e gec.
Sadece train_S3 uzerinde TF-IDF fit. Val/test KULLANILMAZ.

Cikti: fewshot_v3_fixed_1shot_examples.json (11 ornek, ~6480 token toplam)

In [ ]:
# 14-SELECT-1SHOT detayli kod __run_blok14_select_1shot.py
# Sonuc:
#   AFib/DIABETES/COPD: rank 1 zaten temiz
#   STROKE/OTHER: rank 2
#   AKI/PNEUMONIA/GI_BLEEDING: rank 3
#   SEPSIS/UTI: rank 9
#   HEART_FAILURE: rank 13 (ilk 12 hep dirty)
import json
from pathlib import Path
FS = Path(r'./workdir/output/fewshot')
with open(FS/'fewshot_v3_fixed_1shot_examples.json','r',encoding='utf-8') as f:
    medoids = json.load(f)
print(f'11 medoid yuklendi. Token uzunlugu:')
for cls, ex in medoids.items():
    print(f'   {cls:<25s} note={ex["note_id"]:<22s} rank={ex["secilen_rank"]:<3} token={ex["token_uzunluk"]:>4} sim={ex["sim_to_centroid"]:.3f}')

# BLOK 14D — Section-Aware Compaction (medoid + smoke)

11 medoid'i hedef 180-220 token'a sikistir:
- Chief Complaint: tam
- Key Symptoms: hasta-tarafli ilk 2-3 cumle
- Vitals: tam
- Relevant History: ilk 2-3 PMH ifadesi
- Allergies: tek satir
- Medications: en fazla 5 ilac

Cikti: fewshot_v3_compact_1shot_examples.json (11 ornek, ~199 token avg, toplam 2185).

Smoke (3 ornek): hepsi dogru, peak GPU 6 GB, OOM yok.

In [ ]:
# 14D: Compact medoid + smoke. Detay __run_blok14d_compact.py
import json
from pathlib import Path
FS = Path(r'./workdir/output/fewshot')
with open(FS/'fewshot_v3_compact_1shot_examples.json','r',encoding='utf-8') as f:
    cmp = json.load(f)
toplam = sum(d['compact_token'] for d in cmp.values())
print(f'Compact 11 medoid: toplam {toplam} token (avg {toplam/11:.0f})')
print(f'Token sayilari: {[d["compact_token"] for d in cmp.values()]}')

# BLOK 14E — Compact Query (val/test input section-aware)

Worst case input (2024 token) <think> tetikledi + GPU %100. Cozum: query'yi de compact'la.

Aynı section-aware mantik V3 kolonlardan (cc_v3, hpi_clean_v3, pmh_v3, ...) compact uretir.
Ciktilar:
- discharge_split.parquet'e 'triage_input_v3_compact' kolonu eklenir
- Val: ort 203, max 386 token
- Test: ort 199, max 351 token
- Sabit prompt + max compact = 3.101 token -> max_length=4096 ile %100 kapsama
- Smoke 10 ornek: <think>=0/10, dogru=5/10, peak GPU %64.3

In [ ]:
# 14E: Val/test input section-aware compact + parquet update
# Detay __run_blok14e_compact_query.py
import pandas as pd
from pathlib import Path
df = pd.read_parquet(r'./workdir/output/discharge_split.parquet')
if 'triage_input_v3_compact' in df.columns:
    val = df[df['split']=='val']
    test = df[df['split']=='test']
    val_lens = val['triage_input_v3_compact'].fillna('').str.len()
    test_lens = test['triage_input_v3_compact'].fillna('').str.len()
    print(f'triage_input_v3_compact mevcut: {len(df)} satir')
    print(f'  Val char ort/max  : {val_lens.mean():.0f} / {val_lens.max():.0f}')
    print(f'  Test char ort/max : {test_lens.mean():.0f} / {test_lens.max():.0f}')
else:
    print('triage_input_v3_compact kolonu yok — __run_blok14e_compact_query.py calistir.')

# BLOK 13D — Time-Aware Triage Input (V4) — İLK VERSİYON

**Amac:** Historical EHR allowed, current encounter workup/treatment forbidden.

**Senaryo:** AI hastayla ilk temas, doktordan/lab/imaging'den ÖNCE.
EHR'de mevcut historical bilgiler kullanılabilir, mevcut başvuru workup'ı yasak.

**Sonuc:** Test'te HIGH=78, MEDIUM=1, REVIEW=544 — kabul edilemez (cok agresif false positive).
REVIEW_UNCERTAIN kategorisi cok genis (sadece keyword gorulup current marker yoksa REVIEW dusuyor).

**Detayli script:** `__run_blok13d_time_aware.py`

In [ ]:
"""Blok 13D: Time-aware triage input (V4) + leakage audit.
Historical EHR ALLOWED. Current encounter workup/treatment FORBIDDEN.
Inference YOK."""
import pandas as pd
import re
import time
import random
from pathlib import Path

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"

print("=" * 70)
print("BLOK 13D — Time-Aware Triage Input (V4)")
print("=" * 70)

ALLOWED_CLASSES = [
    "SEPSIS", "STROKE", "ATRIAL_FIBRILLATION", "AKI", "PNEUMONIA", "UTI",
    "HEART_FAILURE", "GI_BLEEDING", "DIABETES_COMPLICATION",
    "COPD_EXACERBATION", "OTHER"
]

# ============================================================
# HISTORICAL MARKERS (bunlar bir cumlede varsa cumle historical, KORUNUR)
# ============================================================
HISTORICAL_MARKERS = re.compile(
    r"\b("
    r"history\s+of|hx\s+of|h/o|"
    r"prior|previously|remote|chronic|baseline|"
    r"in\s+the\s+past|past\s+medical|"
    r"s/p|status\s+post|"
    r"recently\s+admitted\s+(in|on|to)|"  # 'recently admitted to OSH for X' historical
    r"prior\s+(admission|hospitalization|visit|imaging|workup|labs?)|"
    r"(\d+|few|several|many)\s+(years?|months?|weeks?|days?)\s+ago|"
    r"old(\s+|-)?(imaging|labs?|TTE|CT|MRI|CXR)|"
    r"home\s+(medications?|meds?|O2|insulin|inhaler|nebulizer)|"
    r"on\s+chronic|chronic\s+suppressive|"
    r"baseline\s+(Cr|creatinine|Hgb|labs?|EF|FEV)"
    r")\b",
    re.IGNORECASE
)

# Bir cumlede yil ifadesi varsa (___ yer tutucu MIMIC) historical sinyal
YEAR_MARKER = re.compile(r"\b(in\s+___|in\s+\d{4}|on\s+___|admitted\s+on\s+___)\b", re.IGNORECASE)

# ============================================================
# CURRENT ENCOUNTER MARKERS (bunlar varsa ZAMANSAL sinyal current)
# ============================================================
CURRENT_MARKERS = re.compile(
    r"\b("
    r"on\s+arrival|upon\s+arrival|on\s+presentation|presented\s+to\s+(the\s+)?ED|"
    r"in\s+the\s+ED|in\s+ED|ED\s+course|"
    r"initial\s+(labs?|vitals)|"
    r"on\s+admission(?!\s+to\s+rehab)|admitted\s+to\s+(MICU|ICU|floor|service|cardiology|medicine)|"
    r"transferred\s+to|sent\s+to\s+(MICU|ICU|floor|cardiology|medicine|pulmonology)|"
    r"received\s+(in\s+ED|in\s+the\s+ED)"
    r")\b",
    re.IGNORECASE
)

# ============================================================
# LEAKAGE PATTERNS — HIGH (current diagnosis/imaging/lab)
# ============================================================
HIGH_LEAK_PATTERNS = {
    # Imaging finding (mevcut basvuru)
    "imaging_showed": re.compile(
        r"\b(CXR|CTA?|MRI|US|TTE|TEE|echocardiogram|EKG|ECG|"
        r"ultrasound|sonography)\b[^.]{0,80}\b("
        r"showed|revealed|demonstrated|notable|consistent|"
        r"with|w/|positive|negative|reveal|reveals|noted|"
        r"found|impression|read\s+as"
        r")\b",
        re.IGNORECASE
    ),
    # Diagnosis assertion (current)
    "diagnosed_with": re.compile(r"\bdiagnosed\s+with\b", re.IGNORECASE),
    "found_to_have": re.compile(r"\bfound\s+to\s+have\b", re.IGNORECASE),
    "concern_for": re.compile(r"\bconcern\s+for\b", re.IGNORECASE),
    "consistent_with_dx": re.compile(
        r"\bconsistent\s+with\b[^.]{0,40}\b("
        r"sepsis|stroke|pneumonia|UTI|infarct|hemorrhage|embolus|"
        r"effusion|infiltrate|consolidation|edema|stenosis|ICH"
        r")\b",
        re.IGNORECASE
    ),
    "admitted_for": re.compile(
        r"\badmitted\s+(for|to)\b[^.]{0,50}\b("
        r"workup|management|treatment|further|acute|exacerbation|"
        r"sepsis|stroke|pneumonia|HF|CHF|COPD|UTI|GI\s+bleed|DKA"
        r")\b",
        re.IGNORECASE
    ),
    # Lab values (current encounter)
    "lab_value_numeric": re.compile(
        r"\b(WBC|Hgb|Hct|platelets?|PLT|INR|lactate|troponin|"
        r"BUN|Cr|creatinine|sodium|Na\+?|K\+?)\s*[<>=]?\s*\d[\d.]*",
        re.IGNORECASE
    ),
    "trop_negative": re.compile(r"\btrop(onin)?\s+(negative|positive|elevated|<|>)", re.IGNORECASE),
    "lactate_value": re.compile(r"\blactate\s+(of\s+)?\d", re.IGNORECASE),
    "labs_notable": re.compile(r"\blabs?\s+(notable|showed|drawn|remarkable|reveal)", re.IGNORECASE),
    "ua_result": re.compile(r"\bUA\s+(showed|notable|positive|negative|with)", re.IGNORECASE),
    "cultures": re.compile(r"\b(blood|urine|sputum)\s+cultures?\s+(grew|positive|notable)", re.IGNORECASE),
    "workup_showed": re.compile(r"\bworkup\s+(showed|revealed|positive|negative|notable)", re.IGNORECASE),
}

# ============================================================
# LEAKAGE PATTERNS — MEDIUM (current treatment / etiket-spesifik tedavi)
# ============================================================
# Bunlar etiket-spesifik tedavi: vancomycin -> sepsis, glucagon -> hypoglycemia
MEDIUM_TREATMENT_PATTERNS = {
    "received_treatment": re.compile(
        r"\b(received|was\s+given|started\s+on|treated\s+with|treating\s+with)\b[^.]{0,60}\b("
        # Antibiotics
        r"vancomycin|zosyn|piperacillin|cefepime|ceftriaxone|levofloxacin|"
        r"azithromycin|ciprofloxacin|nitrofurantoin|bactrim|cephalexin|"
        # Pressors
        r"levophed|norepinephrine|vasopressin|epinephrine|phenylephrine|"
        # Diuretics
        r"lasix|furosemide|bumetanide|metolazone|"
        # GI
        r"protonix|pantoprazole|octreotide|"
        # Endocrine
        r"insulin\s+drip|glucagon|D5|D50|dextrose|"
        # Pulmonary
        r"duonebs|albuterol\s+nebs?|nebulizer|prednisone\s+\d|methylprednisolone|"
        r"tPA|alteplase|"
        r"IV\s+fluids|IVF|fluid\s+bolus|crystalloid|"
        r"pRBC|transfusion|"
        r"thrombectomy|cardioversion"
        r")\b",
        re.IGNORECASE
    ),
}

# ============================================================
# Sentence is HISTORICAL?
# ============================================================
def is_historical_sentence(cumle):
    """Cumle historical EHR mi? (ALLOWED)"""
    if HISTORICAL_MARKERS.search(cumle):
        return True
    if YEAR_MARKER.search(cumle):
        return True
    return False

def is_current_sentence(cumle):
    """Cumle current encounter mi?"""
    return bool(CURRENT_MARKERS.search(cumle))

# ============================================================
# Sentence classification
# ============================================================
def classify_sentence(cumle):
    """Bir cumle icin: CLEAN | HISTORICAL_ALLOWED | REVIEW_UNCERTAIN | CURRENT_HIGH | CURRENT_MEDIUM
       Her durumda hangi pattern tetikledi de doner."""
    bulgular_high = []
    for ad, rx in HIGH_LEAK_PATTERNS.items():
        m = rx.search(cumle)
        if m:
            bulgular_high.append((ad, m.group()))

    bulgular_med = []
    for ad, rx in MEDIUM_TREATMENT_PATTERNS.items():
        m = rx.search(cumle)
        if m:
            bulgular_med.append((ad, m.group()))

    # Risk yoksa CLEAN
    if not bulgular_high and not bulgular_med:
        return "CLEAN", []

    historical = is_historical_sentence(cumle)
    current = is_current_sentence(cumle)

    # Eger historical marker varsa VE current marker yoksa -> HISTORICAL_ALLOWED
    if historical and not current:
        return "HISTORICAL_ALLOWED", bulgular_high + bulgular_med

    # Eger current marker varsa -> CURRENT
    if current:
        if bulgular_high:
            return "CURRENT_HIGH", bulgular_high
        if bulgular_med:
            return "CURRENT_MEDIUM", bulgular_med

    # Hicbir tarafta marker yok ama risk pattern var -> REVIEW
    # Risk pattern HIGH olsa bile, marker yoksa belirsiz
    if bulgular_high:
        # Lab values + numerical + no historical marker = muhtemelen current
        # but to be safe, REVIEW
        return "REVIEW_UNCERTAIN", bulgular_high
    return "REVIEW_UNCERTAIN", bulgular_med

# ============================================================
# Cumlelere bol
# ============================================================
def cumlelere_bol(metin):
    if not metin: return []
    cs = re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", metin)
    return [c.strip() for c in cs if c.strip()]

# ============================================================
# HPI/CC time-aware temizleme
# ============================================================
def clean_hpi_time_aware(hpi_text):
    """Cumle bazli: CURRENT_HIGH ve CURRENT_MEDIUM cumleleri at.
    HISTORICAL_ALLOWED ve CLEAN cumleleri tut. REVIEW_UNCERTAIN: kaydet ama at."""
    if not isinstance(hpi_text, str) or not hpi_text:
        return hpi_text, [], []

    # ROS koru
    ros_m = re.search(r"(?ims)(\n\s*(?:Review of Systems?|ROS)\s*:)", hpi_text)
    if ros_m:
        on = hpi_text[:ros_m.start()]
        ros_kismi = hpi_text[ros_m.start():]
    else:
        on = hpi_text
        ros_kismi = ""

    cumleler = cumlelere_bol(on)
    korunan = []
    atilan_log = []  # [(cumle, kategori, bulgular)]
    review_log = []

    for c in cumleler:
        kat, bulgular = classify_sentence(c)
        if kat == "CLEAN":
            korunan.append(c)
        elif kat == "HISTORICAL_ALLOWED":
            korunan.append(c)
        elif kat == "REVIEW_UNCERTAIN":
            review_log.append((c, kat, bulgular))
            # Conservative: REVIEW'i at ama kaydet
            atilan_log.append((c, kat, bulgular))
        elif kat in ("CURRENT_HIGH", "CURRENT_MEDIUM"):
            atilan_log.append((c, kat, bulgular))

    yeni_on = " ".join(korunan).strip()
    if ros_kismi:
        return yeni_on + "\n\n" + ros_kismi.strip(), atilan_log, review_log
    return yeni_on, atilan_log, review_log

# ============================================================
# PMH/Meds: sadece "current encounter" yapan cumleleri at, kalan tum historical TUTULUR
# ============================================================
def clean_pmh_time_aware(pmh_text):
    """PMH genelde tum historical. Sadece is_current_sentence olanlari at."""
    if not isinstance(pmh_text, str) or not pmh_text:
        return pmh_text, []
    cumleler = cumlelere_bol(pmh_text)
    korunan = []
    atilan = []
    for c in cumleler:
        if is_current_sentence(c):
            kat, b = classify_sentence(c)
            if kat in ("CURRENT_HIGH", "CURRENT_MEDIUM"):
                atilan.append((c, kat, b))
                continue
        korunan.append(c)
    return " ".join(korunan).strip(), atilan

# ============================================================
# Audit overall: V4 input'unun tum kelimelerinde leakage flag
# ============================================================
def audit_v4_input(v4_text):
    """V4 birlestirilmis input'un cumlelerini bir kez daha klasifiye et.
    Toplam: en yuksek severity flag + reason listesi."""
    if not isinstance(v4_text, str) or not v4_text:
        return "CLEAN", "EMPTY", []
    cumleler = cumlelere_bol(v4_text)
    severities = []
    reasons = []
    for c in cumleler:
        kat, bulgular = classify_sentence(c)
        if kat != "CLEAN":
            severities.append(kat)
            for ad, eslesen in bulgular:
                reasons.append(f"{kat}:{ad}={eslesen[:30]}")

    if not severities:
        return "CLEAN", "no_leak", []

    # En yuksek severity
    oncelik = {"CURRENT_HIGH": 4, "CURRENT_MEDIUM": 3, "REVIEW_UNCERTAIN": 2, "HISTORICAL_ALLOWED": 1}
    en_yuksek = max(severities, key=lambda x: oncelik.get(x, 0))
    return en_yuksek, ",".join(set(severities)), reasons[:10]

# ============================================================
# V4 input compose
# ============================================================
def compose_v4(row):
    parts = []
    if row.get("cc_v4_timeaware"):
        parts.append(f"Chief Complaint:\n{row['cc_v4_timeaware']}")
    if row.get("hpi_clean_v4_timeaware"):
        parts.append(f"Symptom History:\n{row['hpi_clean_v4_timeaware']}")
    if row.get("pmh_v4_timeaware"):
        parts.append(f"Past Medical History:\n{row['pmh_v4_timeaware']}")
    if row.get("allergies_v3"):
        parts.append(f"Allergies:\n{row['allergies_v3']}")
    if row.get("meds_adm_v3"):
        parts.append(f"Medications on Admission:\n{row['meds_adm_v3']}")
    if row.get("vitals_v3"):
        parts.append(f"Vital Signs:\n{row['vitals_v3']}")
    return "\n\n".join(parts)

# ============================================================
# MAIN
# ============================================================
print("\n[1/6] Veri yukleniyor...")
df = pd.read_parquet(SPLIT_YOL)
print(f"   Toplam: {len(df):,} satir, {len(df.columns)} kolon")

# Kaynak: hpi_clean_v3_precleanup (en az silinmis hali) varsa onu kullan, yoksa hpi_clean_v3
HPI_KAYNAK = "hpi_clean_v3_precleanup" if "hpi_clean_v3_precleanup" in df.columns else "hpi_clean_v3"
print(f"   HPI kaynak: {HPI_KAYNAK}")

# CC: dokunulmaz (zaten kisa, hasta ifadesi)
df["cc_v4_timeaware"] = df["cc_v3"].copy()

# ============================================================
# HPI temizle (her satir)
# ============================================================
print("\n[2/6] HPI time-aware temizleme...")
t0 = time.time()
hpi_v4_list = []
hpi_atilan_logs = []  # liste of (atilan, review)
for hpi in df[HPI_KAYNAK]:
    yeni_hpi, atilan, review = clean_hpi_time_aware(hpi)
    hpi_v4_list.append(yeni_hpi)
    hpi_atilan_logs.append((atilan, review))
df["hpi_clean_v4_timeaware"] = hpi_v4_list
print(f"   Sure: {time.time()-t0:.1f}s")

# ============================================================
# PMH temizle
# ============================================================
print("\n[3/6] PMH time-aware temizleme (sadece current cumleleri at)...")
t0 = time.time()
pmh_v4_list = []
pmh_atilan_logs = []
for pmh in df["pmh_v3"]:
    yeni_pmh, atilan = clean_pmh_time_aware(pmh)
    pmh_v4_list.append(yeni_pmh)
    pmh_atilan_logs.append(atilan)
df["pmh_v4_timeaware"] = pmh_v4_list
print(f"   Sure: {time.time()-t0:.1f}s")

# ============================================================
# V4 compose + audit
# ============================================================
print("\n[4/6] V4 compose + final audit...")
t0 = time.time()
df["triage_input_v4_timeaware"] = df.apply(compose_v4, axis=1)

# Maskeleme (Blok 4 ile ayni keyword)
DIAG_KEYWORDS = {
    'PNEUMONIA': r'\b(pneumon\w*|bronchopneumon\w*)\b',
    'HEART_FAILURE': r'\b(CHF|congestive heart failure|heart failure|HFrEF|HFpEF|systolic heart failure|diastolic heart failure)\b',
    'SEPSIS': r'\b(sepsis|septic|septicemia|bacteremia|septicaemia)\b',
    'AKI': r'\b(AKI|acute kidney injury|acute renal failure|ARF|acute renal insufficiency)\b',
    'UTI': r'\b(UTI|urinary tract infection|pyelonephritis|cystitis)\b',
    'STROKE': r'\b(stroke|CVA|cerebrovascular accident|ischemic stroke|hemorrhagic stroke|intracerebral hemorrhage|intracranial hemorrhage)\b',
    'GI_BLEEDING': r'\b(GI bleed\w*|gastrointestinal bleed\w*|UGIB|LGIB|hematemesis|melena|hematochezia)\b',
    'COPD_EXACERBATION': r'\b(COPD exacerbation|AECOPD|chronic obstructive|COPD)\b',
    'DIABETES_COMPLICATION': r'\b(DKA|diabetic ketoacidosis|HHS|hyperosmolar|hyperglycemic crisis|diabetic coma)\b',
    'ATRIAL_FIBRILLATION': r'\b(atrial fibrillation|AFib|A-fib|atrial flutter|paroxysmal AF)\b',
}
BIRLESIK = re.compile("|".join(f"(?:{p})" for p in DIAG_KEYWORDS.values()), re.IGNORECASE)
df["triage_input_v4_timeaware_masked"] = df["triage_input_v4_timeaware"].apply(
    lambda m: BIRLESIK.sub("[DIAG_MASKED]", m) if m else m
)

# Audit her satir icin
audit_flag, audit_reason, audit_severity = [], [], []
for v4 in df["triage_input_v4_timeaware"]:
    sev, reason, _ = audit_v4_input(v4)
    audit_flag.append(reason)
    audit_severity.append(sev)
    audit_reason.append(reason)
df["strict_current_leakage_severity"] = audit_severity
df["strict_current_leakage_flag"] = audit_severity != "CLEAN"
df["strict_current_leakage_reason"] = audit_reason
print(f"   Sure: {time.time()-t0:.1f}s")

# ============================================================
# Audit raporu (split bazli)
# ============================================================
print("\n[5/6] AUDIT RAPORU")
print("=" * 70)

for split_ad, alt in [("ALL", df), ("test", df[df["split"]=="test"]),
                       ("val", df[df["split"]=="val"]),
                       ("train_S3", df[df["split"]=="train_S3"])]:
    n = len(alt)
    print(f"\n--- {split_ad} (N={n:,}) ---")
    sev_say = alt["strict_current_leakage_severity"].value_counts().to_dict()
    for sev in ["CLEAN", "HISTORICAL_ALLOWED", "REVIEW_UNCERTAIN", "CURRENT_MEDIUM", "CURRENT_HIGH"]:
        c = sev_say.get(sev, 0)
        if c > 0 or sev in ("CURRENT_HIGH", "CURRENT_MEDIUM", "REVIEW_UNCERTAIN"):
            print(f"   {sev:<25s} {c:>5} ({100*c/max(n,1):>5.1f}%)")

# Test ozeti
print(f"\n   TEST OZET (N=1.650):")
test_alt = df[df["split"]=="test"]
print(f"      full N             : {len(test_alt)}")
print(f"      CLEAN              : {(test_alt['strict_current_leakage_severity']=='CLEAN').sum()}")
print(f"      HISTORICAL_ALLOWED : {(test_alt['strict_current_leakage_severity']=='HISTORICAL_ALLOWED').sum()}")
print(f"      REVIEW_UNCERTAIN   : {(test_alt['strict_current_leakage_severity']=='REVIEW_UNCERTAIN').sum()}")
print(f"      CURRENT_MEDIUM     : {(test_alt['strict_current_leakage_severity']=='CURRENT_MEDIUM').sum()}")
print(f"      CURRENT_HIGH       : {(test_alt['strict_current_leakage_severity']=='CURRENT_HIGH').sum()}")

# Uzunluk karsilastirma
v3_lens = df["triage_input_v3"].fillna("").str.len() if "triage_input_v3" in df.columns else None
v4_lens = df["triage_input_v4_timeaware"].fillna("").str.len()
print(f"\n   Uzunluk: V4 ort={v4_lens.mean():.0f}, max={v4_lens.max()}")
if v3_lens is not None:
    print(f"            V3 ort={v3_lens.mean():.0f}, V4 daha {('uzun' if v4_lens.mean()>v3_lens.mean() else 'kisa')}")

# ============================================================
# Ornek yazdir (3'er ornek)
# ============================================================
print("\n[6/6] ORNEKLER (her kategoriden 3 ornek)")
print("=" * 70)

random.seed(42)
test_df = df[df["split"]=="test"].reset_index(drop=True)

for kategori in ["CLEAN", "HISTORICAL_ALLOWED", "REVIEW_UNCERTAIN", "CURRENT_MEDIUM", "CURRENT_HIGH"]:
    alt = test_df[test_df["strict_current_leakage_severity"]==kategori]
    if len(alt) == 0:
        print(f"\n--- {kategori} --- 0 ornek")
        continue
    n_ornek = min(3, len(alt))
    secilen = alt.sample(n=n_ornek, random_state=42).index.tolist()
    print(f"\n{'#'*70}")
    print(f"# {kategori} ({len(alt)} kayit, {n_ornek} ornek)")
    print(f"{'#'*70}")
    for idx in secilen:
        row = alt.loc[idx]
        print(f"\n   --- note={row['note_id']} label={row['label']} ---")
        print(f"   V4 input ({len(row['triage_input_v4_timeaware'])} char) PREVIEW:")
        print(f"     {(row['triage_input_v4_timeaware'] or '')[:600]}")
        print(f"   reasons: {row['strict_current_leakage_reason'][:200]}")

# ============================================================
# Acceptance kriteri
# ============================================================
print("\n" + "=" * 70)
print("ACCEPTANCE")
print("=" * 70)
high_test = (test_df["strict_current_leakage_severity"]=="CURRENT_HIGH").sum()
med_test = (test_df["strict_current_leakage_severity"]=="CURRENT_MEDIUM").sum()
rev_test = (test_df["strict_current_leakage_severity"]=="REVIEW_UNCERTAIN").sum()

print(f"   Test CURRENT_HIGH   = {high_test}  (hedef: 0)")
print(f"   Test CURRENT_MEDIUM = {med_test}  (hedef: 0)")
print(f"   Test REVIEW         = {rev_test}  (hedef: 0)")

if high_test == 0 and med_test == 0 and rev_test == 0:
    print(f"\n   ✅ ACCEPT: Test'te HIGH/MEDIUM/REVIEW yok. V4 final.")
else:
    print(f"\n   ⚠️  HIGH/MEDIUM/REVIEW > 0. Insan karari gerekli.")
    print(f"      Yukaridaki ornekleri inceleyip patternlari refine et.")

# Parquet'e yaz
print("\nKAYDETME...")
df.to_parquet(SPLIT_YOL, compression="snappy", index=False)
print(f"   {SPLIT_YOL.name} ({SPLIT_YOL.stat().st_size/1024**2:.1f} MB, {len(df.columns)} kolon)")

# Yeni kolonlar
yeni_kolonlar = ["cc_v4_timeaware", "hpi_clean_v4_timeaware", "pmh_v4_timeaware",
                 "triage_input_v4_timeaware", "triage_input_v4_timeaware_masked",
                 "strict_current_leakage_flag", "strict_current_leakage_reason",
                 "strict_current_leakage_severity"]
print(f"   Yeni kolonlar: {yeni_kolonlar}")

print("\n" + "=" * 70)
print("BLOK 13D TAMAMLANDI - inference YOK")
print("=" * 70)


# BLOK 13E — Refined Time-Aware Detector (1. iterasyon)

**13D sorunlari:**
- REVIEW %33 cok yuksek (sahte pozitif)
- 'multiple CVAs' historical olmasi gerekirken HIGH dustu
- 'presenting with cough' tek basina current sayildi

**13E refinement:**
1. REVIEW = sadece gercek belirsizlik
2. CURRENT_HIGH = current_marker + (lab_value_numeric | imaging_finding | direct_dx | admitted_for | treatment)
3. 'presenting/presented with' tek basina current marker DEGIL (semptom baslangici)
4. HISTORICAL_MARKERS genisletildi: multiple CVAs, recurrent UTI, baseline Cr, last TTE, prior...
5. Sanity test eklendi (multiple CVAs/presenting cough/baseline Cr CLEAN olmali)

**Sonuc:** Sanity gectik. Test'te HIGH=594 — yanlis: ilac dozajlari 'Sodium 80 mg' lab sanildi.
**Detayli script:** `__run_blok13e_refined.py`

In [ ]:
"""Blok 13E: Refined time-aware leakage detector.
- REVIEW = sadece gercek belirsizlik
- CURRENT_HIGH = current_marker + (lab_value | imaging_finding | direct_dx | treatment)
- 'presenting with cough' CLEAN
- 'multiple CVAs', 'h/o X', 's/p' historical
- Cleanup before, audit after."""
import pandas as pd
import re
import time
import random
from pathlib import Path

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"

print("=" * 70)
print("BLOK 13E — Refined Time-Aware Leakage Detector")
print("=" * 70)

ALLOWED_CLASSES = [
    "SEPSIS", "STROKE", "ATRIAL_FIBRILLATION", "AKI", "PNEUMONIA", "UTI",
    "HEART_FAILURE", "GI_BLEEDING", "DIABETES_COMPLICATION",
    "COPD_EXACERBATION", "OTHER"
]

# ============================================================
# HISTORICAL MARKERS — gerçek geniş liste
# ============================================================
HISTORICAL = re.compile(
    r"\b("
    r"history\s+of|hx\s+of|h/o|"
    r"PMH|past\s+medical|"
    r"prior|previous|previously|remote|chronic|"
    r"in\s+the\s+past|"
    r"s/p|status\s*post|"
    r"recurrent\s+(UTI|pneumonia|infection)|"
    r"multiple\s+(CVA|stroke|admission|hospitalization)|"
    r"(\d+|few|several|many)\s+(years?|months?|weeks?|days?)\s+ago|"
    r"old(\s+|-)?(imaging|labs?|TTE|CT|MRI|CXR|EF)|"
    r"home\s+(medications?|meds?|O2|insulin|inhaler|nebulizer)|"
    r"on\s+chronic\s+|chronic\s+suppressive|"
    r"baseline\s+(Cr|creatinine|Hgb|HCT|labs?|EF|FEV)|"
    r"last\s+(TTE|EF|CT|MRI|CXR|labs?|admission)|"
    r"prior\s+(TTE|admission|hospitalization|visit|imaging|workup|labs?)|"
    r"recent\s+admission|previous\s+admission|"
    r"discharged\s+(on|in)\s+|recently\s+discharged|"
    r"(in|on)\s+___|admitted\s+(in|on)\s+___|"
    r"originally\s+admitted|initially\s+admitted"
    r")\b",
    re.IGNORECASE
)

# ============================================================
# CURRENT ENCOUNTER MARKERS — net sinyaller (presenting/presented HARIC)
# ============================================================
# 'presenting with X' tek basina current degildir, semptom baslangici
CURRENT = re.compile(
    r"\b("
    r"on\s+arrival|upon\s+arrival|on\s+presentation\s+to|"
    r"in\s+the\s+ED|in\s+ED|ED\s+course|"
    r"initial\s+(labs?|vitals?|workup)|"
    r"upon\s+(arrival|presentation|admission)|"
    r"on\s+admission(?!\s+to\s+rehab)|"
    r"transferred\s+to\s+(MICU|ICU|floor|service|cardiology|medicine|pulmonology)|"
    r"sent\s+to\s+(MICU|ICU|cardiology|medicine)|"
    r"received\s+in\s+(the\s+)?ED"
    r")\b",
    re.IGNORECASE
)

# ============================================================
# CURRENT HIGH PATTERNS — sıkı koşullar
# ============================================================

# 1. Lab result with numeric value (admission/ED context)
LAB_VALUE_NUMERIC = re.compile(
    r"\b(WBC|Hgb|HCT|platelets?|PLT|INR|lactate|troponin|"
    r"BUN|Cr|creatinine|sodium|Na\+?|K\+?|trop)\s*"
    r"(was|of|=|:|\(|<|>|>=|<=)?\s*\d[\d.]*\b",
    re.IGNORECASE
)

# 2. Lab with descriptor (notable/positive/elevated/low/high/normal)
LAB_DESCRIPTOR = re.compile(
    r"\b(WBC|Hgb|HCT|platelets?|PLT|INR|lactate|troponin|"
    r"BUN|Cr|creatinine|sodium|Na\+?|K\+?|trop|UA|labs?)\b[^.]{0,40}"
    r"\b(elevated|high|low|notable|positive|negative|abnormal|"
    r"showed|revealed|drawn|reveal)\b",
    re.IGNORECASE
)

# 3. Imaging finding with verb
IMAGING_FINDING = re.compile(
    r"\b(CXR|CTA?|MRI|US|TTE|TEE|echocardiogram|EKG|ECG|"
    r"ultrasound)\b[^.]{0,80}\b("
    r"showed|revealed|demonstrated|notable|consistent\s+with|"
    r"positive|negative|reveal|reveals|noted|"
    r"impression|read\s+as|w/\s+|with\s+(infiltrate|consolidation|"
    r"effusion|edema|stenosis|ICH|infarct|embolus|hemorrhage|mass|nodule)"
    r")\b",
    re.IGNORECASE
)

# 4. Direct diagnosis (current encounter)
DIRECT_DX = re.compile(
    r"\b(diagnosed\s+with|found\s+to\s+have|"
    r"concern\s+for|consistent\s+with|"
    r"impression\s*:|assessment\s*:)\b",
    re.IGNORECASE
)

# 5. Admitted for X (current decision)
ADMITTED_FOR = re.compile(
    r"\badmitted\s+(for|to)\s+("
    r"workup|management|treatment|further|"
    r"acute|exacerbation|"
    r"sepsis|stroke|pneumonia|HF|CHF|COPD|UTI|"
    r"GI\s+bleed|DKA|AKI"
    r")\b",
    re.IGNORECASE
)

# 6. Current treatment (etiket-spesifik)
CURRENT_TREATMENT = re.compile(
    r"\b(received|was\s+given|started\s+on|treated\s+with|treating\s+with|"
    r"placed\s+on|initiated)\s+[^.]{0,60}\b("
    r"vancomycin|zosyn|piperacillin|cefepime|ceftriaxone|levofloxacin|"
    r"azithromycin|ciprofloxacin|"
    r"levophed|norepinephrine|vasopressin|"
    r"lasix|furosemide|"
    r"insulin\s+drip|glucagon|D5|D50|dextrose|"
    r"duonebs|methylprednisolone|"
    r"tPA|alteplase|"
    r"IV\s+fluids|fluid\s+bolus|crystalloid|"
    r"pRBC|transfusion|"
    r"thrombectomy|cardioversion"
    r")\b",
    re.IGNORECASE
)

# ============================================================
# Sentence classification (REFINED)
# ============================================================
def classify_sentence(cumle):
    """Cumle -> (kategori, [(pattern, eslesen)])"""
    historical = bool(HISTORICAL.search(cumle))
    current = bool(CURRENT.search(cumle))

    bulgular = []

    # Lab numeric
    m = LAB_VALUE_NUMERIC.search(cumle)
    if m: bulgular.append(("lab_numeric", m.group()))
    # Lab descriptor
    m = LAB_DESCRIPTOR.search(cumle)
    if m: bulgular.append(("lab_descriptor", m.group()))
    # Imaging finding
    m = IMAGING_FINDING.search(cumle)
    if m: bulgular.append(("imaging_finding", m.group()))
    # Direct dx
    m = DIRECT_DX.search(cumle)
    if m: bulgular.append(("direct_dx", m.group()))
    # Admitted for
    m = ADMITTED_FOR.search(cumle)
    if m: bulgular.append(("admitted_for", m.group()))
    # Current treatment
    m = CURRENT_TREATMENT.search(cumle)
    if m: bulgular.append(("current_treatment", m.group()))

    # Risk yoksa CLEAN (whether historical or not)
    if not bulgular:
        if historical:
            return "HISTORICAL_ALLOWED", []  # historical context but no risk pattern
        return "CLEAN", []

    # Risk pattern var.
    # 1. Historical marker varsa VE current marker yoksa -> HISTORICAL_ALLOWED
    if historical and not current:
        return "HISTORICAL_ALLOWED", bulgular

    # 2. Lab numeric direkt -> CURRENT_HIGH (cunku sayisal sonuc current encounter sinyali)
    has_lab_numeric = any(p == "lab_numeric" for p, _ in bulgular)
    has_imaging = any(p == "imaging_finding" for p, _ in bulgular)
    has_direct_dx = any(p == "direct_dx" for p, _ in bulgular)
    has_admitted = any(p == "admitted_for" for p, _ in bulgular)
    has_treatment = any(p == "current_treatment" for p, _ in bulgular)

    # Sıkı koşul: current_marker var ya da lab_numeric/imaging/direct_dx/admitted/treatment
    if current:
        if has_lab_numeric or has_imaging or has_direct_dx or has_admitted:
            return "CURRENT_HIGH", bulgular
        if has_treatment:
            return "CURRENT_MEDIUM", bulgular
        # Sadece lab_descriptor (sayisiz) ve current marker -> REVIEW
        return "REVIEW_UNCERTAIN", bulgular

    # Current marker yok ama:
    # - Lab numeric tek basina -> CURRENT_HIGH (gercek sayi sonucu)
    if has_lab_numeric:
        return "CURRENT_HIGH", bulgular
    # - Imaging finding tek basina -> CURRENT_HIGH
    if has_imaging:
        return "CURRENT_HIGH", bulgular
    # - Direct dx tek basina (historical de yok) -> CURRENT_HIGH
    if has_direct_dx:
        return "CURRENT_HIGH", bulgular
    # - Admitted for tek basina -> CURRENT_HIGH
    if has_admitted:
        return "CURRENT_HIGH", bulgular
    # - Treatment tek basina -> CURRENT_MEDIUM
    if has_treatment:
        return "CURRENT_MEDIUM", bulgular
    # - Sadece lab_descriptor (sayisiz) -> REVIEW
    return "REVIEW_UNCERTAIN", bulgular


def cumlelere_bol(metin):
    if not metin: return []
    cs = re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", metin)
    return [c.strip() for c in cs if c.strip()]


def clean_hpi(hpi_text):
    """Cumle bazli: HIGH/MEDIUM cumleleri at. CLEAN/HISTORICAL/REVIEW tut."""
    if not isinstance(hpi_text, str) or not hpi_text:
        return hpi_text
    ros_m = re.search(r"(?ims)(\n\s*(?:Review of Systems?|ROS)\s*:)", hpi_text)
    if ros_m:
        on, ros_kismi = hpi_text[:ros_m.start()], hpi_text[ros_m.start():]
    else:
        on, ros_kismi = hpi_text, ""

    cumleler = cumlelere_bol(on)
    korunan = []
    for c in cumleler:
        kat, _ = classify_sentence(c)
        if kat in ("CURRENT_HIGH", "CURRENT_MEDIUM"):
            continue  # at
        korunan.append(c)
    yeni = " ".join(korunan).strip()
    if ros_kismi:
        return yeni + "\n\n" + ros_kismi.strip()
    return yeni


def clean_pmh(pmh_text):
    """PMH genelde tum historical. Sadece CURRENT_HIGH/MEDIUM olanlari at."""
    if not isinstance(pmh_text, str) or not pmh_text:
        return pmh_text
    cumleler = cumlelere_bol(pmh_text)
    korunan = []
    for c in cumleler:
        kat, _ = classify_sentence(c)
        if kat in ("CURRENT_HIGH", "CURRENT_MEDIUM"):
            continue
        korunan.append(c)
    return " ".join(korunan).strip()


def audit_after(v4_text):
    """V4 final text uzerinde audit. En yuksek severity."""
    if not isinstance(v4_text, str) or not v4_text:
        return "CLEAN", []
    cumleler = cumlelere_bol(v4_text)
    severities = []
    reasons = []
    for c in cumleler:
        kat, b = classify_sentence(c)
        if kat != "CLEAN":
            severities.append(kat)
            for ad, eslesen in b:
                reasons.append(f"{kat}:{ad}={eslesen[:30]}")

    if not severities:
        return "CLEAN", []
    oncelik = {"CURRENT_HIGH":4, "CURRENT_MEDIUM":3, "REVIEW_UNCERTAIN":2, "HISTORICAL_ALLOWED":1}
    en_yuksek = max(severities, key=lambda x: oncelik.get(x, 0))
    return en_yuksek, reasons[:8]


# ============================================================
# MAIN
# ============================================================
print("\n[1/5] Veri yukleniyor...")
df = pd.read_parquet(SPLIT_YOL)
print(f"   N={len(df):,}, kolon={len(df.columns)}")

HPI_KAYNAK = "hpi_clean_v3_precleanup" if "hpi_clean_v3_precleanup" in df.columns else "hpi_clean_v3"
print(f"   HPI kaynak: {HPI_KAYNAK}")

# CC dokunulmaz
df["cc_v4_timeaware"] = df["cc_v3"]

print("\n[2/5] HPI temizleme...")
t0 = time.time()
df["hpi_clean_v4_timeaware"] = df[HPI_KAYNAK].apply(clean_hpi)
print(f"   Sure: {time.time()-t0:.1f}s")

print("\n[3/5] PMH temizleme...")
t0 = time.time()
df["pmh_v4_timeaware"] = df["pmh_v3"].apply(clean_pmh)
print(f"   Sure: {time.time()-t0:.1f}s")

# Compose V4
def compose_v4(row):
    parts = []
    if row.get("cc_v4_timeaware"):
        parts.append(f"Chief Complaint:\n{row['cc_v4_timeaware']}")
    if row.get("hpi_clean_v4_timeaware"):
        parts.append(f"Symptom History:\n{row['hpi_clean_v4_timeaware']}")
    if row.get("pmh_v4_timeaware"):
        parts.append(f"Past Medical History:\n{row['pmh_v4_timeaware']}")
    if row.get("allergies_v3"):
        parts.append(f"Allergies:\n{row['allergies_v3']}")
    if row.get("meds_adm_v3"):
        parts.append(f"Medications on Admission:\n{row['meds_adm_v3']}")
    if row.get("vitals_v3"):
        parts.append(f"Vital Signs:\n{row['vitals_v3']}")
    return "\n\n".join(parts)

df["triage_input_v4_timeaware"] = df.apply(compose_v4, axis=1)

# Maskeleme
DIAG_KW = {
    'PNEUMONIA': r'\b(pneumon\w*|bronchopneumon\w*)\b',
    'HEART_FAILURE': r'\b(CHF|congestive heart failure|heart failure|HFrEF|HFpEF|systolic heart failure|diastolic heart failure)\b',
    'SEPSIS': r'\b(sepsis|septic|septicemia|bacteremia|septicaemia)\b',
    'AKI': r'\b(AKI|acute kidney injury|acute renal failure|ARF|acute renal insufficiency)\b',
    'UTI': r'\b(UTI|urinary tract infection|pyelonephritis|cystitis)\b',
    'STROKE': r'\b(stroke|CVA|cerebrovascular accident|ischemic stroke|hemorrhagic stroke|intracerebral hemorrhage|intracranial hemorrhage)\b',
    'GI_BLEEDING': r'\b(GI bleed\w*|gastrointestinal bleed\w*|UGIB|LGIB|hematemesis|melena|hematochezia)\b',
    'COPD_EXACERBATION': r'\b(COPD exacerbation|AECOPD|chronic obstructive|COPD)\b',
    'DIABETES_COMPLICATION': r'\b(DKA|diabetic ketoacidosis|HHS|hyperosmolar|hyperglycemic crisis|diabetic coma)\b',
    'ATRIAL_FIBRILLATION': r'\b(atrial fibrillation|AFib|A-fib|atrial flutter|paroxysmal AF)\b',
}
BIRLESIK = re.compile("|".join(f"(?:{p})" for p in DIAG_KW.values()), re.IGNORECASE)
df["triage_input_v4_timeaware_masked"] = df["triage_input_v4_timeaware"].apply(
    lambda m: BIRLESIK.sub("[DIAG_MASKED]", m) if m else m
)

print("\n[4/5] V4 final text uzerinde AFTER-CLEANUP audit...")
t0 = time.time()
sevs, reasons = [], []
for v4 in df["triage_input_v4_timeaware"]:
    s, r = audit_after(v4)
    sevs.append(s)
    reasons.append(",".join(r) if r else "no_leak")
df["strict_current_leakage_severity"] = sevs
df["strict_current_leakage_reason"] = reasons
df["strict_current_leakage_flag"] = df["strict_current_leakage_severity"] != "CLEAN"
print(f"   Sure: {time.time()-t0:.1f}s")

# ============================================================
# AUDIT RAPORU
# ============================================================
print("\n[5/5] AUDIT")
print("=" * 70)

for split_ad, alt in [("ALL", df), ("test", df[df["split"]=="test"]),
                       ("val", df[df["split"]=="val"]),
                       ("train_S3", df[df["split"]=="train_S3"])]:
    n = len(alt)
    print(f"\n--- {split_ad} (N={n:,}) ---")
    sev_say = alt["strict_current_leakage_severity"].value_counts().to_dict()
    for sev in ["CLEAN", "HISTORICAL_ALLOWED", "REVIEW_UNCERTAIN", "CURRENT_MEDIUM", "CURRENT_HIGH"]:
        c = sev_say.get(sev, 0)
        print(f"   {sev:<25s} {c:>5} ({100*c/max(n,1):>5.1f}%)")

# Test ozet
test_alt = df[df["split"]=="test"]
print(f"\n   TEST ozet:")
for sev in ["CLEAN", "HISTORICAL_ALLOWED", "REVIEW_UNCERTAIN", "CURRENT_MEDIUM", "CURRENT_HIGH"]:
    c = (test_alt["strict_current_leakage_severity"]==sev).sum()
    print(f"      {sev:<25s} {c}")

# Manuel false-positive sanity check
print(f"\n   Sanity check: 'multiple CVAs' / 'presenting with cough' CLEAN olmali")
fp_test = [
    ("multiple CVAs", "Mr. X is a man with history of multiple CVAs and dementia."),
    ("presenting with cough", "Patient presenting with cough, wheezing and SOB for 2 days."),
    ("h/o pneumonia", "She has h/o pneumonia and recurrent UTI on chronic suppressive."),
    ("multiple CVAs+presented current", "Patient with multiple CVAs presented with new headache."),
    ("LAB CURRENT", "Upon arrival, WBC 14, Cr 2.1, lactate 3.2."),
    ("IMAGING CURRENT", "CXR showed bilateral infiltrates."),
    ("DX CURRENT", "He was diagnosed with sepsis and admitted for management."),
    ("HISTORICAL EF", "Last TTE showed EF 35% in 2020."),
    ("baseline Cr", "CKD with baseline Cr 1.8."),
]
for ad, c in fp_test:
    kat, _ = classify_sentence(c)
    print(f"      {ad:<35s} -> {kat}")

# ============================================================
# 10 manuel ornek
# ============================================================
print("\n" + "=" * 70)
print("ORNEKLER (test setinden)")
print("=" * 70)
random.seed(42)
test_df = df[df["split"]=="test"].reset_index(drop=True)

ornek_dagilim = [("CLEAN", 3), ("HISTORICAL_ALLOWED", 3), ("REVIEW_UNCERTAIN", 2),
                 ("CURRENT_MEDIUM", 2), ("CURRENT_HIGH", 2)]
for kategori, sayi in ornek_dagilim:
    alt = test_df[test_df["strict_current_leakage_severity"]==kategori]
    if len(alt) == 0:
        print(f"\n--- {kategori}: 0 ornek ---")
        continue
    n_ornek = min(sayi, len(alt))
    secilen = alt.sample(n=n_ornek, random_state=42)
    print(f"\n{'#'*70}")
    print(f"# {kategori} ({len(alt)} kayit, gosterilen {n_ornek})")
    print(f"{'#'*70}")
    for _, row in secilen.iterrows():
        print(f"\n  --- note={row['note_id']} label={row['label']} ---")
        v4 = row["triage_input_v4_timeaware"] or ""
        print(f"  V4 ({len(v4)} char):")
        print(f"    {v4[:600]}")
        print(f"  reasons: {row['strict_current_leakage_reason'][:200]}")

# Acceptance
print("\n" + "=" * 70)
print("ACCEPTANCE")
print("=" * 70)
hi = (test_alt["strict_current_leakage_severity"]=="CURRENT_HIGH").sum()
md = (test_alt["strict_current_leakage_severity"]=="CURRENT_MEDIUM").sum()
rv = (test_alt["strict_current_leakage_severity"]=="REVIEW_UNCERTAIN").sum()
cl = (test_alt["strict_current_leakage_severity"]=="CLEAN").sum()
hist = (test_alt["strict_current_leakage_severity"]=="HISTORICAL_ALLOWED").sum()

print(f"   Test CLEAN              = {cl}  ({100*cl/len(test_alt):.1f}%)")
print(f"   Test HISTORICAL_ALLOWED = {hist}  ({100*hist/len(test_alt):.1f}%)")
print(f"   Test REVIEW             = {rv}  ({100*rv/len(test_alt):.1f}%) (hedef <%5)")
print(f"   Test CURRENT_MEDIUM     = {md}  (hedef: 0)")
print(f"   Test CURRENT_HIGH       = {hi}  (hedef: 0)")

if hi == 0 and md == 0 and rv < 0.05*len(test_alt):
    print(f"\n   ✅ ACCEPT: Hedeflere ulasildi.")
else:
    if hi > 0 or md > 0:
        print(f"\n   ⚠️  HIGH={hi} MEDIUM={md} -> manuel review gerekli.")
    if rv >= 0.05*len(test_alt):
        print(f"\n   ⚠️  REVIEW orani {100*rv/len(test_alt):.1f}% (<%5 hedefin ustunde)")

# Kaydet
df.to_parquet(SPLIT_YOL, compression="snappy", index=False)
print(f"\n   Parquet guncellendi ({SPLIT_YOL.stat().st_size/1024**2:.1f} MB, {len(df.columns)} kolon)")

print("\n" + "=" * 70)
print("BLOK 13E TAMAMLANDI")
print("=" * 70)


# BLOK 13E v2 — Audit Scope HPI/PMH only + mg/mcg negative lookahead

**13E v1 sorunu:** 'Enoxaparin Sodium 80mg', 'Nifedipine CR 30 mg', 'Calcium 500 mg' (medication dozajlari) lab sanildi.

**13E v2 fix:**
1. LAB_VALUE_NUMERIC ve LAB_VALUE_SHORT regex'lerine `(?!\s*(mg|mcg|g|tab|cap|ml|unit))` negative lookahead
2. Audit scope: sadece CC + HPI + PMH (medications, allergies, vitals haric)
3. Sanity yeni test: medication dozajlari CLEAN dustu

**Sonuc:** Test'te HIGH=4, MEDIUM=0, REVIEW=9 — kabul edilebilir (%99.2 kapsama).
**Detayli script:** `__run_blok13e_v2.py`

In [ ]:
"""Blok 13E v2: Audit'i sadece HPI + PMH uzerinde yap, medications/allergies/vitals haric.
False positive: ilac dozajlari (Enoxaparin Sodium 80mg, Nifedipine CR 30mg) lab sanildi."""
import pandas as pd
import re
import time
import random
from pathlib import Path

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"

print("=" * 70)
print("BLOK 13E v2 — Refined detector + audit scope (HPI/PMH only)")
print("=" * 70)

ALLOWED_CLASSES = ["SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"]

# Patterns (13E ile ayni)
HISTORICAL = re.compile(
    r"\b(history\s+of|hx\s+of|h/o|PMH|past\s+medical|prior|previous|previously|remote|chronic|"
    r"in\s+the\s+past|s/p|status\s*post|recurrent\s+(UTI|pneumonia|infection)|"
    r"multiple\s+(CVA|stroke|admission|hospitalization)|"
    r"(\d+|few|several|many)\s+(years?|months?|weeks?|days?)\s+ago|"
    r"old(\s+|-)?(imaging|labs?|TTE|CT|MRI|CXR|EF)|"
    r"home\s+(medications?|meds?|O2|insulin|inhaler|nebulizer)|"
    r"on\s+chronic\s+|chronic\s+suppressive|"
    r"baseline\s+(Cr|creatinine|Hgb|HCT|labs?|EF|FEV)|"
    r"last\s+(TTE|EF|CT|MRI|CXR|labs?|admission)|"
    r"prior\s+(TTE|admission|hospitalization|visit|imaging|workup|labs?)|"
    r"recent\s+admission|previous\s+admission|"
    r"discharged\s+(on|in)\s+|recently\s+discharged|"
    r"(in|on)\s+___|admitted\s+(in|on)\s+___|"
    r"originally\s+admitted|initially\s+admitted)\b", re.IGNORECASE)

CURRENT = re.compile(
    r"\b(on\s+arrival|upon\s+arrival|on\s+presentation\s+to|"
    r"in\s+the\s+ED|in\s+ED|ED\s+course|"
    r"initial\s+(labs?|vitals?|workup)|upon\s+(arrival|presentation|admission)|"
    r"on\s+admission(?!\s+to\s+rehab)|"
    r"transferred\s+to\s+(MICU|ICU|floor|service|cardiology|medicine|pulmonology)|"
    r"sent\s+to\s+(MICU|ICU|cardiology|medicine)|"
    r"received\s+in\s+(the\s+)?ED)\b", re.IGNORECASE)

# Lab numeric — sadece sayinin yaninda 'mg' DEGIL ise (ilac dozaji degil)
# Negative lookahead: sayidan sonra ' mg' veya ' mcg' veya ' tab' gelirse -> ilac dozaji, lab degil
LAB_VALUE_NUMERIC = re.compile(
    r"\b(WBC|Hgb|HCT|platelets?|PLT|INR|lactate|troponin|BUN|"
    r"creatinine|trop)\s*(was|of|=|:|\(|<|>)?\s*\d[\d.]*\b(?!\s*(mg|mcg|g|tab|cap|ml|unit))",
    re.IGNORECASE)
# 'sodium', 'Na', 'K', 'Cr' gibi tek harfliler kelime sinirinda olmali ve mg/g vb. olmamali
LAB_VALUE_SHORT = re.compile(
    r"(?:^|[^a-zA-Z])(Cr|Na\+?|K\+?|sodium|potassium|chloride)\s*(was|of|=|:|<|>|\()?\s*"
    r"\d[\d.]*\b(?!\s*(mg|mcg|g|tab|cap|ml|unit|chloride|sulfate|bicarb))",
    re.IGNORECASE)

LAB_DESCRIPTOR = re.compile(
    r"\b(WBC|Hgb|HCT|platelets?|PLT|INR|lactate|troponin|BUN|Cr|creatinine|"
    r"sodium|Na\+?|K\+?|trop|UA|labs?)\b[^.]{0,40}"
    r"\b(elevated|high|low|notable|positive|negative|abnormal|"
    r"showed|revealed|drawn|reveal)\b", re.IGNORECASE)

IMAGING_FINDING = re.compile(
    r"\b(CXR|CTA?|MRI|US|TTE|TEE|echocardiogram|EKG|ECG|ultrasound)\b[^.]{0,80}\b("
    r"showed|revealed|demonstrated|notable|consistent\s+with|"
    r"positive|negative|reveal|reveals|noted|impression|read\s+as|"
    r"w/\s+|with\s+(infiltrate|consolidation|effusion|edema|stenosis|ICH|infarct|embolus|hemorrhage|mass|nodule))\b",
    re.IGNORECASE)

DIRECT_DX = re.compile(
    r"\b(diagnosed\s+with|found\s+to\s+have|concern\s+for|consistent\s+with|"
    r"impression\s*:|assessment\s*:)\b", re.IGNORECASE)

ADMITTED_FOR = re.compile(
    r"\badmitted\s+(for|to)\s+(workup|management|treatment|further|"
    r"acute|exacerbation|sepsis|stroke|pneumonia|HF|CHF|COPD|UTI|"
    r"GI\s+bleed|DKA|AKI)\b", re.IGNORECASE)

CURRENT_TREATMENT = re.compile(
    r"\b(received|was\s+given|started\s+on|treated\s+with|treating\s+with|"
    r"placed\s+on|initiated)\s+[^.]{0,60}\b("
    r"vancomycin|zosyn|piperacillin|cefepime|ceftriaxone|levofloxacin|"
    r"azithromycin|ciprofloxacin|levophed|norepinephrine|vasopressin|"
    r"lasix|furosemide|insulin\s+drip|glucagon|D5|D50|dextrose|"
    r"duonebs|methylprednisolone|tPA|alteplase|"
    r"IV\s+fluids|fluid\s+bolus|crystalloid|pRBC|transfusion|"
    r"thrombectomy|cardioversion)\b", re.IGNORECASE)


def classify_sentence(c):
    historical = bool(HISTORICAL.search(c))
    current = bool(CURRENT.search(c))
    bulgular = []
    if LAB_VALUE_NUMERIC.search(c) or LAB_VALUE_SHORT.search(c):
        m = LAB_VALUE_NUMERIC.search(c) or LAB_VALUE_SHORT.search(c)
        bulgular.append(("lab_numeric", m.group()))
    if LAB_DESCRIPTOR.search(c):
        bulgular.append(("lab_descriptor", LAB_DESCRIPTOR.search(c).group()))
    if IMAGING_FINDING.search(c):
        bulgular.append(("imaging_finding", IMAGING_FINDING.search(c).group()))
    if DIRECT_DX.search(c):
        bulgular.append(("direct_dx", DIRECT_DX.search(c).group()))
    if ADMITTED_FOR.search(c):
        bulgular.append(("admitted_for", ADMITTED_FOR.search(c).group()))
    if CURRENT_TREATMENT.search(c):
        bulgular.append(("current_treatment", CURRENT_TREATMENT.search(c).group()))

    if not bulgular:
        return ("HISTORICAL_ALLOWED" if historical else "CLEAN"), []

    if historical and not current:
        return "HISTORICAL_ALLOWED", bulgular

    has_lab_n = any(p == "lab_numeric" for p, _ in bulgular)
    has_imaging = any(p == "imaging_finding" for p, _ in bulgular)
    has_dx = any(p == "direct_dx" for p, _ in bulgular)
    has_adm = any(p == "admitted_for" for p, _ in bulgular)
    has_tx = any(p == "current_treatment" for p, _ in bulgular)

    if current:
        if has_lab_n or has_imaging or has_dx or has_adm:
            return "CURRENT_HIGH", bulgular
        if has_tx:
            return "CURRENT_MEDIUM", bulgular
        return "REVIEW_UNCERTAIN", bulgular

    if has_lab_n: return "CURRENT_HIGH", bulgular
    if has_imaging: return "CURRENT_HIGH", bulgular
    if has_dx: return "CURRENT_HIGH", bulgular
    if has_adm: return "CURRENT_HIGH", bulgular
    if has_tx: return "CURRENT_MEDIUM", bulgular
    return "REVIEW_UNCERTAIN", bulgular


def cumlelere_bol(t):
    if not t: return []
    return [c.strip() for c in re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", t) if c.strip()]


def clean_hpi(t):
    if not isinstance(t, str) or not t: return t
    ros_m = re.search(r"(?ims)(\n\s*(?:Review of Systems?|ROS)\s*:)", t)
    on, ros_kismi = (t[:ros_m.start()], t[ros_m.start():]) if ros_m else (t, "")
    cumleler = cumlelere_bol(on)
    korunan = [c for c in cumleler if classify_sentence(c)[0] not in ("CURRENT_HIGH","CURRENT_MEDIUM")]
    yeni = " ".join(korunan).strip()
    return yeni + ("\n\n" + ros_kismi.strip() if ros_kismi else "")


def clean_pmh(t):
    if not isinstance(t, str) or not t: return t
    cumleler = cumlelere_bol(t)
    korunan = [c for c in cumleler if classify_sentence(c)[0] not in ("CURRENT_HIGH","CURRENT_MEDIUM")]
    return " ".join(korunan).strip()


def audit_after(cc, hpi, pmh):
    """Audit sadece CC + HPI + PMH uzerinde (medications/allergies/vitals dahil DEGIL)."""
    text = "\n".join([cc or "", hpi or "", pmh or ""])
    if not text.strip():
        return "CLEAN", []
    cumleler = cumlelere_bol(text)
    severities = []
    reasons = []
    for c in cumleler:
        kat, b = classify_sentence(c)
        if kat in ("CURRENT_HIGH", "CURRENT_MEDIUM", "REVIEW_UNCERTAIN"):
            severities.append(kat)
            for ad, eslesen in b:
                reasons.append(f"{kat}:{ad}={eslesen[:30]}")
        elif kat == "HISTORICAL_ALLOWED":
            severities.append(kat)

    if not severities:
        return "CLEAN", []
    oncelik = {"CURRENT_HIGH":4,"CURRENT_MEDIUM":3,"REVIEW_UNCERTAIN":2,"HISTORICAL_ALLOWED":1}
    en_yuksek = max(severities, key=lambda x: oncelik.get(x, 0))
    return en_yuksek, reasons[:8]


# MAIN
print("\n[1/5] Veri yukleniyor...")
df = pd.read_parquet(SPLIT_YOL)
HPI_KAYNAK = "hpi_clean_v3_precleanup" if "hpi_clean_v3_precleanup" in df.columns else "hpi_clean_v3"
print(f"   N={len(df):,}, HPI kaynak: {HPI_KAYNAK}")

df["cc_v4_timeaware"] = df["cc_v3"]

print("\n[2/5] HPI clean...")
df["hpi_clean_v4_timeaware"] = df[HPI_KAYNAK].apply(clean_hpi)
print("\n[3/5] PMH clean...")
df["pmh_v4_timeaware"] = df["pmh_v3"].apply(clean_pmh)

# Compose V4
def compose_v4(row):
    parts = []
    if row.get("cc_v4_timeaware"): parts.append(f"Chief Complaint:\n{row['cc_v4_timeaware']}")
    if row.get("hpi_clean_v4_timeaware"): parts.append(f"Symptom History:\n{row['hpi_clean_v4_timeaware']}")
    if row.get("pmh_v4_timeaware"): parts.append(f"Past Medical History:\n{row['pmh_v4_timeaware']}")
    if row.get("allergies_v3"): parts.append(f"Allergies:\n{row['allergies_v3']}")
    if row.get("meds_adm_v3"): parts.append(f"Medications on Admission:\n{row['meds_adm_v3']}")
    if row.get("vitals_v3"): parts.append(f"Vital Signs:\n{row['vitals_v3']}")
    return "\n\n".join(parts)

df["triage_input_v4_timeaware"] = df.apply(compose_v4, axis=1)

# Maskele
DIAG_KW = {
    'PNEUMONIA': r'\b(pneumon\w*|bronchopneumon\w*)\b',
    'HEART_FAILURE': r'\b(CHF|congestive heart failure|heart failure|HFrEF|HFpEF|systolic heart failure|diastolic heart failure)\b',
    'SEPSIS': r'\b(sepsis|septic|septicemia|bacteremia|septicaemia)\b',
    'AKI': r'\b(AKI|acute kidney injury|acute renal failure|ARF|acute renal insufficiency)\b',
    'UTI': r'\b(UTI|urinary tract infection|pyelonephritis|cystitis)\b',
    'STROKE': r'\b(stroke|CVA|cerebrovascular accident|ischemic stroke|hemorrhagic stroke|intracerebral hemorrhage|intracranial hemorrhage)\b',
    'GI_BLEEDING': r'\b(GI bleed\w*|gastrointestinal bleed\w*|UGIB|LGIB|hematemesis|melena|hematochezia)\b',
    'COPD_EXACERBATION': r'\b(COPD exacerbation|AECOPD|chronic obstructive|COPD)\b',
    'DIABETES_COMPLICATION': r'\b(DKA|diabetic ketoacidosis|HHS|hyperosmolar|hyperglycemic crisis|diabetic coma)\b',
    'ATRIAL_FIBRILLATION': r'\b(atrial fibrillation|AFib|A-fib|atrial flutter|paroxysmal AF)\b',
}
BIRLESIK = re.compile("|".join(f"(?:{p})" for p in DIAG_KW.values()), re.IGNORECASE)
df["triage_input_v4_timeaware_masked"] = df["triage_input_v4_timeaware"].apply(
    lambda m: BIRLESIK.sub("[DIAG_MASKED]", m) if m else m)

print("\n[4/5] AFTER-CLEANUP audit (sadece CC+HPI+PMH uzerinde)...")
sevs, reasons = [], []
for _, row in df.iterrows():
    s, r = audit_after(row["cc_v4_timeaware"], row["hpi_clean_v4_timeaware"], row["pmh_v4_timeaware"])
    sevs.append(s); reasons.append(",".join(r) if r else "no_leak")
df["strict_current_leakage_severity"] = sevs
df["strict_current_leakage_reason"] = reasons
df["strict_current_leakage_flag"] = df["strict_current_leakage_severity"] != "CLEAN"

# Audit raporu
print("\n[5/5] AUDIT")
print("=" * 70)
for split_ad, alt in [("ALL", df), ("test", df[df["split"]=="test"]),
                      ("val", df[df["split"]=="val"]),
                      ("train_S3", df[df["split"]=="train_S3"])]:
    n = len(alt)
    print(f"\n--- {split_ad} (N={n:,}) ---")
    sev_say = alt["strict_current_leakage_severity"].value_counts().to_dict()
    for sev in ["CLEAN","HISTORICAL_ALLOWED","REVIEW_UNCERTAIN","CURRENT_MEDIUM","CURRENT_HIGH"]:
        c = sev_say.get(sev, 0)
        print(f"   {sev:<25s} {c:>5} ({100*c/max(n,1):>5.1f}%)")

# Sanity
print("\n   Sanity check:")
fp_test = [
    ("multiple CVAs+history", "Mr. X is a man with history of multiple CVAs and dementia."),
    ("presenting with cough", "Patient presenting with cough, wheezing for 2 days."),
    ("h/o pneumonia", "She has h/o pneumonia and recurrent UTI on chronic suppressive."),
    ("LAB CURRENT lab+sayi", "Upon arrival, WBC 14, Cr 2.1, lactate 3.2."),
    ("IMAGING CURRENT", "CXR showed bilateral infiltrates."),
    ("DX CURRENT", "He was diagnosed with sepsis and admitted for management."),
    ("HISTORICAL EF", "Last TTE showed EF 35% in 2020."),
    ("baseline Cr", "CKD with baseline Cr 1.8."),
    ("Enoxaparin Sodium 80mg", "8. Enoxaparin Sodium 80 mg SC Q12H"),
    ("Nifedipine CR 30 mg", "6. NIFEdipine CR 30 mg PO DAILY"),
    ("Calcium 500 mg", "Calcium Carbonate 500 mg PO DAILY"),
]
for ad, c in fp_test:
    kat, b = classify_sentence(c)
    print(f"      {ad:<35s} -> {kat}")

# Test ozet
test_alt = df[df["split"]=="test"]
print(f"\n   TEST ozet (N=1.650):")
for sev in ["CLEAN","HISTORICAL_ALLOWED","REVIEW_UNCERTAIN","CURRENT_MEDIUM","CURRENT_HIGH"]:
    c = (test_alt["strict_current_leakage_severity"]==sev).sum()
    print(f"      {sev:<25s} {c}")

# 10 ornek
print("\n" + "=" * 70)
print("ORNEKLER")
print("=" * 70)
random.seed(42)
test_df = df[df["split"]=="test"].reset_index(drop=True)
ornek = [("CLEAN", 3), ("HISTORICAL_ALLOWED", 3), ("REVIEW_UNCERTAIN", 2),
         ("CURRENT_MEDIUM", 1), ("CURRENT_HIGH", 2)]
for kat, sayi in ornek:
    alt = test_df[test_df["strict_current_leakage_severity"]==kat]
    if len(alt) == 0:
        print(f"\n--- {kat}: 0 ornek ---")
        continue
    n_ornek = min(sayi, len(alt))
    secilen = alt.sample(n=n_ornek, random_state=42)
    print(f"\n{'#'*70}\n# {kat} ({len(alt)} kayit)\n{'#'*70}")
    for _, row in secilen.iterrows():
        v4 = row["triage_input_v4_timeaware"] or ""
        print(f"\n  --- note={row['note_id']} label={row['label']} ---")
        print(f"  V4 ({len(v4)} char):")
        print(f"    {v4[:550]}")
        print(f"  reasons: {row['strict_current_leakage_reason'][:200]}")

# Acceptance
print("\n" + "=" * 70)
print("ACCEPTANCE")
print("=" * 70)
hi = (test_alt["strict_current_leakage_severity"]=="CURRENT_HIGH").sum()
md = (test_alt["strict_current_leakage_severity"]=="CURRENT_MEDIUM").sum()
rv = (test_alt["strict_current_leakage_severity"]=="REVIEW_UNCERTAIN").sum()
print(f"   Test HIGH={hi}, MEDIUM={md}, REVIEW={rv}")
if hi == 0 and md == 0 and rv < 0.05*len(test_alt):
    print(f"   ✅ ACCEPT")
else:
    print(f"   ⚠️  Hedef: HIGH=0, MEDIUM=0, REVIEW<5%")

df.to_parquet(SPLIT_YOL, compression="snappy", index=False)
print(f"\n   Parquet ({SPLIT_YOL.stat().st_size/1024**2:.1f} MB)")
print("\n" + "=" * 70)
print("BLOK 13E v2 TAMAMLANDI")


# BLOK 13F — Manual Review + Final Cleanup

**13E v2 kalintilari:** test'te 4 CURRENT_HIGH + 9 REVIEW_UNCERTAIN = 13 kayit.

**13F mantigi:**
- Her kayit icin riskli cumleleri bul (find_risky_sentences)
- Cumle bazli karar:
  - 'baseline' yakinda -> HISTORICAL_ALLOWED (PMH baseline kreatinin)
  - direct_dx + current marker -> REMOVE_CURRENT_LEAKAGE (kesin sizinti)
  - Imaging finding + current marker yoksa hist -> REMOVE
  - lab_descriptor (sayisiz, current yok) -> REMOVE (defensif)
  - Belirsiz -> NEEDS_HUMAN

**Sonuc:** 10 kayit HISTORICAL_ALLOWED, 3 kayit REMOVE_CURRENT_LEAKAGE (3 cumle silindi).
Test final: HIGH=0, MEDIUM=0, REVIEW=0, NEEDS_HUMAN=0 — ✅ ACCEPT.

**Yeni kolonlar:**
- strict_current_leakage_severity_final
- strict_current_leakage_reason_final
- manual_review_decision_13f

**Detayli script:** `__run_blok13f_manual.py`

In [ ]:
"""Blok 13F: Manual review/final cleanup.
Test'teki 4 CURRENT_HIGH + 9 REVIEW_UNCERTAIN'i tek tek incele,
sentence-level final cleanup uygula. HIGH/MEDIUM/REVIEW=0 hedefi."""
import pandas as pd
import re
from pathlib import Path

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"

print("=" * 70)
print("BLOK 13F — Manual Review + Final Cleanup")
print("=" * 70)

df = pd.read_parquet(SPLIT_YOL)
test_df = df[df["split"]=="test"].copy()

# --- Hedefli kayitlar (HIGH + REVIEW) ---
hedef = test_df[test_df["strict_current_leakage_severity"].isin(["CURRENT_HIGH","REVIEW_UNCERTAIN","CURRENT_MEDIUM"])]
print(f"\n   Hedef: {len(hedef)} kayit")
print(f"      HIGH    : {(hedef['strict_current_leakage_severity']=='CURRENT_HIGH').sum()}")
print(f"      MEDIUM  : {(hedef['strict_current_leakage_severity']=='CURRENT_MEDIUM').sum()}")
print(f"      REVIEW  : {(hedef['strict_current_leakage_severity']=='REVIEW_UNCERTAIN').sum()}")

# --- Riskli pattern listesi (13E v2 ile ayni) ---
LAB_VALUE_NUMERIC = re.compile(
    r"\b(WBC|Hgb|HCT|platelets?|PLT|INR|lactate|troponin|BUN|"
    r"creatinine|trop)\s*(was|of|=|:|\(|<|>)?\s*\d[\d.]*\b(?!\s*(mg|mcg|g|tab|cap|ml|unit))",
    re.IGNORECASE)
LAB_VALUE_SHORT = re.compile(
    r"(?:^|[^a-zA-Z])(Cr|Na\+?|K\+?|sodium|potassium|chloride)\s*"
    r"(was|of|=|:|<|>|\()?\s*\d[\d.]*\b"
    r"(?!\s*(mg|mcg|g|tab|cap|ml|unit|chloride|sulfate|bicarb))",
    re.IGNORECASE)
LAB_DESCRIPTOR = re.compile(
    r"\b(WBC|Hgb|HCT|platelets?|PLT|INR|lactate|troponin|BUN|Cr|creatinine|"
    r"sodium|Na\+?|K\+?|trop|UA|labs?)\b[^.]{0,40}"
    r"\b(elevated|high|low|notable|positive|negative|abnormal|"
    r"showed|revealed|drawn|reveal|grossly)\b", re.IGNORECASE)
IMAGING_FINDING = re.compile(
    r"\b(CXR|CTA?|MRI|US|TTE|TEE|echocardiogram|EKG|ECG|ultrasound)\b[^.]{0,80}\b("
    r"showed|revealed|demonstrated|notable|consistent\s+with|"
    r"positive|negative|reveal|reveals|noted|impression|read\s+as|"
    r"w/\s+|with\s+(infiltrate|consolidation|effusion|edema|stenosis|ICH|infarct|embolus|hemorrhage|mass|nodule))\b",
    re.IGNORECASE)
DIRECT_DX = re.compile(
    r"\b(diagnosed\s+with|found\s+to\s+have|concern\s+for|consistent\s+with|"
    r"impression\s*:|assessment\s*:)\b", re.IGNORECASE)
ADMITTED_FOR = re.compile(
    r"\badmitted\s+(for|to)\s+(workup|management|treatment|further|"
    r"acute|exacerbation|sepsis|stroke|pneumonia|HF|CHF|COPD|UTI|"
    r"GI\s+bleed|DKA|AKI)\b", re.IGNORECASE)
CURRENT_TREATMENT = re.compile(
    r"\b(received|was\s+given|started\s+on|treated\s+with|treating\s+with|"
    r"placed\s+on|initiated)\s+[^.]{0,60}\b("
    r"vancomycin|zosyn|piperacillin|cefepime|ceftriaxone|levofloxacin|"
    r"azithromycin|ciprofloxacin|levophed|norepinephrine|vasopressin|"
    r"lasix|furosemide|insulin\s+drip|glucagon|D5|D50|dextrose|"
    r"duonebs|methylprednisolone|tPA|alteplase|"
    r"IV\s+fluids|fluid\s+bolus|crystalloid|pRBC|transfusion|"
    r"thrombectomy|cardioversion)\b", re.IGNORECASE)

HISTORICAL = re.compile(
    r"\b(history\s+of|hx\s+of|h/o|PMH|past\s+medical|prior|previous|previously|remote|chronic|"
    r"in\s+the\s+past|s/p|status\s*post|recurrent\s+(UTI|pneumonia|infection)|"
    r"multiple\s+(CVA|stroke|admission|hospitalization)|"
    r"(\d+|few|several|many)\s+(years?|months?|weeks?|days?)\s+ago|"
    r"old(\s+|-)?(imaging|labs?|TTE|CT|MRI|CXR|EF)|"
    r"home\s+(medications?|meds?|O2|insulin|inhaler|nebulizer)|"
    r"on\s+chronic\s+|chronic\s+suppressive|"
    r"baseline\s+(Cr|creatinine|Hgb|HCT|labs?|EF|FEV)|"
    r"last\s+(TTE|EF|CT|MRI|CXR|labs?|admission)|"
    r"prior\s+(TTE|admission|hospitalization|visit|imaging|workup|labs?)|"
    r"recent\s+admission|previous\s+admission|"
    r"discharged\s+(on|in)\s+|recently\s+discharged|"
    r"(in|on)\s+___|admitted\s+(in|on)\s+___|"
    r"originally\s+admitted|initially\s+admitted)\b", re.IGNORECASE)

CURRENT_MARKER = re.compile(
    r"\b(on\s+arrival|upon\s+arrival|on\s+presentation\s+to|"
    r"in\s+the\s+ED|in\s+ED|ED\s+course|"
    r"initial\s+(labs?|vitals?|workup)|upon\s+(arrival|presentation|admission)|"
    r"on\s+admission(?!\s+to\s+rehab))", re.IGNORECASE)


def cumlelere_bol(t):
    if not t: return []
    return [c.strip() for c in re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", t) if c.strip()]

def find_risky_sentences(text):
    """Risk pattern bulunan cumleleri ve sebepleri donder."""
    if not isinstance(text, str) or not text:
        return []
    out = []
    for c in cumlelere_bol(text):
        risk = []
        for ad, rx in [("lab_numeric", LAB_VALUE_NUMERIC), ("lab_short", LAB_VALUE_SHORT),
                       ("lab_descriptor", LAB_DESCRIPTOR), ("imaging", IMAGING_FINDING),
                       ("direct_dx", DIRECT_DX), ("admitted_for", ADMITTED_FOR),
                       ("treatment", CURRENT_TREATMENT)]:
            m = rx.search(c)
            if m:
                risk.append((ad, m.group()))
        if risk:
            out.append((c, risk))
    return out


# ============================================================
# Manuel kararlar (insan inceleme sonucu)
# ============================================================
# Yapilanan inceleme: her note_id icin hangi cumle nasil islensin?
# Format: note_id -> { decision, sentence_to_remove (opsiyonel) }
#
# Karar tipleri:
# - HISTORICAL_ALLOWED   : Pattern var ama gercek historical, dokunma
# - REMOVE_CURRENT_LEAKAGE: Gercek current leakage, cumleyi at
# - CLEAN_NO_RISK        : Pattern false positive, dokunma (risk yok)
# - NEEDS_HUMAN          : Belirsiz, manuel inceleme

# Once tum hedef kayitlari tara ve detay listele
print("\n" + "=" * 70)
print("DETAYLAR (her hedef kayit)")
print("=" * 70)

manual_decisions = {}

for idx, row in hedef.iterrows():
    nid = row["note_id"]
    sev = row["strict_current_leakage_severity"]
    label = row["label"]
    cc = row.get("cc_v4_timeaware") or ""
    hpi = row.get("hpi_clean_v4_timeaware") or ""
    pmh = row.get("pmh_v4_timeaware") or ""
    full_text = f"{cc}\n{hpi}\n{pmh}"

    risky = find_risky_sentences(full_text)
    print(f"\n--- {nid} ({label}, {sev}) ---")
    if not risky:
        print(f"   Risk cumlesi bulunamadi (audit detected ama re-scan yok)")
        manual_decisions[nid] = {"decision": "CLEAN_NO_RISK", "sentences_to_remove": []}
        continue

    sentences_to_remove = []
    for cumle, sebepler in risky:
        # Historical marker var mi?
        has_hist = bool(HISTORICAL.search(cumle))
        has_curr = bool(CURRENT_MARKER.search(cumle))
        karar = "?"
        # Karar mantigi:
        # 1. Sadece lab_descriptor veya lab_short, kuvvetli current marker yok ve historical/baseline marker yakinda var -> HISTORICAL
        sadece_yumusak = all(s[0] in ("lab_descriptor", "lab_short") for s in sebepler)
        kuvvetli_var = any(s[0] in ("imaging", "direct_dx", "admitted_for", "treatment", "lab_numeric") for s in sebepler)

        # 2. Cumlede 'baseline' veya 'h/o' veya 's/p' yakin -> historical
        if re.search(r"\bbaseline\b", cumle, re.IGNORECASE):
            karar = "HISTORICAL_ALLOWED"
        elif has_hist and not has_curr and not kuvvetli_var:
            karar = "HISTORICAL_ALLOWED"
        # 3. lab_short olup mg/mcg/dose kontekstinde -> CLEAN_NO_RISK (false positive)
        elif sadece_yumusak and not has_curr:
            # Cumle kisa ve "Cr 1.1-1.5" gibi range yapisinda mi?
            if re.search(r"\b(baseline|range)\b", cumle, re.IGNORECASE):
                karar = "HISTORICAL_ALLOWED"
            else:
                # Belirsiz - REVIEW olarak isaretle
                karar = "NEEDS_HUMAN"
        # 4. Direct_dx + current marker -> kesin remove
        elif any(s[0] == "direct_dx" for s in sebepler) and has_curr:
            karar = "REMOVE_CURRENT_LEAKAGE"
            sentences_to_remove.append(cumle)
        # 5. Imaging finding kuvvetli, historical yok -> remove
        elif any(s[0] == "imaging" for s in sebepler) and not has_hist:
            karar = "REMOVE_CURRENT_LEAKAGE"
            sentences_to_remove.append(cumle)
        # 6. lab_numeric kuvvetli, historical yok ve baseline yok -> remove
        elif any(s[0] == "lab_numeric" for s in sebepler) and not has_hist:
            karar = "REMOVE_CURRENT_LEAKAGE"
            sentences_to_remove.append(cumle)
        # 7. found to have / direct_dx tek basina -> remove
        elif any(s[0] == "direct_dx" for s in sebepler):
            karar = "REMOVE_CURRENT_LEAKAGE"
            sentences_to_remove.append(cumle)
        # 8. lab_descriptor + current marker -> remove
        elif any(s[0] == "lab_descriptor" for s in sebepler) and has_curr:
            karar = "REMOVE_CURRENT_LEAKAGE"
            sentences_to_remove.append(cumle)
        # 9. lab_descriptor tek basina (historical/current yok) -> remove (defensif)
        elif sadece_yumusak:
            karar = "REMOVE_CURRENT_LEAKAGE"
            sentences_to_remove.append(cumle)
        else:
            karar = "NEEDS_HUMAN"

        sebep_str = ",".join(f"{s[0]}={s[1][:30]}" for s in sebepler)
        print(f"   Cumle: {cumle[:200]}")
        print(f"   Sebep: {sebep_str}")
        print(f"   has_hist={has_hist}, has_curr={has_curr}")
        print(f"   KARAR: {karar}")

    # Kayit icin nihai karar
    if sentences_to_remove:
        manual_decisions[nid] = {"decision": "REMOVE_CURRENT_LEAKAGE", "sentences_to_remove": sentences_to_remove}
    else:
        # Risk var ama hicbir cumle remove edilmedi -> historical
        manual_decisions[nid] = {"decision": "HISTORICAL_ALLOWED", "sentences_to_remove": []}

print("\n" + "=" * 70)
print(f"MANUAL KARAR DAGILIMI ({len(manual_decisions)} kayit)")
print("=" * 70)
karar_sayim = {}
for d in manual_decisions.values():
    karar_sayim[d["decision"]] = karar_sayim.get(d["decision"], 0) + 1
for k, v in sorted(karar_sayim.items()):
    print(f"   {k:<28s} {v}")

# ============================================================
# Decision uygula: REMOVE_CURRENT_LEAKAGE icin cumleleri sil
# ============================================================
print("\n" + "=" * 70)
print("UYGULA: REMOVE cumleleri V4 input'tan sil")
print("=" * 70)

# Yeni kolonlar - default olarak mevcut degerleri kopyala
df["strict_current_leakage_severity_final"] = df["strict_current_leakage_severity"].copy()
df["strict_current_leakage_reason_final"] = df["strict_current_leakage_reason"].copy()
df["manual_review_decision_13f"] = "AUTO"

silinen_toplam = 0
for nid, d in manual_decisions.items():
    mask = df["note_id"] == nid
    if not mask.any():
        continue

    karar = d["decision"]
    df.loc[mask, "manual_review_decision_13f"] = karar

    if karar == "HISTORICAL_ALLOWED":
        df.loc[mask, "strict_current_leakage_severity_final"] = "HISTORICAL_ALLOWED"
        df.loc[mask, "strict_current_leakage_reason_final"] = "manual: historical context"
    elif karar == "CLEAN_NO_RISK":
        df.loc[mask, "strict_current_leakage_severity_final"] = "CLEAN"
        df.loc[mask, "strict_current_leakage_reason_final"] = "manual: false positive"
    elif karar == "REMOVE_CURRENT_LEAKAGE":
        # Cumleleri V4 input ve alt-bolum kolonlarindan sil
        idx = df.index[mask][0]
        for cumle in d["sentences_to_remove"]:
            for col in ["hpi_clean_v4_timeaware", "pmh_v4_timeaware",
                        "triage_input_v4_timeaware", "triage_input_v4_timeaware_masked"]:
                eski = df.at[idx, col] or ""
                yeni = eski.replace(cumle, "")
                # cift bosluk temizligi
                yeni = re.sub(r"\s{2,}", " ", yeni).strip()
                df.at[idx, col] = yeni
            silinen_toplam += 1
        df.loc[mask, "strict_current_leakage_severity_final"] = "CLEAN"
        df.loc[mask, "strict_current_leakage_reason_final"] = "manual: current leakage removed"

print(f"   Toplam {silinen_toplam} cumle silindi.")

# ============================================================
# FINAL AUDIT
# ============================================================
print("\n" + "=" * 70)
print("FINAL AUDIT (test, post-13F)")
print("=" * 70)

# Tum test'i ve diger split'leri yeniden audit et (final severity ile)
test_alt_post = df[df["split"]=="test"]
val_post = df[df["split"]=="val"]
train_post = df[df["split"]=="train_S3"]

for split_ad, alt in [("test", test_alt_post), ("val", val_post), ("train_S3", train_post)]:
    n = len(alt)
    print(f"\n--- {split_ad} (N={n:,}) ---")
    sev_say = alt["strict_current_leakage_severity_final"].value_counts().to_dict()
    for sev in ["CLEAN","HISTORICAL_ALLOWED","REVIEW_UNCERTAIN","CURRENT_MEDIUM","CURRENT_HIGH"]:
        c = sev_say.get(sev, 0)
        print(f"   {sev:<25s} {c:>5} ({100*c/max(n,1):>5.1f}%)")

# Acceptance
print("\n" + "=" * 70)
print("ACCEPTANCE")
print("=" * 70)
hi = (test_alt_post["strict_current_leakage_severity_final"]=="CURRENT_HIGH").sum()
md = (test_alt_post["strict_current_leakage_severity_final"]=="CURRENT_MEDIUM").sum()
rv = (test_alt_post["strict_current_leakage_severity_final"]=="REVIEW_UNCERTAIN").sum()
nh = (test_alt_post["manual_review_decision_13f"]=="NEEDS_HUMAN").sum()

print(f"   Test CURRENT_HIGH      = {hi}  (hedef: 0)")
print(f"   Test CURRENT_MEDIUM    = {md}  (hedef: 0)")
print(f"   Test REVIEW            = {rv}  (hedef: 0)")
print(f"   Test NEEDS_HUMAN       = {nh}")

if hi == 0 and md == 0 and rv == 0 and nh == 0:
    print(f"\n   ✅ ACCEPT: Tum hedefler tutturuldu. V4 final.")
elif nh > 0:
    print(f"\n   ⚠️  NEEDS_HUMAN={nh} kayit var - asagida ornekleri:")
    for _, row in test_alt_post[test_alt_post["manual_review_decision_13f"]=="NEEDS_HUMAN"].iterrows():
        print(f"      {row['note_id']} ({row['label']}) - {row['strict_current_leakage_reason']}")
else:
    print(f"\n   ⚠️  Bazi sayilar > 0:")

# Parquet kaydet
df.to_parquet(SPLIT_YOL, compression="snappy", index=False)
print(f"\n   Parquet ({SPLIT_YOL.stat().st_size/1024**2:.1f} MB, {len(df.columns)} kolon)")

print("\n" + "=" * 70)
print("BLOK 13F TAMAMLANDI")
print("=" * 70)


# BLOK 13B-V4-VAL — V4 TIME-AWARE Zero-shot Val

**Input:** `triage_input_v4_timeaware_masked` (V4 time-aware, historical EHR allowed).

**Konfigurasyon:** Aynı 13B-VAL kurulumu — Qwen3.5-4B 4-bit, chat_template + `enable_thinking=False`,
max_length=2048, max_new_tokens=20, do_sample=False, checkpoint her 50 ornekte.

**Sonuc:**
- Val (n=550): **acc=0.4818, macro-F1=0.4367**
- Parser: direct=548 (%99.6), fuzzy=0, invalid=2 (%0.4)
- V3 vs V4 (val): delta=-0.0036 (gurultu icinde, eşdeger)

**Cikti dosyalari (output/zeroshot/):**
- `zeroshot_v4_val.csv`
- `zeroshot_v4_val_audit.txt`
- `zeroshot_v4_val_results_summary.csv`

**Detayli script:** `__run_blok13b_v4_val.py`

In [ ]:
"""Blok 13B-V4-VAL: V4 (TIME-AWARE TRIAGE) zero-shot val inference.
Aynı 13B-VAL kurulumu, input olarak triage_input_v4_timeaware_masked."""
import pandas as pd
import numpy as np
import torch
import time
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
ZS_DIR     = OUTPUT_DIR / "zeroshot"
ZS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("BLOK 13B-V4-VAL — V4 TIME-AWARE TRIAGE Zero-shot Val")
print("=" * 70)
print("   Input: triage_input_v4_timeaware_masked (historical EHR allowed)")
print("   KURAL: Sadece val. Test'e otomatik gecmez.")

ALLOWED_CLASSES = [
    "SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"
]

PROMPT_TEMPLATE = """You are a clinical NLP system. Given a patient's triage information collected at first contact (chief complaint, symptoms, brief history, medications, allergies, vital signs), classify the most likely PRIMARY reason for admission into ONE of these 11 categories:

SEPSIS - Systemic infection, septic shock, bacteremia
STROKE - Ischemic or hemorrhagic stroke, CVA
ATRIAL_FIBRILLATION - Atrial fibrillation/flutter (acute)
AKI - Acute kidney injury, acute renal failure
PNEUMONIA - Lung infection (bacterial, viral, aspiration)
UTI - Urinary tract infection, pyelonephritis, cystitis
HEART_FAILURE - Congestive heart failure, decompensated CHF
GI_BLEEDING - Upper or lower GI hemorrhage
DIABETES_COMPLICATION - DKA, HHS, severe hyper/hypoglycemia
COPD_EXACERBATION - Acute exacerbation of COPD
OTHER - None of the above

Rules:
- Output exactly one class label from the list, using the exact uppercase label string.
- Choose the SINGLE most likely PRIMARY reason for admission.

TRIAGE NOTE:
{input}

PRIMARY DIAGNOSIS:"""

FUZZY_MAP = {
    "SEPS":"SEPSIS","SEPT":"SEPSIS","PNEUM":"PNEUMONIA",
    "STROK":"STROKE","CVA":"STROKE",
    "ATRIAL":"ATRIAL_FIBRILLATION","AFIB":"ATRIAL_FIBRILLATION","FIBRIL":"ATRIAL_FIBRILLATION",
    "AKI":"AKI","KIDNEY":"AKI","RENAL":"AKI",
    "UTI":"UTI","URIN":"UTI",
    "HEART":"HEART_FAILURE","CHF":"HEART_FAILURE",
    "GI":"GI_BLEEDING","BLEED":"GI_BLEEDING",
    "DIAB":"DIABETES_COMPLICATION","DKA":"DIABETES_COMPLICATION",
    "COPD":"COPD_EXACERBATION","OTHER":"OTHER","NONE":"OTHER",
}

def parse_class(raw):
    if not raw: return "INVALID","empty"
    text = raw.strip().upper()
    for cls in sorted(ALLOWED_CLASSES, key=len, reverse=True):
        if cls in text[:80]: return cls,"direct"
    for k,v in FUZZY_MAP.items():
        if k in text[:80]: return v,"fuzzy"
    return "INVALID","invalid"

# Model
print("\n[1/4] Model yukleme...")
t0 = time.time()
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-4B", quantization_config=bnb,
    device_map="auto", torch_dtype=torch.float16, local_files_only=True)
model.eval()
print(f"   Yuklendi {time.time()-t0:.1f}s, GPU {torch.cuda.memory_allocated()/1024**2:.0f}MB")

def inference(input_text):
    prompt = PROMPT_TEMPLATE.format(input=input_text)
    msgs = [{"role":"user","content":prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return raw, inputs.input_ids.shape[1]

# Veri
print("\n[2/4] V4 val seti...")
df = pd.read_parquet(SPLIT_YOL)
val_df = df[df["split"]=="val"].reset_index(drop=True)
print(f"   Val: {len(val_df):,}")

# Inference
print("\n[3/4] V4 val inference (checkpoint/resume)...")
cp_yol  = ZS_DIR / "zeroshot_v4_val_checkpoint.csv"
final_yol = ZS_DIR / "zeroshot_v4_val.csv"

if cp_yol.exists():
    cp = pd.read_csv(cp_yol)
    islenen = set(cp["note_id"].astype(str))
    kayitlar = cp.to_dict("records")
    print(f"   Resume: {len(islenen)} kayit zaten var.")
else:
    islenen = set()
    kayitlar = []

t_baslangic = time.time()
sayac_yeni = 0

for i, row in val_df.iterrows():
    nid = str(row["note_id"])
    if nid in islenen: continue
    inp = row["triage_input_v4_timeaware_masked"] if pd.notna(row["triage_input_v4_timeaware_masked"]) else ""
    t0 = time.time()
    raw, in_n = inference(inp)
    sure = time.time() - t0
    parsed, kat = parse_class(raw)
    kayitlar.append({
        "note_id": nid, "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]), "label": row["label"],
        "raw_output": raw, "parsed_pred": parsed,
        "parse_kategori": kat, "in_tokens": int(in_n),
        "sure_s": round(sure, 3),
        "leakage_severity_final": str(row.get("strict_current_leakage_severity_final","UNK")),
    })
    sayac_yeni += 1
    if len(kayitlar) % 50 == 0:
        pd.DataFrame(kayitlar).to_csv(cp_yol, index=False)
        gecen = time.time() - t_baslangic
        ort = gecen / sayac_yeni if sayac_yeni > 0 else 0
        kalan = (len(val_df) - len(kayitlar)) * ort
        print(f"   [{len(kayitlar):>3}/{len(val_df)}] gecen {gecen/60:.1f}dk ort {ort:.2f}s kalan ~{kalan/60:.1f}dk")

df_son = pd.DataFrame(kayitlar)
df_son.to_csv(final_yol, index=False)
df_son.to_csv(cp_yol, index=False)
toplam = time.time() - t_baslangic
print(f"   BITTI. {len(kayitlar)} kayit, {toplam/60:.1f}dk")

# AUDIT
print("\n[4/4] V4 VAL AUDIT")
print("=" * 70)
n = len(df_son)
ks = df_son["parse_kategori"].value_counts().to_dict()
direct = ks.get("direct", 0); fuzzy = ks.get("fuzzy", 0)
invalid = ks.get("invalid", 0) + ks.get("empty", 0)
print(f"\nParser: direct={direct} ({100*direct/n:.1f}%) fuzzy={fuzzy} invalid={invalid} ({100*invalid/n:.1f}%)")

y_true = df_son["label"].values
y_pred = df_son["parsed_pred"].values
valid = y_pred != "INVALID"

acc_full = accuracy_score(y_true, y_pred)
mf1_full = f1_score(y_true, y_pred, average="macro", zero_division=0)
wf1_full = f1_score(y_true, y_pred, average="weighted", zero_division=0)
acc_v = accuracy_score(y_true[valid], y_pred[valid]) if valid.sum() else 0.0
mf1_v = f1_score(y_true[valid], y_pred[valid], average="macro", zero_division=0) if valid.sum() else 0.0

print(f"\nV4 Val metrikleri:")
print(f"   {'Subset':<22s} {'N':>5s} {'Acc':>8s} {'MacroF1':>10s} {'WF1':>8s}")
print("-" * 60)
print(f"   {'Full':<22s} {n:>5} {acc_full:>8.4f} {mf1_full:>10.4f} {wf1_full:>8.4f}")
print(f"   {'Valid only':<22s} {valid.sum():>5} {acc_v:>8.4f} {mf1_v:>10.4f}")

# Per-class
labels = sorted(set(y_true) | (set(y_pred) - {"INVALID"}))
p, r, f, supp = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
print(f"\nPer-class:")
for cls, pi, ri, fi, si in zip(labels, p, r, f, supp):
    print(f"   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}")

# Tahmin dagilimi (bias)
print(f"\nTahmin dagilimi (bias):")
for cls, sayi in df_son["parsed_pred"].value_counts().items():
    print(f"   {cls:<25s} {sayi:>4} ({100*sayi/n:>5.1f}%)")

# V3 ile karsilastirma
print(f"\nV3 vs V4 (val):")
v3_summary_yol = ZS_DIR / "zeroshot_v3_val_results_summary.csv"
if v3_summary_yol.exists():
    v3 = pd.read_csv(v3_summary_yol)
    v3_full = float(v3[v3["subset"]=="full"]["accuracy"].iloc[0])
    v3_mf1 = float(v3[v3["subset"]=="full"]["macro_f1"].iloc[0])
    print(f"   {'':<22s} {'V3':>8s} {'V4':>8s} {'Delta':>8s}")
    print(f"   {'Full Acc':<22s} {v3_full:>8.4f} {acc_full:>8.4f} {acc_full-v3_full:>+8.4f}")
    print(f"   {'Full MacroF1':<22s} {v3_mf1:>8.4f} {mf1_full:>8.4f} {mf1_full-v3_mf1:>+8.4f}")

# Summary kaydet
ozet = pd.DataFrame({
    "subset": ["full","valid_only"],
    "n": [n, valid.sum()],
    "accuracy": [acc_full, acc_v],
    "macro_f1": [mf1_full, mf1_v],
    "weighted_f1": [wf1_full, 0.0],
})
ozet.to_csv(ZS_DIR / "zeroshot_v4_val_results_summary.csv", index=False)

audit_yol = ZS_DIR / "zeroshot_v4_val_audit.txt"
with open(audit_yol, "w", encoding="utf-8") as fh:
    fh.write("BLOK 13B-V4-VAL AUDIT (TIME-AWARE TRIAGE)\n")
    fh.write("=" * 70 + "\n")
    fh.write(f"Val n: {n}\n")
    fh.write(f"Parser: direct={direct}, fuzzy={fuzzy}, invalid={invalid}\n\n")
    fh.write(f"Full      : acc={acc_full:.4f}, macroF1={mf1_full:.4f}, wF1={wf1_full:.4f}\n")
    fh.write(f"Valid only: acc={acc_v:.4f}, macroF1={mf1_v:.4f}\n\n")
    fh.write("Per-class:\n")
    for cls, pi, ri, fi, si in zip(labels, p, r, f, supp):
        fh.write(f"   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}\n")

# Karar
print("\n" + "=" * 70)
print("KARAR")
print("=" * 70)
print(f"   V4 val full acc: {acc_full:.4f}")
if acc_full < 0.20:
    print(f"   ⚠️  Acc < %20: TEST BASLATILMASIN.")
elif acc_full <= 0.40:
    print(f"   📊 Acc %20-40: Beklenen aralık. Test icin kullanici onayi iste.")
else:
    print(f"   ✅ Acc > %40: Iyi sonuc. Test icin kullanici onayi iste.")
print(f"   TEST'e otomatik gecis YOK.")
print("=" * 70)


# BLOK 13B-V4-TEST — V4 TIME-AWARE Zero-shot TEST (FINAL)

**Final V4 zero-shot sonucu.** Aynı V4-VAL kurulumu, test split (n=1.650).

**Sonuc:**
- **Full (n=1.650):** acc=0.5152, macro-F1=0.4681, weighted-F1=0.5107
- **Valid only (n=1.639):** acc=0.5186, macro-F1=0.5121
- **Time-aware clean (n=1.650):** = Full (testin %100'u zaten time-aware clean)
- **Keyword-clean (V3 ref, n=948):** acc=0.4082, macro-F1=0.3775
- Parser: direct=1639 (%99.3), invalid=11 (%0.7)

**V3 vs V4 (test):**
| Metric | V3 | V4 | Delta |
|---|---:|---:|---:|
| Full Acc | 0.5158 | 0.5152 | -0.0006 |
| Full MacroF1 | 0.4702 | 0.4681 | -0.0021 |
| Keyword-clean Acc | 0.4167 | 0.4082 | -0.0084 |

→ V4 ≈ V3 sayısal olarak. **Metodolojik ustunluk:** test'in %100'u time-aware clean (V3'te %57).

**Per-class delta:**
- V4 iyilesen: PNEUMONIA (+0.033), AKI (+0.018), GI_BLEEDING (+0.006)
- V4 dusen: UTI (-0.019), DIABETES (-0.018), ATRIAL_FIB (-0.016)

**Cikti dosyalari (output/zeroshot/):**
- `zeroshot_v4_test.csv` (1.650 satir)
- `zeroshot_v4_test_audit.txt`
- `zeroshot_v4_test_results_summary.csv`

**Detayli script:** `__run_blok13b_v4_test.py`

In [ ]:
"""Blok 13B-V4-TEST: V4 (TIME-AWARE) zero-shot TEST. Final V4 sonucu."""
import pandas as pd
import numpy as np
import torch
import time
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
ZS_DIR     = OUTPUT_DIR / "zeroshot"

print("=" * 70)
print("BLOK 13B-V4-TEST — V4 TIME-AWARE Zero-shot TEST (FINAL)")
print("=" * 70)

ALLOWED_CLASSES = [
    "SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"
]

PROMPT_TEMPLATE = """You are a clinical NLP system. Given a patient's triage information collected at first contact (chief complaint, symptoms, brief history, medications, allergies, vital signs), classify the most likely PRIMARY reason for admission into ONE of these 11 categories:

SEPSIS - Systemic infection, septic shock, bacteremia
STROKE - Ischemic or hemorrhagic stroke, CVA
ATRIAL_FIBRILLATION - Atrial fibrillation/flutter (acute)
AKI - Acute kidney injury, acute renal failure
PNEUMONIA - Lung infection (bacterial, viral, aspiration)
UTI - Urinary tract infection, pyelonephritis, cystitis
HEART_FAILURE - Congestive heart failure, decompensated CHF
GI_BLEEDING - Upper or lower GI hemorrhage
DIABETES_COMPLICATION - DKA, HHS, severe hyper/hypoglycemia
COPD_EXACERBATION - Acute exacerbation of COPD
OTHER - None of the above

Rules:
- Output exactly one class label from the list, using the exact uppercase label string.
- Choose the SINGLE most likely PRIMARY reason for admission.

TRIAGE NOTE:
{input}

PRIMARY DIAGNOSIS:"""

FUZZY_MAP = {
    "SEPS":"SEPSIS","SEPT":"SEPSIS","PNEUM":"PNEUMONIA",
    "STROK":"STROKE","CVA":"STROKE",
    "ATRIAL":"ATRIAL_FIBRILLATION","AFIB":"ATRIAL_FIBRILLATION","FIBRIL":"ATRIAL_FIBRILLATION",
    "AKI":"AKI","KIDNEY":"AKI","RENAL":"AKI",
    "UTI":"UTI","URIN":"UTI",
    "HEART":"HEART_FAILURE","CHF":"HEART_FAILURE",
    "GI":"GI_BLEEDING","BLEED":"GI_BLEEDING",
    "DIAB":"DIABETES_COMPLICATION","DKA":"DIABETES_COMPLICATION",
    "COPD":"COPD_EXACERBATION","OTHER":"OTHER","NONE":"OTHER",
}

def parse_class(raw):
    if not raw: return "INVALID","empty"
    text = raw.strip().upper()
    for cls in sorted(ALLOWED_CLASSES, key=len, reverse=True):
        if cls in text[:80]: return cls,"direct"
    for k,v in FUZZY_MAP.items():
        if k in text[:80]: return v,"fuzzy"
    return "INVALID","invalid"

# Model
print("\n[1/4] Model yukleme...")
t0 = time.time()
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-4B", quantization_config=bnb,
    device_map="auto", torch_dtype=torch.float16, local_files_only=True)
model.eval()
print(f"   Yuklendi {time.time()-t0:.1f}s, GPU {torch.cuda.memory_allocated()/1024**2:.0f}MB")

def inference(input_text):
    prompt = PROMPT_TEMPLATE.format(input=input_text)
    msgs = [{"role":"user","content":prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return raw, inputs.input_ids.shape[1]

# Veri
print("\n[2/4] V4 test seti...")
df = pd.read_parquet(SPLIT_YOL)
test_df = df[df["split"]=="test"].reset_index(drop=True)
print(f"   Test: {len(test_df):,}")

# Inference
print("\n[3/4] V4 TEST inference (checkpoint/resume)...")
cp_yol  = ZS_DIR / "zeroshot_v4_test_checkpoint.csv"
final_yol = ZS_DIR / "zeroshot_v4_test.csv"

if cp_yol.exists():
    cp = pd.read_csv(cp_yol)
    islenen = set(cp["note_id"].astype(str))
    kayitlar = cp.to_dict("records")
    print(f"   Resume: {len(islenen)} kayit zaten var.")
else:
    islenen = set()
    kayitlar = []

t_baslangic = time.time()
sayac_yeni = 0

for i, row in test_df.iterrows():
    nid = str(row["note_id"])
    if nid in islenen: continue
    inp = row["triage_input_v4_timeaware_masked"] if pd.notna(row["triage_input_v4_timeaware_masked"]) else ""
    t0 = time.time()
    raw, in_n = inference(inp)
    sure = time.time() - t0
    parsed, kat = parse_class(raw)
    kayitlar.append({
        "note_id": nid, "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]), "label": row["label"],
        "raw_output": raw, "parsed_pred": parsed,
        "parse_kategori": kat, "in_tokens": int(in_n),
        "sure_s": round(sure, 3),
        "leakage_severity_final": str(row.get("strict_current_leakage_severity_final","UNK")),
        "has_own_label_term_v3": bool(row.get("has_own_label_term_v3", False)),
    })
    sayac_yeni += 1
    if len(kayitlar) % 50 == 0:
        pd.DataFrame(kayitlar).to_csv(cp_yol, index=False)
        gecen = time.time() - t_baslangic
        ort = gecen / sayac_yeni if sayac_yeni > 0 else 0
        kalan = (len(test_df) - len(kayitlar)) * ort
        print(f"   [{len(kayitlar):>4}/{len(test_df)}] gecen {gecen/60:.1f}dk ort {ort:.2f}s kalan ~{kalan/60:.1f}dk")

df_son = pd.DataFrame(kayitlar)
df_son.to_csv(final_yol, index=False)
df_son.to_csv(cp_yol, index=False)
toplam = time.time() - t_baslangic
print(f"   BITTI. {len(kayitlar)} kayit, {toplam/60:.1f}dk")

# AUDIT
print("\n[4/4] V4 TEST FINAL AUDIT")
print("=" * 70)
n = len(df_son)
ks = df_son["parse_kategori"].value_counts().to_dict()
direct = ks.get("direct", 0); fuzzy = ks.get("fuzzy", 0)
invalid = ks.get("invalid", 0) + ks.get("empty", 0)
print(f"\nParser: direct={direct} ({100*direct/n:.1f}%) fuzzy={fuzzy} invalid={invalid} ({100*invalid/n:.1f}%)")

y_true = df_son["label"].values
y_pred = df_son["parsed_pred"].values
valid = y_pred != "INVALID"
clean_v3_subset = (~df_son["has_own_label_term_v3"]).values
timeaware_clean = df_son["leakage_severity_final"].isin(["CLEAN","HISTORICAL_ALLOWED"]).values

acc_full = accuracy_score(y_true, y_pred)
mf1_full = f1_score(y_true, y_pred, average="macro", zero_division=0)
wf1_full = f1_score(y_true, y_pred, average="weighted", zero_division=0)
acc_v = accuracy_score(y_true[valid], y_pred[valid]) if valid.sum() else 0.0
mf1_v = f1_score(y_true[valid], y_pred[valid], average="macro", zero_division=0) if valid.sum() else 0.0
wf1_v = f1_score(y_true[valid], y_pred[valid], average="weighted", zero_division=0) if valid.sum() else 0.0

# Time-aware clean alt-set (testte zaten 1650/1650)
acc_ta = accuracy_score(y_true[timeaware_clean], y_pred[timeaware_clean]) if timeaware_clean.sum() else 0.0
mf1_ta = f1_score(y_true[timeaware_clean], y_pred[timeaware_clean], average="macro", zero_division=0) if timeaware_clean.sum() else 0.0

# V3 keyword-clean alt-set (V4 ile karsilastirma icin)
acc_kw = accuracy_score(y_true[clean_v3_subset], y_pred[clean_v3_subset]) if clean_v3_subset.sum() else 0.0
mf1_kw = f1_score(y_true[clean_v3_subset], y_pred[clean_v3_subset], average="macro", zero_division=0) if clean_v3_subset.sum() else 0.0

print(f"\nV4 TEST METRIKLERI:")
print(f"   {'Subset':<26s} {'N':>5s} {'Acc':>8s} {'MacroF1':>10s} {'WF1':>8s}")
print("-" * 65)
print(f"   {'Full':<26s} {n:>5} {acc_full:>8.4f} {mf1_full:>10.4f} {wf1_full:>8.4f}")
print(f"   {'Valid only':<26s} {valid.sum():>5} {acc_v:>8.4f} {mf1_v:>10.4f} {wf1_v:>8.4f}")
print(f"   {'Time-aware clean':<26s} {timeaware_clean.sum():>5} {acc_ta:>8.4f} {mf1_ta:>10.4f}")
print(f"   {'Keyword-clean (V3 ref)':<26s} {clean_v3_subset.sum():>5} {acc_kw:>8.4f} {mf1_kw:>10.4f}")

# Per-class
labels = sorted(set(y_true) | (set(y_pred) - {"INVALID"}))
p, r, f, supp = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
print(f"\nPer-class:")
for cls, pi, ri, fi, si in zip(labels, p, r, f, supp):
    print(f"   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}")

# Confusion matrix
labels_full = ALLOWED_CLASSES + ["INVALID"]
cm = confusion_matrix(y_true, y_pred, labels=labels_full)
ks_short = {"SEPSIS":"SEP","STROKE":"STR","ATRIAL_FIBRILLATION":"AFB","AKI":"AKI",
    "PNEUMONIA":"PNE","UTI":"UTI","HEART_FAILURE":"HF","GI_BLEEDING":"GIB",
    "DIABETES_COMPLICATION":"DIA","COPD_EXACERBATION":"CPD","OTHER":"OTH","INVALID":"INV"}
print(f"\nConfusion matrix:")
print("gercek\\pred  " + " ".join(f"{ks_short[c]:>4}" for c in labels_full))
for i, g in enumerate(labels_full):
    if cm[i].sum() == 0: continue
    print(f"{ks_short[g]:<13s}" + " ".join(f"{cm[i][j]:>4}" for j in range(len(labels_full))))

# Bias
print(f"\nTahmin dagilimi (bias):")
for cls, sayi in df_son["parsed_pred"].value_counts().items():
    print(f"   {cls:<25s} {sayi:>5} ({100*sayi/n:>5.1f}%)")

# V3 vs V4 (test) karsilastirma
print(f"\nV3 vs V4 (test):")
v3_sum_yol = ZS_DIR / "zeroshot_v3_test_results_summary.csv"
if v3_sum_yol.exists():
    v3 = pd.read_csv(v3_sum_yol)
    v3_full = float(v3[v3["subset"]=="full"]["accuracy"].iloc[0])
    v3_mf1 = float(v3[v3["subset"]=="full"]["macro_f1"].iloc[0])
    v3_kw = float(v3[v3["subset"]=="clean_own"]["accuracy"].iloc[0])
    v3_kw_mf1 = float(v3[v3["subset"]=="clean_own"]["macro_f1"].iloc[0])
    print(f"   {'':<26s} {'V3':>8s} {'V4':>8s} {'Delta':>8s}")
    print(f"   {'Full Acc':<26s} {v3_full:>8.4f} {acc_full:>8.4f} {acc_full-v3_full:>+8.4f}")
    print(f"   {'Full MacroF1':<26s} {v3_mf1:>8.4f} {mf1_full:>8.4f} {mf1_full-v3_mf1:>+8.4f}")
    print(f"   {'Keyword-clean Acc':<26s} {v3_kw:>8.4f} {acc_kw:>8.4f} {acc_kw-v3_kw:>+8.4f}")
    print(f"   {'Keyword-clean MacroF1':<26s} {v3_kw_mf1:>8.4f} {mf1_kw:>8.4f} {mf1_kw-v3_kw_mf1:>+8.4f}")

# Val-test consistency
print(f"\nV4 Val vs Test consistency:")
v4_val_yol = ZS_DIR / "zeroshot_v4_val_results_summary.csv"
if v4_val_yol.exists():
    val_s = pd.read_csv(v4_val_yol)
    val_full = float(val_s[val_s["subset"]=="full"]["accuracy"].iloc[0])
    val_mf1 = float(val_s[val_s["subset"]=="full"]["macro_f1"].iloc[0])
    print(f"   Val Acc  : {val_full:.4f} (test: {acc_full:.4f}, delta: {acc_full-val_full:+.4f})")
    print(f"   Val MF1  : {val_mf1:.4f} (test: {mf1_full:.4f}, delta: {mf1_full-val_mf1:+.4f})")

# Summary kaydet
ozet = pd.DataFrame({
    "subset": ["full","valid_only","timeaware_clean","keyword_clean"],
    "n": [n, valid.sum(), timeaware_clean.sum(), clean_v3_subset.sum()],
    "accuracy": [acc_full, acc_v, acc_ta, acc_kw],
    "macro_f1": [mf1_full, mf1_v, mf1_ta, mf1_kw],
    "weighted_f1": [wf1_full, wf1_v, 0.0, 0.0],
})
ozet.to_csv(ZS_DIR / "zeroshot_v4_test_results_summary.csv", index=False)

audit_yol = ZS_DIR / "zeroshot_v4_test_audit.txt"
with open(audit_yol, "w", encoding="utf-8") as fh:
    fh.write("BLOK 13B-V4-TEST FINAL AUDIT (TIME-AWARE)\n")
    fh.write("=" * 70 + "\n")
    fh.write(f"Test n: {n}\n")
    fh.write(f"Parser: direct={direct}, fuzzy={fuzzy}, invalid={invalid}\n\n")
    fh.write(f"Full          : acc={acc_full:.4f}, macroF1={mf1_full:.4f}, wF1={wf1_full:.4f}\n")
    fh.write(f"Valid only    : acc={acc_v:.4f}, macroF1={mf1_v:.4f}\n")
    fh.write(f"Time-aware clean: acc={acc_ta:.4f}, macroF1={mf1_ta:.4f}, n={timeaware_clean.sum()}\n")
    fh.write(f"Keyword-clean : acc={acc_kw:.4f}, macroF1={mf1_kw:.4f}, n={clean_v3_subset.sum()}\n\n")
    fh.write("Per-class:\n")
    for cls, pi, ri, fi, si in zip(labels, p, r, f, supp):
        fh.write(f"   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}\n")

print("\n" + "=" * 70)
print("V4 TEST TAMAMLANDI - FINAL")
print("=" * 70)


In [ ]:
"""Blok 14-V4-SELECT: V4 1-shot ve 3-shot medoid + MMR seclim.
Sadece train_S3 + leakage_severity_final in [CLEAN, HISTORICAL_ALLOWED].
Inference YOK."""
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
FS_DIR     = OUTPUT_DIR / "fewshot"
FS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("BLOK 14-V4-SELECT — V4 1-shot + 3-shot medoid + MMR")
print("=" * 70)

ALLOWED_CLASSES = [
    "SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"
]

# --- Veri ---
print("\n[1/6] Veri yukleniyor...")
df = pd.read_parquet(SPLIT_YOL)
df["note_id"] = df["note_id"].astype(str)
print(f"   N={len(df):,}, kolon={len(df.columns)}")

# Pool: train_S3 + leakage clean
pool_full = df[df["split"]=="train_S3"].reset_index(drop=True)
print(f"   train_S3 pool: {len(pool_full):,}")

# Filtre: leakage_severity_final in [CLEAN, HISTORICAL_ALLOWED]
LEAKAGE_KOLON = "strict_current_leakage_severity_final"
if LEAKAGE_KOLON not in pool_full.columns:
    print(f"   UYARI: {LEAKAGE_KOLON} yok, {pool_full.columns.tolist()}")
    LEAKAGE_KOLON = "strict_current_leakage_severity"

clean_mask = pool_full[LEAKAGE_KOLON].isin(["CLEAN","HISTORICAL_ALLOWED"])
pool = pool_full[clean_mask].reset_index(drop=True)
print(f"   Filtrelenmis pool (CLEAN/HISTORICAL_ALLOWED): {len(pool):,}")
print(f"   Disardakiler: {(~clean_mask).sum()} (REVIEW/HIGH/MEDIUM)")

# Sinif dagilimi
print(f"\n   Pool sinif dagilimi (V4 clean):")
for cls in ALLOWED_CLASSES:
    n = (pool["label"]==cls).sum()
    n_orig = (pool_full["label"]==cls).sum()
    print(f"      {cls:<25s} {n:>3} (V4 clean) / {n_orig:>3} (train_S3 toplam)")

# --- TF-IDF fit ---
print("\n[2/6] TF-IDF fit (sadece V4 clean pool)...")
docs = pool["triage_input_v4_timeaware_masked"].fillna("").tolist()
vec = TfidfVectorizer(
    max_features=20000, ngram_range=(1, 2),
    min_df=2, stop_words="english", sublinear_tf=True
)
X = vec.fit_transform(docs)
print(f"   Shape: {X.shape}, vocab: {len(vec.vocabulary_)}")

# Tokenizer (Qwen)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)

def token_say(metin):
    if not isinstance(metin, str) or not metin: return 0
    return len(tok.encode(metin, add_special_tokens=False))

# --- 1-shot/class secim (medoid, en yakin centroid) ---
print("\n[3/6] V4 1-shot/class medoid seclim...")
shot1_kayitlar = []

for cls in ALLOWED_CLASSES:
    cls_idx = np.where(pool["label"].values == cls)[0]
    if len(cls_idx) == 0:
        print(f"   UYARI: {cls} icin pool'da clean ornek yok!")
        continue

    X_cls = X[cls_idx]
    centroid = np.asarray(X_cls.mean(axis=0)).reshape(1, -1)
    sim_cent = cosine_similarity(X_cls, centroid).flatten()
    rank_lokal = np.argsort(-sim_cent)

    # En yakin (rank 1) — leakage filtresi zaten pool'da uygulanmis
    secilen = int(rank_lokal[0])
    global_idx = cls_idx[secilen]
    row = pool.iloc[global_idx]

    triage = row["triage_input_v4_timeaware_masked"] or ""
    n_token = token_say(triage)

    shot1_kayitlar.append({
        "label": cls,
        "rank": 1,
        "note_id": str(row["note_id"]),
        "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]),
        "sim_to_centroid": round(float(sim_cent[secilen]), 4),
        "leakage_severity": str(row[LEAKAGE_KOLON]),
        "char_uzunluk": len(triage),
        "token_uzunluk": n_token,
        "preview": triage[:200].replace("\n", " "),
        "triage_input_v4_timeaware_masked": triage,
    })
    print(f"   {cls:<25s} note={row['note_id']:<22s} sim={sim_cent[secilen]:.3f} sev={str(row[LEAKAGE_KOLON]):<22s} tok={n_token}")

# --- 3-shot/class secim (medoid + MMR) ---
print("\n[4/6] V4 3-shot/class secim (centroid + MMR diversity)...")
ALPHA = 0.7  # centroid agirlik
BETA  = 0.3  # diversity penalty

shot3_kayitlar = []

for cls in ALLOWED_CLASSES:
    cls_idx = np.where(pool["label"].values == cls)[0]
    if len(cls_idx) < 3:
        print(f"   UYARI: {cls} icin clean pool'da {len(cls_idx)} ornek var (3 gerekli)")
        continue

    X_cls = X[cls_idx]
    centroid = np.asarray(X_cls.mean(axis=0)).reshape(1, -1)
    sim_cent = cosine_similarity(X_cls, centroid).flatten()

    # 1. ornek: en yakin centroid
    secilen_lokal = [int(np.argmax(sim_cent))]

    # 2. ve 3. ornek: MMR
    for _ in range(2):
        secilen_X = X_cls[secilen_lokal]
        sim_to_sel = cosine_similarity(X_cls, secilen_X)
        max_sim_sel = sim_to_sel.max(axis=1)
        skor = ALPHA * sim_cent - BETA * max_sim_sel
        for s in secilen_lokal: skor[s] = -np.inf
        secilen_lokal.append(int(np.argmax(skor)))

    for sira, lokal_idx in enumerate(secilen_lokal, 1):
        global_idx = cls_idx[lokal_idx]
        row = pool.iloc[global_idx]
        # Max sim to previously selected
        if sira == 1:
            max_sim_prev = 0.0
        else:
            secilen_oncekiler = X_cls[secilen_lokal[:sira-1]]
            max_sim_prev = float(cosine_similarity(X_cls[lokal_idx:lokal_idx+1], secilen_oncekiler).max())

        triage = row["triage_input_v4_timeaware_masked"] or ""
        n_token = token_say(triage)

        shot3_kayitlar.append({
            "label": cls,
            "rank": sira,
            "note_id": str(row["note_id"]),
            "subject_id": int(row["subject_id"]),
            "hadm_id": int(row["hadm_id"]),
            "sim_to_centroid": round(float(sim_cent[lokal_idx]), 4),
            "max_sim_to_prev_selected": round(max_sim_prev, 4),
            "leakage_severity": str(row[LEAKAGE_KOLON]),
            "char_uzunluk": len(triage),
            "token_uzunluk": n_token,
            "preview": triage[:200].replace("\n", " "),
            "triage_input_v4_timeaware_masked": triage,
        })

    sims = [shot3_kayitlar[-3]["sim_to_centroid"],
            shot3_kayitlar[-2]["sim_to_centroid"],
            shot3_kayitlar[-1]["sim_to_centroid"]]
    print(f"   {cls:<25s} sim=[{sims[0]:.3f}, {sims[1]:.3f}, {sims[2]:.3f}]")

# --- Audit ---
print("\n[5/6] AUDIT")
print("=" * 70)

shot1_df = pd.DataFrame(shot1_kayitlar)
shot3_df = pd.DataFrame(shot3_kayitlar)

# 1-shot ozeti
print(f"\n1-SHOT SECILEN (11 ornek):")
print(f"   {'Sinif':<25s} {'note_id':<22s} {'sim_cent':>9s} {'sev':<22s} {'tok':>5s}")
print("   " + "-" * 90)
for _, r in shot1_df.iterrows():
    print(f"   {r['label']:<25s} {r['note_id']:<22s} {r['sim_to_centroid']:>9.3f} {r['leakage_severity']:<22s} {r['token_uzunluk']:>5}")

print(f"\n   1-shot token toplami: {shot1_df['token_uzunluk'].sum():,}")
print(f"   1-shot ortalama: {shot1_df['token_uzunluk'].mean():.0f}")
print(f"   1-shot min/max : {shot1_df['token_uzunluk'].min()} / {shot1_df['token_uzunluk'].max()}")

# 3-shot ozeti
print(f"\n3-SHOT SECILEN (33 ornek):")
print(f"   {'Sinif':<25s} {'rank':>4s} {'note_id':<22s} {'sim_cent':>9s} {'max_sim_prev':>12s} {'sev':<22s} {'tok':>5s}")
print("   " + "-" * 110)
for _, r in shot3_df.iterrows():
    print(f"   {r['label']:<25s} {r['rank']:>4} {r['note_id']:<22s} {r['sim_to_centroid']:>9.3f} {r['max_sim_to_prev_selected']:>12.3f} {r['leakage_severity']:<22s} {r['token_uzunluk']:>5}")

print(f"\n   3-shot token toplami: {shot3_df['token_uzunluk'].sum():,}")
print(f"   3-shot ortalama: {shot3_df['token_uzunluk'].mean():.0f}")
print(f"   3-shot min/max : {shot3_df['token_uzunluk'].min()} / {shot3_df['token_uzunluk'].max()}")

# MMR diversity check
print(f"\n   3-shot MMR diversity (rank 2+ icin max_sim_to_prev):")
for sira in [2, 3]:
    alt = shot3_df[shot3_df["rank"]==sira]
    print(f"      rank {sira}: ort={alt['max_sim_to_prev_selected'].mean():.3f}, max={alt['max_sim_to_prev_selected'].max():.3f}")

# --- Token butce hesabi ---
print(f"\n[6/6] TOKEN BUTCESI HESAPLAMA")
print("=" * 70)

# Sabit kisim (header + sinif tanimlari + trailer wrapper) — ~280 token
HEADER_TOKEN = 280
EX_WRAPPER_TOKEN = 25  # "EXAMPLE i: ... PRIMARY DIAGNOSIS:" wrapper

# 1-shot full
shot1_full = HEADER_TOKEN + shot1_df["token_uzunluk"].sum() + 11 * EX_WRAPPER_TOKEN
# 1-shot compact (180-220 token/example hedef, var olan 14D mantigi)
SHOT1_COMPACT_TARGET = 200
shot1_compact = HEADER_TOKEN + 11 * SHOT1_COMPACT_TARGET + 11 * EX_WRAPPER_TOKEN

# 3-shot full
shot3_full = HEADER_TOKEN + shot3_df["token_uzunluk"].sum() + 33 * EX_WRAPPER_TOKEN
# 3-shot compact (daha agresif, 100-120 token/example hedef cunku 33 var)
SHOT3_COMPACT_TARGET = 120
shot3_compact = HEADER_TOKEN + 33 * SHOT3_COMPACT_TARGET + 33 * EX_WRAPPER_TOKEN

print(f"\n   {'Yontem':<22s} {'Full prompt':>12s} {'Compact (target)':>17s}")
print("   " + "-" * 55)
print(f"   {'1-shot (11 ornek)':<22s} {shot1_full:>12,} {shot1_compact:>17,}")
print(f"   {'3-shot (33 ornek)':<22s} {shot3_full:>12,} {shot3_compact:>17,}")

print(f"\n   Test input ortalamasi (V4): ~580 token")
print(f"   1-shot full + test_avg     : ~{shot1_full + 580:,}")
print(f"   1-shot compact + test_avg  : ~{shot1_compact + 580:,}  (max_length=4096 sigar)")
print(f"   3-shot full + test_avg     : ~{shot3_full + 580:,}  (8192 sigmaz)")
print(f"   3-shot compact + test_avg  : ~{shot3_compact + 580:,}  (max_length=8192 hedeflenir)")

# Leakage severity dagilimi (secilen)
print(f"\n   Leakage severity (secilen ornekler):")
print(f"   1-shot: {shot1_df['leakage_severity'].value_counts().to_dict()}")
print(f"   3-shot: {shot3_df['leakage_severity'].value_counts().to_dict()}")

# Acceptance: sadece CLEAN/HISTORICAL olmali
shot1_kabul = shot1_df["leakage_severity"].isin(["CLEAN","HISTORICAL_ALLOWED"]).all()
shot3_kabul = shot3_df["leakage_severity"].isin(["CLEAN","HISTORICAL_ALLOWED"]).all()
print(f"\n   1-shot tum CLEAN/HISTORICAL: {'✅' if shot1_kabul else '❌'}")
print(f"   3-shot tum CLEAN/HISTORICAL: {'✅' if shot3_kabul else '❌'}")

# Kaydet
print("\nKAYDETME...")
shot1_csv = FS_DIR / "fewshot_v4_1shot_examples.csv"
shot1_json = FS_DIR / "fewshot_v4_1shot_examples.json"
shot3_csv = FS_DIR / "fewshot_v4_3shot_examples.csv"
shot3_json = FS_DIR / "fewshot_v4_3shot_examples.json"

shot1_df.to_csv(shot1_csv, index=False)
shot3_df.to_csv(shot3_csv, index=False)

# JSON: label -> (1-shot tek dict, 3-shot list of 3)
shot1_json_yapi = {}
for _, r in shot1_df.iterrows():
    shot1_json_yapi[r["label"]] = {
        "note_id": r["note_id"], "subject_id": r["subject_id"], "hadm_id": r["hadm_id"],
        "sim_to_centroid": float(r["sim_to_centroid"]),
        "leakage_severity": r["leakage_severity"],
        "char_uzunluk": int(r["char_uzunluk"]),
        "token_uzunluk": int(r["token_uzunluk"]),
        "triage_input_v4_timeaware_masked": r["triage_input_v4_timeaware_masked"],
    }

shot3_json_yapi = {}
for cls in ALLOWED_CLASSES:
    shot3_json_yapi[cls] = []
    for _, r in shot3_df[shot3_df["label"]==cls].iterrows():
        shot3_json_yapi[cls].append({
            "rank": int(r["rank"]),
            "note_id": r["note_id"], "subject_id": r["subject_id"], "hadm_id": r["hadm_id"],
            "sim_to_centroid": float(r["sim_to_centroid"]),
            "max_sim_to_prev_selected": float(r["max_sim_to_prev_selected"]),
            "leakage_severity": r["leakage_severity"],
            "char_uzunluk": int(r["char_uzunluk"]),
            "token_uzunluk": int(r["token_uzunluk"]),
            "triage_input_v4_timeaware_masked": r["triage_input_v4_timeaware_masked"],
        })

with open(shot1_json, "w", encoding="utf-8") as f:
    json.dump(shot1_json_yapi, f, ensure_ascii=False, indent=2)
with open(shot3_json, "w", encoding="utf-8") as f:
    json.dump(shot3_json_yapi, f, ensure_ascii=False, indent=2)

print(f"   {shot1_csv.name} ({shot1_csv.stat().st_size/1024:.1f} KB)")
print(f"   {shot1_json.name} ({shot1_json.stat().st_size/1024:.1f} KB)")
print(f"   {shot3_csv.name} ({shot3_csv.stat().st_size/1024:.1f} KB)")
print(f"   {shot3_json.name} ({shot3_json.stat().st_size/1024:.1f} KB)")

# Preview
print("\n" + "=" * 70)
print("PREVIEW (1-shot)")
print("=" * 70)
for cls in ALLOWED_CLASSES:
    if cls not in shot1_json_yapi: continue
    ex = shot1_json_yapi[cls]
    print(f"\n   {cls} (note: {ex['note_id']}, tok: {ex['token_uzunluk']}, sev: {ex['leakage_severity']})")
    print(f"   {ex['triage_input_v4_timeaware_masked'][:250]}...")

print("\n" + "=" * 70)
print("BLOK 14-V4-SELECT TAMAMLANDI - inference YOK")
print("=" * 70)


# BLOK 14-V4-SELECT — V4 1-shot + 3-shot Few-shot Medoid Seçimi

**Pool:** train_S3 (4.400 örnek) → leakage filtresi (CLEAN/HISTORICAL_ALLOWED) → 4.348 örnek
**Yöntem:** TF-IDF centroid (1-shot) + MMR diversity (3-shot)

**Sonuç:**
- 1-shot (11 örnek): toplam **8.534 token** (avg 776/medoid). Hepsi CLEAN/HISTORICAL.
- 3-shot (33 örnek): toplam **22.286 token** (avg 675/medoid). MMR diversity rank 2: 0.104, rank 3: 0.132.

**Token bütçesi:**
- 1-shot full: ~9.089 → **compact target: ~2.755** → +query: ~3.335 (max_length=4096 ✅)
- 3-shot full: ~23.391 → compact target: ~5.065 → +query: ~5.645 (max_length=8192 hedef)

**Çıktı:** `output/fewshot/fewshot_v4_1shot_examples.{csv,json}` ve `fewshot_v4_3shot_examples.{csv,json}`

**Detaylı script:** `__run_blok14_v4_select.py`

# BLOK 14-V4-COMPACT-1SHOT — V4 1-shot Section-Aware Compaction

**Mantık:** 14D'deki section-aware compaction V4 1-shot medoid'lere uygulandı.

**Format:** Chief Complaint + Key Symptoms (semptom-skorlu ilk 2-3 cümle) + Vitals + Relevant History + Allergies + Medications (max 5).

**Sonuç:**
- Toplam: **8.534 → 2.601 token** (%30 sıkıştırma)
- Ortalama: 236 token/medoid (hedef 180-220, biraz aşıldı ama 4096'ya rahat sığar)
- Tüm 11 medoid V4 clean (CLEAN veya HISTORICAL_ALLOWED)

**Çıktı:** `output/fewshot/fewshot_v4_compact_1shot_examples.json`

In [ ]:
"""Blok 14-V4-COMPACT-1SHOT: V4 1-shot medoid'i section-aware compact.
Aynı 14D mantığı, V4 input için. Hedef: ~200 token/medoid."""
import pandas as pd
import json
import re
from pathlib import Path
from transformers import AutoTokenizer

OUTPUT_DIR = Path(r"./workdir/output")
FS_DIR     = OUTPUT_DIR / "fewshot"

print("=" * 70)
print("BLOK 14-V4-COMPACT-1SHOT — Section-aware compaction")
print("=" * 70)

ALLOWED_CLASSES = ["SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"]

with open(FS_DIR / "fewshot_v4_1shot_examples.json", "r", encoding="utf-8") as f:
    v4_data = json.load(f)

# Section parser
SECTION_RE = re.compile(
    r"(Chief Complaint|Symptom History|Past Medical History|Allergies|"
    r"Medications on Admission|Vital Signs):\s*\n?", re.IGNORECASE)

def parse_sections(text):
    if not text: return {}
    parts = SECTION_RE.split(text)
    sections = {}
    for i in range(1, len(parts) - 1, 2):
        sections[parts[i].strip()] = parts[i+1].strip()
    return sections

SYMPTOM_KEYWORDS = re.compile(
    r"\b(presents?|reports?|complains?|denies|developed|noted|noticed|started|began|"
    r"pain|cough|fever|dyspnea|SOB|fatigue|weakness|nausea|vomit|diarrhea|bleeding|"
    r"dizz|confusion|swelling|edema|chest|abdomen|headache|hypoxic|hypotens|tachycard|"
    r"palpitations|orthopnea|wheez|melena|hemato|dysuria|frequen|urgency)\b", re.IGNORECASE)

def cumlelere_bol(t):
    if not t: return []
    return [c.strip() for c in re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", t) if c.strip()]

def hasta_skor(c):
    return len(SYMPTOM_KEYWORDS.findall(c))

def compact_input(full_text):
    """14D mantığı: section-aware sıkıştırma."""
    sections = parse_sections(full_text)
    parts = []

    cc = sections.get("Chief Complaint", "").strip()
    if cc:
        parts.append(f"Chief Complaint: {cc}")

    sh = sections.get("Symptom History", "").strip()
    if sh:
        ros_idx = re.search(r"(?im)^\s*(Review of Systems?|ROS)\s*:", sh)
        if ros_idx:
            sh = sh[:ros_idx.start()]
        cumleler = cumlelere_bol(sh)
        skorli = [(i, c, hasta_skor(c)) for i, c in enumerate(cumleler[:5])]
        secilen = [c for i, c, s in skorli if s > 0][:3]
        if not secilen:
            secilen = [c for i, c, s in skorli][:2]
        else:
            if cumleler and cumleler[0] not in secilen:
                secilen = [cumleler[0]] + secilen
                secilen = secilen[:3]
        if secilen:
            parts.append(f"Key Symptoms: {' '.join(secilen)}")

    vs = sections.get("Vital Signs", "").strip()
    if vs:
        parts.append(f"Vitals: {vs}")

    pmh = sections.get("Past Medical History", "").strip()
    if pmh:
        pmh_satirlar = [s.strip() for s in re.split(r"[\n;]+", pmh) if s.strip()]
        kisa = [s for s in pmh_satirlar if 5 < len(s) < 100][:3]
        if not kisa:
            kisa = [c for c in cumlelere_bol(pmh)[:2] if len(c) < 200]
        if kisa:
            parts.append(f"Relevant History: " + "; ".join(kisa))

    al = sections.get("Allergies", "").strip()
    if al:
        al_kisa = al.split("\n")[0].strip()[:100]
        if al_kisa and al_kisa.upper() not in ("NKDA", "NONE", "NKA"):
            parts.append(f"Allergies: {al_kisa}")

    meds = sections.get("Medications on Admission", "").strip()
    if meds:
        ilac_satirlar = re.findall(r"\d+\.\s*([A-Z][A-Za-z][^.\n]{2,80})", meds)
        if ilac_satirlar:
            kisa_ilaclar = []
            for s in ilac_satirlar[:5]:
                ilac = s.split()[:3]
                kisa_ilaclar.append(" ".join(ilac))
            parts.append(f"Medications: " + ", ".join(kisa_ilaclar))

    return "\n".join(parts)

# Compact uygula
print("\n[1/2] V4 1-shot medoid compact'la...")
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)

compact_data = {}
for cls in ALLOWED_CLASSES:
    full = v4_data[cls]["triage_input_v4_timeaware_masked"]
    compact = compact_input(full)
    n_full = len(tok.encode(full, add_special_tokens=False))
    n_comp = len(tok.encode(compact, add_special_tokens=False))
    compact_data[cls] = {
        "note_id": v4_data[cls]["note_id"],
        "leakage_severity": v4_data[cls]["leakage_severity"],
        "full_token": n_full,
        "compact_token": n_comp,
        "compact_char": len(compact),
        "compact_text": compact,
    }
    print(f"   {cls:<25s} full={n_full:>4} -> compact={n_comp:>4} ({100*n_comp/n_full:.0f}%) sev={v4_data[cls]['leakage_severity']}")

toplam_full = sum(d["full_token"] for d in compact_data.values())
toplam_comp = sum(d["compact_token"] for d in compact_data.values())
print(f"\n   Toplam: full={toplam_full:,} -> compact={toplam_comp:,} ({100*toplam_comp/toplam_full:.0f}%)")
print(f"   Hedef: 11x ~200 token = ~2200")

# Kaydet
out_yol = FS_DIR / "fewshot_v4_compact_1shot_examples.json"
with open(out_yol, "w", encoding="utf-8") as f:
    json.dump(compact_data, f, ensure_ascii=False, indent=2)
print(f"\n   Kaydedildi: {out_yol.name}")

# Preview
print("\n[2/2] PREVIEW (her sinif compact)")
print("=" * 70)
for cls in ALLOWED_CLASSES:
    d = compact_data[cls]
    print(f"\n   {cls} (note: {d['note_id']}, tok: {d['compact_token']}, sev: {d['leakage_severity']}):")
    print(f"   {d['compact_text'][:400]}")

print("\n" + "=" * 70)
print("V4 1-shot compact TAMAMLANDI")
print("=" * 70)


# BLOK 14C-V4-VAL — V4 Few-shot 1-shot Val Inference

**Konfigürasyon:** 11 V4 compact medoid + V4 query (runtime compact), max_length=4096.

**Sonuç:**
- Süre: 45.5 dk (550 örnek × 4.96 s/örnek)
- **Full (n=550): acc=0.4873, macro-F1=0.4675**
- Valid only (n=544): acc=0.4926, macro-F1=0.5131
- Parser: direct=541 (%98.4), fuzzy=3, invalid=6 (%1.1)

**Karşılaştırma (val):**
| Yöntem | Acc | Macro-F1 |
|---|---:|---:|
| V4 zero-shot | 0.4818 | 0.4367 |
| **V4 few-shot 1-shot** | **0.4873** | **0.4675** (+0.0308) |
| V3 few-shot 1-shot (referans) | 0.5182 | 0.4909 |

**Per-class delta (V4 few-shot vs V4 zero-shot):**
- ⬆ AKI (+0.090), PNEUMONIA (+0.087), UTI (+0.110), HF (+0.058), SEPSIS (+0.055)
- ⬇ OTHER (-0.056), GI_BLEEDING (-0.025)

→ V4 few-shot V4 zeroshot'tan iyi, özellikle zayıf sınıflarda (AKI/PNEUMONIA/UTI) belirgin iyileşme.

**Çıktı:** `output/fewshot/fewshot_v4_compact_1shot_val.csv` + audit + summary
**Detaylı script:** `__run_blok14c_v4_val.py`

In [ ]:
"""Blok 14C-V4-VAL: V4 few-shot 1-shot val inference (compact medoid + V4 query).
max_length=4096, V4 input + 11 V4 compact medoid."""
import pandas as pd
import numpy as np
import torch
import json
import re
import time
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
FS_DIR     = OUTPUT_DIR / "fewshot"

print("=" * 70)
print("BLOK 14C-V4-VAL — V4 Few-shot 1-shot val (compact)")
print("=" * 70)
print("   KURAL: Sadece val. Test'e otomatik gecmez.")

# V4 compact medoid
with open(FS_DIR / "fewshot_v4_compact_1shot_examples.json", "r", encoding="utf-8") as f:
    compact_data = json.load(f)

ALLOWED_CLASSES = ["SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"]

# V4 query'yi de compact uretelim — eskisi V3 compact (triage_input_v3_compact),
# yeni V4 compact'i runtime'da uretiyoruz. Ya da V3 compact zaten tum input'tan
# uretildigi icin V4-spesifik bir compact lazim. Basit yol: V4 input'u parse edip compact uret.

SECTION_RE = re.compile(
    r"(Chief Complaint|Symptom History|Past Medical History|Allergies|"
    r"Medications on Admission|Vital Signs):\s*\n?", re.IGNORECASE)
SYMPTOM_KEYWORDS = re.compile(
    r"\b(presents?|reports?|complains?|denies|developed|noted|noticed|started|began|"
    r"pain|cough|fever|dyspnea|SOB|fatigue|weakness|nausea|vomit|diarrhea|bleeding|"
    r"dizz|confusion|swelling|edema|chest|abdomen|headache|hypoxic|hypotens|tachycard|"
    r"palpitations|orthopnea|wheez|melena|hemato|dysuria|frequen|urgency)\b", re.IGNORECASE)

def parse_sections(text):
    if not text: return {}
    parts = SECTION_RE.split(text)
    s = {}
    for i in range(1, len(parts) - 1, 2):
        s[parts[i].strip()] = parts[i+1].strip()
    return s

def cumlelere_bol(t):
    if not t: return []
    return [c.strip() for c in re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", t) if c.strip()]

def hasta_skor(c):
    return len(SYMPTOM_KEYWORDS.findall(c))

def compact_v4_query(full_text):
    sections = parse_sections(full_text)
    parts = []
    cc = sections.get("Chief Complaint", "").strip()
    if cc: parts.append(f"Chief Complaint: {cc}")
    sh = sections.get("Symptom History", "").strip()
    if sh:
        ros = re.search(r"(?im)^\s*(Review of Systems?|ROS)\s*:", sh)
        if ros: sh = sh[:ros.start()]
        cumleler = cumlelere_bol(sh)
        skorli = [(i, c, hasta_skor(c)) for i, c in enumerate(cumleler[:5])]
        secilen = [c for i, c, s in skorli if s > 0][:3]
        if not secilen:
            secilen = [c for i, c, s in skorli][:2]
        else:
            if cumleler and cumleler[0] not in secilen:
                secilen = [cumleler[0]] + secilen
                secilen = secilen[:3]
        if secilen: parts.append(f"Key Symptoms: {' '.join(secilen)}")
    vs = sections.get("Vital Signs", "").strip()
    if vs: parts.append(f"Vitals: {vs}")
    pmh = sections.get("Past Medical History", "").strip()
    if pmh:
        pmh_satirlar = [s.strip() for s in re.split(r"[\n;]+", pmh) if s.strip()]
        kisa = [s for s in pmh_satirlar if 5 < len(s) < 100][:3]
        if not kisa:
            kisa = [c for c in cumlelere_bol(pmh)[:2] if len(c) < 200]
        if kisa: parts.append(f"Relevant History: " + "; ".join(kisa))
    al = sections.get("Allergies", "").strip()
    if al:
        al_kisa = al.split("\n")[0].strip()[:100]
        if al_kisa and al_kisa.upper() not in ("NKDA","NONE","NKA"):
            parts.append(f"Allergies: {al_kisa}")
    meds = sections.get("Medications on Admission", "").strip()
    if meds:
        ilac_satirlar = re.findall(r"\d+\.\s*([A-Z][A-Za-z][^.\n]{2,80})", meds)
        if ilac_satirlar:
            kisa_ilaclar = []
            for s in ilac_satirlar[:5]:
                ilac = s.split()[:3]
                kisa_ilaclar.append(" ".join(ilac))
            parts.append(f"Medications: " + ", ".join(kisa_ilaclar))
    return "\n".join(parts)

HEADER = """You are a clinical NLP system. Given a patient's triage information collected at first contact (chief complaint, symptoms, brief history, medications, allergies, vital signs), classify the most likely PRIMARY reason for admission into ONE of these 11 categories:

SEPSIS - Systemic infection, septic shock, bacteremia
STROKE - Ischemic or hemorrhagic stroke, CVA
ATRIAL_FIBRILLATION - Atrial fibrillation/flutter (acute)
AKI - Acute kidney injury, acute renal failure
PNEUMONIA - Lung infection (bacterial, viral, aspiration)
UTI - Urinary tract infection, pyelonephritis, cystitis
HEART_FAILURE - Congestive heart failure, decompensated CHF
GI_BLEEDING - Upper or lower GI hemorrhage
DIABETES_COMPLICATION - DKA, HHS, severe hyper/hypoglycemia
COPD_EXACERBATION - Acute exacerbation of COPD
OTHER - None of the above

Rules:
- Output exactly one class label from the list, using the exact uppercase label string.
- Choose the SINGLE most likely PRIMARY reason for admission.

Below are 11 compact example triage notes (one per class):
"""
EX_TMPL = "\nEXAMPLE {i} ({label}):\n{input}\nPRIMARY DIAGNOSIS: {label}\n"
TT = "\nNow classify the following triage note:\nTRIAGE NOTE:\n{input}\nPRIMARY DIAGNOSIS:"

SABIT = HEADER
for i, cls in enumerate(ALLOWED_CLASSES, 1):
    SABIT += EX_TMPL.format(i=i, label=cls, input=compact_data[cls]["compact_text"])

FUZZY_MAP = {
    "SEPS":"SEPSIS","SEPT":"SEPSIS","PNEUM":"PNEUMONIA",
    "STROK":"STROKE","CVA":"STROKE",
    "ATRIAL":"ATRIAL_FIBRILLATION","AFIB":"ATRIAL_FIBRILLATION","FIBRIL":"ATRIAL_FIBRILLATION",
    "AKI":"AKI","KIDNEY":"AKI","RENAL":"AKI",
    "UTI":"UTI","URIN":"UTI",
    "HEART":"HEART_FAILURE","CHF":"HEART_FAILURE",
    "GI":"GI_BLEEDING","BLEED":"GI_BLEEDING",
    "DIAB":"DIABETES_COMPLICATION","DKA":"DIABETES_COMPLICATION",
    "COPD":"COPD_EXACERBATION","OTHER":"OTHER","NONE":"OTHER",
}

def parse_class(raw):
    if not raw: return "INVALID","empty"
    text = raw.strip().upper()
    for cls in sorted(ALLOWED_CLASSES, key=len, reverse=True):
        if cls in text[:80]: return cls,"direct"
    for k,v in FUZZY_MAP.items():
        if k in text[:80]: return v,"fuzzy"
    return "INVALID","invalid"

print("\n[1/4] Model yukleme...")
t0 = time.time()
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-4B", quantization_config=bnb,
    device_map="auto", torch_dtype=torch.float16, local_files_only=True)
model.eval()
print(f"   Yuklendi {time.time()-t0:.1f}s, GPU {torch.cuda.memory_allocated()/1024**2:.0f}MB")

MAX_LEN = 4096

def inference(test_input):
    prompt = SABIT + TT.format(input=test_input)
    msgs = [{"role":"user","content":prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return raw, inputs.input_ids.shape[1]

print("\n[2/4] V4 val seti...")
df = pd.read_parquet(SPLIT_YOL)
val_df = df[df["split"]=="val"].reset_index(drop=True)
print(f"   Val: {len(val_df):,}")

print("\n[3/4] V4 compact 1-shot inference...")
cp_yol = FS_DIR / "fewshot_v4_compact_1shot_val_checkpoint.csv"
final_yol = FS_DIR / "fewshot_v4_compact_1shot_val.csv"

if cp_yol.exists():
    cp = pd.read_csv(cp_yol)
    islenen = set(cp["note_id"].astype(str))
    kayitlar = cp.to_dict("records")
    print(f"   Resume: {len(islenen)} kayit zaten var.")
else:
    islenen = set()
    kayitlar = []

t_baslangic = time.time()
sayac_yeni = 0

for i, row in val_df.iterrows():
    nid = str(row["note_id"])
    if nid in islenen: continue
    # V4 query'yi runtime'da compact et
    v4_full = row["triage_input_v4_timeaware_masked"] or ""
    inp = compact_v4_query(v4_full)
    t0 = time.time()
    raw, in_n = inference(inp)
    sure = time.time() - t0
    parsed, kat = parse_class(raw)
    kayitlar.append({
        "note_id": nid, "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]), "label": row["label"],
        "raw_output": raw, "parsed_pred": parsed,
        "parse_kategori": kat, "in_tokens": int(in_n),
        "sure_s": round(sure, 3),
        "leakage_severity_final": str(row.get("strict_current_leakage_severity_final","UNK")),
    })
    sayac_yeni += 1
    if len(kayitlar) % 25 == 0:
        pd.DataFrame(kayitlar).to_csv(cp_yol, index=False)
        gecen = time.time() - t_baslangic
        ort = gecen / sayac_yeni if sayac_yeni > 0 else 0
        kalan = (len(val_df) - len(kayitlar)) * ort
        print(f"   [{len(kayitlar):>3}/{len(val_df)}] gecen {gecen/60:.1f}dk ort {ort:.2f}s kalan ~{kalan/60:.1f}dk")

df_son = pd.DataFrame(kayitlar)
df_son.to_csv(final_yol, index=False)
df_son.to_csv(cp_yol, index=False)
toplam = time.time() - t_baslangic
print(f"   BITTI. {len(kayitlar)} kayit, {toplam/60:.1f}dk")

# AUDIT
print("\n[4/4] V4 FEW-SHOT VAL AUDIT")
print("=" * 70)
n = len(df_son)
ks = df_son["parse_kategori"].value_counts().to_dict()
direct = ks.get("direct", 0); fuzzy = ks.get("fuzzy", 0)
invalid = ks.get("invalid", 0) + ks.get("empty", 0)
print(f"\nParser: direct={direct} ({100*direct/n:.1f}%) fuzzy={fuzzy} invalid={invalid} ({100*invalid/n:.1f}%)")

y_true = df_son["label"].values
y_pred = df_son["parsed_pred"].values
valid = y_pred != "INVALID"
acc_full = accuracy_score(y_true, y_pred)
mf1_full = f1_score(y_true, y_pred, average="macro", zero_division=0)
acc_v = accuracy_score(y_true[valid], y_pred[valid]) if valid.sum() else 0.0
mf1_v = f1_score(y_true[valid], y_pred[valid], average="macro", zero_division=0) if valid.sum() else 0.0

print(f"\nV4 Few-shot Val:")
print(f"   Full        n={n}    acc={acc_full:.4f} macroF1={mf1_full:.4f}")
print(f"   Valid only  n={valid.sum()}  acc={acc_v:.4f} macroF1={mf1_v:.4f}")

labels = sorted(set(y_true) | (set(y_pred) - {"INVALID"}))
p, r, f, supp = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
print(f"\nPer-class:")
for cls, pi, ri, fi, si in zip(labels, p, r, f, supp):
    print(f"   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}")

# Karsilastirma: V4 zero-shot + V3 few-shot
print(f"\nKarsilastirma (val):")
v4_zs_yol = OUTPUT_DIR / "zeroshot" / "zeroshot_v4_val_results_summary.csv"
v3_fs_yol = FS_DIR / "fewshot_v3_compact_1shot_val_results_summary.csv"
print(f"   {'':<22s} {'Acc':>8s} {'MacroF1':>10s}")
print(f"   {'V4 few-shot 1-shot':<22s} {acc_full:>8.4f} {mf1_full:>10.4f}")
if v4_zs_yol.exists():
    v4_zs = pd.read_csv(v4_zs_yol)
    v4_zs_acc = float(v4_zs[v4_zs["subset"]=="full"]["accuracy"].iloc[0])
    v4_zs_mf1 = float(v4_zs[v4_zs["subset"]=="full"]["macro_f1"].iloc[0])
    print(f"   {'V4 zero-shot':<22s} {v4_zs_acc:>8.4f} {v4_zs_mf1:>10.4f}  (delta {acc_full-v4_zs_acc:+.4f})")
if v3_fs_yol.exists():
    v3_fs = pd.read_csv(v3_fs_yol)
    v3_fs_acc = float(v3_fs[v3_fs["subset"]=="full"]["accuracy"].iloc[0])
    v3_fs_mf1 = float(v3_fs[v3_fs["subset"]=="full"]["macro_f1"].iloc[0])
    print(f"   {'V3 few-shot 1-shot':<22s} {v3_fs_acc:>8.4f} {v3_fs_mf1:>10.4f}  (delta {acc_full-v3_fs_acc:+.4f})")

# Summary
ozet = pd.DataFrame({
    "subset":["full","valid_only"],
    "n":[n, valid.sum()],
    "accuracy":[acc_full, acc_v],
    "macro_f1":[mf1_full, mf1_v],
})
ozet.to_csv(FS_DIR / "fewshot_v4_compact_1shot_val_results_summary.csv", index=False)

# Karar
print("\n" + "=" * 70)
print("KARAR")
print("=" * 70)
print(f"   V4 few-shot val acc: {acc_full:.4f}")
print(f"   TEST'e otomatik gecis YOK.")
print("=" * 70)


# BLOK 14V4-CLEAN-FEWSHOT-EXAMPLES — Compact Örnek Audit + Temizlik

**Amaç:** `fewshot_v4_compact_1shot_examples.json` içindeki compact örnekleri
ayrıca **forbidden phrase** audit'inden geçirir. Source severity CLEAN/HISTORICAL
yetmez; compact text'in kendisi de current encounter ifadeleri içermemeli.

**Forbidden compact example phrases:**
- Code stroke / CT scanning
- REASON FOR MICU ADMISSION
- scheduled TEE / DCCV
- elevated Cr / abnormal labs
- found in ED / found to have
- given oxygen / CPAP / nebulizer (treatment)
- admitted for / MICU admission / ED course
- CXR/CT/MRI/TTE showed/revealed
- labs showed/notable/revealed

**Strateji:**
1. 11 compact örnek tek tek audit
2. Riskli cümle varsa: cümle çıkar VEYA aynı sınıftan alternatif medoid (rank 2/3...) seç
3. HIGH/MEDIUM/REVIEW = 0 olana kadar devam

**Sonuç:**
- 5 sınıfta cümle silindi (SEPSIS, STROKE, AFIB, AKI)
- 1 sınıfta alternatif medoid (PNEUMONIA → <note_id>)
- 5 sınıf zaten temizdi
- **Tüm 11 örnek temiz, toplam 2.575 token**

**Çıktı:** `output/fewshot/fewshot_v4_compact_1shot_clean_examples.json`
**Detaylı script:** `__run_blok14_v4_clean_fewshot.py`

In [ ]:
"""Blok 14V4-CLEAN-FEWSHOT-EXAMPLES: Compact fewshot ornekleri tek tek audit + temizlik.
Source severity yetmez, compact text de temiz olmali.
Forbidden phrases varsa cumle cikar veya alternatif medoid sec."""
import pandas as pd
import numpy as np
import json
import re
import time
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
FS_DIR     = OUTPUT_DIR / "fewshot"

print("=" * 70)
print("BLOK 14V4-CLEAN-FEWSHOT-EXAMPLES — Compact ornek audit + temizlik")
print("=" * 70)

ALLOWED_CLASSES = ["SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"]

# ============================================================
# FORBIDDEN COMPACT PHRASE PATTERNS (kullanici listesi)
# ============================================================
FORBIDDEN_PATTERNS = [
    ("code_stroke", re.compile(r"\bcode\s+(stroke|MI|sepsis|trauma)\b", re.IGNORECASE)),
    ("ct_scanning", re.compile(r"\b(CT scanning|reviewed images|read images|reviewed CT|reviewed scan)\b", re.IGNORECASE)),
    ("micu_admission", re.compile(r"\b(REASON FOR (MICU|ICU) ADMISSION|MICU admission|ICU admission)\b", re.IGNORECASE)),
    ("scheduled_proc", re.compile(r"\bscheduled\s+(TEE|DCCV|EGD|colonoscopy|cardioversion|TTE)\b", re.IGNORECASE)),
    ("elevated_lab", re.compile(r"\b(elevated\s+(Cr|creatinine|WBC|Hgb|HCT|lactate|troponin|BUN|sodium|Na)|abnormal\s+labs?)\b", re.IGNORECASE)),
    ("found_in_ed", re.compile(r"\b(found\s+in\s+ED|found\s+to\s+have|found\s+down)\b", re.IGNORECASE)),
    ("given_treatment", re.compile(r"\b(given|received|started\s+on|placed\s+on|treated\s+with)\s+[^.]{0,40}\b(oxygen|CPAP|BiPAP|high\s*flow|nebulizer|nebs|albuterol|duonebs|antibiotics|abx|vancomycin|zosyn|lasix|insulin\s+drip|glucagon|tPA|pressors|levophed|norepinephrine|vasopressin|crystalloid|IV\s+fluids|fluid\s+bolus)\b", re.IGNORECASE)),
    ("admitted_for", re.compile(r"\b(admitted\s+(for|to)|MICU\s+admission|ED\s+course)\b", re.IGNORECASE)),
    ("imaging_showed", re.compile(r"\b(CXR|CTA?|MRI|TTE|TEE|US|echocardiogram)\b[^.]{0,40}\b(showed|revealed|demonstrated|consistent\s+with|notable|w/|with\s+(infiltrate|consolidation|effusion|edema|ICH|infarct|hemorrhage))\b", re.IGNORECASE)),
    ("labs_showed", re.compile(r"\bLabs?\b[^.]{0,40}\b(showed|notable|revealed|drawn|remarkable)\b", re.IGNORECASE)),
    # Ek: presented to ED + symptom var ise OK, ama "presented to ED with workup" gibi sentence atilir
    ("presented_workup", re.compile(r"\bpresent(ed|s|ing)\s+to\s+(the\s+)?ED\s+(for|with)\s+(workup|further\s+evaluation|management|admission|treatment)\b", re.IGNORECASE)),
]

def find_forbidden(text):
    """Riskli phrase varsa (pattern_ad, eslesen, konum) listesi don."""
    if not isinstance(text, str) or not text:
        return []
    bulgular = []
    for ad, rx in FORBIDDEN_PATTERNS:
        for m in rx.finditer(text):
            bulgular.append((ad, m.group(), m.start(), m.end()))
    return bulgular

def cumlelere_bol(t):
    if not t: return []
    return [c.strip() for c in re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", t) if c.strip()]

def temizle_cumle_bazli(compact_text):
    """Riskli pattern iceren cumleleri at, kalanlari birlestir.
    Compact format: 'Chief Complaint: X\nKey Symptoms: Y\nVitals: Z\n...' satir bazli."""
    if not isinstance(compact_text, str): return compact_text, 0
    # Section ayraclari korunarak satir-bazli isle
    yeni_satirlar = []
    atilan = 0
    for satir in compact_text.split("\n"):
        if not satir.strip():
            yeni_satirlar.append(satir)
            continue
        # Section header'i koru (Chief Complaint:, Key Symptoms: ...)
        if re.match(r"^[A-Z][A-Za-z ]+:", satir):
            # Header + icerik tek satir genelde — icerigini cumlelere bol
            m = re.match(r"^([A-Z][A-Za-z ]+:\s*)(.*)$", satir)
            if m:
                header = m.group(1)
                icerik = m.group(2)
                cumleler = cumlelere_bol(icerik)
                temiz_cumleler = []
                for c in cumleler:
                    bulg = find_forbidden(c)
                    if bulg:
                        atilan += 1
                    else:
                        temiz_cumleler.append(c)
                if temiz_cumleler:
                    yeni_satirlar.append(header + " ".join(temiz_cumleler))
                # else: sectionun tamami atildi, hic ekleme
            else:
                yeni_satirlar.append(satir)
        else:
            # Header'siz satir — direkt forbidden check
            bulg = find_forbidden(satir)
            if bulg:
                atilan += 1
            else:
                yeni_satirlar.append(satir)
    return "\n".join(yeni_satirlar).strip(), atilan

# ============================================================
# Section-aware compaction (14D mantigi, alternative medoid icin)
# ============================================================
SECTION_RE = re.compile(
    r"(Chief Complaint|Symptom History|Past Medical History|Allergies|"
    r"Medications on Admission|Vital Signs):\s*\n?", re.IGNORECASE)
SYMPTOM_KEYWORDS = re.compile(
    r"\b(presents?|reports?|complains?|denies|developed|noted|noticed|started|began|"
    r"pain|cough|fever|dyspnea|SOB|fatigue|weakness|nausea|vomit|diarrhea|bleeding|"
    r"dizz|confusion|swelling|edema|chest|abdomen|headache|hypoxic|hypotens|tachycard|"
    r"palpitations|orthopnea|wheez|melena|hemato|dysuria|frequen|urgency)\b", re.IGNORECASE)

def parse_sections(text):
    if not text: return {}
    parts = SECTION_RE.split(text)
    s = {}
    for i in range(1, len(parts) - 1, 2):
        s[parts[i].strip()] = parts[i+1].strip()
    return s

def hasta_skor(c):
    return len(SYMPTOM_KEYWORDS.findall(c))

def compact_from_full(full_text):
    sections = parse_sections(full_text)
    parts = []
    cc = sections.get("Chief Complaint", "").strip()
    if cc: parts.append(f"Chief Complaint: {cc}")
    sh = sections.get("Symptom History", "").strip()
    if sh:
        ros = re.search(r"(?im)^\s*(Review of Systems?|ROS)\s*:", sh)
        if ros: sh = sh[:ros.start()]
        cumleler = cumlelere_bol(sh)
        skorli = [(i, c, hasta_skor(c)) for i, c in enumerate(cumleler[:5])]
        secilen = [c for i, c, s in skorli if s > 0][:3]
        if not secilen:
            secilen = [c for i, c, s in skorli][:2]
        else:
            if cumleler and cumleler[0] not in secilen:
                secilen = [cumleler[0]] + secilen
                secilen = secilen[:3]
        if secilen: parts.append(f"Key Symptoms: {' '.join(secilen)}")
    vs = sections.get("Vital Signs", "").strip()
    if vs: parts.append(f"Vitals: {vs}")
    pmh = sections.get("Past Medical History", "").strip()
    if pmh:
        pmh_satirlar = [s.strip() for s in re.split(r"[\n;]+", pmh) if s.strip()]
        kisa = [s for s in pmh_satirlar if 5 < len(s) < 100][:3]
        if not kisa:
            kisa = [c for c in cumlelere_bol(pmh)[:2] if len(c) < 200]
        if kisa: parts.append(f"Relevant History: " + "; ".join(kisa))
    al = sections.get("Allergies", "").strip()
    if al:
        al_kisa = al.split("\n")[0].strip()[:100]
        if al_kisa and al_kisa.upper() not in ("NKDA","NONE","NKA"):
            parts.append(f"Allergies: {al_kisa}")
    meds = sections.get("Medications on Admission", "").strip()
    if meds:
        ilac_satirlar = re.findall(r"\d+\.\s*([A-Z][A-Za-z][^.\n]{2,80})", meds)
        if ilac_satirlar:
            kisa_ilaclar = []
            for s in ilac_satirlar[:5]:
                ilac = s.split()[:3]
                kisa_ilaclar.append(" ".join(ilac))
            parts.append(f"Medications: " + ", ".join(kisa_ilaclar))
    return "\n".join(parts)

# ============================================================
# Mevcut compact'leri yukle
# ============================================================
print("\n[1/5] Mevcut compact 1-shot ornekleri yukleniyor...")
with open(FS_DIR / "fewshot_v4_compact_1shot_examples.json", "r", encoding="utf-8") as f:
    compact_eski = json.load(f)

# Audit her ornek
print("\n[2/5] Forbidden phrase audit (her sinif)...")
print("=" * 70)
audit_sonuc = {}
for cls in ALLOWED_CLASSES:
    txt = compact_eski[cls]["compact_text"]
    bulg = find_forbidden(txt)
    audit_sonuc[cls] = {
        "note_id": compact_eski[cls]["note_id"],
        "forbidden_count": len(bulg),
        "forbidden_details": [(ad, eslesen) for ad, eslesen, _, _ in bulg],
    }
    isaret = "❌ KIRLI" if bulg else "✅ TEMIZ"
    print(f"\n   {cls:<25s} {isaret}")
    print(f"      note: {compact_eski[cls]['note_id']}")
    if bulg:
        for ad, eslesen, _, _ in bulg:
            print(f"      RISK: {ad} = {eslesen!r}")

kirli = [c for c in ALLOWED_CLASSES if audit_sonuc[c]["forbidden_count"] > 0]
print(f"\n   Kirli sinif sayisi: {len(kirli)}/{len(ALLOWED_CLASSES)}")
print(f"   Kirli siniflar: {kirli}")

# ============================================================
# Pool yukle (alternative aday icin)
# ============================================================
print("\n[3/5] Train pool + TF-IDF (alternatif medoid icin)...")
df = pd.read_parquet(SPLIT_YOL)
df["note_id"] = df["note_id"].astype(str)
pool_full = df[df["split"]=="train_S3"].reset_index(drop=True)
LEAKAGE_KOLON = "strict_current_leakage_severity_final"
if LEAKAGE_KOLON not in pool_full.columns:
    LEAKAGE_KOLON = "strict_current_leakage_severity"
clean_mask = pool_full[LEAKAGE_KOLON].isin(["CLEAN","HISTORICAL_ALLOWED"])
pool = pool_full[clean_mask].reset_index(drop=True)
print(f"   Pool: {len(pool):,} V4 clean")

vec = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=2,
                      stop_words="english", sublinear_tf=True)
X = vec.fit_transform(pool["triage_input_v4_timeaware_masked"].fillna("").tolist())
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)

# ============================================================
# Cumle bazli temizleme + alternatif medoid arama
# ============================================================
print("\n[4/5] Temizlik: cumle-bazli + alternatif medoid arama...")
print("=" * 70)

yeni_compact = {}
final_audit = {}

for cls in ALLOWED_CLASSES:
    eski = compact_eski[cls]
    if audit_sonuc[cls]["forbidden_count"] == 0:
        # Zaten temiz
        yeni_compact[cls] = eski.copy()
        final_audit[cls] = {"strategy": "kept_original", "kayit_no": 0,
                            "note_id": eski["note_id"]}
        print(f"\n   {cls:<25s} ZATEN TEMIZ (note: {eski['note_id']})")
        continue

    # 1. Once cumle-bazli temizle
    temiz_text, atilan = temizle_cumle_bazli(eski["compact_text"])
    bulg_after = find_forbidden(temiz_text)
    if not bulg_after and temiz_text.strip() and len(temiz_text) > 100:
        # Cumle temizligi yetti
        n_token = len(tok.encode(temiz_text, add_special_tokens=False))
        yeni_compact[cls] = {
            "note_id": eski["note_id"],
            "leakage_severity": eski["leakage_severity"],
            "full_token": eski["full_token"],
            "compact_token": n_token,
            "compact_char": len(temiz_text),
            "compact_text": temiz_text,
            "cleaning_strategy": "sentence_removed",
            "atilan_cumle": atilan,
        }
        final_audit[cls] = {"strategy":"sentence_removed","atilan":atilan,
                            "note_id": eski["note_id"]}
        print(f"\n   {cls:<25s} CUMLE TEMIZLENDI ({atilan} cumle atildi, kalan: {n_token} tok)")
        continue

    # 2. Cumle temizligi yetmedi -> alternatif medoid ara
    print(f"\n   {cls:<25s} CUMLE TEMIZLIGI YETMEDI - alternatif aday araniyor...")
    cls_idx = np.where(pool["label"].values == cls)[0]
    if len(cls_idx) == 0:
        print(f"      UYARI: pool'da bu sinif yok!")
        continue

    X_cls = X[cls_idx]
    centroid = np.asarray(X_cls.mean(axis=0)).reshape(1,-1)
    sim_cent = cosine_similarity(X_cls, centroid).flatten()
    rank_lokal = np.argsort(-sim_cent)

    # Mevcut note_id'yi atla, sonraki adaylari dene
    eski_nid = eski["note_id"]
    bulundu = False
    rank_taranan = 0
    for r, lokal in enumerate(rank_lokal):
        rank_taranan = r + 1
        global_idx = cls_idx[int(lokal)]
        row = pool.iloc[global_idx]
        if str(row["note_id"]) == str(eski_nid):
            continue  # mevcut atla

        full_text = row["triage_input_v4_timeaware_masked"] or ""
        compact_aday = compact_from_full(full_text)
        bulg = find_forbidden(compact_aday)
        if not bulg and compact_aday.strip() and len(compact_aday) > 100:
            # Temiz aday
            n_token = len(tok.encode(compact_aday, add_special_tokens=False))
            yeni_compact[cls] = {
                "note_id": str(row["note_id"]),
                "leakage_severity": str(row[LEAKAGE_KOLON]),
                "full_token": len(tok.encode(full_text, add_special_tokens=False)),
                "compact_token": n_token,
                "compact_char": len(compact_aday),
                "compact_text": compact_aday,
                "cleaning_strategy": "alternative_medoid",
                "atlanan_rank": rank_taranan - 1,
                "centroid_sim": float(sim_cent[lokal]),
            }
            final_audit[cls] = {"strategy":"alternative_medoid",
                                "atlanan_rank": rank_taranan - 1,
                                "note_id": str(row["note_id"])}
            print(f"      ALTERNATIF rank {rank_taranan}: note={row['note_id']} sim={sim_cent[lokal]:.3f} tok={n_token}")
            bulundu = True
            break

    if not bulundu:
        print(f"      UYARI: temiz alternatif bulunamadi! Cumle temizligi sonucu kullanilacak.")
        # Geri donus: cumle temizligi sonucu (riskli olsa bile)
        n_token = len(tok.encode(temiz_text, add_special_tokens=False))
        yeni_compact[cls] = {
            "note_id": eski["note_id"],
            "leakage_severity": eski["leakage_severity"],
            "full_token": eski["full_token"],
            "compact_token": n_token,
            "compact_char": len(temiz_text),
            "compact_text": temiz_text,
            "cleaning_strategy": "sentence_removed_fallback",
            "atilan_cumle": atilan,
        }
        final_audit[cls] = {"strategy":"fallback","note_id": eski["note_id"]}

# ============================================================
# Final audit (yeni compact'lar uzerinde)
# ============================================================
print("\n[5/5] FINAL AUDIT (yeni compact'lar uzerinde)")
print("=" * 70)

hala_kirli = 0
toplam_token = 0
print(f"\n   {'Sinif':<25s} {'Strateji':<25s} {'note':<22s} {'tok':>5s} {'risk':>5s}")
print("   " + "-" * 90)
for cls in ALLOWED_CLASSES:
    if cls not in yeni_compact:
        print(f"   {cls:<25s} HATA — yok")
        continue
    d = yeni_compact[cls]
    bulg = find_forbidden(d["compact_text"])
    if bulg: hala_kirli += 1
    toplam_token += d["compact_token"]
    print(f"   {cls:<25s} {d.get('cleaning_strategy','?'):<25s} {d['note_id']:<22s} {d['compact_token']:>5} {len(bulg):>5}")
    if bulg:
        for ad, eslesen, _, _ in bulg[:2]:
            print(f"      → KALAN RISK: {ad} = {eslesen!r}")

print(f"\n   Hala kirli ornek: {hala_kirli}/{len(ALLOWED_CLASSES)}")
print(f"   Toplam token: {toplam_token:,}")
print(f"   Ortalama: {toplam_token/len(yeni_compact):.0f}")

# Acceptance
print("\n   ACCEPTANCE:")
if hala_kirli == 0:
    print(f"   ✅ Tum 11 ornek temiz. V4 fewshot final.")
else:
    print(f"   ⚠️ {hala_kirli} ornek hala kirli. Manuel inceleme gerekli.")

# Kaydet (yeni dosya)
out_yol = FS_DIR / "fewshot_v4_compact_1shot_clean_examples.json"
with open(out_yol, "w", encoding="utf-8") as f:
    json.dump(yeni_compact, f, ensure_ascii=False, indent=2)
print(f"\n   Kaydedildi: {out_yol.name}")

# Preview
print("\n" + "=" * 70)
print("PREVIEW (yeni compact ornekleri)")
print("=" * 70)
for cls in ALLOWED_CLASSES:
    if cls not in yeni_compact: continue
    d = yeni_compact[cls]
    print(f"\n   {cls} (note: {d['note_id']}, strategy: {d.get('cleaning_strategy','?')}, tok: {d['compact_token']}):")
    print(f"   {d['compact_text'][:500]}")

print("\n" + "=" * 70)
print("BLOK 14V4-CLEAN-FEWSHOT TAMAMLANDI - inference YOK")
print("=" * 70)


# BLOK 14C-V4-CLEAN-VAL — V4 TEMIZ Few-shot Val Inference

**Konfigürasyon:** 11 V4 **temiz** compact medoid (cümle bazlı temizlik + alternatif medoid uygulanmış)
+ V4 query (runtime compact), max_length=4096, max_new_tokens=20, do_sample=False.

**Beklenen süre:** ~45 dakika (550 örnek × ~5 s/örnek)

**Beklenen karşılaştırma (val):**
| Yöntem | Acc | Macro-F1 |
|---|---:|---:|
| V4 zero-shot | 0.4818 | 0.4367 |
| V4 fewshot ESKİ (kirli medoid) | 0.4873 | 0.4675 |
| **V4 fewshot CLEAN (bu blok)** | **? (çalıştırılacak)** | **?** |
| V3 fewshot 1-shot (referans) | 0.5182 | 0.4909 |

**Çıktılar:**
- `output/fewshot/fewshot_v4_clean_1shot_val.csv`
- `output/fewshot/fewshot_v4_clean_1shot_val_results_summary.csv`
- Checkpoint: `fewshot_v4_clean_1shot_val_checkpoint.csv`

**Detaylı script:** `__run_blok14c_v4_clean_val.py`

⚠️ **Henüz çalıştırılmadı.** Sen çalıştır. Komut:
```bash
python ./workdir/__run_blok14c_v4_clean_val.py
```

In [ ]:
"""Blok 14C-V4-CLEAN-VAL: V4 TEMIZ medoid'lerle few-shot val (550 ornek)."""
import pandas as pd
import numpy as np
import torch
import json
import re
import time
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
FS_DIR     = OUTPUT_DIR / "fewshot"

print("=" * 70)
print("BLOK 14C-V4-CLEAN-VAL — V4 TEMIZ medoid + few-shot val")
print("=" * 70)

# TEMIZ compact medoid (14V4-CLEAN-FEWSHOT cikti)
with open(FS_DIR / "fewshot_v4_compact_1shot_clean_examples.json", "r", encoding="utf-8") as f:
    compact_data = json.load(f)

ALLOWED_CLASSES = ["SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"]

# V4 query'yi runtime'da compact et (ayni mantik)
SECTION_RE = re.compile(
    r"(Chief Complaint|Symptom History|Past Medical History|Allergies|"
    r"Medications on Admission|Vital Signs):\s*\n?", re.IGNORECASE)
SYMPTOM_KEYWORDS = re.compile(
    r"\b(presents?|reports?|complains?|denies|developed|noted|noticed|started|began|"
    r"pain|cough|fever|dyspnea|SOB|fatigue|weakness|nausea|vomit|diarrhea|bleeding|"
    r"dizz|confusion|swelling|edema|chest|abdomen|headache|hypoxic|hypotens|tachycard|"
    r"palpitations|orthopnea|wheez|melena|hemato|dysuria|frequen|urgency)\b", re.IGNORECASE)

def parse_sections(text):
    if not text: return {}
    parts = SECTION_RE.split(text)
    s = {}
    for i in range(1, len(parts) - 1, 2):
        s[parts[i].strip()] = parts[i+1].strip()
    return s

def cumlelere_bol(t):
    if not t: return []
    return [c.strip() for c in re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", t) if c.strip()]

def hasta_skor(c):
    return len(SYMPTOM_KEYWORDS.findall(c))

def compact_v4_query(full_text):
    sections = parse_sections(full_text)
    parts = []
    cc = sections.get("Chief Complaint", "").strip()
    if cc: parts.append(f"Chief Complaint: {cc}")
    sh = sections.get("Symptom History", "").strip()
    if sh:
        ros = re.search(r"(?im)^\s*(Review of Systems?|ROS)\s*:", sh)
        if ros: sh = sh[:ros.start()]
        cumleler = cumlelere_bol(sh)
        skorli = [(i, c, hasta_skor(c)) for i, c in enumerate(cumleler[:5])]
        secilen = [c for i, c, s in skorli if s > 0][:3]
        if not secilen:
            secilen = [c for i, c, s in skorli][:2]
        else:
            if cumleler and cumleler[0] not in secilen:
                secilen = [cumleler[0]] + secilen
                secilen = secilen[:3]
        if secilen: parts.append(f"Key Symptoms: {' '.join(secilen)}")
    vs = sections.get("Vital Signs", "").strip()
    if vs: parts.append(f"Vitals: {vs}")
    pmh = sections.get("Past Medical History", "").strip()
    if pmh:
        pmh_satirlar = [s.strip() for s in re.split(r"[\n;]+", pmh) if s.strip()]
        kisa = [s for s in pmh_satirlar if 5 < len(s) < 100][:3]
        if not kisa:
            kisa = [c for c in cumlelere_bol(pmh)[:2] if len(c) < 200]
        if kisa: parts.append(f"Relevant History: " + "; ".join(kisa))
    al = sections.get("Allergies", "").strip()
    if al:
        al_kisa = al.split("\n")[0].strip()[:100]
        if al_kisa and al_kisa.upper() not in ("NKDA","NONE","NKA"):
            parts.append(f"Allergies: {al_kisa}")
    meds = sections.get("Medications on Admission", "").strip()
    if meds:
        ilac_satirlar = re.findall(r"\d+\.\s*([A-Z][A-Za-z][^.\n]{2,80})", meds)
        if ilac_satirlar:
            kisa_ilaclar = []
            for s in ilac_satirlar[:5]:
                ilac = s.split()[:3]
                kisa_ilaclar.append(" ".join(ilac))
            parts.append(f"Medications: " + ", ".join(kisa_ilaclar))
    return "\n".join(parts)

HEADER = """You are a clinical NLP system. Given a patient's triage information collected at first contact (chief complaint, symptoms, brief history, medications, allergies, vital signs), classify the most likely PRIMARY reason for admission into ONE of these 11 categories:

SEPSIS - Systemic infection, septic shock, bacteremia
STROKE - Ischemic or hemorrhagic stroke, CVA
ATRIAL_FIBRILLATION - Atrial fibrillation/flutter (acute)
AKI - Acute kidney injury, acute renal failure
PNEUMONIA - Lung infection (bacterial, viral, aspiration)
UTI - Urinary tract infection, pyelonephritis, cystitis
HEART_FAILURE - Congestive heart failure, decompensated CHF
GI_BLEEDING - Upper or lower GI hemorrhage
DIABETES_COMPLICATION - DKA, HHS, severe hyper/hypoglycemia
COPD_EXACERBATION - Acute exacerbation of COPD
OTHER - None of the above

Rules:
- Output exactly one class label from the list, using the exact uppercase label string.
- Choose the SINGLE most likely PRIMARY reason for admission.

Below are 11 compact example triage notes (one per class):
"""
EX_TMPL = "\nEXAMPLE {i} ({label}):\n{input}\nPRIMARY DIAGNOSIS: {label}\n"
TT = "\nNow classify the following triage note:\nTRIAGE NOTE:\n{input}\nPRIMARY DIAGNOSIS:"

SABIT = HEADER
for i, cls in enumerate(ALLOWED_CLASSES, 1):
    SABIT += EX_TMPL.format(i=i, label=cls, input=compact_data[cls]["compact_text"])

FUZZY_MAP = {
    "SEPS":"SEPSIS","SEPT":"SEPSIS","PNEUM":"PNEUMONIA",
    "STROK":"STROKE","CVA":"STROKE",
    "ATRIAL":"ATRIAL_FIBRILLATION","AFIB":"ATRIAL_FIBRILLATION","FIBRIL":"ATRIAL_FIBRILLATION",
    "AKI":"AKI","KIDNEY":"AKI","RENAL":"AKI",
    "UTI":"UTI","URIN":"UTI",
    "HEART":"HEART_FAILURE","CHF":"HEART_FAILURE",
    "GI":"GI_BLEEDING","BLEED":"GI_BLEEDING",
    "DIAB":"DIABETES_COMPLICATION","DKA":"DIABETES_COMPLICATION",
    "COPD":"COPD_EXACERBATION","OTHER":"OTHER","NONE":"OTHER",
}

def parse_class(raw):
    if not raw: return "INVALID","empty"
    text = raw.strip().upper()
    for cls in sorted(ALLOWED_CLASSES, key=len, reverse=True):
        if cls in text[:80]: return cls,"direct"
    for k,v in FUZZY_MAP.items():
        if k in text[:80]: return v,"fuzzy"
    return "INVALID","invalid"

print("\n[1/4] Model yukleme...")
t0 = time.time()
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-4B", quantization_config=bnb,
    device_map="auto", torch_dtype=torch.float16, local_files_only=True)
model.eval()
print(f"   Yuklendi {time.time()-t0:.1f}s, GPU {torch.cuda.memory_allocated()/1024**2:.0f}MB")

MAX_LEN = 4096

def inference(test_input):
    prompt = SABIT + TT.format(input=test_input)
    msgs = [{"role":"user","content":prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return raw, inputs.input_ids.shape[1]

print("\n[2/4] V4 val seti...")
df = pd.read_parquet(SPLIT_YOL)
val_df = df[df["split"]=="val"].reset_index(drop=True)
print(f"   Val: {len(val_df):,}")

print("\n[3/4] V4 CLEAN compact 1-shot inference...")
cp_yol = FS_DIR / "fewshot_v4_clean_1shot_val_checkpoint.csv"
final_yol = FS_DIR / "fewshot_v4_clean_1shot_val.csv"

if cp_yol.exists():
    cp = pd.read_csv(cp_yol)
    islenen = set(cp["note_id"].astype(str))
    kayitlar = cp.to_dict("records")
    print(f"   Resume: {len(islenen)} kayit zaten var.")
else:
    islenen = set()
    kayitlar = []

t_baslangic = time.time()
sayac_yeni = 0

for i, row in val_df.iterrows():
    nid = str(row["note_id"])
    if nid in islenen: continue
    v4_full = row["triage_input_v4_timeaware_masked"] or ""
    inp = compact_v4_query(v4_full)
    t0 = time.time()
    raw, in_n = inference(inp)
    sure = time.time() - t0
    parsed, kat = parse_class(raw)
    kayitlar.append({
        "note_id": nid, "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]), "label": row["label"],
        "raw_output": raw, "parsed_pred": parsed,
        "parse_kategori": kat, "in_tokens": int(in_n),
        "sure_s": round(sure, 3),
        "leakage_severity_final": str(row.get("strict_current_leakage_severity_final","UNK")),
    })
    sayac_yeni += 1
    if len(kayitlar) % 25 == 0:
        pd.DataFrame(kayitlar).to_csv(cp_yol, index=False)
        gecen = time.time() - t_baslangic
        ort = gecen / sayac_yeni if sayac_yeni > 0 else 0
        kalan = (len(val_df) - len(kayitlar)) * ort
        print(f"   [{len(kayitlar):>3}/{len(val_df)}] gecen {gecen/60:.1f}dk ort {ort:.2f}s kalan ~{kalan/60:.1f}dk")

df_son = pd.DataFrame(kayitlar)
df_son.to_csv(final_yol, index=False)
df_son.to_csv(cp_yol, index=False)
toplam = time.time() - t_baslangic
print(f"   BITTI. {len(kayitlar)} kayit, {toplam/60:.1f}dk")

# AUDIT
print("\n[4/4] V4 CLEAN FEW-SHOT VAL AUDIT")
print("=" * 70)
n = len(df_son)
ks = df_son["parse_kategori"].value_counts().to_dict()
direct = ks.get("direct", 0); fuzzy = ks.get("fuzzy", 0)
invalid = ks.get("invalid", 0) + ks.get("empty", 0)
print(f"\nParser: direct={direct} ({100*direct/n:.1f}%) fuzzy={fuzzy} invalid={invalid} ({100*invalid/n:.1f}%)")

y_true = df_son["label"].values
y_pred = df_son["parsed_pred"].values
valid = y_pred != "INVALID"
acc_full = accuracy_score(y_true, y_pred)
mf1_full = f1_score(y_true, y_pred, average="macro", zero_division=0)
acc_v = accuracy_score(y_true[valid], y_pred[valid]) if valid.sum() else 0.0
mf1_v = f1_score(y_true[valid], y_pred[valid], average="macro", zero_division=0) if valid.sum() else 0.0

print(f"\nV4 CLEAN Few-shot Val:")
print(f"   Full        n={n}    acc={acc_full:.4f} macroF1={mf1_full:.4f}")
print(f"   Valid only  n={valid.sum()}  acc={acc_v:.4f} macroF1={mf1_v:.4f}")

labels = sorted(set(y_true) | (set(y_pred) - {"INVALID"}))
p, r, f, supp = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
print(f"\nPer-class:")
for cls, pi, ri, fi, si in zip(labels, p, r, f, supp):
    print(f"   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}")

# Karsilastirma
print(f"\nKarsilastirma (val):")
print(f"   {'':<28s} {'Acc':>8s} {'MacroF1':>10s}")
print(f"   {'V4 zero-shot':<28s} {0.4818:>8.4f} {0.4367:>10.4f}")
print(f"   {'V4 fewshot ESKI (kirli)':<28s} {0.4873:>8.4f} {0.4675:>10.4f}")
print(f"   {'V4 fewshot CLEAN':<28s} {acc_full:>8.4f} {mf1_full:>10.4f}  *")
print(f"   {'V3 fewshot 1-shot':<28s} {0.5182:>8.4f} {0.4909:>10.4f}")

# Summary
ozet = pd.DataFrame({
    "subset":["full","valid_only"],
    "n":[n, valid.sum()],
    "accuracy":[acc_full, acc_v],
    "macro_f1":[mf1_full, mf1_v],
})
ozet.to_csv(FS_DIR / "fewshot_v4_clean_1shot_val_results_summary.csv", index=False)
print(f"\n   Kaydedildi: fewshot_v4_clean_1shot_val.csv + summary")

print("\n" + "=" * 70)
print("BLOK 14C-V4-CLEAN-VAL TAMAMLANDI - test'e otomatik gecis YOK")
print("=" * 70)


# BLOK 14C-V4-CLEAN-TEST — V4 TEMIZ Few-shot Test (FINAL)

**Konfigürasyon:** Aynı val ayarları, test split (n=1.650).

**Beklenen süre:** ~125 dakika (1.650 örnek × ~4.5 s)

**Audit:**
- Full test (1.650)
- Valid only
- Time-aware clean (CLEAN + HISTORICAL_ALLOWED): testte 1.650/1.650
- Keyword-clean (V3 referansı): ~948
- Per-class P/R/F1
- Confusion matrix
- Bias
- 4-yöntem final karşılaştırma (V3 zero, V4 zero, V4 fewshot kirli, **V4 fewshot CLEAN**)

**Çıktılar:**
- `output/fewshot/fewshot_v4_clean_1shot_test.csv`
- `output/fewshot/fewshot_v4_clean_1shot_test_audit.txt`
- `output/fewshot/fewshot_v4_clean_1shot_test_results_summary.csv`
- Checkpoint: `fewshot_v4_clean_1shot_test_checkpoint.csv`

**Detaylı script:** `__run_blok14c_v4_clean_test.py`

⚠️ **Henüz çalıştırılmadı.** Sen çalıştır:
```bash
python ./workdir/__run_blok14c_v4_clean_test.py
```

**Önerilen sıra:** Önce val'i çalıştır (45 dk), val sonucuna bakıp test'e geç (125 dk).

In [ ]:
"""Blok 14C-V4-CLEAN-TEST: V4 TEMIZ medoid + few-shot test (1.650 ornek)."""
import pandas as pd
import numpy as np
import torch
import json
import re
import time
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
FS_DIR     = OUTPUT_DIR / "fewshot"

print("=" * 70)
print("BLOK 14C-V4-CLEAN-TEST — V4 TEMIZ medoid + few-shot TEST (FINAL)")
print("=" * 70)

with open(FS_DIR / "fewshot_v4_compact_1shot_clean_examples.json", "r", encoding="utf-8") as f:
    compact_data = json.load(f)

ALLOWED_CLASSES = ["SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"]

SECTION_RE = re.compile(
    r"(Chief Complaint|Symptom History|Past Medical History|Allergies|"
    r"Medications on Admission|Vital Signs):\s*\n?", re.IGNORECASE)
SYMPTOM_KEYWORDS = re.compile(
    r"\b(presents?|reports?|complains?|denies|developed|noted|noticed|started|began|"
    r"pain|cough|fever|dyspnea|SOB|fatigue|weakness|nausea|vomit|diarrhea|bleeding|"
    r"dizz|confusion|swelling|edema|chest|abdomen|headache|hypoxic|hypotens|tachycard|"
    r"palpitations|orthopnea|wheez|melena|hemato|dysuria|frequen|urgency)\b", re.IGNORECASE)

def parse_sections(text):
    if not text: return {}
    parts = SECTION_RE.split(text)
    s = {}
    for i in range(1, len(parts) - 1, 2):
        s[parts[i].strip()] = parts[i+1].strip()
    return s

def cumlelere_bol(t):
    if not t: return []
    return [c.strip() for c in re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", t) if c.strip()]

def hasta_skor(c):
    return len(SYMPTOM_KEYWORDS.findall(c))

def compact_v4_query(full_text):
    sections = parse_sections(full_text)
    parts = []
    cc = sections.get("Chief Complaint", "").strip()
    if cc: parts.append(f"Chief Complaint: {cc}")
    sh = sections.get("Symptom History", "").strip()
    if sh:
        ros = re.search(r"(?im)^\s*(Review of Systems?|ROS)\s*:", sh)
        if ros: sh = sh[:ros.start()]
        cumleler = cumlelere_bol(sh)
        skorli = [(i, c, hasta_skor(c)) for i, c in enumerate(cumleler[:5])]
        secilen = [c for i, c, s in skorli if s > 0][:3]
        if not secilen:
            secilen = [c for i, c, s in skorli][:2]
        else:
            if cumleler and cumleler[0] not in secilen:
                secilen = [cumleler[0]] + secilen
                secilen = secilen[:3]
        if secilen: parts.append(f"Key Symptoms: {' '.join(secilen)}")
    vs = sections.get("Vital Signs", "").strip()
    if vs: parts.append(f"Vitals: {vs}")
    pmh = sections.get("Past Medical History", "").strip()
    if pmh:
        pmh_satirlar = [s.strip() for s in re.split(r"[\n;]+", pmh) if s.strip()]
        kisa = [s for s in pmh_satirlar if 5 < len(s) < 100][:3]
        if not kisa:
            kisa = [c for c in cumlelere_bol(pmh)[:2] if len(c) < 200]
        if kisa: parts.append(f"Relevant History: " + "; ".join(kisa))
    al = sections.get("Allergies", "").strip()
    if al:
        al_kisa = al.split("\n")[0].strip()[:100]
        if al_kisa and al_kisa.upper() not in ("NKDA","NONE","NKA"):
            parts.append(f"Allergies: {al_kisa}")
    meds = sections.get("Medications on Admission", "").strip()
    if meds:
        ilac_satirlar = re.findall(r"\d+\.\s*([A-Z][A-Za-z][^.\n]{2,80})", meds)
        if ilac_satirlar:
            kisa_ilaclar = []
            for s in ilac_satirlar[:5]:
                ilac = s.split()[:3]
                kisa_ilaclar.append(" ".join(ilac))
            parts.append(f"Medications: " + ", ".join(kisa_ilaclar))
    return "\n".join(parts)

HEADER = """You are a clinical NLP system. Given a patient's triage information collected at first contact (chief complaint, symptoms, brief history, medications, allergies, vital signs), classify the most likely PRIMARY reason for admission into ONE of these 11 categories:

SEPSIS - Systemic infection, septic shock, bacteremia
STROKE - Ischemic or hemorrhagic stroke, CVA
ATRIAL_FIBRILLATION - Atrial fibrillation/flutter (acute)
AKI - Acute kidney injury, acute renal failure
PNEUMONIA - Lung infection (bacterial, viral, aspiration)
UTI - Urinary tract infection, pyelonephritis, cystitis
HEART_FAILURE - Congestive heart failure, decompensated CHF
GI_BLEEDING - Upper or lower GI hemorrhage
DIABETES_COMPLICATION - DKA, HHS, severe hyper/hypoglycemia
COPD_EXACERBATION - Acute exacerbation of COPD
OTHER - None of the above

Rules:
- Output exactly one class label from the list, using the exact uppercase label string.
- Choose the SINGLE most likely PRIMARY reason for admission.

Below are 11 compact example triage notes (one per class):
"""
EX_TMPL = "\nEXAMPLE {i} ({label}):\n{input}\nPRIMARY DIAGNOSIS: {label}\n"
TT = "\nNow classify the following triage note:\nTRIAGE NOTE:\n{input}\nPRIMARY DIAGNOSIS:"

SABIT = HEADER
for i, cls in enumerate(ALLOWED_CLASSES, 1):
    SABIT += EX_TMPL.format(i=i, label=cls, input=compact_data[cls]["compact_text"])

FUZZY_MAP = {
    "SEPS":"SEPSIS","SEPT":"SEPSIS","PNEUM":"PNEUMONIA",
    "STROK":"STROKE","CVA":"STROKE",
    "ATRIAL":"ATRIAL_FIBRILLATION","AFIB":"ATRIAL_FIBRILLATION","FIBRIL":"ATRIAL_FIBRILLATION",
    "AKI":"AKI","KIDNEY":"AKI","RENAL":"AKI",
    "UTI":"UTI","URIN":"UTI",
    "HEART":"HEART_FAILURE","CHF":"HEART_FAILURE",
    "GI":"GI_BLEEDING","BLEED":"GI_BLEEDING",
    "DIAB":"DIABETES_COMPLICATION","DKA":"DIABETES_COMPLICATION",
    "COPD":"COPD_EXACERBATION","OTHER":"OTHER","NONE":"OTHER",
}

def parse_class(raw):
    if not raw: return "INVALID","empty"
    text = raw.strip().upper()
    for cls in sorted(ALLOWED_CLASSES, key=len, reverse=True):
        if cls in text[:80]: return cls,"direct"
    for k,v in FUZZY_MAP.items():
        if k in text[:80]: return v,"fuzzy"
    return "INVALID","invalid"

print("\n[1/4] Model yukleme...")
t0 = time.time()
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-4B", quantization_config=bnb,
    device_map="auto", torch_dtype=torch.float16, local_files_only=True)
model.eval()
print(f"   Yuklendi {time.time()-t0:.1f}s, GPU {torch.cuda.memory_allocated()/1024**2:.0f}MB")

MAX_LEN = 4096

def inference(test_input):
    prompt = SABIT + TT.format(input=test_input)
    msgs = [{"role":"user","content":prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return raw, inputs.input_ids.shape[1]

print("\n[2/4] V4 test seti...")
df = pd.read_parquet(SPLIT_YOL)
test_df = df[df["split"]=="test"].reset_index(drop=True)
print(f"   Test: {len(test_df):,}")

print("\n[3/4] V4 CLEAN compact 1-shot TEST inference...")
cp_yol = FS_DIR / "fewshot_v4_clean_1shot_test_checkpoint.csv"
final_yol = FS_DIR / "fewshot_v4_clean_1shot_test.csv"

if cp_yol.exists():
    cp = pd.read_csv(cp_yol)
    islenen = set(cp["note_id"].astype(str))
    kayitlar = cp.to_dict("records")
    print(f"   Resume: {len(islenen)} kayit zaten var.")
else:
    islenen = set()
    kayitlar = []

t_baslangic = time.time()
sayac_yeni = 0

for i, row in test_df.iterrows():
    nid = str(row["note_id"])
    if nid in islenen: continue
    v4_full = row["triage_input_v4_timeaware_masked"] or ""
    inp = compact_v4_query(v4_full)
    t0 = time.time()
    raw, in_n = inference(inp)
    sure = time.time() - t0
    parsed, kat = parse_class(raw)
    kayitlar.append({
        "note_id": nid, "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]), "label": row["label"],
        "raw_output": raw, "parsed_pred": parsed,
        "parse_kategori": kat, "in_tokens": int(in_n),
        "sure_s": round(sure, 3),
        "leakage_severity_final": str(row.get("strict_current_leakage_severity_final","UNK")),
        "has_own_label_term_v3": bool(row.get("has_own_label_term_v3", False)),
    })
    sayac_yeni += 1
    if len(kayitlar) % 50 == 0:
        pd.DataFrame(kayitlar).to_csv(cp_yol, index=False)
        gecen = time.time() - t_baslangic
        ort = gecen / sayac_yeni if sayac_yeni > 0 else 0
        kalan = (len(test_df) - len(kayitlar)) * ort
        print(f"   [{len(kayitlar):>4}/{len(test_df)}] gecen {gecen/60:.1f}dk ort {ort:.2f}s kalan ~{kalan/60:.1f}dk")

df_son = pd.DataFrame(kayitlar)
df_son.to_csv(final_yol, index=False)
df_son.to_csv(cp_yol, index=False)
toplam = time.time() - t_baslangic
print(f"   BITTI. {len(kayitlar)} kayit, {toplam/60:.1f}dk")

# AUDIT
print("\n[4/4] V4 CLEAN FEW-SHOT TEST FINAL AUDIT")
print("=" * 70)
n = len(df_son)
ks = df_son["parse_kategori"].value_counts().to_dict()
direct = ks.get("direct", 0); fuzzy = ks.get("fuzzy", 0)
invalid = ks.get("invalid", 0) + ks.get("empty", 0)
print(f"\nParser: direct={direct} ({100*direct/n:.1f}%) fuzzy={fuzzy} invalid={invalid} ({100*invalid/n:.1f}%)")

y_true = df_son["label"].values
y_pred = df_son["parsed_pred"].values
valid = y_pred != "INVALID"
clean_v3 = (~df_son["has_own_label_term_v3"]).values
timeaware_clean = df_son["leakage_severity_final"].isin(["CLEAN","HISTORICAL_ALLOWED"]).values

acc_full = accuracy_score(y_true, y_pred)
mf1_full = f1_score(y_true, y_pred, average="macro", zero_division=0)
wf1_full = f1_score(y_true, y_pred, average="weighted", zero_division=0)
acc_v = accuracy_score(y_true[valid], y_pred[valid]) if valid.sum() else 0.0
mf1_v = f1_score(y_true[valid], y_pred[valid], average="macro", zero_division=0) if valid.sum() else 0.0
acc_kw = accuracy_score(y_true[clean_v3], y_pred[clean_v3]) if clean_v3.sum() else 0.0
mf1_kw = f1_score(y_true[clean_v3], y_pred[clean_v3], average="macro", zero_division=0) if clean_v3.sum() else 0.0
acc_ta = accuracy_score(y_true[timeaware_clean], y_pred[timeaware_clean]) if timeaware_clean.sum() else 0.0
mf1_ta = f1_score(y_true[timeaware_clean], y_pred[timeaware_clean], average="macro", zero_division=0) if timeaware_clean.sum() else 0.0

print(f"\nV4 CLEAN FEW-SHOT TEST METRIKLERI:")
print(f"   {'Subset':<26s} {'N':>5s} {'Acc':>8s} {'MacroF1':>10s} {'WF1':>8s}")
print("-" * 65)
print(f"   {'Full':<26s} {n:>5} {acc_full:>8.4f} {mf1_full:>10.4f} {wf1_full:>8.4f}")
print(f"   {'Valid only':<26s} {valid.sum():>5} {acc_v:>8.4f} {mf1_v:>10.4f}")
print(f"   {'Time-aware clean':<26s} {timeaware_clean.sum():>5} {acc_ta:>8.4f} {mf1_ta:>10.4f}")
print(f"   {'Keyword-clean (V3 ref)':<26s} {clean_v3.sum():>5} {acc_kw:>8.4f} {mf1_kw:>10.4f}")

# Per-class
labels = sorted(set(y_true) | (set(y_pred) - {"INVALID"}))
p, r, f, supp = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
print(f"\nPer-class:")
for cls, pi, ri, fi, si in zip(labels, p, r, f, supp):
    print(f"   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}")

# Confusion matrix
labels_full = ALLOWED_CLASSES + ["INVALID"]
cm = confusion_matrix(y_true, y_pred, labels=labels_full)
ks_short = {"SEPSIS":"SEP","STROKE":"STR","ATRIAL_FIBRILLATION":"AFB","AKI":"AKI",
    "PNEUMONIA":"PNE","UTI":"UTI","HEART_FAILURE":"HF","GI_BLEEDING":"GIB",
    "DIABETES_COMPLICATION":"DIA","COPD_EXACERBATION":"CPD","OTHER":"OTH","INVALID":"INV"}
print(f"\nConfusion matrix:")
print("gercek\\pred  " + " ".join(f"{ks_short[c]:>4}" for c in labels_full))
for i, g in enumerate(labels_full):
    if cm[i].sum() == 0: continue
    print(f"{ks_short[g]:<13s}" + " ".join(f"{cm[i][j]:>4}" for j in range(len(labels_full))))

# Bias
print(f"\nTahmin dagilimi (bias):")
for cls, sayi in df_son["parsed_pred"].value_counts().items():
    print(f"   {cls:<25s} {sayi:>5} ({100*sayi/n:>5.1f}%)")

# 4-yontem karsilastirma (test)
print(f"\n4-YONTEM KARSILASTIRMA (test):")
print(f"   {'':<28s} {'Acc':>8s} {'MacroF1':>10s}")
print(f"   {'V3 zero-shot':<28s} {0.5158:>8.4f} {0.4702:>10.4f}")
print(f"   {'V4 zero-shot':<28s} {0.5152:>8.4f} {0.4681:>10.4f}")
print(f"   {'V4 fewshot CLEAN (final)':<28s} {acc_full:>8.4f} {mf1_full:>10.4f}  *")

# Summary
ozet = pd.DataFrame({
    "subset":["full","valid_only","timeaware_clean","keyword_clean"],
    "n":[n, valid.sum(), timeaware_clean.sum(), clean_v3.sum()],
    "accuracy":[acc_full, acc_v, acc_ta, acc_kw],
    "macro_f1":[mf1_full, mf1_v, mf1_ta, mf1_kw],
    "weighted_f1":[wf1_full, 0.0, 0.0, 0.0],
})
ozet.to_csv(FS_DIR / "fewshot_v4_clean_1shot_test_results_summary.csv", index=False)

audit_yol = FS_DIR / "fewshot_v4_clean_1shot_test_audit.txt"
with open(audit_yol, "w", encoding="utf-8") as fh:
    fh.write("BLOK 14C-V4-CLEAN-TEST FINAL AUDIT\n")
    fh.write("=" * 70 + "\n")
    fh.write(f"Test n: {n}\n")
    fh.write(f"Parser: direct={direct}, fuzzy={fuzzy}, invalid={invalid}\n\n")
    fh.write(f"Full          : acc={acc_full:.4f}, macroF1={mf1_full:.4f}, wF1={wf1_full:.4f}\n")
    fh.write(f"Valid only    : acc={acc_v:.4f}, macroF1={mf1_v:.4f}\n")
    fh.write(f"Time-aware    : acc={acc_ta:.4f}, macroF1={mf1_ta:.4f}\n")
    fh.write(f"Keyword-clean : acc={acc_kw:.4f}, macroF1={mf1_kw:.4f}\n\n")
    fh.write("Per-class:\n")
    for cls, pi, ri, fi, si in zip(labels, p, r, f, supp):
        fh.write(f"   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}\n")

print("\n" + "=" * 70)
print("V4 CLEAN FEW-SHOT TEST TAMAMLANDI - FINAL")
print("=" * 70)


# BLOK 14V4-RESELECT-OTHER-AND-3SHOT-CLEAN — OTHER smart selection + 3-shot compact

**Amac:** V4 1-shot val'de OTHER over-prediction (P=0.160, R=0.580) tespit edildi.
Mevcut OTHER medoid hedef sınıf benzeri (dyspnea, ascites, CKD).
Bu blok OTHER'ı target-class benzeri örneklerden temizler ve 3-shot/class compact üretir.

**OTHER smart selection:**
- **Dışlanan keyword'ler:** dyspnea, SOB, wheez, dysuria, melena, hematochezia, facial droop, palpitations, hyperglycemia, elevated Cr, sepsis, CHF, volume overload
- **Tercih keyword'leri:** fall, trauma, fracture, malignancy, abdominal pain, psychiatric, post-op, vertigo (non-stroke)
- OTHER pool 394 → excl=0: 174 → pref≥1: 125 (kullanılan havuz)

**3-shot/class:**
- 11 sınıf × 3 örnek = 33 medoid
- Sıra 1: centroid medoid, Sıra 2-3: MMR diversity (α=0.7, β=0.3)
- Compact target: 100-130 token/örnek
- Forbidden phrase audit (tüm 33 örnekte 0 forbidden ✅)

**Sonuç:**
- 33 örnek, **toplam 5.023 token** (avg 152/örnek)
- Severity: 30 HISTORICAL_ALLOWED + 3 CLEAN ✅
- Sabit prompt + max query = ~8.152 token (max_length=8192 sığar)

**Çıktı:** `output/fewshot/fewshot_v4_3shot_clean_examples.{csv,json}`
**Detaylı script:** `__run_blok14_v4_reselect_other_3shot.py`

In [ ]:
"""Blok 14V4-RESELECT-OTHER-AND-3SHOT-CLEAN.
1. OTHER icin smart selection (target-class benzeri ornekleri disla)
2. 3-shot/class compact clean (medoid + MMR diversity)
3. Compact audit: forbidden phrase + leakage severity
Inference YOK."""
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
FS_DIR     = OUTPUT_DIR / "fewshot"

print("=" * 70)
print("BLOK 14V4-RESELECT-OTHER-AND-3SHOT-CLEAN")
print("=" * 70)

ALLOWED_CLASSES = ["SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"]

# ============================================================
# OTHER icin DISLAMA pattern listesi (target sinif benzeri)
# ============================================================
OTHER_EXCLUDE_PATTERNS = re.compile(
    # Respiratory (PNEUMONIA, COPD, HF benzeri)
    r"\b(dyspnea|SOB|shortness\s+of\s+breath|wheezing|wheez\w*|hypox\w*|"
    r"productive\s+cough|cough\s+productive|pneumonia|"
    # GI bleeding
    r"melena|hematochezia|BRBPR|hematemesis|GI\s+bleed|"
    # UTI
    r"dysuria|urinary\s+(tract|frequency|urgency)|UTI|pyelonephritis|cystitis|"
    # Stroke-like
    r"facial\s+droop|aphasia|hemiparesis|hemiplegi|slurred\s+speech|"
    r"acute\s+onset\s+(weakness|numbness)|"
    # AFib
    r"palpitation|atrial\s+fib|AFib|A-fib|atrial\s+flutter|"
    # Diabetes
    r"hyperglyc\w*|hypoglyc\w*|DKA|HHS|hyperosmolar|"
    # AKI
    r"elevated\s+(Cr|creatinine)|acute\s+kidney|AKI|"
    # Sepsis / infection
    r"sepsis|septic|bacteremia|"
    # CHF
    r"CHF|congestive\s+heart\s+failure|HFrEF|HFpEF|volume\s+overload|"
    r"orthopnea|PND|paroxysmal\s+nocturnal)\b",
    re.IGNORECASE
)

# OTHER icin TERCIH pattern (gercek non-target)
OTHER_PREFER_PATTERNS = re.compile(
    r"\b(fall|trauma|fracture|MVA|motor\s+vehicle|"
    r"laceration|burn|abrasion|"
    r"intoxication|alcohol\s+(use|intoxication)|"
    r"psychiatric|depression|anxiety|suicidal|"
    r"malignancy|cancer|metasta|chemotherapy|"
    r"abdominal\s+pain|RUQ|LUQ|pelvic|"
    r"post-?op|procedure|elective|"
    r"electrolyte\s+disturbance|hyponatremia(?!.*HF)|hypernatremia|"
    r"vertigo(?!.*stroke)|syncope(?!.*HF))\b",
    re.IGNORECASE
)

# ============================================================
# Forbidden compact phrases (14V4-CLEAN-FEWSHOT ile ayni)
# ============================================================
FORBIDDEN_PATTERNS = [
    ("code_stroke", re.compile(r"\bcode\s+(stroke|MI|sepsis|trauma)\b", re.IGNORECASE)),
    ("ct_scanning", re.compile(r"\b(CT scanning|reviewed images|read images|reviewed CT|reviewed scan)\b", re.IGNORECASE)),
    ("micu_admission", re.compile(r"\b(REASON FOR (MICU|ICU) ADMISSION|MICU admission|ICU admission)\b", re.IGNORECASE)),
    ("scheduled_proc", re.compile(r"\bscheduled\s+(TEE|DCCV|EGD|colonoscopy|cardioversion|TTE)\b", re.IGNORECASE)),
    ("elevated_lab", re.compile(r"\b(elevated\s+(Cr|creatinine|WBC|Hgb|HCT|lactate|troponin|BUN|sodium|Na)|abnormal\s+labs?)\b", re.IGNORECASE)),
    ("found_in_ed", re.compile(r"\b(found\s+in\s+ED|found\s+to\s+have|found\s+down)\b", re.IGNORECASE)),
    ("given_treatment", re.compile(r"\b(given|received|started\s+on|placed\s+on|treated\s+with)\s+[^.]{0,40}\b(oxygen|CPAP|BiPAP|high\s*flow|nebulizer|nebs|albuterol|duonebs|antibiotics|abx|vancomycin|zosyn|lasix|insulin\s+drip|glucagon|tPA|pressors|levophed|norepinephrine|vasopressin|crystalloid|IV\s+fluids|fluid\s+bolus)\b", re.IGNORECASE)),
    ("admitted_for", re.compile(r"\b(admitted\s+(for|to)|MICU\s+admission|ED\s+course)\b", re.IGNORECASE)),
    ("imaging_showed", re.compile(r"\b(CXR|CTA?|MRI|TTE|TEE|US|echocardiogram)\b[^.]{0,40}\b(showed|revealed|demonstrated|consistent\s+with|notable|w/|with\s+(infiltrate|consolidation|effusion|edema|ICH|infarct|hemorrhage))\b", re.IGNORECASE)),
    ("labs_showed", re.compile(r"\bLabs?\b[^.]{0,40}\b(showed|notable|revealed|drawn|remarkable)\b", re.IGNORECASE)),
    ("presented_workup", re.compile(r"\bpresent(ed|s|ing)\s+to\s+(the\s+)?ED\s+(for|with)\s+(workup|further\s+evaluation|management|admission|treatment)\b", re.IGNORECASE)),
]

def find_forbidden(text):
    if not isinstance(text, str) or not text: return []
    bulgular = []
    for ad, rx in FORBIDDEN_PATTERNS:
        for m in rx.finditer(text):
            bulgular.append((ad, m.group()))
    return bulgular

# ============================================================
# Section-aware compaction
# ============================================================
SECTION_RE = re.compile(
    r"(Chief Complaint|Symptom History|Past Medical History|Allergies|"
    r"Medications on Admission|Vital Signs):\s*\n?", re.IGNORECASE)
SYMPTOM_KEYWORDS = re.compile(
    r"\b(presents?|reports?|complains?|denies|developed|noted|noticed|started|began|"
    r"pain|cough|fever|dyspnea|SOB|fatigue|weakness|nausea|vomit|diarrhea|bleeding|"
    r"dizz|confusion|swelling|edema|chest|abdomen|headache|hypoxic|hypotens|tachycard|"
    r"palpitations|orthopnea|wheez|melena|hemato|dysuria|frequen|urgency)\b", re.IGNORECASE)

def parse_sections(text):
    if not text: return {}
    parts = SECTION_RE.split(text)
    s = {}
    for i in range(1, len(parts) - 1, 2):
        s[parts[i].strip()] = parts[i+1].strip()
    return s

def cumlelere_bol(t):
    if not t: return []
    return [c.strip() for c in re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", t) if c.strip()]

def hasta_skor(c):
    return len(SYMPTOM_KEYWORDS.findall(c))

def compact_for_3shot(full_text, max_token_target=130, tok=None):
    """3-shot icin daha agresif compact: 100-130 token hedef."""
    sections = parse_sections(full_text)
    parts = []
    cc = sections.get("Chief Complaint", "").strip()
    if cc: parts.append(f"Chief Complaint: {cc}")
    sh = sections.get("Symptom History", "").strip()
    if sh:
        ros = re.search(r"(?im)^\s*(Review of Systems?|ROS)\s*:", sh)
        if ros: sh = sh[:ros.start()]
        cumleler = cumlelere_bol(sh)
        skorli = [(i, c, hasta_skor(c)) for i, c in enumerate(cumleler[:5])]
        secilen = [c for i, c, s in skorli if s > 0][:2]  # 3-shot icin max 2 cumle
        if not secilen:
            secilen = [c for i, c, s in skorli][:1]
        if secilen: parts.append(f"Key Symptoms: {' '.join(secilen)}")
    vs = sections.get("Vital Signs", "").strip()
    if vs:
        # Vitals'i kisalt: 100 char max
        parts.append(f"Vitals: {vs[:120]}")
    pmh = sections.get("Past Medical History", "").strip()
    if pmh:
        pmh_satirlar = [s.strip() for s in re.split(r"[\n;]+", pmh) if s.strip()]
        kisa = [s for s in pmh_satirlar if 5 < len(s) < 80][:2]  # 3-shot icin 2 PMH
        if not kisa:
            kisa = [c for c in cumlelere_bol(pmh)[:1] if len(c) < 150]
        if kisa: parts.append(f"Relevant History: " + "; ".join(kisa))
    meds = sections.get("Medications on Admission", "").strip()
    if meds:
        ilac_satirlar = re.findall(r"\d+\.\s*([A-Z][A-Za-z][^.\n]{2,60})", meds)
        if ilac_satirlar:
            kisa_ilaclar = []
            for s in ilac_satirlar[:3]:  # 3-shot icin max 3 ilac
                ilac = s.split()[:2]
                kisa_ilaclar.append(" ".join(ilac))
            parts.append(f"Medications: " + ", ".join(kisa_ilaclar))
    return "\n".join(parts)

# ============================================================
# Veri ve pool
# ============================================================
print("\n[1/6] Veri yukleme ve pool filtre...")
df = pd.read_parquet(SPLIT_YOL)
df["note_id"] = df["note_id"].astype(str)
pool_full = df[df["split"]=="train_S3"].reset_index(drop=True)

LEAKAGE_KOLON = "strict_current_leakage_severity_final"
if LEAKAGE_KOLON not in pool_full.columns:
    LEAKAGE_KOLON = "strict_current_leakage_severity"
clean_mask = pool_full[LEAKAGE_KOLON].isin(["CLEAN","HISTORICAL_ALLOWED"])
pool = pool_full[clean_mask].reset_index(drop=True)
print(f"   train_S3 pool (V4 clean): {len(pool):,}")

vec = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=2,
                      stop_words="english", sublinear_tf=True)
X = vec.fit_transform(pool["triage_input_v4_timeaware_masked"].fillna("").tolist())
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)

# ============================================================
# OTHER icin SMART selection
# ============================================================
print("\n[2/6] OTHER icin smart re-selection...")
print("=" * 70)

other_idx = np.where(pool["label"].values == "OTHER")[0]
print(f"   OTHER pool: {len(other_idx)} ornek")

# OTHER adaylarini skorla
other_kayitlar = []
for lokal_idx in other_idx:
    row = pool.iloc[lokal_idx]
    text = row["triage_input_v4_timeaware_masked"] or ""
    excl_count = len(OTHER_EXCLUDE_PATTERNS.findall(text))
    pref_count = len(OTHER_PREFER_PATTERNS.findall(text))
    other_kayitlar.append({
        "lokal_idx": int(lokal_idx),
        "note_id": str(row["note_id"]),
        "leakage": str(row[LEAKAGE_KOLON]),
        "excl": excl_count,
        "pref": pref_count,
        "skor": pref_count - excl_count,
        "text": text,
    })

# Filtreleme: excl=0 olanlar, pref>=1 olanlar tercih
other_uygun = [o for o in other_kayitlar if o["excl"] == 0]
print(f"   Excl=0 (target benzeri yok): {len(other_uygun)}")
other_yuksek_pref = [o for o in other_uygun if o["pref"] >= 1]
print(f"   Pref>=1 (gercek non-target): {len(other_yuksek_pref)}")

if len(other_yuksek_pref) >= 3:
    aday_havuz_idx = [o["lokal_idx"] for o in other_yuksek_pref]
    print(f"   -> Pref>=1 havuzu kullanilacak ({len(aday_havuz_idx)})")
elif len(other_uygun) >= 3:
    aday_havuz_idx = [o["lokal_idx"] for o in other_uygun]
    print(f"   -> Excl=0 havuzu kullanilacak ({len(aday_havuz_idx)})")
else:
    # Geri donus: en dusuk excl skoru olanlardan al
    other_kayitlar.sort(key=lambda x: x["excl"])
    aday_havuz_idx = [o["lokal_idx"] for o in other_kayitlar[:50]]
    print(f"   -> En dusuk excl havuzu kullanilacak ({len(aday_havuz_idx)})")

print(f"\n   ORNEK reddedilen OTHER (ilk 5):")
for o in other_kayitlar[:5]:
    if o["excl"] > 0:
        # Hangi pattern eslesti
        eslesen = OTHER_EXCLUDE_PATTERNS.search(o["text"])
        m_str = eslesen.group() if eslesen else "?"
        print(f"      {o['note_id']}: excl={o['excl']} pref={o['pref']} (eslesen: {m_str!r})")

print(f"\n   ORNEK secilen OTHER aday (ilk 5):")
for o in other_yuksek_pref[:5]:
    eslesen = OTHER_PREFER_PATTERNS.search(o["text"])
    m_str = eslesen.group() if eslesen else "?"
    print(f"      {o['note_id']}: pref={o['pref']} (eslesen: {m_str!r})")

# ============================================================
# 3-shot/class secim (medoid + MMR)
# ============================================================
print("\n[3/6] 3-shot/class secim (medoid + MMR diversity)...")
print("=" * 70)

ALPHA = 0.7
BETA = 0.3
MAX_RETRIES = 20  # forbidden cikarsa kac aday daha dene

shot3_kayitlar = []

for cls in ALLOWED_CLASSES:
    if cls == "OTHER":
        # Smart aday havuzu
        cls_idx_arr = np.array(aday_havuz_idx)
    else:
        cls_idx_arr = np.where(pool["label"].values == cls)[0]

    if len(cls_idx_arr) < 3:
        print(f"   UYARI: {cls} icin {len(cls_idx_arr)} aday var")
        continue

    X_cls = X[cls_idx_arr]
    centroid = np.asarray(X_cls.mean(axis=0)).reshape(1,-1)
    sim_cent = cosine_similarity(X_cls, centroid).flatten()

    # 3 ornek sec, MMR + forbidden filter
    secilen_lokal_arr = []  # cls_idx_arr icindeki indexler
    rank_taranan = 0

    # Aday sirasi: rank by score (mmr style)
    for hedef_sira in range(3):
        if hedef_sira == 0:
            skor = sim_cent.copy()
        else:
            secilen_X = X_cls[secilen_lokal_arr]
            sim_sel = cosine_similarity(X_cls, secilen_X)
            max_sim_sel = sim_sel.max(axis=1)
            skor = ALPHA * sim_cent - BETA * max_sim_sel
            for s in secilen_lokal_arr: skor[s] = -np.inf

        # Sirala
        rank = np.argsort(-skor)
        bulundu = False
        for r_local, lokal in enumerate(rank):
            if int(lokal) in secilen_lokal_arr: continue
            global_idx = cls_idx_arr[int(lokal)]
            row = pool.iloc[global_idx]
            full_text = row["triage_input_v4_timeaware_masked"] or ""
            compact_aday = compact_for_3shot(full_text, tok=tok)
            bulg = find_forbidden(compact_aday)
            if not bulg and len(compact_aday) > 50:
                secilen_lokal_arr.append(int(lokal))
                bulundu = True
                rank_taranan = r_local
                break
            else:
                rank_taranan = r_local
                if r_local >= MAX_RETRIES: break

        if not bulundu:
            print(f"   UYARI: {cls} sira {hedef_sira+1} icin temiz aday bulunamadi (taranan: {rank_taranan+1})")

    # Kaydet
    for sira, lokal in enumerate(secilen_lokal_arr, 1):
        global_idx = cls_idx_arr[lokal]
        row = pool.iloc[global_idx]
        full_text = row["triage_input_v4_timeaware_masked"] or ""
        compact_text = compact_for_3shot(full_text, tok=tok)

        if sira == 1:
            max_sim_prev = 0.0
        else:
            secilen_X = X_cls[secilen_lokal_arr[:sira-1]]
            max_sim_prev = float(cosine_similarity(X_cls[lokal:lokal+1], secilen_X).max())

        n_token = len(tok.encode(compact_text, add_special_tokens=False))
        bulg = find_forbidden(compact_text)
        shot3_kayitlar.append({
            "label": cls,
            "rank": sira,
            "note_id": str(row["note_id"]),
            "subject_id": int(row["subject_id"]),
            "hadm_id": int(row["hadm_id"]),
            "sim_to_centroid": round(float(sim_cent[lokal]), 4),
            "max_sim_to_prev": round(max_sim_prev, 4),
            "leakage_severity": str(row[LEAKAGE_KOLON]),
            "compact_token": n_token,
            "compact_char": len(compact_text),
            "compact_text": compact_text,
            "forbidden_count": len(bulg),
        })

    sims = [shot3_kayitlar[-3]["sim_to_centroid"],
            shot3_kayitlar[-2]["sim_to_centroid"],
            shot3_kayitlar[-1]["sim_to_centroid"]]
    toks = [shot3_kayitlar[-3]["compact_token"],
            shot3_kayitlar[-2]["compact_token"],
            shot3_kayitlar[-1]["compact_token"]]
    print(f"   {cls:<25s} sim=[{sims[0]:.3f},{sims[1]:.3f},{sims[2]:.3f}] tok=[{toks[0]},{toks[1]},{toks[2]}]")

# ============================================================
# Audit
# ============================================================
print("\n[4/6] FINAL AUDIT")
print("=" * 70)

shot3_df = pd.DataFrame(shot3_kayitlar)
toplam_token = shot3_df["compact_token"].sum()
toplam_kirli = (shot3_df["forbidden_count"] > 0).sum()

print(f"\n   Secilen toplam: {len(shot3_df)}/33")
print(f"   Toplam token  : {toplam_token:,} (avg {shot3_df['compact_token'].mean():.0f}/ornek)")
print(f"   Hala kirli    : {toplam_kirli}")
print(f"   Severity dist : {shot3_df['leakage_severity'].value_counts().to_dict()}")

# Token bütçe hesabi
HEADER_TOKEN = 280
EX_WRAPPER = 25
sabit_total = HEADER_TOKEN + toplam_token + 33 * EX_WRAPPER
print(f"\n   Sabit prompt (header+33 example+wrapper): ~{sabit_total:,} token")
print(f"   Test query avg (~580): toplam ~{sabit_total + 580:,}")
print(f"   Test query max (~2024): toplam ~{sabit_total + 2024:,}")
print(f"   8192 sigar mi? {'EVET' if sabit_total + 2024 < 8192 else 'HAYIR (kesilir)'}")

# MMR diversity
print(f"\n   MMR diversity (rank 2+):")
for sira in [2, 3]:
    alt = shot3_df[shot3_df["rank"]==sira]
    print(f"      rank {sira}: max_sim_prev ort={alt['max_sim_to_prev'].mean():.3f}, max={alt['max_sim_to_prev'].max():.3f}")

# Sinif bazli detay
print(f"\n   Sinif bazli detay:")
print(f"   {'Sinif':<25s} {'rank':>4s} {'note':<22s} {'sim':>6s} {'mmr':>6s} {'tok':>4s} {'fb':>3s}")
print("   " + "-" * 80)
for _, r in shot3_df.iterrows():
    print(f"   {r['label']:<25s} {r['rank']:>4} {r['note_id']:<22s} {r['sim_to_centroid']:>6.3f} {r['max_sim_to_prev']:>6.3f} {r['compact_token']:>4} {r['forbidden_count']:>3}")

# ============================================================
# Kaydet
# ============================================================
print("\n[5/6] Kaydetme...")
shot3_df.to_csv(FS_DIR / "fewshot_v4_3shot_clean_examples.csv", index=False)

shot3_json = {}
for cls in ALLOWED_CLASSES:
    shot3_json[cls] = []
    for _, r in shot3_df[shot3_df["label"]==cls].iterrows():
        shot3_json[cls].append({
            "rank": int(r["rank"]),
            "note_id": r["note_id"], "subject_id": r["subject_id"], "hadm_id": r["hadm_id"],
            "sim_to_centroid": float(r["sim_to_centroid"]),
            "max_sim_to_prev": float(r["max_sim_to_prev"]),
            "leakage_severity": r["leakage_severity"],
            "compact_token": int(r["compact_token"]),
            "compact_text": r["compact_text"],
        })
with open(FS_DIR / "fewshot_v4_3shot_clean_examples.json", "w", encoding="utf-8") as f:
    json.dump(shot3_json, f, ensure_ascii=False, indent=2)

print(f"   fewshot_v4_3shot_clean_examples.csv ({(FS_DIR/'fewshot_v4_3shot_clean_examples.csv').stat().st_size/1024:.1f} KB)")
print(f"   fewshot_v4_3shot_clean_examples.json ({(FS_DIR/'fewshot_v4_3shot_clean_examples.json').stat().st_size/1024:.1f} KB)")

# ============================================================
# Preview
# ============================================================
print("\n[6/6] PREVIEW (her sinif rank 1)")
print("=" * 70)
for cls in ALLOWED_CLASSES:
    if cls not in shot3_json or not shot3_json[cls]: continue
    ex = shot3_json[cls][0]
    print(f"\n   {cls} (rank 1, note: {ex['note_id']}, tok: {ex['compact_token']}, sev: {ex['leakage_severity']}):")
    print(f"   {ex['compact_text'][:400]}")

print("\n" + "=" * 70)
print("BLOK 14V4-RESELECT-OTHER-AND-3SHOT-CLEAN TAMAMLANDI - inference YOK")
print("=" * 70)


# BLOK 14C-V4-3SHOT-CLEAN-VAL — V4 3-shot CLEAN Val Inference

**⚠️ ÇOK ÖNEMLİ: Bu hücreyi SEN çalıştır (Jupyter → Run Cell)**
**Tahmini süre:** ~70-90 dakika (550 örnek × ~8-9 s/örnek, prompt uzun)

**Konfigürasyon:**
- Examples: 33 V4 3-shot CLEAN compact medoid (`fewshot_v4_3shot_clean_examples.json`)
- Query: V4 compact (runtime, normal compact öncelik, >8050 tok ise aggressive fallback)
- max_length=8192, max_new_tokens=20, do_sample=False
- chat_template + enable_thinking=False
- Checkpoint her 25 örnekte

**Yeni rule eklendi prompt'a:**
> OTHER is NOT an uncertainty label. Choose OTHER ONLY when none of the 10 named diagnoses is clinically supported. If one of the 10 named classes is plausible, choose that named class instead of OTHER.

**Token safety:**
- Normal compact: 3 sentences/3 PMH/5 meds, query ~150-300 token
- Aggressive compact (sadece prompt > 8050 tok ise): 2 sentences/2 PMH/3 meds
- Aggressive kullanım sayısı log'lanır

**Karşılaştırma hedefi:**
| Yöntem | Acc | Macro-F1 | OTHER% |
|---|---:|---:|---:|
| V4 zero-shot | 0.4818 | 0.4367 | 15.8% |
| V4 1-shot CLEAN | 0.4727 | 0.4479 | ? |
| **V4 3-shot CLEAN (bu hücre)** | **?** | **?** | **?** |

**Çıktılar:**
- `output/fewshot/fewshot_v4_3shot_clean_val.csv`
- `output/fewshot/fewshot_v4_3shot_clean_val_checkpoint.csv`
- `output/fewshot/fewshot_v4_3shot_clean_val_results_summary.csv`

**Detaylı script (terminal alternatifi):** `__run_blok14c_v4_3shot_clean_val.py`

TEST'e otomatik geçiş YOK. Val sonucundan sonra ayrı hücre.

In [ ]:
"""Blok 14C-V4-3SHOT-CLEAN-VAL: V4 3-shot CLEAN few-shot val (550 ornek).
- 11 sinif x 3 ornek = 33 compact medoid
- Query V4 compact (aggressive fallback varsa)
- max_length=8192
- OTHER rule: only when none of 10 named diagnoses plausible"""
import pandas as pd
import numpy as np
import torch
import json
import re
import time
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
FS_DIR     = OUTPUT_DIR / "fewshot"

print("=" * 70)
print("BLOK 14C-V4-3SHOT-CLEAN-VAL — V4 3-shot CLEAN few-shot val")
print("=" * 70)

with open(FS_DIR / "fewshot_v4_3shot_clean_examples.json", "r", encoding="utf-8") as f:
    shot3_data = json.load(f)

ALLOWED_CLASSES = ["SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"]

# ============================================================
# Section-aware compaction (NORMAL ve AGGRESSIVE versiyonlari)
# ============================================================
SECTION_RE = re.compile(
    r"(Chief Complaint|Symptom History|Past Medical History|Allergies|"
    r"Medications on Admission|Vital Signs):\s*\n?", re.IGNORECASE)
SYMPTOM_KEYWORDS = re.compile(
    r"\b(presents?|reports?|complains?|denies|developed|noted|noticed|started|began|"
    r"pain|cough|fever|dyspnea|SOB|fatigue|weakness|nausea|vomit|diarrhea|bleeding|"
    r"dizz|confusion|swelling|edema|chest|abdomen|headache|hypoxic|hypotens|tachycard|"
    r"palpitations|orthopnea|wheez|melena|hemato|dysuria|frequen|urgency)\b", re.IGNORECASE)

def parse_sections(text):
    if not text: return {}
    parts = SECTION_RE.split(text)
    s = {}
    for i in range(1, len(parts) - 1, 2):
        s[parts[i].strip()] = parts[i+1].strip()
    return s

def cumlelere_bol(t):
    if not t: return []
    return [c.strip() for c in re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", t) if c.strip()]

def hasta_skor(c):
    return len(SYMPTOM_KEYWORDS.findall(c))

def compact_normal(full_text):
    """Standart V4 compact (Key Symptoms 3 cumle, PMH 3, Meds 5)."""
    sections = parse_sections(full_text)
    parts = []
    cc = sections.get("Chief Complaint", "").strip()
    if cc: parts.append(f"Chief Complaint: {cc}")
    sh = sections.get("Symptom History", "").strip()
    if sh:
        ros = re.search(r"(?im)^\s*(Review of Systems?|ROS)\s*:", sh)
        if ros: sh = sh[:ros.start()]
        cumleler = cumlelere_bol(sh)
        skorli = [(i, c, hasta_skor(c)) for i, c in enumerate(cumleler[:5])]
        secilen = [c for i, c, s in skorli if s > 0][:3]
        if not secilen:
            secilen = [c for i, c, s in skorli][:2]
        else:
            if cumleler and cumleler[0] not in secilen:
                secilen = [cumleler[0]] + secilen
                secilen = secilen[:3]
        if secilen: parts.append(f"Key Symptoms: {' '.join(secilen)}")
    vs = sections.get("Vital Signs", "").strip()
    if vs: parts.append(f"Vitals: {vs}")
    pmh = sections.get("Past Medical History", "").strip()
    if pmh:
        pmh_satirlar = [s.strip() for s in re.split(r"[\n;]+", pmh) if s.strip()]
        kisa = [s for s in pmh_satirlar if 5 < len(s) < 100][:3]
        if not kisa:
            kisa = [c for c in cumlelere_bol(pmh)[:2] if len(c) < 200]
        if kisa: parts.append(f"Relevant History: " + "; ".join(kisa))
    al = sections.get("Allergies", "").strip()
    if al:
        al_kisa = al.split("\n")[0].strip()[:100]
        if al_kisa and al_kisa.upper() not in ("NKDA","NONE","NKA"):
            parts.append(f"Allergies: {al_kisa}")
    meds = sections.get("Medications on Admission", "").strip()
    if meds:
        ilac_satirlar = re.findall(r"\d+\.\s*([A-Z][A-Za-z][^.\n]{2,80})", meds)
        if ilac_satirlar:
            kisa = [" ".join(s.split()[:3]) for s in ilac_satirlar[:5]]
            parts.append(f"Medications: " + ", ".join(kisa))
    return "\n".join(parts)

def compact_aggressive(full_text):
    """Aggressive compact: Key Symptoms 2 cumle, PMH 2, Meds 3."""
    sections = parse_sections(full_text)
    parts = []
    cc = sections.get("Chief Complaint", "").strip()
    if cc: parts.append(f"Chief Complaint: {cc}")
    sh = sections.get("Symptom History", "").strip()
    if sh:
        ros = re.search(r"(?im)^\s*(Review of Systems?|ROS)\s*:", sh)
        if ros: sh = sh[:ros.start()]
        cumleler = cumlelere_bol(sh)
        skorli = [(i, c, hasta_skor(c)) for i, c in enumerate(cumleler[:4])]
        secilen = [c for i, c, s in skorli if s > 0][:2]  # max 2
        if not secilen:
            secilen = [c for i, c, s in skorli][:1]
        if secilen: parts.append(f"Key Symptoms: {' '.join(secilen)}")
    vs = sections.get("Vital Signs", "").strip()
    if vs: parts.append(f"Vitals: {vs[:100]}")  # vitals da kisalt
    pmh = sections.get("Past Medical History", "").strip()
    if pmh:
        pmh_satirlar = [s.strip() for s in re.split(r"[\n;]+", pmh) if s.strip()]
        kisa = [s for s in pmh_satirlar if 5 < len(s) < 80][:2]  # max 2
        if kisa: parts.append(f"Relevant History: " + "; ".join(kisa))
    meds = sections.get("Medications on Admission", "").strip()
    if meds:
        ilac_satirlar = re.findall(r"\d+\.\s*([A-Z][A-Za-z][^.\n]{2,60})", meds)
        if ilac_satirlar:
            kisa = [" ".join(s.split()[:2]) for s in ilac_satirlar[:3]]  # max 3
            parts.append(f"Medications: " + ", ".join(kisa))
    return "\n".join(parts)

# ============================================================
# Prompt build (33 examples)
# ============================================================
HEADER = """You are a clinical NLP system. Given a patient's triage information collected at first contact (chief complaint, symptoms, brief history, medications, allergies, vital signs), classify the most likely PRIMARY reason for admission into ONE of these 11 categories:

SEPSIS - Systemic infection, septic shock, bacteremia
STROKE - Ischemic or hemorrhagic stroke, CVA
ATRIAL_FIBRILLATION - Atrial fibrillation/flutter (acute)
AKI - Acute kidney injury, acute renal failure
PNEUMONIA - Lung infection (bacterial, viral, aspiration)
UTI - Urinary tract infection, pyelonephritis, cystitis
HEART_FAILURE - Congestive heart failure, decompensated CHF
GI_BLEEDING - Upper or lower GI hemorrhage
DIABETES_COMPLICATION - DKA, HHS, severe hyper/hypoglycemia
COPD_EXACERBATION - Acute exacerbation of COPD
OTHER - None of the above

Rules:
- Output exactly one class label from the list, using the exact uppercase label string.
- Choose the SINGLE most likely PRIMARY reason for admission.
- OTHER is NOT an uncertainty label. Choose OTHER ONLY when none of the 10 named diagnoses is clinically supported.
- If one of the 10 named classes is plausible, choose that named class instead of OTHER.

Below are 33 compact example triage notes (3 per class):
"""
EX_TMPL = "\nEXAMPLE {i} ({label}, ex{rank}):\n{input}\nPRIMARY DIAGNOSIS: {label}\n"
TT = "\nNow classify the following triage note:\nTRIAGE NOTE:\n{input}\nPRIMARY DIAGNOSIS:"

SABIT = HEADER
ex_idx = 1
for cls in ALLOWED_CLASSES:
    for ex in shot3_data[cls]:
        SABIT += EX_TMPL.format(i=ex_idx, label=cls, rank=ex["rank"], input=ex["compact_text"])
        ex_idx += 1

FUZZY_MAP = {
    "SEPS":"SEPSIS","SEPT":"SEPSIS","PNEUM":"PNEUMONIA",
    "STROK":"STROKE","CVA":"STROKE",
    "ATRIAL":"ATRIAL_FIBRILLATION","AFIB":"ATRIAL_FIBRILLATION","FIBRIL":"ATRIAL_FIBRILLATION",
    "AKI":"AKI","KIDNEY":"AKI","RENAL":"AKI",
    "UTI":"UTI","URIN":"UTI",
    "HEART":"HEART_FAILURE","CHF":"HEART_FAILURE",
    "GI":"GI_BLEEDING","BLEED":"GI_BLEEDING",
    "DIAB":"DIABETES_COMPLICATION","DKA":"DIABETES_COMPLICATION",
    "COPD":"COPD_EXACERBATION","OTHER":"OTHER","NONE":"OTHER",
}

def parse_class(raw):
    if not raw: return "INVALID","empty"
    text = raw.strip().upper()
    for cls in sorted(ALLOWED_CLASSES, key=len, reverse=True):
        if cls in text[:80]: return cls,"direct"
    for k,v in FUZZY_MAP.items():
        if k in text[:80]: return v,"fuzzy"
    return "INVALID","invalid"

# ============================================================
# Model
# ============================================================
print("\n[1/4] Model yukleme...")
t0 = time.time()
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-4B", quantization_config=bnb,
    device_map="auto", torch_dtype=torch.float16, local_files_only=True)
model.eval()
print(f"   Yuklendi {time.time()-t0:.1f}s, GPU {torch.cuda.memory_allocated()/1024**2:.0f}MB")

MAX_LEN = 8192
TOKEN_SAFETY = 8050  # eger normal compact prompt > bu sayi -> aggressive

def build_prompt_with_query(test_input):
    return SABIT + TT.format(input=test_input)

def count_prompt_tokens(test_input):
    """Tam prompt token sayisi (chat-wrapped)."""
    prompt = build_prompt_with_query(test_input)
    msgs = [{"role":"user","content":prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return len(tok.encode(text, add_special_tokens=False))

def inference(test_input):
    prompt = build_prompt_with_query(test_input)
    msgs = [{"role":"user","content":prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return raw, inputs.input_ids.shape[1]

# ============================================================
# Veri
# ============================================================
print("\n[2/4] V4 val seti...")
df = pd.read_parquet(SPLIT_YOL)
val_df = df[df["split"]=="val"].reset_index(drop=True)
print(f"   Val: {len(val_df):,}")

# ============================================================
# Inference
# ============================================================
print("\n[3/4] V4 3-SHOT CLEAN val inference...")
cp_yol = FS_DIR / "fewshot_v4_3shot_clean_val_checkpoint.csv"
final_yol = FS_DIR / "fewshot_v4_3shot_clean_val.csv"

if cp_yol.exists():
    cp = pd.read_csv(cp_yol)
    islenen = set(cp["note_id"].astype(str))
    kayitlar = cp.to_dict("records")
    print(f"   Resume: {len(islenen)} kayit zaten var.")
else:
    islenen = set()
    kayitlar = []

t_baslangic = time.time()
sayac_yeni = 0
aggressive_count = 0

for i, row in val_df.iterrows():
    nid = str(row["note_id"])
    if nid in islenen: continue
    v4_full = row["triage_input_v4_timeaware_masked"] or ""

    # Once normal compact dene
    inp_normal = compact_normal(v4_full)
    n_norm = count_prompt_tokens(inp_normal)
    if n_norm > TOKEN_SAFETY:
        # Aggressive
        inp = compact_aggressive(v4_full)
        compact_strategy = "aggressive"
        aggressive_count += 1
    else:
        inp = inp_normal
        compact_strategy = "normal"

    t0 = time.time()
    raw, in_n = inference(inp)
    sure = time.time() - t0
    parsed, kat = parse_class(raw)
    kayitlar.append({
        "note_id": nid, "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]), "label": row["label"],
        "raw_output": raw, "parsed_pred": parsed,
        "parse_kategori": kat, "in_tokens": int(in_n),
        "sure_s": round(sure, 3),
        "compact_strategy": compact_strategy,
        "leakage_severity_final": str(row.get("strict_current_leakage_severity_final","UNK")),
    })
    sayac_yeni += 1
    if len(kayitlar) % 25 == 0:
        pd.DataFrame(kayitlar).to_csv(cp_yol, index=False)
        gecen = time.time() - t_baslangic
        ort = gecen / sayac_yeni if sayac_yeni > 0 else 0
        kalan = (len(val_df) - len(kayitlar)) * ort
        print(f"   [{len(kayitlar):>3}/{len(val_df)}] gecen {gecen/60:.1f}dk ort {ort:.2f}s kalan ~{kalan/60:.1f}dk aggr={aggressive_count}")

df_son = pd.DataFrame(kayitlar)
df_son.to_csv(final_yol, index=False)
df_son.to_csv(cp_yol, index=False)
toplam = time.time() - t_baslangic
print(f"   BITTI. {len(kayitlar)} kayit, {toplam/60:.1f}dk")
print(f"   Aggressive compact kullanildi: {aggressive_count} ornek")

# ============================================================
# AUDIT
# ============================================================
print("\n[4/4] V4 3-SHOT CLEAN VAL AUDIT")
print("=" * 70)
n = len(df_son)
ks = df_son["parse_kategori"].value_counts().to_dict()
direct = ks.get("direct", 0); fuzzy = ks.get("fuzzy", 0)
invalid = ks.get("invalid", 0) + ks.get("empty", 0)
print(f"\nParser: direct={direct} ({100*direct/n:.1f}%) fuzzy={fuzzy} invalid={invalid} ({100*invalid/n:.1f}%)")

# Compact strategy
strat = df_son["compact_strategy"].value_counts().to_dict()
print(f"Compact strategy: {strat}")

y_true = df_son["label"].values
y_pred = df_son["parsed_pred"].values
valid = y_pred != "INVALID"
acc_full = accuracy_score(y_true, y_pred)
mf1_full = f1_score(y_true, y_pred, average="macro", zero_division=0)
acc_v = accuracy_score(y_true[valid], y_pred[valid]) if valid.sum() else 0.0
mf1_v = f1_score(y_true[valid], y_pred[valid], average="macro", zero_division=0) if valid.sum() else 0.0

print(f"\nV4 3-SHOT CLEAN Val:")
print(f"   Full        n={n}    acc={acc_full:.4f} macroF1={mf1_full:.4f}")
print(f"   Valid only  n={valid.sum()}  acc={acc_v:.4f} macroF1={mf1_v:.4f}")

# Per-class
labels = sorted(set(y_true) | (set(y_pred) - {"INVALID"}))
p, r, f, supp = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
print(f"\nPer-class:")
for cls, pi, ri, fi, si in zip(labels, p, r, f, supp):
    print(f"   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}")

# Bias dagilimi (ozellikle OTHER)
print(f"\nTahmin dagilimi (bias):")
for cls, sayi in df_son["parsed_pred"].value_counts().items():
    print(f"   {cls:<25s} {sayi:>4} ({100*sayi/n:>5.1f}%)")

# Karsilastirma
print(f"\nKARSILASTIRMA (val):")
print(f"   {'':<32s} {'Acc':>8s} {'MacroF1':>10s} {'OTHER%':>8s}")
print(f"   {'V4 zero-shot':<32s} {0.4818:>8.4f} {0.4367:>10.4f} {15.8:>7.1f}%")
print(f"   {'V4 1-shot CLEAN':<32s} {0.4727:>8.4f} {0.4479:>10.4f} {'?':>7s}")
print(f"   {'V4 3-shot CLEAN (this)':<32s} {acc_full:>8.4f} {mf1_full:>10.4f} {100*(df_son['parsed_pred']=='OTHER').sum()/n:>7.1f}%")

# Kaydet summary
ozet = pd.DataFrame({
    "subset":["full","valid_only"],
    "n":[n, valid.sum()],
    "accuracy":[acc_full, acc_v],
    "macro_f1":[mf1_full, mf1_v],
})
ozet.to_csv(FS_DIR / "fewshot_v4_3shot_clean_val_results_summary.csv", index=False)

print("\n" + "=" * 70)
print("BLOK 14C-V4-3SHOT-CLEAN-VAL TAMAMLANDI - test'e otomatik gecis YOK")
print("=" * 70)


# BLOK 14V4-3SHOT-SMOKE — Gerçek tokenizer audit + 6 örnek smoke (NORMAL compact)

**Amaç:** 3-shot 33-örnek normal compact prompt'un 8 GB GPU'ya uygunluğunu test etmek.

**Sonuç (KRITIK):**
- Sabit prompt: 6.213 token
- Total prompt avg: 6.424 (max 6.619)
- ✅ 8050 üstü 0, 8192 üstü 0 → token açısından sığar
- ❌ **Smoke süre: 33-46 s/örnek** (1-shot 5s'e karşı)
- ❌ **Peak GPU: 11.85-12.45 GB** (8 GB GPU'da shared system memory'e taşıyor)
- Val tahmini: 5.8 saat, test: 17.4 saat → pratik DEĞİL

**Karar:** Normal 3-shot compact yetmez, ULTRA-COMPACT gerekli.

In [ ]:
"""Blok 14V4-3SHOT-SMOKE: Gercek tokenizer audit + 6 ornek smoke test (no full val).
- 3 longest + 3 random val
- Token audit: fixed + query + total
- Aggressive compact fallback (>8050)
- OOM/think check"""
import pandas as pd
import numpy as np
import torch
import json
import re
import time
import random
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
FS_DIR     = OUTPUT_DIR / "fewshot"

print("=" * 70)
print("BLOK 14V4-3SHOT-SMOKE — Token audit + 6 ornek smoke test")
print("=" * 70)

with open(FS_DIR / "fewshot_v4_3shot_clean_examples.json", "r", encoding="utf-8") as f:
    shot3_data = json.load(f)

ALLOWED_CLASSES = ["SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"]

# ============================================================
# Compaction (normal + aggressive)
# ============================================================
SECTION_RE = re.compile(
    r"(Chief Complaint|Symptom History|Past Medical History|Allergies|"
    r"Medications on Admission|Vital Signs):\s*\n?", re.IGNORECASE)
SYMPTOM_KEYWORDS = re.compile(
    r"\b(presents?|reports?|complains?|denies|developed|noted|noticed|started|began|"
    r"pain|cough|fever|dyspnea|SOB|fatigue|weakness|nausea|vomit|diarrhea|bleeding|"
    r"dizz|confusion|swelling|edema|chest|abdomen|headache|hypoxic|hypotens|tachycard|"
    r"palpitations|orthopnea|wheez|melena|hemato|dysuria|frequen|urgency)\b", re.IGNORECASE)

def parse_sections(text):
    if not text: return {}
    parts = SECTION_RE.split(text)
    s = {}
    for i in range(1, len(parts) - 1, 2):
        s[parts[i].strip()] = parts[i+1].strip()
    return s

def cumlelere_bol(t):
    if not t: return []
    return [c.strip() for c in re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", t) if c.strip()]

def hasta_skor(c):
    return len(SYMPTOM_KEYWORDS.findall(c))

def compact_normal(full_text):
    """Normal: KS 3 cumle, PMH 3, Meds 5."""
    sections = parse_sections(full_text)
    parts = []
    cc = sections.get("Chief Complaint", "").strip()
    if cc: parts.append(f"Chief Complaint: {cc}")
    sh = sections.get("Symptom History", "").strip()
    if sh:
        ros = re.search(r"(?im)^\s*(Review of Systems?|ROS)\s*:", sh)
        if ros: sh = sh[:ros.start()]
        cumleler = cumlelere_bol(sh)
        skorli = [(i, c, hasta_skor(c)) for i, c in enumerate(cumleler[:5])]
        secilen = [c for i, c, s in skorli if s > 0][:3]
        if not secilen:
            secilen = [c for i, c, s in skorli][:2]
        else:
            if cumleler and cumleler[0] not in secilen:
                secilen = [cumleler[0]] + secilen
                secilen = secilen[:3]
        if secilen: parts.append(f"Key Symptoms: {' '.join(secilen)}")
    vs = sections.get("Vital Signs", "").strip()
    if vs: parts.append(f"Vitals: {vs}")
    pmh = sections.get("Past Medical History", "").strip()
    if pmh:
        pmh_satirlar = [s.strip() for s in re.split(r"[\n;]+", pmh) if s.strip()]
        kisa = [s for s in pmh_satirlar if 5 < len(s) < 100][:3]
        if not kisa:
            kisa = [c for c in cumlelere_bol(pmh)[:2] if len(c) < 200]
        if kisa: parts.append(f"Relevant History: " + "; ".join(kisa))
    al = sections.get("Allergies", "").strip()
    if al:
        al_kisa = al.split("\n")[0].strip()[:100]
        if al_kisa and al_kisa.upper() not in ("NKDA","NONE","NKA"):
            parts.append(f"Allergies: {al_kisa}")
    meds = sections.get("Medications on Admission", "").strip()
    if meds:
        ilac_satirlar = re.findall(r"\d+\.\s*([A-Z][A-Za-z][^.\n]{2,80})", meds)
        if ilac_satirlar:
            kisa = [" ".join(s.split()[:3]) for s in ilac_satirlar[:5]]
            parts.append(f"Medications: " + ", ".join(kisa))
    return "\n".join(parts)

def compact_aggressive(full_text):
    """Aggressive: KS 2 cumle, PMH 2, Meds 3."""
    sections = parse_sections(full_text)
    parts = []
    cc = sections.get("Chief Complaint", "").strip()
    if cc: parts.append(f"Chief Complaint: {cc}")
    sh = sections.get("Symptom History", "").strip()
    if sh:
        ros = re.search(r"(?im)^\s*(Review of Systems?|ROS)\s*:", sh)
        if ros: sh = sh[:ros.start()]
        cumleler = cumlelere_bol(sh)
        skorli = [(i, c, hasta_skor(c)) for i, c in enumerate(cumleler[:4])]
        secilen = [c for i, c, s in skorli if s > 0][:2]
        if not secilen:
            secilen = [c for i, c, s in skorli][:1]
        if secilen: parts.append(f"Key Symptoms: {' '.join(secilen)}")
    vs = sections.get("Vital Signs", "").strip()
    if vs: parts.append(f"Vitals: {vs[:100]}")
    pmh = sections.get("Past Medical History", "").strip()
    if pmh:
        pmh_satirlar = [s.strip() for s in re.split(r"[\n;]+", pmh) if s.strip()]
        kisa = [s for s in pmh_satirlar if 5 < len(s) < 80][:2]
        if kisa: parts.append(f"Relevant History: " + "; ".join(kisa))
    meds = sections.get("Medications on Admission", "").strip()
    if meds:
        ilac_satirlar = re.findall(r"\d+\.\s*([A-Z][A-Za-z][^.\n]{2,60})", meds)
        if ilac_satirlar:
            kisa = [" ".join(s.split()[:2]) for s in ilac_satirlar[:3]]
            parts.append(f"Medications: " + ", ".join(kisa))
    return "\n".join(parts)

# ============================================================
# Prompt build
# ============================================================
HEADER = """You are a clinical NLP system. Given a patient's triage information collected at first contact (chief complaint, symptoms, brief history, medications, allergies, vital signs), classify the most likely PRIMARY reason for admission into ONE of these 11 categories:

SEPSIS - Systemic infection, septic shock, bacteremia
STROKE - Ischemic or hemorrhagic stroke, CVA
ATRIAL_FIBRILLATION - Atrial fibrillation/flutter (acute)
AKI - Acute kidney injury, acute renal failure
PNEUMONIA - Lung infection (bacterial, viral, aspiration)
UTI - Urinary tract infection, pyelonephritis, cystitis
HEART_FAILURE - Congestive heart failure, decompensated CHF
GI_BLEEDING - Upper or lower GI hemorrhage
DIABETES_COMPLICATION - DKA, HHS, severe hyper/hypoglycemia
COPD_EXACERBATION - Acute exacerbation of COPD
OTHER - None of the above

Rules:
- Output exactly one class label from the list, using the exact uppercase label string.
- Choose the SINGLE most likely PRIMARY reason for admission.
- OTHER is NOT an uncertainty label. Choose OTHER ONLY when none of the 10 named diagnoses is clinically supported.
- If one of the 10 named classes is plausible, choose that named class instead of OTHER.

Below are 33 compact example triage notes (3 per class):
"""
EX_TMPL = "\nEXAMPLE {i} ({label}, ex{rank}):\n{input}\nPRIMARY DIAGNOSIS: {label}\n"
TT = "\nNow classify the following triage note:\nTRIAGE NOTE:\n{input}\nPRIMARY DIAGNOSIS:"

SABIT = HEADER
ex_idx = 1
for cls in ALLOWED_CLASSES:
    for ex in shot3_data[cls]:
        SABIT += EX_TMPL.format(i=ex_idx, label=cls, rank=ex["rank"], input=ex["compact_text"])
        ex_idx += 1

# ============================================================
# Tokenizer & gercek token audit
# ============================================================
print("\n[1/4] Gercek tokenizer audit...")
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def chat_token_count(prompt):
    msgs = [{"role":"user","content":prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return len(tok.encode(text, add_special_tokens=False)), text

def build_prompt(query):
    return SABIT + TT.format(input=query)

# Sabit prompt token (query="" ile)
fixed_prompt = build_prompt("")
fixed_n, _ = chat_token_count(fixed_prompt)
print(f"   Sabit prompt (header+33 examples+trailer wrap, query bos): {fixed_n} token")

# ============================================================
# Val query token istatistigi
# ============================================================
print("\n[2/4] Val query token istatistigi...")
df = pd.read_parquet(SPLIT_YOL)
val_df = df[df["split"]=="val"].reset_index(drop=True)

query_tokens = []
total_tokens = []
total_after_aggr = []
aggr_count = 0
exceed_count = 0

for i, row in val_df.iterrows():
    v4_full = row["triage_input_v4_timeaware_masked"] or ""
    inp_n = compact_normal(v4_full)
    full_prompt = build_prompt(inp_n)
    total_n, _ = chat_token_count(full_prompt)
    query_only = total_n - fixed_n  # yaklasik query token
    query_tokens.append(query_only)
    total_tokens.append(total_n)

    if total_n > 8050:
        aggr_count += 1
        inp_a = compact_aggressive(v4_full)
        total_a, _ = chat_token_count(build_prompt(inp_a))
        total_after_aggr.append(total_a)
        if total_a > 8192:
            exceed_count += 1
    else:
        total_after_aggr.append(total_n)

qa = np.array(query_tokens)
ta = np.array(total_tokens)
taa = np.array(total_after_aggr)

print(f"\n   Val query token (sadece query, normal compact):")
print(f"      Avg: {qa.mean():.0f}  P50: {np.median(qa):.0f}  P95: {np.percentile(qa,95):.0f}  Max: {qa.max():.0f}")
print(f"\n   Total prompt token (normal compact):")
print(f"      Avg: {ta.mean():.0f}  P50: {np.median(ta):.0f}  P95: {np.percentile(ta,95):.0f}  Max: {ta.max():.0f}")
print(f"      8050 ustu (aggressive gerek): {aggr_count} ({100*aggr_count/len(val_df):.1f}%)")
print(f"      8192 ustu (truncate riski, normal): {(ta > 8192).sum()}")

print(f"\n   Total prompt token (aggressive uygulandiktan sonra):")
print(f"      Max: {taa.max():.0f}")
print(f"      8192 ustu (truncate, post-aggressive): {(taa > 8192).sum()}")

# ============================================================
# Smoke test (3 longest + 3 random)
# ============================================================
print("\n[3/4] Smoke test (3 longest + 3 random)...")

# Indeksler
val_with_idx = [(i, ta[i]) for i in range(len(val_df))]
val_with_idx.sort(key=lambda x: -x[1])
longest_3 = [i for i,_ in val_with_idx[:3]]
random.seed(42)
remaining = [i for i in range(len(val_df)) if i not in longest_3]
random_3 = random.sample(remaining, 3)
secilen_idx = longest_3 + random_3

# Model yukle
print("\n   Model yukleniyor...")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-4B", quantization_config=bnb,
    device_map="auto", torch_dtype=torch.float16, local_files_only=True)
model.eval()
print(f"   Yuklendi {time.time()-t0:.1f}s, GPU {torch.cuda.memory_allocated()/1024**2:.0f}MB")

MAX_LEN = 8192

def parse_class_simple(raw):
    if not raw: return "INVALID"
    text = raw.strip().upper()
    for cls in sorted(ALLOWED_CLASSES, key=len, reverse=True):
        if cls in text[:80]: return cls
    return "INVALID"

print(f"\n   {'idx':<5s} {'kategori':<7s} {'tok_pre':>7s} {'strat':<10s} {'tok_post':>8s} {'sure':>7s} {'peak_GB':>8s} {'<think>':>7s} {'parsed':<22s} {'OK':>3s}")
print("   " + "-" * 110)

for kategori, idx_list in [("UZUN", longest_3), ("RAND", random_3)]:
    for idx in idx_list:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        row = val_df.iloc[idx]
        v4_full = row["triage_input_v4_timeaware_masked"] or ""

        # Normal once
        inp_n = compact_normal(v4_full)
        prompt_n = build_prompt(inp_n)
        n_n, _ = chat_token_count(prompt_n)
        if n_n > 8050:
            inp = compact_aggressive(v4_full)
            stratezi = "aggr"
            prompt = build_prompt(inp)
        else:
            inp = inp_n
            stratezi = "normal"
            prompt = prompt_n

        msgs = [{"role":"user","content":prompt}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to("cuda")
        in_n_post = inputs.input_ids.shape[1]

        try:
            t1 = time.time()
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tok.eos_token_id)
            sure = time.time() - t1
            peak_gb = torch.cuda.max_memory_allocated()/1024**3
            raw = tok.decode(out[0][in_n_post:], skip_special_tokens=True).strip()
            parsed = parse_class_simple(raw)
            think_var = "<think>" in raw or "</think>" in raw
            dogru = "OK" if parsed == row["label"] else "X"
            raw_short = raw[:30].replace("\n","\\n")
            print(f"   {idx:<5d} {kategori:<7s} {n_n:>7d} {stratezi:<10s} {in_n_post:>8d} {sure:>6.1f}s {peak_gb:>7.2f}G {str(think_var):>7s} {parsed:<22s} {dogru:>3s}")
            print(f"      raw={raw_short!r}, label={row['label']}")
        except torch.cuda.OutOfMemoryError as e:
            print(f"   {idx:<5d} OOM!")
            torch.cuda.empty_cache()
        except Exception as e:
            print(f"   {idx:<5d} ERROR: {type(e).__name__}: {str(e)[:100]}")

# ============================================================
# Acceptance
# ============================================================
print("\n[4/4] ACCEPTANCE")
print("=" * 70)
print(f"   8192 ustu (post-aggressive): {(taa > 8192).sum()}")
print(f"   Aggressive gereken: {aggr_count}/{len(val_df)} ({100*aggr_count/len(val_df):.1f}%)")
print(f"   OOM/think yukaridaki smoke testlerinde gorulebilir.")
print(f"   Eger smoke testte OOM yok, <think> yok ve aggressive yeterliyse:")
print(f"   -> 14C-V4-3SHOT-CLEAN-VAL kabul edilebilir.")
print("=" * 70)


# BLOK 14V4-3SHOT-ULTRACOMPACT — Ultra-compact 33 örnek (60-80 token hedef)

**Amaç:** 3-shot 33 örnek hedefini 8 GB GPU'ya sığdırmak.

**Format:** CC + Sx (1 cümle) + Hx (1 phrase) + VS (kısa)
Medications ve Allergies çıkarıldı.

**Sonuç:**
- 33 örnek toplam: **959 token** (avg 29/örnek)
- Sabit prompt: **1.805 token**
- Total prompt avg: **2.016**, max **2.211** ✅
- Forbidden phrase: **0/33** ✅
- Smoke süre: **2.3-3.3 s/örnek** ✅
- Peak GPU: **3.97-4.19 GB** (8 GB içinde rahat) ✅
- `<think>`: yok ✅

**Smoke accuracy:** 3/6 (1-shot ile aynı, az örnekte istatistik anlamsız)

**Çıktı:** `output/fewshot/fewshot_v4_3shot_ultracompact_examples.json`

In [ ]:
"""Blok 14V4-3SHOT-ULTRACOMPACT: Ultra compact 3-shot examples (60-80 token).
Hedef: 8 GB GPU'ya uygun, total prompt < 4000 token."""
import pandas as pd
import numpy as np
import torch
import json
import re
import time
import random
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
FS_DIR     = OUTPUT_DIR / "fewshot"

print("=" * 70)
print("BLOK 14V4-3SHOT-ULTRACOMPACT — 60-80 token/example, 8GB-uyumlu")
print("=" * 70)

# 33 clean examples (mevcut)
with open(FS_DIR / "fewshot_v4_3shot_clean_examples.json", "r", encoding="utf-8") as f:
    shot3_data = json.load(f)

ALLOWED_CLASSES = ["SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"]

# ============================================================
# Forbidden phrases (audit)
# ============================================================
FORBIDDEN_PATTERNS = [
    ("code_stroke", re.compile(r"\bcode\s+(stroke|MI|sepsis|trauma)\b", re.IGNORECASE)),
    ("ct_scanning", re.compile(r"\b(CT scanning|reviewed images|read images|reviewed CT|reviewed scan)\b", re.IGNORECASE)),
    ("micu_admission", re.compile(r"\b(REASON FOR (MICU|ICU) ADMISSION|MICU admission|ICU admission)\b", re.IGNORECASE)),
    ("scheduled_proc", re.compile(r"\bscheduled\s+(TEE|DCCV|EGD|colonoscopy|cardioversion|TTE)\b", re.IGNORECASE)),
    ("elevated_lab", re.compile(r"\b(elevated\s+(Cr|creatinine|WBC|Hgb|HCT|lactate|troponin|BUN|sodium|Na)|abnormal\s+labs?)\b", re.IGNORECASE)),
    ("found_in_ed", re.compile(r"\b(found\s+in\s+ED|found\s+to\s+have|found\s+down)\b", re.IGNORECASE)),
    ("given_treatment", re.compile(r"\b(given|received|started\s+on|placed\s+on|treated\s+with)\s+[^.]{0,40}\b(oxygen|CPAP|BiPAP|nebulizer|nebs|albuterol|duonebs|antibiotics|abx|vancomycin|zosyn|lasix|insulin\s+drip|glucagon|tPA|pressors|levophed|crystalloid|IV\s+fluids|fluid\s+bolus)\b", re.IGNORECASE)),
    ("admitted_for", re.compile(r"\b(admitted\s+(for|to)|MICU\s+admission|ED\s+course)\b", re.IGNORECASE)),
    ("imaging_showed", re.compile(r"\b(CXR|CTA?|MRI|TTE|TEE|US|echocardiogram)\b[^.]{0,40}\b(showed|revealed|demonstrated|consistent\s+with|notable|w/|with\s+(infiltrate|consolidation|effusion|edema|ICH|infarct|hemorrhage))\b", re.IGNORECASE)),
    ("labs_showed", re.compile(r"\bLabs?\b[^.]{0,40}\b(showed|notable|revealed|drawn|remarkable)\b", re.IGNORECASE)),
]

def find_forbidden(text):
    if not isinstance(text, str) or not text: return []
    return [(ad, m.group()) for ad, rx in FORBIDDEN_PATTERNS for m in rx.finditer(text)]

# ============================================================
# ULTRA-COMPACT: CC: ... / Sx: 1 cumle / Hx: 1 phrase / VS: kisa
# ============================================================
SECTION_RE = re.compile(
    r"(Chief Complaint|Symptom History|Past Medical History|Allergies|"
    r"Medications on Admission|Vital Signs):\s*\n?", re.IGNORECASE)
SYMPTOM_KEYWORDS = re.compile(
    r"\b(presents?|reports?|complains?|denies|developed|noted|noticed|started|began|"
    r"pain|cough|fever|dyspnea|SOB|fatigue|weakness|nausea|vomit|diarrhea|bleeding|"
    r"dizz|confusion|swelling|edema|chest|abdomen|headache|hypoxic|hypotens|tachycard|"
    r"palpitations|orthopnea|wheez|melena|hemato|dysuria|frequen|urgency)\b", re.IGNORECASE)

def parse_sections(text):
    if not text: return {}
    parts = SECTION_RE.split(text)
    s = {}
    for i in range(1, len(parts) - 1, 2):
        s[parts[i].strip()] = parts[i+1].strip()
    return s

def cumlelere_bol(t):
    if not t: return []
    return [c.strip() for c in re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", t) if c.strip()]

def hasta_skor(c):
    return len(SYMPTOM_KEYWORDS.findall(c))

def ultra_compact(full_text):
    """Ultra compact: CC + 1 sx cumle + 1 hx + kisa VS. Meds/allergies YOK."""
    sections = parse_sections(full_text)
    parts = []

    # CC: tam (zaten kisa)
    cc = sections.get("Chief Complaint", "").strip()
    if cc:
        # CC'i 80 char max
        cc_kisa = cc[:80].strip()
        parts.append(f"CC: {cc_kisa}")

    # Sx: 1 ana semptom cumle (en yuksek skorlu, max 200 char)
    sh = sections.get("Symptom History", "").strip()
    if sh:
        ros = re.search(r"(?im)^\s*(Review of Systems?|ROS)\s*:", sh)
        if ros: sh = sh[:ros.start()]
        cumleler = cumlelere_bol(sh)
        if cumleler:
            # Skoru en yuksek olan ilk cumleyi al (ya da ilk cumle)
            skorlu = sorted([(i, c, hasta_skor(c)) for i, c in enumerate(cumleler[:4])],
                            key=lambda x: (-x[2], x[0]))
            ana = skorlu[0][1] if skorlu else cumleler[0]
            ana_kisa = ana[:200].strip()
            parts.append(f"Sx: {ana_kisa}")

    # Hx: 1 onemli kisa phrase (PMH'den, max 80 char)
    pmh = sections.get("Past Medical History", "").strip()
    if pmh:
        pmh_satirlar = [s.strip() for s in re.split(r"[\n;,]+", pmh) if s.strip()]
        # Onemli liste eleman (5-60 char)
        kisa = [s for s in pmh_satirlar if 5 < len(s) < 60][:1]
        if kisa:
            parts.append(f"Hx: {kisa[0]}")

    # VS: sadece kisa vital ozeti
    vs = sections.get("Vital Signs", "").strip()
    if vs:
        # VS satirini 60 char max
        vs_kisa = vs.replace("\n"," ")[:60].strip()
        if vs_kisa:
            parts.append(f"VS: {vs_kisa}")

    return "\n".join(parts)

# ============================================================
# Ultra compact tum 33 ornek
# ============================================================
print("\n[1/5] 33 ornek ultra compact...")
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

ultra_data = {}
toplam_kirli = 0
toplam_token = 0

for cls in ALLOWED_CLASSES:
    ultra_data[cls] = []
    for ex in shot3_data[cls]:
        # Compact'i parse edip ultra hale getir
        # Ama compact_text section header'i farkli (Chief Complaint:, Key Symptoms:)
        # ultra_compact full V4 input istiyor — eski compact'tan re-parse et
        # Compact_text'te zaten section header var, parse_sections calisir
        ultra_text = ultra_compact(ex["compact_text"])
        # Eger ultra_compact bos donduriyorsa (parse failed), full V4'e bak
        if not ultra_text or len(ultra_text) < 30:
            # df'den note_id ile full V4 al
            df = pd.read_parquet(SPLIT_YOL)
            row_match = df[df["note_id"]==ex["note_id"]]
            if len(row_match) > 0:
                v4_full = row_match.iloc[0]["triage_input_v4_timeaware_masked"] or ""
                ultra_text = ultra_compact(v4_full)

        n_tok = len(tok.encode(ultra_text, add_special_tokens=False))
        bulg = find_forbidden(ultra_text)
        toplam_kirli += len(bulg)
        toplam_token += n_tok
        ultra_data[cls].append({
            "rank": ex["rank"],
            "note_id": ex["note_id"],
            "leakage_severity": ex["leakage_severity"],
            "ultra_token": n_tok,
            "ultra_char": len(ultra_text),
            "ultra_text": ultra_text,
            "forbidden_count": len(bulg),
        })

# Audit
print(f"\n   Ultra compact 33 ornek:")
print(f"   {'Sinif':<25s} {'rank':>4s} {'note':<22s} {'ultra_tok':>10s} {'fb':>3s}")
print("   " + "-" * 70)
for cls in ALLOWED_CLASSES:
    for ex in ultra_data[cls]:
        print(f"   {cls:<25s} {ex['rank']:>4} {ex['note_id']:<22s} {ex['ultra_token']:>10} {ex['forbidden_count']:>3}")

print(f"\n   Toplam token: {toplam_token:,}")
print(f"   Avg/example : {toplam_token/33:.0f} (hedef 60-80)")
print(f"   Toplam kirli: {toplam_kirli}")

# ============================================================
# Prompt build
# ============================================================
HEADER = """You are a clinical NLP system. Given a patient's triage information collected at first contact (chief complaint, symptoms, brief history, medications, allergies, vital signs), classify the most likely PRIMARY reason for admission into ONE of these 11 categories:

SEPSIS - Systemic infection, septic shock, bacteremia
STROKE - Ischemic or hemorrhagic stroke, CVA
ATRIAL_FIBRILLATION - Atrial fibrillation/flutter (acute)
AKI - Acute kidney injury, acute renal failure
PNEUMONIA - Lung infection (bacterial, viral, aspiration)
UTI - Urinary tract infection, pyelonephritis, cystitis
HEART_FAILURE - Congestive heart failure, decompensated CHF
GI_BLEEDING - Upper or lower GI hemorrhage
DIABETES_COMPLICATION - DKA, HHS, severe hyper/hypoglycemia
COPD_EXACERBATION - Acute exacerbation of COPD
OTHER - None of the above

Rules:
- Output exactly one class label from the list, using the exact uppercase label string.
- Choose the SINGLE most likely PRIMARY reason for admission.
- OTHER is NOT an uncertainty label. Choose OTHER ONLY when none of the 10 named diagnoses is clinically supported.
- If one of the 10 named classes is plausible, choose that named class instead of OTHER.

Below are 33 ultra-compact example triage notes (3 per class):
"""
EX_TMPL = "\nEx{i} ({label}): {input}\n=> {label}\n"
TT = "\nNow classify:\n{input}\n=> "

SABIT = HEADER
ex_idx = 1
for cls in ALLOWED_CLASSES:
    for ex in ultra_data[cls]:
        SABIT += EX_TMPL.format(i=ex_idx, label=cls, input=ex["ultra_text"].replace("\n"," | "))
        ex_idx += 1

# Sabit token
def chat_token_count(prompt):
    msgs = [{"role":"user","content":prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return len(tok.encode(text, add_special_tokens=False)), text

print("\n[2/5] Sabit prompt token audit...")
fixed_prompt = SABIT + TT.format(input="")
fixed_n, _ = chat_token_count(fixed_prompt)
print(f"   Sabit prompt (header + 33 ultra example + trailer wrap): {fixed_n} token")

# ============================================================
# Val query token istatistigi
# ============================================================
print("\n[3/5] Val query token istatistigi (V4 compact query, runtime)...")

# Mevcut V3 compact query mantigi (14C-V4-VAL ile ayni)
def compact_v4_query(full_text):
    sections = parse_sections(full_text)
    parts = []
    cc = sections.get("Chief Complaint", "").strip()
    if cc: parts.append(f"Chief Complaint: {cc}")
    sh = sections.get("Symptom History", "").strip()
    if sh:
        ros = re.search(r"(?im)^\s*(Review of Systems?|ROS)\s*:", sh)
        if ros: sh = sh[:ros.start()]
        cumleler = cumlelere_bol(sh)
        skorli = [(i, c, hasta_skor(c)) for i, c in enumerate(cumleler[:5])]
        secilen = [c for i, c, s in skorli if s > 0][:3]
        if not secilen:
            secilen = [c for i, c, s in skorli][:2]
        else:
            if cumleler and cumleler[0] not in secilen:
                secilen = [cumleler[0]] + secilen
                secilen = secilen[:3]
        if secilen: parts.append(f"Key Symptoms: {' '.join(secilen)}")
    vs = sections.get("Vital Signs", "").strip()
    if vs: parts.append(f"Vitals: {vs}")
    pmh = sections.get("Past Medical History", "").strip()
    if pmh:
        pmh_satirlar = [s.strip() for s in re.split(r"[\n;]+", pmh) if s.strip()]
        kisa = [s for s in pmh_satirlar if 5 < len(s) < 100][:3]
        if not kisa:
            kisa = [c for c in cumlelere_bol(pmh)[:2] if len(c) < 200]
        if kisa: parts.append(f"Relevant History: " + "; ".join(kisa))
    al = sections.get("Allergies", "").strip()
    if al:
        al_kisa = al.split("\n")[0].strip()[:100]
        if al_kisa and al_kisa.upper() not in ("NKDA","NONE","NKA"):
            parts.append(f"Allergies: {al_kisa}")
    meds = sections.get("Medications on Admission", "").strip()
    if meds:
        ilac_satirlar = re.findall(r"\d+\.\s*([A-Z][A-Za-z][^.\n]{2,80})", meds)
        if ilac_satirlar:
            kisa = [" ".join(s.split()[:3]) for s in ilac_satirlar[:5]]
            parts.append(f"Medications: " + ", ".join(kisa))
    return "\n".join(parts)

df = pd.read_parquet(SPLIT_YOL)
val_df = df[df["split"]=="val"].reset_index(drop=True)

total_tokens = []
query_tokens = []
for _, row in val_df.iterrows():
    v4_full = row["triage_input_v4_timeaware_masked"] or ""
    inp = compact_v4_query(v4_full)
    full_prompt = SABIT + TT.format(input=inp)
    total_n, _ = chat_token_count(full_prompt)
    query_tokens.append(total_n - fixed_n)
    total_tokens.append(total_n)

ta = np.array(total_tokens)
qa = np.array(query_tokens)
print(f"\n   Query (sadece V4 compact query):")
print(f"      Avg: {qa.mean():.0f}  P50: {np.median(qa):.0f}  P95: {np.percentile(qa,95):.0f}  Max: {qa.max():.0f}")
print(f"\n   Total prompt:")
print(f"      Avg: {ta.mean():.0f}  P50: {np.median(ta):.0f}  P95: {np.percentile(ta,95):.0f}  Max: {ta.max():.0f}")
print(f"      4000 ustu: {(ta > 4000).sum()}")
print(f"      4096 ustu: {(ta > 4096).sum()}")
print(f"      8192 ustu: {(ta > 8192).sum()}")

# ============================================================
# Smoke test
# ============================================================
print("\n[4/5] Smoke test (3 longest + 3 random)...")

val_with_idx = [(i, ta[i]) for i in range(len(val_df))]
val_with_idx.sort(key=lambda x: -x[1])
longest_3 = [i for i,_ in val_with_idx[:3]]
random.seed(42)
remaining = [i for i in range(len(val_df)) if i not in longest_3]
random_3 = random.sample(remaining, 3)

print("\n   Model yukleniyor...")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-4B", quantization_config=bnb,
    device_map="auto", torch_dtype=torch.float16, local_files_only=True)
model.eval()
print(f"   Yuklendi {time.time()-t0:.1f}s, GPU {torch.cuda.memory_allocated()/1024**2:.0f}MB")

MAX_LEN = 4096

def parse_class_simple(raw):
    if not raw: return "INVALID"
    text = raw.strip().upper()
    for cls in sorted(ALLOWED_CLASSES, key=len, reverse=True):
        if cls in text[:80]: return cls
    return "INVALID"

print(f"\n   {'idx':<5s} {'cat':<6s} {'tok':>5s} {'sure':>7s} {'peak_GB':>8s} {'<think>':>7s} {'parsed':<22s} {'OK':>3s}")
print("   " + "-" * 90)

for kategori, idx_list in [("UZUN", longest_3), ("RAND", random_3)]:
    for idx in idx_list:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        row = val_df.iloc[idx]
        v4_full = row["triage_input_v4_timeaware_masked"] or ""
        inp = compact_v4_query(v4_full)
        prompt = SABIT + TT.format(input=inp)
        msgs = [{"role":"user","content":prompt}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to("cuda")
        in_n = inputs.input_ids.shape[1]

        try:
            t1 = time.time()
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tok.eos_token_id)
            sure = time.time() - t1
            peak_gb = torch.cuda.max_memory_allocated()/1024**3
            raw = tok.decode(out[0][in_n:], skip_special_tokens=True).strip()
            parsed = parse_class_simple(raw)
            think_var = "<think>" in raw or "</think>" in raw
            dogru = "OK" if parsed == row["label"] else "X"
            print(f"   {idx:<5d} {kategori:<6s} {in_n:>5d} {sure:>6.1f}s {peak_gb:>7.2f}G {str(think_var):>7s} {parsed:<22s} {dogru:>3s}")
            print(f"      raw={raw[:30]!r}, label={row['label']}")
        except torch.cuda.OutOfMemoryError:
            print(f"   {idx:<5d} OOM!")
            torch.cuda.empty_cache()
        except Exception as e:
            print(f"   {idx:<5d} ERROR: {type(e).__name__}: {str(e)[:100]}")

# ============================================================
# Acceptance + Kaydet
# ============================================================
print("\n[5/5] ACCEPTANCE + KAYDETME")
print("=" * 70)
print(f"   Hedef: total prompt avg < 4000")
print(f"   Sonuc: avg={ta.mean():.0f}, max={ta.max():.0f}")
print(f"   Hedef: smoke sure < 12 s/ornek (yukarida)")
print(f"   Hedef: peak GPU shared'a tasmasin (8 GB altinda)")
print(f"   Hedef: <think> yok (yukarida)")
print(f"   Hedef: forbidden 0/33: {'EVET' if toplam_kirli == 0 else f'HAYIR ({toplam_kirli})'}")

# JSON kaydet
out_yol = FS_DIR / "fewshot_v4_3shot_ultracompact_examples.json"
with open(out_yol, "w", encoding="utf-8") as f:
    json.dump(ultra_data, f, ensure_ascii=False, indent=2)
print(f"\n   Kaydedildi: {out_yol.name}")

# Preview
print("\n" + "=" * 70)
print("PREVIEW (her sinif rank 1)")
print("=" * 70)
for cls in ALLOWED_CLASSES:
    if not ultra_data[cls]: continue
    ex = ultra_data[cls][0]
    print(f"\n   {cls} (note: {ex['note_id']}, tok: {ex['ultra_token']}):")
    print(f"   {ex['ultra_text']}")

print("\n" + "=" * 70)
print("BLOK 14V4-3SHOT-ULTRACOMPACT TAMAMLANDI - val'e otomatik gecis YOK")
print("=" * 70)


# BLOK 14C-V4-3SHOT-ULTRA-VAL — V4 3-shot ULTRA-COMPACT Val Inference

**⚠️ ÇOK ÖNEMLİ: Bu hücreyi SEN çalıştır (Jupyter → Run Cell)**
**Tahmini süre:** ~25-35 dakika (550 örnek × ~2.5-3 s/örnek)

**Konfigürasyon:**
- Examples: 33 V4 3-shot ULTRA-COMPACT medoid (`fewshot_v4_3shot_ultracompact_examples.json`, avg 29 tok/örnek)
- Query: V4 compact (runtime, normal — aggressive gerekmiyor)
- max_length=4096, max_new_tokens=20, do_sample=False
- chat_template + enable_thinking=False
- Checkpoint her 25 örnekte

**Total prompt boyutu:** avg 2.016, max 2.211 (smoke onayli)
**Peak GPU beklenen:** ~4.2 GB (8 GB içinde rahat)

**Karşılaştırma hedefi (val):**
| Yöntem | Acc | Macro-F1 | OTHER% | HF% |
|---|---:|---:|---:|---:|
| V4 zero-shot | 0.4818 | 0.4367 | 15.8% | 20.2% |
| V4 1-shot CLEAN | 0.4727 | 0.4479 | ? | ? |
| **V4 3-shot ULTRA (bu hücre)** | **?** | **?** | **?** | **?** |

**Çıktılar:**
- `output/fewshot/fewshot_v4_3shot_ultracompact_val.csv` (550 satır)
- `output/fewshot/fewshot_v4_3shot_ultracompact_val_checkpoint.csv`
- `output/fewshot/fewshot_v4_3shot_ultracompact_val_results_summary.csv`

**Detaylı script (terminal alternatifi):** `__run_blok14c_v4_3shot_ultra_val.py`

TEST'e otomatik geçiş YOK. Val sonucundan sonra ayrı hücre.

In [ ]:
"""Blok 14C-V4-3SHOT-ULTRA-VAL: V4 3-shot ULTRA-COMPACT few-shot val (550 ornek).
- 33 ultra-compact medoid (avg 29 tok)
- Query V4 compact (runtime)
- max_length=4096, sure ~25-35 dk
- Test'e otomatik gecmez."""
import pandas as pd
import numpy as np
import torch
import json
import re
import time
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
FS_DIR     = OUTPUT_DIR / "fewshot"

print("=" * 70)
print("BLOK 14C-V4-3SHOT-ULTRA-VAL — V4 3-shot ULTRA-COMPACT val")
print("=" * 70)
print("   KURAL: Sadece val. Test'e otomatik gecmez.")

# Ultra-compact 33 medoid
with open(FS_DIR / "fewshot_v4_3shot_ultracompact_examples.json", "r", encoding="utf-8") as f:
    ultra_data = json.load(f)

ALLOWED_CLASSES = ["SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"]

# ============================================================
# V4 query compact (smoke ile ayni)
# ============================================================
SECTION_RE = re.compile(
    r"(Chief Complaint|Symptom History|Past Medical History|Allergies|"
    r"Medications on Admission|Vital Signs):\s*\n?", re.IGNORECASE)
SYMPTOM_KEYWORDS = re.compile(
    r"\b(presents?|reports?|complains?|denies|developed|noted|noticed|started|began|"
    r"pain|cough|fever|dyspnea|SOB|fatigue|weakness|nausea|vomit|diarrhea|bleeding|"
    r"dizz|confusion|swelling|edema|chest|abdomen|headache|hypoxic|hypotens|tachycard|"
    r"palpitations|orthopnea|wheez|melena|hemato|dysuria|frequen|urgency)\b", re.IGNORECASE)

def parse_sections(text):
    if not text: return {}
    parts = SECTION_RE.split(text)
    s = {}
    for i in range(1, len(parts) - 1, 2):
        s[parts[i].strip()] = parts[i+1].strip()
    return s

def cumlelere_bol(t):
    if not t: return []
    return [c.strip() for c in re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", t) if c.strip()]

def hasta_skor(c):
    return len(SYMPTOM_KEYWORDS.findall(c))

def compact_v4_query(full_text):
    sections = parse_sections(full_text)
    parts = []
    cc = sections.get("Chief Complaint", "").strip()
    if cc: parts.append(f"Chief Complaint: {cc}")
    sh = sections.get("Symptom History", "").strip()
    if sh:
        ros = re.search(r"(?im)^\s*(Review of Systems?|ROS)\s*:", sh)
        if ros: sh = sh[:ros.start()]
        cumleler = cumlelere_bol(sh)
        skorli = [(i, c, hasta_skor(c)) for i, c in enumerate(cumleler[:5])]
        secilen = [c for i, c, s in skorli if s > 0][:3]
        if not secilen:
            secilen = [c for i, c, s in skorli][:2]
        else:
            if cumleler and cumleler[0] not in secilen:
                secilen = [cumleler[0]] + secilen
                secilen = secilen[:3]
        if secilen: parts.append(f"Key Symptoms: {' '.join(secilen)}")
    vs = sections.get("Vital Signs", "").strip()
    if vs: parts.append(f"Vitals: {vs}")
    pmh = sections.get("Past Medical History", "").strip()
    if pmh:
        pmh_satirlar = [s.strip() for s in re.split(r"[\n;]+", pmh) if s.strip()]
        kisa = [s for s in pmh_satirlar if 5 < len(s) < 100][:3]
        if not kisa:
            kisa = [c for c in cumlelere_bol(pmh)[:2] if len(c) < 200]
        if kisa: parts.append(f"Relevant History: " + "; ".join(kisa))
    al = sections.get("Allergies", "").strip()
    if al:
        al_kisa = al.split("\n")[0].strip()[:100]
        if al_kisa and al_kisa.upper() not in ("NKDA","NONE","NKA"):
            parts.append(f"Allergies: {al_kisa}")
    meds = sections.get("Medications on Admission", "").strip()
    if meds:
        ilac_satirlar = re.findall(r"\d+\.\s*([A-Z][A-Za-z][^.\n]{2,80})", meds)
        if ilac_satirlar:
            kisa = [" ".join(s.split()[:3]) for s in ilac_satirlar[:5]]
            parts.append(f"Medications: " + ", ".join(kisa))
    return "\n".join(parts)

# ============================================================
# Prompt build (smoke ile ayni format)
# ============================================================
HEADER = """You are a clinical NLP system. Given a patient's triage information collected at first contact (chief complaint, symptoms, brief history, medications, allergies, vital signs), classify the most likely PRIMARY reason for admission into ONE of these 11 categories:

SEPSIS - Systemic infection, septic shock, bacteremia
STROKE - Ischemic or hemorrhagic stroke, CVA
ATRIAL_FIBRILLATION - Atrial fibrillation/flutter (acute)
AKI - Acute kidney injury, acute renal failure
PNEUMONIA - Lung infection (bacterial, viral, aspiration)
UTI - Urinary tract infection, pyelonephritis, cystitis
HEART_FAILURE - Congestive heart failure, decompensated CHF
GI_BLEEDING - Upper or lower GI hemorrhage
DIABETES_COMPLICATION - DKA, HHS, severe hyper/hypoglycemia
COPD_EXACERBATION - Acute exacerbation of COPD
OTHER - None of the above

Rules:
- Output exactly one class label from the list, using the exact uppercase label string.
- Choose the SINGLE most likely PRIMARY reason for admission.
- OTHER is NOT an uncertainty label. Choose OTHER ONLY when none of the 10 named diagnoses is clinically supported.
- If one of the 10 named classes is plausible, choose that named class instead of OTHER.

Below are 33 ultra-compact example triage notes (3 per class):
"""
EX_TMPL = "\nEx{i} ({label}): {input}\n=> {label}\n"
TT = "\nNow classify:\n{input}\n=> "

SABIT = HEADER
ex_idx = 1
for cls in ALLOWED_CLASSES:
    for ex in ultra_data[cls]:
        SABIT += EX_TMPL.format(i=ex_idx, label=cls, input=ex["ultra_text"].replace("\n"," | "))
        ex_idx += 1

FUZZY_MAP = {
    "SEPS":"SEPSIS","SEPT":"SEPSIS","PNEUM":"PNEUMONIA",
    "STROK":"STROKE","CVA":"STROKE",
    "ATRIAL":"ATRIAL_FIBRILLATION","AFIB":"ATRIAL_FIBRILLATION","FIBRIL":"ATRIAL_FIBRILLATION",
    "AKI":"AKI","KIDNEY":"AKI","RENAL":"AKI",
    "UTI":"UTI","URIN":"UTI",
    "HEART":"HEART_FAILURE","CHF":"HEART_FAILURE",
    "GI":"GI_BLEEDING","BLEED":"GI_BLEEDING",
    "DIAB":"DIABETES_COMPLICATION","DKA":"DIABETES_COMPLICATION",
    "COPD":"COPD_EXACERBATION","OTHER":"OTHER","NONE":"OTHER",
}

def parse_class(raw):
    if not raw: return "INVALID","empty"
    text = raw.strip().upper()
    for cls in sorted(ALLOWED_CLASSES, key=len, reverse=True):
        if cls in text[:80]: return cls,"direct"
    for k,v in FUZZY_MAP.items():
        if k in text[:80]: return v,"fuzzy"
    return "INVALID","invalid"

# ============================================================
# Model
# ============================================================
print("\n[1/4] Model yukleme...")
t0 = time.time()
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-4B", quantization_config=bnb,
    device_map="auto", torch_dtype=torch.float16, local_files_only=True)
model.eval()
print(f"   Yuklendi {time.time()-t0:.1f}s, GPU {torch.cuda.memory_allocated()/1024**2:.0f}MB")

MAX_LEN = 4096

def inference(query):
    prompt = SABIT + TT.format(input=query)
    msgs = [{"role":"user","content":prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return raw, inputs.input_ids.shape[1]

# ============================================================
# Veri ve inference
# ============================================================
print("\n[2/4] Val seti...")
df = pd.read_parquet(SPLIT_YOL)
val_df = df[df["split"]=="val"].reset_index(drop=True)
print(f"   Val: {len(val_df):,}")

print("\n[3/4] Inference (checkpoint/resume)...")
cp_yol = FS_DIR / "fewshot_v4_3shot_ultracompact_val_checkpoint.csv"
final_yol = FS_DIR / "fewshot_v4_3shot_ultracompact_val.csv"

if cp_yol.exists():
    cp = pd.read_csv(cp_yol)
    islenen = set(cp["note_id"].astype(str))
    kayitlar = cp.to_dict("records")
    print(f"   Resume: {len(islenen)} kayit zaten var.")
else:
    islenen = set()
    kayitlar = []

t_baslangic = time.time()
sayac_yeni = 0

for i, row in val_df.iterrows():
    nid = str(row["note_id"])
    if nid in islenen: continue
    # NaN guvenligi
    v4_full = row["triage_input_v4_timeaware_masked"]
    if not isinstance(v4_full, str):
        v4_full = ""
    inp = compact_v4_query(v4_full)

    t0 = time.time()
    raw, in_n = inference(inp)
    sure = time.time() - t0
    parsed, kat = parse_class(raw)
    kayitlar.append({
        "note_id": nid, "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]), "label": row["label"],
        "raw_output": raw, "parsed_pred": parsed,
        "parse_kategori": kat, "in_tokens": int(in_n),
        "sure_s": round(sure, 3),
        "has_think": ("<think>" in raw or "</think>" in raw),
        "leakage_severity_final": str(row.get("strict_current_leakage_severity_final","UNK")),
    })
    sayac_yeni += 1
    if len(kayitlar) % 25 == 0:
        pd.DataFrame(kayitlar).to_csv(cp_yol, index=False)
        gecen = time.time() - t_baslangic
        ort = gecen / sayac_yeni if sayac_yeni > 0 else 0
        kalan = (len(val_df) - len(kayitlar)) * ort
        print(f"   [{len(kayitlar):>3}/{len(val_df)}] gecen {gecen/60:.1f}dk ort {ort:.2f}s kalan ~{kalan/60:.1f}dk")

df_son = pd.DataFrame(kayitlar)
df_son.to_csv(final_yol, index=False)
df_son.to_csv(cp_yol, index=False)
toplam = time.time() - t_baslangic
print(f"   BITTI. {len(kayitlar)} kayit, {toplam/60:.1f}dk")

# ============================================================
# AUDIT
# ============================================================
print("\n[4/4] V4 3-SHOT ULTRA VAL AUDIT")
print("=" * 70)
n = len(df_son)
ks = df_son["parse_kategori"].value_counts().to_dict()
direct = ks.get("direct", 0); fuzzy = ks.get("fuzzy", 0)
invalid = ks.get("invalid", 0) + ks.get("empty", 0)
print(f"\nParser: direct={direct} ({100*direct/n:.1f}%) fuzzy={fuzzy} invalid={invalid} ({100*invalid/n:.1f}%)")

# Think + token istatistik
think_count = df_son["has_think"].sum() if "has_think" in df_son.columns else 0
print(f"Think output: {think_count}/{n} ({100*think_count/n:.1f}%)")
print(f"Input token avg/max: {df_son['in_tokens'].mean():.0f} / {df_son['in_tokens'].max()}")

y_true = df_son["label"].values
y_pred = df_son["parsed_pred"].values
valid = y_pred != "INVALID"
acc_full = accuracy_score(y_true, y_pred)
mf1_full = f1_score(y_true, y_pred, average="macro", zero_division=0)
acc_v = accuracy_score(y_true[valid], y_pred[valid]) if valid.sum() else 0.0
mf1_v = f1_score(y_true[valid], y_pred[valid], average="macro", zero_division=0) if valid.sum() else 0.0

print(f"\nV4 3-SHOT ULTRA Val:")
print(f"   Full        n={n}    acc={acc_full:.4f} macroF1={mf1_full:.4f}")
print(f"   Valid only  n={valid.sum()}  acc={acc_v:.4f} macroF1={mf1_v:.4f}")

# Per-class
labels = sorted(set(y_true) | (set(y_pred) - {"INVALID"}))
p, r, f, supp = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
print(f"\nPer-class:")
for cls, pi, ri, fi, si in zip(labels, p, r, f, supp):
    print(f"   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}")

# Bias
print(f"\nTahmin dagilimi (bias, ozellikle OTHER ve HEART_FAILURE):")
for cls, sayi in df_son["parsed_pred"].value_counts().items():
    print(f"   {cls:<25s} {sayi:>4} ({100*sayi/n:>5.1f}%)")

# Karsilastirma
other_pct = 100*(df_son['parsed_pred']=='OTHER').sum()/n
hf_pct = 100*(df_son['parsed_pred']=='HEART_FAILURE').sum()/n
print(f"\nKARSILASTIRMA (val):")
print(f"   {'':<32s} {'Acc':>8s} {'MacroF1':>10s} {'OTHER%':>8s} {'HF%':>7s}")
print(f"   {'V4 zero-shot':<32s} {0.4818:>8.4f} {0.4367:>10.4f} {15.8:>7.1f}% {20.2:>6.1f}%")
print(f"   {'V4 1-shot CLEAN':<32s} {0.4727:>8.4f} {0.4479:>10.4f} {'?':>7s} {'?':>6s}")
print(f"   {'V4 3-shot ULTRA (this)':<32s} {acc_full:>8.4f} {mf1_full:>10.4f} {other_pct:>7.1f}% {hf_pct:>6.1f}%")

# Summary
ozet = pd.DataFrame({
    "subset":["full","valid_only"],
    "n":[n, valid.sum()],
    "accuracy":[acc_full, acc_v],
    "macro_f1":[mf1_full, mf1_v],
})
ozet.to_csv(FS_DIR / "fewshot_v4_3shot_ultracompact_val_results_summary.csv", index=False)

print("\n" + "=" * 70)
print("BLOK 14C-V4-3SHOT-ULTRA-VAL TAMAMLANDI - test'e otomatik gecis YOK")
print("=" * 70)


In [ ]:
import json
import re
import pandas as pd
from pathlib import Path

JSON_DIR = Path(r"./workdir/output/qlora_json")

FILES = {
    "train_S3": JSON_DIR / "qlora_v4_train_S3.json",
    "val": JSON_DIR / "qlora_v4_val.json",
    "test": JSON_DIR / "qlora_v4_test.json",
}

RISK_PHRASES = re.compile(
    r"\b("
    r"admitted\s+for|being\s+admitted\s+for|directly\s+admitted\s+for|"
    r"found\s+to\s+have|currently\s+(being\s+)?treated\s+for|"
    r"work[\s-]?up\s+of\s+suspected|work[\s-]?up\s+for\s+suspected|"
    r"concern\s+for|consistent\s+with"
    r")\b",
    re.IGNORECASE
)

HISTORICAL_MARKERS = re.compile(
    r"\b("
    r"history\s+of|hx\s+of|h/o|pmh|past\s+medical|"
    r"prior|previous|previously|remote|chronic|baseline|"
    r"s/p|status\s+post|recurrent|"
    r"last\s+(admission|hospitalization|visit|tte|echo|ct|mri|cxr|labs?)|"
    r"previous\s+(admission|hospitalization|visit)|"
    r"prior\s+(admission|hospitalization|visit|workup|imaging|labs?)|"
    r"recently\s+(admitted|discharged|hospitalized)|"
    r"recent\s+(admission|hospitalization)|"
    r"discharged\s+(from|on|with)|since\s+discharge|"
    r"hospitalized\s+(in|on|at)?\s*___|admitted\s+(in|on|at)?\s*___|"
    r"(\d+|few|several|many|___)\s+(days?|weeks?|months?|years?)\s+ago|"
    r"in\s+___|on\s+___"
    r")\b",
    re.IGNORECASE
)

CURRENT_MARKERS = re.compile(
    r"\b("
    r"now\s+admitted|currently\s+admitted|this\s+admission|"
    r"on\s+arrival|upon\s+arrival|on\s+presentation|"
    r"in\s+the\s+ed|in\s+ed|ed\s+course|"
    r"initial\s+(labs?|workup|evaluation)|"
    r"on\s+admission|upon\s+admission|"
    r"presents?\s+(with|to)|presented\s+(with|to)|"
    r"transferred\s+from|sent\s+from|brought\s+in"
    r")\b",
    re.IGNORECASE
)

SECTION_RE = re.compile(
    r"(Chief Complaint|Symptom History|Past Medical History|Allergies|"
    r"Medications on Admission|Vital Signs):\s*\n?",
    re.IGNORECASE
)

HISTORICAL_SECTIONS = {"past medical history", "medications on admission", "allergies"}
CURRENT_SECTIONS = {"chief complaint", "symptom history", "vital signs"}

def split_sentences(text):
    if not isinstance(text, str) or not text.strip():
        return []
    text = text.replace("\r", "\n")
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9_\[])", text)
    out = []
    for p in parts:
        for q in p.split("\n"):
            q = q.strip()
            if q:
                out.append(q)
    return out

def iter_section_sentences(text):
    parts = SECTION_RE.split(text or "")
    if len(parts) <= 1:
        for s in split_sentences(text):
            yield "unknown", s
        return

    for i in range(1, len(parts) - 1, 2):
        section = parts[i].strip().lower()
        body = parts[i + 1]
        for s in split_sentences(body):
            yield section, s

def classify_temporal(sentence, section):
    risk_match = RISK_PHRASES.search(sentence)
    if not risk_match:
        return None

    has_hist = bool(HISTORICAL_MARKERS.search(sentence))
    has_curr = bool(CURRENT_MARKERS.search(sentence))
    sec = (section or "unknown").lower()

    if sec in HISTORICAL_SECTIONS and not has_curr:
        return "HISTORICAL_ALLOWED", "risk phrase inside historical section"

    if has_hist and not has_curr:
        return "HISTORICAL_ALLOWED", "explicit historical marker"

    if has_hist and has_curr:
        return "REVIEW_MIXED", "both historical and current markers"

    if has_curr:
        return "CURRENT_RISK", "explicit current/admission marker"

    if sec in CURRENT_SECTIONS:
        return "REVIEW_CURRENT_SECTION", "risk phrase inside current-facing section"

    return "REVIEW_UNCERTAIN", "risk phrase but temporal context unclear"

def audit_records(records, split_name):
    rows = []

    for r in records:
        text = r.get("input", "")
        for section, sent in iter_section_sentences(text):
            decision = classify_temporal(sent, section)
            if decision is None:
                continue

            temporal_label, reason = decision
            phrase = RISK_PHRASES.search(sent).group(0)

            rows.append({
                "split": split_name,
                "note_id": r.get("note_id"),
                "gold_label": r.get("output"),
                "section": section,
                "phrase": phrase,
                "decision": temporal_label,
                "reason": reason,
                "sentence": sent,
            })

    return rows

all_rows = []
split_sizes = {}

for split_name, path in FILES.items():
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    split_sizes[split_name] = len(records)
    all_rows.extend(audit_records(records, split_name))

audit = pd.DataFrame(all_rows)

print("=" * 80)
print("V4 TRAIN/VAL/TEST TEMPORAL LEAKAGE AUDIT")
print("=" * 80)

print("\nSplit sizes:")
for split_name, n in split_sizes.items():
    print(f"  {split_name:<8s}: {n:,}")

print("\nToplam riskli cumle:", len(audit))

print("\nDecision dagilimi - genel:")
print(audit["decision"].value_counts(dropna=False))

print("\nDecision dagilimi - split bazli:")
print(pd.crosstab(audit["split"], audit["decision"], margins=True))

print("\nPhrase dagilimi - genel:")
print(audit["phrase"].str.lower().value_counts().head(30))

print("\nPhrase dagilimi - split bazli:")
print(pd.crosstab(audit["split"], audit["phrase"].str.lower()))

# Risk oranlarini kayit sayisina gore hesapla
risk_summary = []
for split_name, n in split_sizes.items():
    sub = audit[audit["split"] == split_name]
    risky = sub[sub["decision"] != "HISTORICAL_ALLOWED"]
    current = sub[sub["decision"] == "CURRENT_RISK"]

    risk_summary.append({
        "split": split_name,
        "records": n,
        "risk_sentences_total": len(sub),
        "non_historical_review_or_current": len(risky),
        "current_risk": len(current),
        "review_or_current_per_record_pct": 100 * len(risky) / n,
        "current_risk_per_record_pct": 100 * len(current) / n,
    })

risk_summary = pd.DataFrame(risk_summary)

print("\nRisk oranlari:")
display(risk_summary)

# Incelenecek asil liste
review = audit[audit["decision"] != "HISTORICAL_ALLOWED"].copy()

print("\nIncelenecek cumle sayisi:", len(review))
display(
    review[
        ["split", "note_id", "gold_label", "section", "phrase", "decision", "reason", "sentence"]
    ].head(80)
)

# Istersen CSV olarak kaydetmek icin bunu ac:
# OUT = JSON_DIR / "v4_train_val_test_temporal_leakage_audit.csv"
# audit.to_csv(OUT, index=False, encoding="utf-8-sig")
# print("Audit CSV kaydedildi:", OUT)


In [ ]:
import json
import re
import statistics
import pandas as pd
from pathlib import Path

JSON_DIR = Path(r"./workdir/output/qlora_json")
V5_DIR = Path(r"./workdir/output/v5_complaint_only")
V5_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILES = {
    "train_S1": JSON_DIR / "qlora_v4_train_S1.json",
    "train_S2": JSON_DIR / "qlora_v4_train_S2.json",
    "train_S3": JSON_DIR / "qlora_v4_train_S3.json",
    "val": JSON_DIR / "qlora_v4_val.json",
    "test": JSON_DIR / "qlora_v4_test.json",
}

OUTPUT_FILES = {
    "train_S1": V5_DIR / "v5_3_chief_complaint_only_train_S1.json",
    "train_S2": V5_DIR / "v5_3_chief_complaint_only_train_S2.json",
    "train_S3": V5_DIR / "v5_3_chief_complaint_only_train_S3.json",
    "val": V5_DIR / "v5_3_chief_complaint_only_val.json",
    "test": V5_DIR / "v5_3_chief_complaint_only_test.json",
}

SECTION_RE = re.compile(
    r"(Chief Complaint|Symptom History|Past Medical History|Allergies|"
    r"Medications on Admission|Vital Signs):\s*\n?",
    re.IGNORECASE
)

# V5.3 icin bunlarin inputta kalmasi kesin yasak.
FORBIDDEN = re.compile(
    r"\b("
    # history / EHR context
    r"history\s+of|hx\s+of|h/o|pmh|past\s+medical|past\s+medical\s+history|"
    r"prior|previous|previously|baseline|s/p|status\s+post|recurrent|chronic|"
    r"home\s+meds?|home\s+medications?|medications?\s+on\s+admission|allergies|"
    r"family\s+history|social\s+history|"
    # clinician decision / current admission
    r"admitted\s+for|being\s+admitted\s+for|directly\s+admitted\s+for|"
    r"found\s+to\s+have|diagnosed\s+with|diagnosis|diagnoses|"
    r"concern\s+for|consistent\s+with|rule\s+out|r/o|work[\s-]?up|"
    r"assessment|impression|evaluation\s+for|transfer\s+for|transferred\s+for|"
    # labs / imaging / tests
    r"labs?|wbc|hgb|hct|platelets?|plt|inr|lactate|troponin|bun|creatinine|"
    r"\bcr\b|sodium|\bna\b|\bk\b|ua|urinalysis|abg|vbg|"
    r"cxr|ct|cta|mri|tte|tee|ekg|ecg|xray|x-ray|ultrasound|echo|"
    # treatment
    r"treated\s+with|started\s+on|received|given|administered|placed\s+on|"
    r"vancomycin|zosyn|ceftriaxone|cefepime|lasix|furosemide|duonebs|"
    r"insulin\s+drip|glucagon|tpa|alteplase|pressors|levophed|norepinephrine|"
    r"iv\s+fluids|fluid\s+bolus|"
    # discharge artifacts
    r"discharge|discharged|hospital\s+course|brief\s+hospital\s+course|"
    r"follow[\s-]?up|"
    # direct class / diagnosis terms
    r"sepsis|septic|bacteremia|stroke|cva|infarct|hemorrhage|"
    r"atrial\s+fibrillation|afib|a-fib|flutter|rvr|"
    r"aki|acute\s+kidney\s+injury|renal\s+failure|"
    r"pneumonia|pna|uti|urinary\s+tract\s+infection|pyelonephritis|"
    r"heart\s+failure|chf|hfref|hfpef|"
    r"gi\s+bleed|gib|gastrointestinal\s+bleed|"
    r"dka|hhs|diabetic\s+ketoacidosis|hyperglycemia|hypoglycemia|"
    r"copd\s+exacerbation"
    r")\b",
    re.IGNORECASE
)

SYMPTOM_ANCHOR = re.compile(
    r"\b("
    r"pain|dyspnea|shortness\s+of\s+breath|sob|cough|fever|chills|weakness|"
    r"numbness|facial\s+droop|slurred\s+speech|confusion|altered\s+mental\s+status|ams|"
    r"dizziness|syncope|palpitations|nausea|vomit|vomiting|diarrhea|melena|"
    r"hematemesis|bleeding|blood|dysuria|frequency|urgency|swelling|edema|"
    r"wheez|fatigue|malaise|seizure|fall|headache|flank|abdominal|chest|"
    r"back|leg|arm|weak|tired|unresponsive|lethargy"
    r")\b",
    re.IGNORECASE
)

def parse_sections(text):
    parts = SECTION_RE.split(text or "")
    sections = {}
    for i in range(1, len(parts) - 1, 2):
        sections[parts[i].strip().lower()] = parts[i + 1].strip()
    return sections

def normalize_piece(text):
    text = text or ""
    text = text.replace("[DIAG_MASKED]", " ")
    text = text.replace("DIAG_MASKED", " ")
    text = re.sub(r"[_]{2,}", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip(" |;/,.-")
    return text

def split_complaint_lines(cc):
    if not cc:
        return []
    raw_parts = []
    for line in cc.splitlines():
        line = line.strip()
        if not line:
            continue
        # Chief complaint satirlari bazen slash/semicolon ile birden fazla sikayet tasiyor.
        raw_parts.extend(re.split(r"\s*/\s*|\s*;\s*|\s+\|\s+", line))
    return [normalize_piece(p) for p in raw_parts if normalize_piece(p)]

def safe_complaint_from_v4(v4_input):
    sections = parse_sections(v4_input)
    cc = sections.get("chief complaint", "")

    safe_parts = []
    for part in split_complaint_lines(cc):
        if not part:
            continue
        if FORBIDDEN.search(part):
            continue
        # Chief Complaint oldugu icin symptom anchor yoksa da tutulabilir,
        # ama cok uzun serbest metin varsa kisaltilir.
        part = part[:120].strip(" |;/,.-")
        if part:
            safe_parts.append(part)

    safe_parts = safe_parts[:3]
    if safe_parts:
        text = "Patient Complaint: " + "; ".join(safe_parts)
    else:
        text = "Patient Complaint: unspecified symptoms"

    # Son guvenlik gecisi: yasak bir sey kaldiysa fallback.
    if FORBIDDEN.search(text):
        text = "Patient Complaint: unspecified symptoms"

    return text

def convert_records(records):
    out = []
    for r in records:
        v5_input = safe_complaint_from_v4(r.get("input", ""))
        out.append({
            "note_id": r["note_id"],
            "subject_id": r["subject_id"],
            "hadm_id": r["hadm_id"],
            "instruction": (
                "Classify the likely primary admission diagnosis using only the patient's "
                "chief complaint. Output only the class name."
            ),
            "input": v5_input,
            "output": r["output"],
        })
    return out

def audit_records(records, split_name):
    rows = []
    for r in records:
        text = r.get("input", "") or ""
        forbidden = FORBIDDEN.search(text)
        symptom = SYMPTOM_ANCHOR.search(text)
        rows.append({
            "split": split_name,
            "note_id": r.get("note_id"),
            "label": r.get("output"),
            "chars": len(text),
            "is_unspecified": text.strip().lower() == "patient complaint: unspecified symptoms",
            "has_forbidden": forbidden is not None,
            "forbidden_match": forbidden.group(0) if forbidden else "",
            "has_symptom_anchor": symptom is not None,
            "input": text,
        })
    return pd.DataFrame(rows)

all_audits = []
all_sizes = {}

for split_name, in_path in INPUT_FILES.items():
    with open(in_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    converted = convert_records(records)
    audit = audit_records(converted, split_name)

    forbidden_n = int(audit["has_forbidden"].sum())
    empty_n = int((audit["chars"] == 0).sum())

    print("=" * 80)
    print(split_name)
    print("n:", len(converted))
    print("forbidden:", forbidden_n)
    print("empty:", empty_n)
    print("unspecified:", int(audit["is_unspecified"].sum()))
    print("no symptom anchor:", int((~audit["has_symptom_anchor"]).sum()))
    print(
        "char mean/p50/max:",
        round(audit["chars"].mean(), 1),
        audit["chars"].median(),
        audit["chars"].max()
    )

    assert forbidden_n == 0, f"{split_name}: forbidden leakage kaldi!"
    assert empty_n == 0, f"{split_name}: empty input var!"

    with open(OUTPUT_FILES[split_name], "w", encoding="utf-8") as f:
        json.dump(converted, f, ensure_ascii=False, indent=2)

    print("yazildi:", OUTPUT_FILES[split_name])

    all_audits.append(audit)
    all_sizes[split_name] = len(converted)

audit_all = pd.concat(all_audits, ignore_index=True)
audit_path = V5_DIR / "v5_3_chief_complaint_only_audit.csv"
audit_all.to_csv(audit_path, index=False, encoding="utf-8-sig")

print("\n" + "=" * 80)
print("V5.3 CHIEF-COMPLAINT-ONLY FINAL AUDIT")
print("=" * 80)
print("total n:", len(audit_all))
print("total forbidden:", int(audit_all["has_forbidden"].sum()))
print("total empty:", int((audit_all["chars"] == 0).sum()))
print("\nForbidden by split:")
print(pd.crosstab(audit_all["split"], audit_all["has_forbidden"], margins=True))
print("\nUnspecified by split:")
print(audit_all.groupby("split")["is_unspecified"].sum())
print("\nNo symptom anchor by split:")
print(audit_all.groupby("split")["has_symptom_anchor"].apply(lambda s: int((~s).sum())))
print("\nLabel distribution check:")
for split_name in INPUT_FILES:
    print("\n", split_name)
    print(audit_all[audit_all["split"] == split_name]["label"].value_counts().sort_index())

assert int(audit_all["has_forbidden"].sum()) == 0, "V5.3 final audit failed: forbidden leakage var!"
assert int((audit_all["chars"] == 0).sum()) == 0, "V5.3 final audit failed: empty input var!"

print("\nV5.3 TEMIZ: forbidden leakage = 0")
print("Audit CSV:", audit_path)

display(
    audit_all[
        ["split", "note_id", "label", "chars", "is_unspecified", "has_symptom_anchor", "input"]
    ].sample(30, random_state=42)
)


In [ ]:
import json
import time
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

V5_DIR = Path(r"./workdir/output/v5_complaint_only")

VAL_JSON = V5_DIR / "v5_3_chief_complaint_only_val.json"
CP_CSV = V5_DIR / "v5_3_zeroshot_val_checkpoint.csv"
OUT_CSV = V5_DIR / "v5_3_zeroshot_val.csv"
SUMMARY_CSV = V5_DIR / "v5_3_zeroshot_val_summary.csv"
AUDIT_TXT = V5_DIR / "v5_3_zeroshot_val_audit.txt"

MODEL_ID = "Qwen/Qwen3.5-4B"
MAX_LEN = 512
MAX_NEW = 20

ALLOWED = [
    "SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"
]

PROMPT = """You are a clinical text classifier.

You receive ONLY the patient's chief complaint.
You do NOT have medical history, medications, vital signs, labs, imaging, physician assessment, treatment, or discharge information.

Classify the most likely PRIMARY reason for admission into exactly ONE class:

SEPSIS
STROKE
ATRIAL_FIBRILLATION
AKI
PNEUMONIA
UTI
HEART_FAILURE
GI_BLEEDING
DIABETES_COMPLICATION
COPD_EXACERBATION
OTHER

Rules:
- Output exactly one class label.
- Use the exact uppercase label string.
- Do not explain.
- If the chief complaint is unspecified or insufficient, choose OTHER.

CHIEF COMPLAINT:
{input}

PRIMARY CLASS:"""

FUZZY = {
    "SEPS":"SEPSIS", "SEPT":"SEPSIS", "PNEUM":"PNEUMONIA",
    "STROK":"STROKE", "CVA":"STROKE",
    "ATRIAL":"ATRIAL_FIBRILLATION", "AFIB":"ATRIAL_FIBRILLATION", "FIBRIL":"ATRIAL_FIBRILLATION",
    "AKI":"AKI", "RENAL":"AKI", "KIDNEY":"AKI",
    "UTI":"UTI", "URIN":"UTI",
    "HEART":"HEART_FAILURE", "CHF":"HEART_FAILURE",
    "GI":"GI_BLEEDING", "BLEED":"GI_BLEEDING",
    "DIAB":"DIABETES_COMPLICATION", "DKA":"DIABETES_COMPLICATION",
    "COPD":"COPD_EXACERBATION",
    "OTHER":"OTHER", "NONE":"OTHER"
}

def parse_class(raw):
    if not raw:
        return "INVALID", "empty"
    t = raw.strip().upper()
    for c in sorted(ALLOWED, key=len, reverse=True):
        if c in t[:80]:
            return c, "direct"
    for k, v in FUZZY.items():
        if k in t[:80]:
            return v, "fuzzy"
    return "INVALID", "invalid"

print("=" * 70)
print("V5.3 CHIEF-COMPLAINT-ONLY ZERO-SHOT VAL")
print("=" * 70)

with open(VAL_JSON, "r", encoding="utf-8") as f:
    val = json.load(f)

print("Val n:", len(val))

if "model" not in globals() or "tok" not in globals():
    print("\n[1/4] Model yukleniyor...")
    t0 = time.time()
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tok = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb,
        device_map="auto",
        torch_dtype=torch.float16,
        local_files_only=True,
        trust_remote_code=True,
    )
    model.eval()
    print(f"Model yuklendi: {time.time()-t0:.1f}s")
    print(f"GPU mem: {torch.cuda.memory_allocated()/1024**2:.0f} MB")
else:
    print("\n[1/4] Model bellekte mevcut, tekrar yuklenmedi.")

def infer_one(text):
    prompt = PROMPT.format(input=text)
    msgs = [{"role": "user", "content": prompt}]
    try:
        chat = tok.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        chat = tok.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
        )

    inputs = tok(
        chat,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )

    raw = tok.decode(
        out[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    return raw, inputs.input_ids.shape[1]

print("\n[2/4] Checkpoint kontrolu...")
if CP_CSV.exists():
    old = pd.read_csv(CP_CSV)
    done = set(old["note_id"].astype(str))
    rows = old.to_dict("records")
    print("Resume:", len(done), "kayit")
else:
    done = set()
    rows = []
    print("Yeni run.")

print("\n[3/4] Inference basliyor...")
t_start = time.time()

for i, r in enumerate(val):
    nid = str(r["note_id"])
    if nid in done:
        continue

    t0 = time.time()
    raw, tok_n = infer_one(r["input"])
    pred, parse = parse_class(raw)

    rows.append({
        "idx": i,
        "note_id": nid,
        "gold": r["output"],
        "pred": pred,
        "parse": parse,
        "raw": raw,
        "input": r["input"],
        "input_tokens": tok_n,
        "input_chars": len(r["input"]),
        "sec": time.time() - t0,
    })

    if (len(rows) % 25 == 0) or (i == len(val) - 1):
        pd.DataFrame(rows).to_csv(CP_CSV, index=False, encoding="utf-8-sig")
        elapsed = (time.time() - t_start) / 60
        print(f"{len(rows)}/{len(val)} | elapsed={elapsed:.1f} dk | pred={pred} gold={r['output']} raw={raw[:50]}")

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print("\n[4/4] Metrikler...")
valid = df[df["pred"].isin(ALLOWED)].copy()

full_acc = accuracy_score(df["gold"], df["pred"])
full_mf1 = f1_score(df["gold"], df["pred"], labels=ALLOWED, average="macro", zero_division=0)
full_wf1 = f1_score(df["gold"], df["pred"], labels=ALLOWED, average="weighted", zero_division=0)

if len(valid):
    valid_acc = accuracy_score(valid["gold"], valid["pred"])
    valid_mf1 = f1_score(valid["gold"], valid["pred"], labels=ALLOWED, average="macro", zero_division=0)
    valid_wf1 = f1_score(valid["gold"], valid["pred"], labels=ALLOWED, average="weighted", zero_division=0)
else:
    valid_acc = valid_mf1 = valid_wf1 = 0.0

summary = pd.DataFrame([
    {"subset": "full", "n": len(df), "accuracy": full_acc, "macro_f1": full_mf1, "weighted_f1": full_wf1},
    {"subset": "valid_only", "n": len(valid), "accuracy": valid_acc, "macro_f1": valid_mf1, "weighted_f1": valid_wf1},
])

summary.to_csv(SUMMARY_CSV, index=False)

per = precision_recall_fscore_support(
    df["gold"], df["pred"],
    labels=ALLOWED,
    zero_division=0
)

with open(AUDIT_TXT, "w", encoding="utf-8") as f:
    f.write("V5.3 CHIEF-COMPLAINT-ONLY ZERO-SHOT VAL\n")
    f.write("=" * 70 + "\n")
    f.write(f"n={len(df)}\n")
    f.write(f"model={MODEL_ID}\n")
    f.write(f"max_len={MAX_LEN}, max_new={MAX_NEW}\n\n")
    f.write(summary.to_string(index=False))
    f.write("\n\nParser:\n")
    f.write(df["parse"].value_counts(dropna=False).to_string())
    f.write("\n\nPer-class:\n")
    for cls, p, r, f1, sup in zip(ALLOWED, per[0], per[1], per[2], per[3]):
        f.write(f"\n{cls:<25s} P={p:.3f} R={r:.3f} F1={f1:.3f} N={sup}")

print("\n" + "=" * 70)
print("V5.3 ZERO-SHOT VAL BITTI")
print("=" * 70)
print(summary)
print("\nParser:")
print(df["parse"].value_counts(dropna=False))
print("\nPred distribution:")
print(df["pred"].value_counts())
print("\nSure dk:", round((time.time() - t_start) / 60, 1))
print("Cikti:", OUT_CSV)
print("Ozet:", SUMMARY_CSV)
print("Audit:", AUDIT_TXT)


In [ ]:
import gc
import torch

for name in ["model", "tok", "tokenizer", "trainer", "loaded_model", "base2"]:
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.reset_peak_memory_stats()
    print("GPU cache bosaltildi.")
    print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
    print("Reserved  GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))
else:
    print("CUDA yok.")


In [ ]:
import os, gc, torch

print("PID:", os.getpid())
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print("Allocated GB:", round(torch.cuda.memory_allocated()/1024**3, 3))
    print("Reserved  GB:", round(torch.cuda.memory_reserved()/1024**3, 3))

!nvidia-smi


In [ ]:
import json, random
from pathlib import Path
from collections import defaultdict, Counter

OUTPUT_DIR = Path(r"./workdir/output")
JSON_DIR = OUTPUT_DIR / "qlora_json"
ADAPTER_DIR = OUTPUT_DIR / "qlora_adapters" / "v4_s3_smoke_safe"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_JSON = JSON_DIR / "qlora_v4_train_S3.json"
VAL_JSON = JSON_DIR / "qlora_v4_val.json"

ALLOWED = [
    "SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"
]

SEED = 42
random.seed(SEED)

with open(TRAIN_JSON, "r", encoding="utf-8") as f:
    train_all = json.load(f)
with open(VAL_JSON, "r", encoding="utf-8") as f:
    val_all = json.load(f)

by_label = defaultdict(list)
for r in train_all:
    by_label[r["output"]].append(r)

smoke_train = []
for cls in ALLOWED:
    smoke_train.extend(random.sample(by_label[cls], 2))
random.shuffle(smoke_train)
smoke_train = smoke_train[:20]

val_by_label = defaultdict(list)
for r in val_all:
    val_by_label[r["output"]].append(r)

smoke_val = [val_by_label[cls][0] for cls in ALLOWED[:5]]

print("Train S3:", len(train_all))
print("Smoke train:", len(smoke_train), dict(Counter(r["output"] for r in smoke_train)))
print("Smoke val:", len(smoke_val), [r["output"] for r in smoke_val])


In [ ]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen3.5-4B"
MAX_SEQ_LEN = 1024   # SAFE smoke. Gecerse 1536/2048 denenir.

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Model yukleniyor...")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    local_files_only=True,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    local_files_only=True,
    trust_remote_code=True,
)

model.config.use_cache = False

try:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
except TypeError:
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

print("Yuklendi:", round(time.time() - t0, 1), "s")
print("GPU allocated GB:", round(torch.cuda.memory_allocated()/1024**3, 2))


In [ ]:
import json
import time
import random
import gc
from pathlib import Path
from collections import defaultdict, Counter

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# ============================================================
# CONFIG - V4 SPEED SMOKE
# ============================================================
OUTPUT_DIR = Path(r"./workdir/output")
JSON_DIR = OUTPUT_DIR / "qlora_json"
ADAPTER_DIR = OUTPUT_DIR / "qlora_adapters" / "v4_s3_speed_smoke"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen3.5-4B"
TRAIN_JSON = JSON_DIR / "qlora_v4_train_S3.json"
VAL_JSON = JSON_DIR / "qlora_v4_val.json"

SEED = 42
SMOKE_TRAIN_N = 8
SMOKE_VAL_N = 5

MAX_SEQ_LEN = 512
INPUT_TOKEN_BUDGET = 260
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 4
LR = 2e-4

ALLOWED = [
    "SEPSIS", "STROKE", "ATRIAL_FIBRILLATION", "AKI", "PNEUMONIA", "UTI",
    "HEART_FAILURE", "GI_BLEEDING", "DIABETES_COMPLICATION",
    "COPD_EXACERBATION", "OTHER"
]

print("=" * 80)
print("V4 QLoRA S3 SPEED SMOKE")
print("=" * 80)
print("Full S3 train yapmaz.")
print("SMOKE_TRAIN_N:", SMOKE_TRAIN_N)
print("MAX_SEQ_LEN:", MAX_SEQ_LEN)
print("INPUT_TOKEN_BUDGET:", INPUT_TOKEN_BUDGET)
print("Batch:", PER_DEVICE_BATCH, "Grad accum:", GRAD_ACCUM)
print("LoRA r=4 alpha=8 targets=q_proj,v_proj")

random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# CLEANUP
# ============================================================
for name in ["model", "tokenizer", "trainer"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print("GPU before allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))

# ============================================================
# DATA
# ============================================================
print("\n[1/7] Veri yukleniyor...")

with open(TRAIN_JSON, "r", encoding="utf-8") as f:
    train_all = json.load(f)

with open(VAL_JSON, "r", encoding="utf-8") as f:
    val_all = json.load(f)

by_label = defaultdict(list)
for r in train_all:
    by_label[r["output"]].append(r)

# 8 farkli siniftan 1'er ornek
smoke_train = []
for cls in ALLOWED:
    if len(smoke_train) < SMOKE_TRAIN_N:
        smoke_train.append(random.choice(by_label[cls]))

random.shuffle(smoke_train)

val_by_label = defaultdict(list)
for r in val_all:
    val_by_label[r["output"]].append(r)

smoke_val = []
for cls in ALLOWED:
    if cls in val_by_label and len(smoke_val) < SMOKE_VAL_N:
        smoke_val.append(val_by_label[cls][0])

print("Train all:", len(train_all))
print("Smoke train:", len(smoke_train), dict(Counter(r["output"] for r in smoke_train)))
print("Smoke val:", len(smoke_val), [r["output"] for r in smoke_val])

# ============================================================
# MODEL
# ============================================================
print("\n[2/7] Model/tokenizer yukleniyor...")

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    local_files_only=True,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    local_files_only=True,
    trust_remote_code=True,
)

model.config.use_cache = False

try:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
except TypeError:
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()

lora_cfg = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

print("Model loaded sec:", round(time.time() - t0, 1))
print("GPU after load allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))

# ============================================================
# HELPERS
# ============================================================
def trim_input_to_budget(text, budget=INPUT_TOKEN_BUDGET):
    ids = tokenizer(text or "", add_special_tokens=False)["input_ids"]
    ids = ids[:budget]
    return tokenizer.decode(ids, skip_special_tokens=True).strip()

def make_user_text(r):
    input_short = trim_input_to_budget(r["input"])
    return f"{r['instruction']}\n\n{input_short}"

def apply_template(messages, add_generation_prompt):
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

def build_texts(r):
    user_text = make_user_text(r)

    prompt_text = apply_template(
        [{"role": "user", "content": user_text}],
        add_generation_prompt=True,
    )

    full_text = apply_template(
        [
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": r["output"]},
        ],
        add_generation_prompt=False,
    )

    return prompt_text, full_text

def parse_class(raw):
    t = (raw or "").strip().upper()
    for cls in sorted(ALLOWED, key=len, reverse=True):
        if cls in t[:80]:
            return cls
    return "INVALID"

# ============================================================
# DATASET
# ============================================================
print("\n[3/7] Dataset hazirlaniyor...")

class SmokeDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        prompt_text, full_text = build_texts(self.records[idx])

        prompt_ids = tokenizer(
            prompt_text,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            add_special_tokens=False,
        )["input_ids"]

        enc = tokenizer(
            full_text,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            padding="max_length",
            return_tensors="pt",
            add_special_tokens=False,
        )

        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        labels[:min(len(prompt_ids), MAX_SEQ_LEN)] = -100

        if (labels != -100).sum().item() == 0:
            last_real = int(attention_mask.sum().item()) - 1
            if last_real >= 0:
                labels[last_real] = input_ids[last_real]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

def collate_fn(features):
    return {
        "input_ids": torch.stack([f["input_ids"] for f in features]),
        "attention_mask": torch.stack([f["attention_mask"] for f in features]),
        "labels": torch.stack([f["labels"] for f in features]),
    }

train_ds = SmokeDataset(smoke_train)

lengths = []
label_lengths = []
for i in range(len(train_ds)):
    item = train_ds[i]
    lengths.append(int(item["attention_mask"].sum().item()))
    label_lengths.append(int((item["labels"] != -100).sum().item()))

print("Dataset size:", len(train_ds))
print("Input token mean/max:", round(sum(lengths) / len(lengths), 1), max(lengths))
print("Label token mean/min/max:", round(sum(label_lengths) / len(label_lengths), 1), min(label_lengths), max(label_lengths))

# ============================================================
# TRAIN
# ============================================================
print("\n[4/7] Train basliyor...")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

args = TrainingArguments(
    output_dir=str(ADAPTER_DIR / "checkpoints"),
    num_train_epochs=1,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    report_to=[],
    seed=SEED,
    gradient_checkpointing=True,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=collate_fn,
)

t0 = time.time()
result = trainer.train()
train_sec = time.time() - t0

peak_gb = torch.cuda.max_memory_allocated() / 1024**3
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]

print("\nTrain bitti.")
print("Train sec:", round(train_sec, 1))
print("Peak GPU GB:", round(peak_gb, 3))
print("Final loss:", result.training_loss)
print("Losses:", losses)

# ============================================================
# SAVE
# ============================================================
print("\n[5/7] Adapter kaydediliyor...")

adapter_path = ADAPTER_DIR / "adapter"
trainer.save_model(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))

print("Adapter:", adapter_path)
for p in sorted(adapter_path.glob("*")):
    print(" ", p.name, round(p.stat().st_size / 1024, 1), "KB")

# ============================================================
# INFERENCE
# ============================================================
print("\n[6/7] 5 val inference...")

model.eval()
tokenizer.padding_side = "left"

def infer_one(r):
    user_text = make_user_text(r)
    messages = [{"role": "user", "content": user_text}]
    prompt = apply_template(messages, add_generation_prompt=True)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    raw = tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    ).strip()

    return raw, parse_class(raw), int(inputs.input_ids.shape[1])

valid = 0
for i, r in enumerate(smoke_val, 1):
    raw, pred, tok_n = infer_one(r)
    ok = pred in ALLOWED
    valid += int(ok)
    print(f"[{i}] gold={r['output']} pred={pred} valid={ok} tok={tok_n} raw={raw[:100]}")

# ============================================================
# FINAL
# ============================================================
print("\n[7/7] Speed smoke sonucu")
print("=" * 80)
print("Peak GPU GB:", round(peak_gb, 3))
print("Train sec:", round(train_sec, 1))
print("Losses:", losses)
print("Valid outputs:", valid, "/", len(smoke_val))
print("Adapter saved:", adapter_path)
print("Full S3 train'e gecilmedi.")

if peak_gb < 7.6 and valid == len(smoke_val) and train_sec < 600:
    print("SPEED SMOKE ACCEPT: Full S3 icin bu hiz ayarlari aday.")
else:
    print("SPEED SMOKE NOT ACCEPT: ya yavas ya format/memory sorunu var.")


# BLOK 15A-SMOKE - V4 QLoRA Final Smoke (80 ornek)

**Amac:** Full S3 oncesi son saglik kontrolu. Hizli smoke (8 ornek, r=4) loss flat ve mode collapse gosterdi (5/5 valid ama 4'u OTHER). Bu smoke loss egrisi gercekten dusuyor mu gorur.

**Final ayar (Full S3 ile birebir ayni):**
- 80 ornek (11 sinif x ~7-8)
- 1 epoch, batch=1, grad_accum=8 (eff=8) -> 10 optimizer step
- MAX_SEQ_LEN=512, INPUT_TOKEN_BUDGET=320 (260->320, biraz daha klinik baglam)
- LoRA r=8, alpha=16, target=q_proj+v_proj only (k/o eklenmedi: bellek+sure)
- LR=2e-4, optim=adamw_torch, gradient_checkpointing

**Hedef:**
- Peak GPU < 7.6 GB
- Train sure < 20 dk
- Loss DUSEN egri (hizli smoke flat'ti)
- 5 val: 5/5 valid + en az 2 dogru (5/5 OTHER mode collapse degil)

**Cikti:** `output/qlora_adapters/v4_s3_final_smoke/adapter/`


In [ ]:
import json
import time
import random
import gc
from pathlib import Path
from collections import defaultdict, Counter

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# ============================================================
# CONFIG - V4 FINAL SMOKE (Full S3 ayarlari, 80 ornek)
# ============================================================
OUTPUT_DIR = Path(r"./workdir/output")
JSON_DIR = OUTPUT_DIR / "qlora_json"
ADAPTER_DIR = OUTPUT_DIR / "qlora_adapters" / "v4_s3_final_smoke"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen3.5-4B"
TRAIN_JSON = JSON_DIR / "qlora_v4_train_S3.json"
VAL_JSON = JSON_DIR / "qlora_v4_val.json"

SEED = 42
SMOKE_TRAIN_N = 80
SMOKE_VAL_N = 5

# Codex final ayarlari
MAX_SEQ_LEN = 512
INPUT_TOKEN_BUDGET = 320
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
LR = 2e-4
LORA_R = 8
LORA_ALPHA = 16

ALLOWED = [
    "SEPSIS", "STROKE", "ATRIAL_FIBRILLATION", "AKI", "PNEUMONIA", "UTI",
    "HEART_FAILURE", "GI_BLEEDING", "DIABETES_COMPLICATION",
    "COPD_EXACERBATION", "OTHER"
]

print("=" * 80)
print("V4 QLoRA FINAL SMOKE (Full S3 ayarlari, 80 ornek)")
print("=" * 80)
print("MAX_SEQ_LEN:", MAX_SEQ_LEN, "INPUT_TOKEN_BUDGET:", INPUT_TOKEN_BUDGET)
print("Batch:", PER_DEVICE_BATCH, "Grad accum:", GRAD_ACCUM, "(eff", PER_DEVICE_BATCH*GRAD_ACCUM, ")")
print("LoRA r:", LORA_R, "alpha:", LORA_ALPHA, "target: q_proj, v_proj")
print("Optim: adamw_torch, gradient_checkpointing=True")

random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# CLEANUP
# ============================================================
for name in ["model", "tokenizer", "trainer"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print("GPU before allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))

# ============================================================
# DATA
# ============================================================
print("\n[1/7] Veri yukleniyor...")

with open(TRAIN_JSON, "r", encoding="utf-8") as f:
    train_all = json.load(f)

with open(VAL_JSON, "r", encoding="utf-8") as f:
    val_all = json.load(f)

by_label = defaultdict(list)
for r in train_all:
    by_label[r["output"]].append(r)

# 11 siniftan dengeli 80 ornek (~7-8/sinif)
per_class = SMOKE_TRAIN_N // len(ALLOWED)  # 7
extras = SMOKE_TRAIN_N - per_class * len(ALLOWED)  # 3
smoke_train = []
for i, cls in enumerate(ALLOWED):
    n = per_class + (1 if i < extras else 0)
    smoke_train.extend(random.sample(by_label[cls], n))
random.shuffle(smoke_train)

val_by_label = defaultdict(list)
for r in val_all:
    val_by_label[r["output"]].append(r)

# 5 farkli siniftan val
smoke_val = []
for cls in ALLOWED:
    if cls in val_by_label and len(smoke_val) < SMOKE_VAL_N:
        smoke_val.append(val_by_label[cls][0])

print("Train all:", len(train_all))
print("Smoke train:", len(smoke_train))
print("Sinif dagilimi:", dict(Counter(r["output"] for r in smoke_train)))
print("Smoke val:", len(smoke_val), [r["output"] for r in smoke_val])

# ============================================================
# MODEL
# ============================================================
print("\n[2/7] Model/tokenizer yukleniyor...")

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    local_files_only=True,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    local_files_only=True,
    trust_remote_code=True,
)

model.config.use_cache = False

try:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
except TypeError:
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

print("Model loaded sec:", round(time.time() - t0, 1))
print("GPU after load allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))

# ============================================================
# HELPERS
# ============================================================
def trim_input_to_budget(text, budget=INPUT_TOKEN_BUDGET):
    ids = tokenizer(text or "", add_special_tokens=False)["input_ids"]
    ids = ids[:budget]
    return tokenizer.decode(ids, skip_special_tokens=True).strip()

def make_user_text(r):
    input_short = trim_input_to_budget(r["input"])
    return f"{r['instruction']}\n\n{input_short}"

def apply_template(messages, add_generation_prompt):
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

def build_texts(r):
    user_text = make_user_text(r)
    prompt_text = apply_template(
        [{"role": "user", "content": user_text}],
        add_generation_prompt=True,
    )
    full_text = apply_template(
        [
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": r["output"]},
        ],
        add_generation_prompt=False,
    )
    return prompt_text, full_text

def parse_class(raw):
    t = (raw or "").strip().upper()
    for cls in sorted(ALLOWED, key=len, reverse=True):
        if cls in t[:80]:
            return cls
    return "INVALID"

# ============================================================
# DATASET
# ============================================================
print("\n[3/7] Dataset hazirlaniyor...")

class SmokeDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        prompt_text, full_text = build_texts(self.records[idx])

        prompt_ids = tokenizer(
            prompt_text,
            truncation=True, max_length=MAX_SEQ_LEN,
            add_special_tokens=False,
        )["input_ids"]

        enc = tokenizer(
            full_text,
            truncation=True, max_length=MAX_SEQ_LEN,
            padding="max_length", return_tensors="pt",
            add_special_tokens=False,
        )

        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        labels[:min(len(prompt_ids), MAX_SEQ_LEN)] = -100

        if (labels != -100).sum().item() == 0:
            last_real = int(attention_mask.sum().item()) - 1
            if last_real >= 0:
                labels[last_real] = input_ids[last_real]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

def collate_fn(features):
    return {
        "input_ids": torch.stack([f["input_ids"] for f in features]),
        "attention_mask": torch.stack([f["attention_mask"] for f in features]),
        "labels": torch.stack([f["labels"] for f in features]),
    }

train_ds = SmokeDataset(smoke_train)

lengths = []
label_lengths = []
for i in range(min(len(train_ds), 20)):
    item = train_ds[i]
    lengths.append(int(item["attention_mask"].sum().item()))
    label_lengths.append(int((item["labels"] != -100).sum().item()))

print("Dataset size:", len(train_ds))
print("Input token mean/max (ilk 20):", round(sum(lengths) / len(lengths), 1), max(lengths))
print("Label token mean/min/max (ilk 20):", round(sum(label_lengths) / len(label_lengths), 1), min(label_lengths), max(label_lengths))

# ============================================================
# TRAIN
# ============================================================
print("\n[4/7] Train basliyor (80 ornek, 1 epoch, eff_batch=8 -> 10 step)...")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

args = TrainingArguments(
    output_dir=str(ADAPTER_DIR / "checkpoints"),
    num_train_epochs=1,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    report_to=[],
    seed=SEED,
    gradient_checkpointing=True,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=collate_fn,
)

t0 = time.time()
result = trainer.train()
train_sec = time.time() - t0

peak_gb = torch.cuda.max_memory_allocated() / 1024**3
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]

print("\nTrain bitti.")
print("Train sec:", round(train_sec, 1), "(", round(train_sec/60, 2), "dk)")
print("Peak GPU GB:", round(peak_gb, 3))
print("Final loss:", round(result.training_loss, 4))
print("Loss trend (her step):")
for i, l in enumerate(losses, 1):
    print(f"   step {i}: {l:.4f}")

# Loss gercekten dustu mu?
if len(losses) >= 2:
    delta = losses[-1] - losses[0]
    print(f"Loss delta (son - ilk): {delta:+.4f}", "DUSUYOR" if delta < -0.05 else ("FLAT" if abs(delta) < 0.05 else "YUKSELIYOR"))

# ============================================================
# SAVE
# ============================================================
print("\n[5/7] Adapter kaydediliyor...")

adapter_path = ADAPTER_DIR / "adapter"
trainer.save_model(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))

print("Adapter:", adapter_path)
for p in sorted(adapter_path.glob("*")):
    print(" ", p.name, round(p.stat().st_size / 1024, 1), "KB")

# ============================================================
# INFERENCE
# ============================================================
print("\n[6/7] 5 val inference...")

model.eval()
tokenizer.padding_side = "left"

def infer_one(r):
    user_text = make_user_text(r)
    messages = [{"role": "user", "content": user_text}]
    prompt = apply_template(messages, add_generation_prompt=True)

    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=MAX_SEQ_LEN,
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    raw = tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    ).strip()

    return raw, parse_class(raw), int(inputs.input_ids.shape[1])

valid = 0
preds = []
for i, r in enumerate(smoke_val, 1):
    raw, pred, tok_n = infer_one(r)
    preds.append(pred)
    ok = pred in ALLOWED
    valid += int(ok)
    is_correct = pred == r["output"]
    mark = "OK" if is_correct else ("VALID" if ok else "BAD")
    print(f"[{i}] gold={r['output']:<22s} pred={pred:<22s} {mark} tok={tok_n} raw={raw[:60]}")

# ============================================================
# FINAL
# ============================================================
print("\n[7/7] FINAL SMOKE sonucu")
print("=" * 80)
print("Peak GPU GB:", round(peak_gb, 3))
print("Train sec:", round(train_sec, 1), "(", round(train_sec/60, 2), "dk)")
print("Loss trend:", [round(l, 4) for l in losses])
print("Valid outputs:", valid, "/", len(smoke_val))
print("Predictions:", preds)
unique_pred = len(set(preds))
all_other = all(p == "OTHER" for p in preds)
print("Unique predictions:", unique_pred, "All OTHER?", all_other)
print("Adapter saved:", adapter_path)

# Acceptance kriteri (smoke'ta accuracy YOK — collapse + format + loss + GPU)
loss_dusuyor = len(losses) >= 2 and (losses[-1] - losses[0]) < -0.05
gpu_ok = peak_gb < 7.6
sure_ok = train_sec < 1200
format_ok = valid == len(smoke_val)
collapse_ok = unique_pred >= 2 and not all_other  # tek sinifa cokmemis

if gpu_ok and format_ok and sure_ok and loss_dusuyor and collapse_ok:
    print("\nFINAL SMOKE ACCEPT: Full S3'e gecilebilir.")
else:
    sebepler = []
    if not gpu_ok: sebepler.append(f"peak_gpu>{7.6}")
    if not sure_ok: sebepler.append(f"sure>1200s")
    if not format_ok: sebepler.append(f"format invalid (valid<{len(smoke_val)})")
    if not loss_dusuyor: sebepler.append("loss flat/yukseliyor")
    if not collapse_ok: sebepler.append("mode collapse (tek sinif/hepsi OTHER)")
    print("\nFINAL SMOKE NOT ACCEPT:", ", ".join(sebepler))


# BLOK 15B - V4 QLoRA Full S3 Train (4.400 ornek)

**ZORUNLU ON KOSUL:** Blok 15A-SMOKE 'FINAL SMOKE ACCEPT' yazmali. Yoksa BU HUCREYI CALISTIRMA.
Bu kodun ayarlari hizli smoke'tan daha guclu (r=8, budget=320, grad_accum=8). 80 ornek final smoke onceki adim.

**Ayarlar (gece run icin Codex onerisi — 2 epoch + dusuk LR):**
- Train: qlora_v4_train_S3.json (4.400 ornek)
- **2 epoch**, batch=1, grad_accum=8 (eff=8) -> **1100 optimizer step**
- MAX_SEQ_LEN=512, INPUT_TOKEN_BUDGET=320 (15A-SMOKE ile ayni)
- LoRA r=8, alpha=16, target=q_proj+v_proj (15A-SMOKE ile ayni)
- **LR=1e-4** (2e-4 yerine — 2 epoch + dusuk LR klasik QLoRA recetesi, kararli convergence)
- optim=adamw_torch, gradient_checkpointing

**Guvenlik:**
- save_strategy='steps', save_steps=50, save_total_limit=2 -> 7 saatte crash olursa max ~20 dk kayip
- Train oncesi 500 ornek label truncation audit + assert (bad_prompt=0, label_min>0 sart)
- resume_from_checkpoint otomatik (varsa kaldigi yerden)

**Tahmini sure:** ~7 saat (15A-SMOKE'taki 3.69 dk / 10 step × 1100 step = 406 dk)
**Cikti:** `output/qlora_adapters/v4_s3_full_2epoch_lr1e4/adapter/`

**Sonra:** Blok 15-EVAL (val + test inference + accuracy/F1/per-class)


In [ ]:
import json
import time
import random
import gc
from pathlib import Path
from collections import defaultdict, Counter

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# ============================================================
# CONFIG - V4 FULL S3 (4.400 ornek)
# ============================================================
OUTPUT_DIR = Path(r"./workdir/output")
JSON_DIR = OUTPUT_DIR / "qlora_json"
ADAPTER_DIR = OUTPUT_DIR / "qlora_adapters" / "v4_s3_full_2epoch_lr1e4"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen3.5-4B"
TRAIN_JSON = JSON_DIR / "qlora_v4_train_S3.json"
VAL_JSON = JSON_DIR / "qlora_v4_val.json"

SEED = 42
MAX_SEQ_LEN = 512
INPUT_TOKEN_BUDGET = 320
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
LR = 1e-4
LORA_R = 8
LORA_ALPHA = 16
EPOCHS = 2

ALLOWED = [
    "SEPSIS", "STROKE", "ATRIAL_FIBRILLATION", "AKI", "PNEUMONIA", "UTI",
    "HEART_FAILURE", "GI_BLEEDING", "DIABETES_COMPLICATION",
    "COPD_EXACERBATION", "OTHER"
]

print("=" * 80)
print("V4 QLoRA FULL S3 (4.400 ornek, 2 epoch, eff_batch=8 -> 1100 step)")
print("=" * 80)
print("ON KOSUL: 15A-SMOKE 'ACCEPT' yazdiysa devam et.")
print("Tahmini sure: ~7 saat (gece run icin)")

random.seed(SEED)
torch.manual_seed(SEED)

# Cleanup
for name in ["model", "tokenizer", "trainer"]:
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# ============================================================
# DATA
# ============================================================
print("\n[1/6] Veri yukleniyor...")
with open(TRAIN_JSON, "r", encoding="utf-8") as f:
    train_recs = json.load(f)
with open(VAL_JSON, "r", encoding="utf-8") as f:
    val_recs = json.load(f)
assert len(train_recs) == 4400, f"S3 4400 olmali, {len(train_recs)} bulundu"
print("Train:", len(train_recs), "Val:", len(val_recs))
print("Sinif dagilimi:", dict(Counter(r["output"] for r in train_recs)))

# ============================================================
# MODEL
# ============================================================
print("\n[2/6] Model/tokenizer yukleniyor (4-bit NF4)...")
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, local_files_only=True, trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto",
    torch_dtype=torch.float16, local_files_only=True, trust_remote_code=True,
)
model.config.use_cache = False

try:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
except TypeError:
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print("Load sec:", round(time.time()-t0, 1))
print("GPU after load GB:", round(torch.cuda.memory_allocated()/1024**3, 3))

# ============================================================
# HELPERS + DATASET
# ============================================================
def trim_input_to_budget(text, budget=INPUT_TOKEN_BUDGET):
    ids = tokenizer(text or "", add_special_tokens=False)["input_ids"][:budget]
    return tokenizer.decode(ids, skip_special_tokens=True).strip()

def make_user_text(r):
    return f"{r['instruction']}\n\n{trim_input_to_budget(r['input'])}"

def apply_template(messages, add_generation_prompt):
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=add_generation_prompt, enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=add_generation_prompt,
        )

def build_texts(r):
    user_text = make_user_text(r)
    prompt_text = apply_template(
        [{"role":"user","content":user_text}], add_generation_prompt=True,
    )
    full_text = apply_template(
        [{"role":"user","content":user_text},
         {"role":"assistant","content":r["output"]}],
        add_generation_prompt=False,
    )
    return prompt_text, full_text

class S3Dataset(Dataset):
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        prompt_text, full_text = build_texts(self.records[idx])
        prompt_ids = tokenizer(
            prompt_text, truncation=True, max_length=MAX_SEQ_LEN,
            add_special_tokens=False,
        )["input_ids"]
        enc = tokenizer(
            full_text, truncation=True, max_length=MAX_SEQ_LEN,
            padding="max_length", return_tensors="pt",
            add_special_tokens=False,
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        labels[:min(len(prompt_ids), MAX_SEQ_LEN)] = -100
        if (labels != -100).sum().item() == 0:
            last_real = int(attention_mask.sum().item()) - 1
            if last_real >= 0:
                labels[last_real] = input_ids[last_real]
        return {"input_ids":input_ids, "attention_mask":attention_mask, "labels":labels}

def collate_fn(features):
    return {
        "input_ids": torch.stack([f["input_ids"] for f in features]),
        "attention_mask": torch.stack([f["attention_mask"] for f in features]),
        "labels": torch.stack([f["labels"] for f in features]),
    }

print("\n[3/6] Dataset hazirlaniyor...")
train_ds = S3Dataset(train_recs)
print("Train ds:", len(train_ds))

# ------------------------------------------------------------
# LABEL TRUNCATION AUDIT (500 ornek, train oncesi guvenlik)
# ------------------------------------------------------------
print("\nLabel truncation audit (500 ornek)...")
bad_prompt = 0
near_limit = 0
label_lens = []
for r in train_recs[:500]:
    prompt_text, full_text = build_texts(r)
    prompt_len = len(tokenizer(prompt_text, add_special_tokens=False)["input_ids"])
    full_len = len(tokenizer(full_text, add_special_tokens=False)["input_ids"])
    if prompt_len >= MAX_SEQ_LEN - 8:
        bad_prompt += 1
    if full_len >= MAX_SEQ_LEN:
        near_limit += 1
    label_lens.append(max(0, min(full_len, MAX_SEQ_LEN) - min(prompt_len, MAX_SEQ_LEN)))

print("Audit sample n:", len(label_lens))
print("bad_prompt (prompt assistant alanini yiyor):", bad_prompt)
print("near_limit (full_len >= MAX_SEQ_LEN):", near_limit)
print("label token min/mean/max:", min(label_lens), round(sum(label_lens)/len(label_lens), 1), max(label_lens))

assert bad_prompt == 0, "Prompt assistant label alanini yiyor; INPUT_TOKEN_BUDGET dusur (300 dene)."
assert min(label_lens) > 0, "Bazi kayitlarda assistant label kayboluyor; MAX_SEQ_LEN/budget revize."

# ============================================================
# TRAIN
# ============================================================
print("\n[4/6] Train basliyor (2 epoch, ~1100 step, tahmini ~7 saat)...")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

ckpt_dir = ADAPTER_DIR / "checkpoints"

args = TrainingArguments(
    output_dir=str(ckpt_dir),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    fp16=True,
    logging_steps=10,
    save_strategy="steps",        # crash dayanikli
    save_steps=50,                 # her 50 optimizer step (~%4.5'te bir, 7 saatte crash olursa max ~20 dk kayip)
    save_total_limit=2,            # disk dostu
    report_to=[],
    seed=SEED,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, data_collator=collate_fn,
)

# Resume kontrolu
last_ckpt = None
if ckpt_dir.exists():
    last_ckpt = get_last_checkpoint(str(ckpt_dir))
print("Resume checkpoint:", last_ckpt if last_ckpt else "(yok, sifirdan basliyor)")

t0 = time.time()
result = trainer.train(resume_from_checkpoint=last_ckpt if last_ckpt else None)
train_sec = time.time() - t0
peak_gb = torch.cuda.max_memory_allocated() / 1024**3

losses = [(h.get("step"), h["loss"]) for h in trainer.state.log_history if "loss" in h]
print("\nTrain bitti.")
print("Train sec:", round(train_sec, 0), "(", round(train_sec/60, 1), "dk )")
print("Peak GPU GB:", round(peak_gb, 3))
print("Final loss:", round(result.training_loss, 4))
print("\nLoss trend (her 10 step):")
for step, l in losses[:5]:
    print(f"   step {step}: {l:.4f}")
if len(losses) > 10:
    print(f"   ...({len(losses)-10} ara step atlandi)...")
for step, l in losses[-5:]:
    print(f"   step {step}: {l:.4f}")

# ============================================================
# SAVE
# ============================================================
print("\n[5/6] Adapter kaydediliyor...")
adapter_path = ADAPTER_DIR / "adapter"
trainer.save_model(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))
print("Adapter:", adapter_path)
for p in sorted(adapter_path.glob("*")):
    print(" ", p.name, round(p.stat().st_size / 1024, 1), "KB")

# Training curve kaydet (scaling analizi icin)
curve_yol = ADAPTER_DIR / "training_curve.json"
with open(curve_yol, "w", encoding="utf-8") as f:
    json.dump({
        "scenario": "S3_full_2epoch_lr1e4",
        "train_size": len(train_recs),
        "epochs": EPOCHS,
        "train_sec": train_sec,
        "peak_gb": peak_gb,
        "final_loss": result.training_loss,
        "losses": [(s, l) for s, l in losses],
        "hyperparams": {
            "model": MODEL_ID,
            "max_seq_len": MAX_SEQ_LEN,
            "input_token_budget": INPUT_TOKEN_BUDGET,
            "batch": PER_DEVICE_BATCH, "grad_accum": GRAD_ACCUM,
            "lr": LR, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
            "target_modules": ["q_proj","v_proj"],
            "optim": "adamw_torch", "scheduler": "cosine",
            "warmup_ratio": 0.03,
        },
    }, f, indent=2)
print("Training curve:", curve_yol)

# ============================================================
# QUICK VAL CHECK (5 ornek, sadece sanity)
# ============================================================
print("\n[6/6] 5 val ornegi quick check...")
model.eval()
tokenizer.padding_side = "left"

def parse_class(raw):
    t = (raw or "").strip().upper()
    for cls in sorted(ALLOWED, key=len, reverse=True):
        if cls in t[:80]:
            return cls
    return "INVALID"

val_by_label = defaultdict(list)
for r in val_recs:
    val_by_label[r["output"]].append(r)
quick = []
for cls in ALLOWED[:5]:
    if cls in val_by_label:
        quick.append(val_by_label[cls][0])

valid = 0
correct = 0
for i, r in enumerate(quick, 1):
    user_text = make_user_text(r)
    prompt = apply_template([{"role":"user","content":user_text}], add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    pred = parse_class(raw)
    ok = pred in ALLOWED
    is_correct = pred == r["output"]
    valid += int(ok); correct += int(is_correct)
    mark = "OK" if is_correct else ("VALID" if ok else "BAD")
    print(f"[{i}] gold={r['output']:<22s} pred={pred:<22s} {mark}")

print("\n" + "=" * 80)
print("FULL S3 TAMAMLANDI.")
print("=" * 80)
print("Adapter:", adapter_path)
print("Train sure:", round(train_sec/60, 1), "dk")
print("Peak GPU:", round(peak_gb, 2), "GB")
print("Quick val: valid=" + str(valid) + "/5, correct=" + str(correct) + "/5")
print("\nSiradaki: Blok 15-EVAL (full val + test, accuracy/F1/per-class)")


# BLOK 15C-SPOT - V4 QLoRA-S3 Quick Spot Check (10 val ornegi)

**Amac:** Full eval'a (val+test ~1.5 saat) gecmeden once 10 ornek sanity. Adapter dogru yukleniyor mu, format temiz mi, 'OTHER' bias var mi.

**Adapter:** `output/qlora_adapters/v4_s3_full_2epoch_lr1e4/adapter/`
**Sure:** ~1-2 dk (model yukleme 30s + 10 ornek inference 30-60s)

**Hedef:**
- 10/10 valid format (ALLOWED setinde)
- Unique >= 3 (mode collapse degil)
- En az 4 dogru (zero-shot baseline 5/10 idi, QLoRA daha iyi olmali)

**ON KOSUL:** Restart Kernel sonrasi calistir (model GPU'da kalmasin).


In [ ]:
import json
import gc
import time
from pathlib import Path
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# CONFIG
ADAPTER_PATH = Path(r"./workdir/output/qlora_adapters/v4_s3_full_2epoch_lr1e4/adapter")
VAL_JSON = Path(r"./workdir/output/qlora_json/qlora_v4_val.json")
MODEL_ID = "Qwen/Qwen3.5-4B"
MAX_SEQ_LEN = 512
INPUT_TOKEN_BUDGET = 320
SPOT_N = 10

ALLOWED = [
    "SEPSIS", "STROKE", "ATRIAL_FIBRILLATION", "AKI", "PNEUMONIA", "UTI",
    "HEART_FAILURE", "GI_BLEEDING", "DIABETES_COMPLICATION",
    "COPD_EXACERBATION", "OTHER"
]

print("=" * 80)
print("BLOK 15C-SPOT - V4 QLoRA-S3 Quick Spot (10 val)")
print("=" * 80)

# Cleanup
for name in ["model", "base", "tokenizer", "peft_model"]:
    if name in globals():
        try: globals()[name] = None
        except: pass
        try: del globals()[name]
        except: pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
print(f"GPU after cleanup: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# Veri
with open(VAL_JSON, "r", encoding="utf-8") as f:
    val_all = json.load(f)
# 11 siniftan ilk 10
seen = set()
spot = []
for r in val_all:
    if r["output"] not in seen and len(spot) < SPOT_N:
        seen.add(r["output"])
        spot.append(r)
# 10 dolmadiysa diger siniflardan
for r in val_all:
    if len(spot) >= SPOT_N: break
    if r["output"] in seen: continue
    spot.append(r)
print(f"Spot orneklem: {len(spot)} ornek, etiketler: {[r['output'] for r in spot]}")

# Model
print("\n[1/3] Model + adapter yukleniyor...")
t0 = time.time()
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto",
    torch_dtype=torch.float16, local_files_only=True, trust_remote_code=True,
)
model = PeftModel.from_pretrained(base, str(ADAPTER_PATH))
model.eval()
print(f"   Yuklendi. Sure: {time.time()-t0:.1f}s, GPU: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# Helpers (train ile birebir ayni)
def trim_input_to_budget(text, budget=INPUT_TOKEN_BUDGET):
    ids = tokenizer(text or "", add_special_tokens=False)["input_ids"][:budget]
    return tokenizer.decode(ids, skip_special_tokens=True).strip()

def make_user_text(r):
    return f"{r['instruction']}\n\n{trim_input_to_budget(r['input'])}"

def apply_template(messages):
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def parse_class(raw):
    t = (raw or "").strip().upper()
    for cls in sorted(ALLOWED, key=len, reverse=True):
        if cls in t[:80]:
            return cls
    return "INVALID"

# Inference
print("\n[2/3] 10 ornek inference...")
preds = []
valid = 0
correct = 0
t0 = time.time()
for i, r in enumerate(spot, 1):
    user_text = make_user_text(r)
    prompt = apply_template([{"role":"user","content":user_text}])
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    pred = parse_class(raw)
    preds.append(pred)
    ok = pred in ALLOWED
    is_correct = pred == r["output"]
    valid += int(ok)
    correct += int(is_correct)
    mark = "OK" if is_correct else ("VALID" if ok else "BAD")
    print(f"   [{i}] gold={r['output']:<22s} pred={pred:<22s} {mark} raw='{raw[:50]}'")
print(f"   Inference sure: {time.time()-t0:.1f}s ({(time.time()-t0)/len(spot):.2f}s/ornek)")

# Audit
print("\n[3/3] Spot audit")
print("=" * 80)
unique = len(set(preds))
all_other = all(p == "OTHER" for p in preds)
print(f"Valid:   {valid}/{len(spot)} ({100*valid/len(spot):.0f}%)")
print(f"Correct: {correct}/{len(spot)} ({100*correct/len(spot):.0f}%) -- zero-shot baseline ~5/10")
print(f"Unique predictions: {unique}, All OTHER? {all_other}")
print(f"Tahmin dagilimi: {dict(Counter(preds))}")
print(f"Peak GPU: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

# Acceptance
format_ok = valid == len(spot)
collapse_ok = unique >= 3 and not all_other
quality_ok = correct >= 4

print("\n" + ("-" * 80))
if format_ok and collapse_ok and quality_ok:
    print("SPOT ACCEPT: Full eval'a gecilebilir (Blok 15D-EVAL).")
else:
    sebepler = []
    if not format_ok: sebepler.append(f"format invalid ({valid}/{len(spot)})")
    if not collapse_ok: sebepler.append(f"mode collapse (unique={unique}, all_other={all_other})")
    if not quality_ok: sebepler.append(f"accuracy dusuk ({correct}/{len(spot)}, hedef >=4)")
    print(f"SPOT NOT ACCEPT: {', '.join(sebepler)}")


# BLOK 15D-EVAL - V4 QLoRA-S3 Full Eval (val 550 + test 1650)

**ON KOSUL:** Blok 15C-SPOT 'SPOT ACCEPT' yazmali.
**Restart Kernel** yapilmasi gerekmiyor (spot zaten model'i yukledi); sadece bu hucre devam eder.

**Adapter:** `output/qlora_adapters/v4_s3_full_2epoch_lr1e4/adapter/`
**Tahmini sure:** val ~30 dk + test ~90 dk = **~2 saat**

**Cikti:**
- `output/qlora_eval/qlora_v4_s3_val.csv` / `_test.csv` (raw + parsed predictions)
- `output/qlora_eval/qlora_v4_s3_{val,test}_results_summary.csv` (full + valid_only metrics)
- `output/qlora_eval/qlora_v4_s3_{val,test}_audit.txt` (per-class P/R/F1 + ZS karsilastirma)
- `output/qlora_eval/qlora_v4_s3_{val,test}_confusion.csv` (12x12 with INVALID)

**Metrikler:** Accuracy, Macro-F1, Weighted-F1, Per-class P/R/F1, Confusion Matrix, Tahmin bias
**Karsilastirma:** Zero-shot V4 ile delta (val acc=0.4818 MF1=0.4367, test acc=0.5152 MF1=0.4681)

**Checkpoint/resume:** Her 50 ornekte CSV'ye yaz, kesinti olursa kaldigi yerden devam (`*_checkpoint.csv`).


In [ ]:
import json
import gc
import time
from pathlib import Path
from collections import Counter

import torch
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# CONFIG
DATASET_DIR = Path(r"./workdir/output")
ADAPTER_PATH = DATASET_DIR / "qlora_adapters" / "v4_s3_full_2epoch_lr1e4" / "adapter"
SPLIT_YOL = DATASET_DIR / "discharge_split.parquet"
OUT_DIR = DATASET_DIR / "qlora_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ZS_DIR = DATASET_DIR / "zeroshot"

MODEL_ID = "Qwen/Qwen3.5-4B"
MAX_SEQ_LEN = 512
INPUT_TOKEN_BUDGET = 320
MAX_NEW_TOKENS = 8

ALLOWED = [
    "SEPSIS", "STROKE", "ATRIAL_FIBRILLATION", "AKI", "PNEUMONIA", "UTI",
    "HEART_FAILURE", "GI_BLEEDING", "DIABETES_COMPLICATION",
    "COPD_EXACERBATION", "OTHER"
]

INSTRUCTION = (
    "Classify the following admission note into exactly one of these 11 classes: "
    "SEPSIS, STROKE, ATRIAL_FIBRILLATION, AKI, PNEUMONIA, UTI, HEART_FAILURE, "
    "GI_BLEEDING, DIABETES_COMPLICATION, COPD_EXACERBATION, OTHER. "
    "Output only the class name."
)

FUZZY_MAP = {
    "SEPS":"SEPSIS","SEPT":"SEPSIS","PNEUM":"PNEUMONIA",
    "STROK":"STROKE","CVA":"STROKE",
    "ATRIAL":"ATRIAL_FIBRILLATION","AFIB":"ATRIAL_FIBRILLATION","FIBRIL":"ATRIAL_FIBRILLATION",
    "AKI":"AKI","KIDNEY":"AKI","RENAL":"AKI",
    "UTI":"UTI","URIN":"UTI",
    "HEART":"HEART_FAILURE","CHF":"HEART_FAILURE",
    "GI":"GI_BLEEDING","BLEED":"GI_BLEEDING",
    "DIAB":"DIABETES_COMPLICATION","DKA":"DIABETES_COMPLICATION",
    "COPD":"COPD_EXACERBATION","OTHER":"OTHER","NONE":"OTHER",
}

print("=" * 80)
print("BLOK 15D-EVAL - V4 QLoRA-S3 Full (val 550 + test 1650)")
print("=" * 80)
print(f"Adapter: {ADAPTER_PATH}")
assert ADAPTER_PATH.exists(), f"Adapter yok: {ADAPTER_PATH}"

# Model (spot'tan kalmadiysa yukle)
if "model" not in globals() or "tokenizer" not in globals():
    print("\n[Model yukleniyor]")
    for name in ["model", "base", "tokenizer", "peft_model"]:
        if name in globals():
            try: globals()[name] = None
            except: pass
            try: del globals()[name]
            except: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb, device_map="auto",
        torch_dtype=torch.float16, local_files_only=True, trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base, str(ADAPTER_PATH))
    model.eval()
    print(f"Model yuklendi: {time.time()-t0:.1f}s, GPU: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
else:
    print("\nModel zaten yuklu (Blok 15C-SPOT'tan), spot ile ayni adapter olmali.")
    model.eval()
    tokenizer.padding_side = "left"

# Helpers (train ile birebir ayni)
def trim_input_to_budget(text, budget=INPUT_TOKEN_BUDGET):
    ids = tokenizer(text or "", add_special_tokens=False)["input_ids"][:budget]
    return tokenizer.decode(ids, skip_special_tokens=True).strip()

def make_user_text_from_row(row):
    inp = row.get("triage_input_v4_timeaware_masked", "") or ""
    return f"{INSTRUCTION}\n\n{trim_input_to_budget(inp)}"

def apply_template(messages):
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def parse_class(raw):
    if not raw: return "INVALID", "empty"
    t = raw.strip().upper()
    for cls in sorted(ALLOWED, key=len, reverse=True):
        if cls in t[:80]:
            return cls, "direct"
    for k, v in FUZZY_MAP.items():
        if k in t[:80]:
            return v, "fuzzy"
    return "INVALID", "invalid"

def inference(input_text):
    user_text = f"{INSTRUCTION}\n\n{trim_input_to_budget(input_text)}"
    prompt = apply_template([{"role":"user","content":user_text}])
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return raw, int(inputs.input_ids.shape[1])


def run_split(split_name, hedef_n):
    print(f"\n{'='*80}")
    print(f"[SPLIT: {split_name}] hedef n={hedef_n}")
    print(f"{'='*80}")

    df = pd.read_parquet(SPLIT_YOL)
    sub_df = df[df["split"] == split_name].reset_index(drop=True)
    assert len(sub_df) == hedef_n, f"{split_name}: {len(sub_df)} bulundu, hedef {hedef_n}"

    cp_yol = OUT_DIR / f"qlora_v4_s3_{split_name}_checkpoint.csv"
    final_yol = OUT_DIR / f"qlora_v4_s3_{split_name}.csv"

    if cp_yol.exists():
        cp = pd.read_csv(cp_yol)
        islenen = set(cp["note_id"].astype(str))
        kayitlar = cp.to_dict("records")
        print(f"Resume: {len(islenen)} kayit zaten var.")
    else:
        islenen = set()
        kayitlar = []

    t_baslangic = time.time()
    sayac_yeni = 0
    for i, row in sub_df.iterrows():
        nid = str(row["note_id"])
        if nid in islenen: continue
        inp = row["triage_input_v4_timeaware_masked"] if pd.notna(row["triage_input_v4_timeaware_masked"]) else ""
        t0 = time.time()
        raw, in_n = inference(inp)
        sure = time.time() - t0
        parsed, kat = parse_class(raw)
        kayitlar.append({
            "note_id": nid,
            "subject_id": int(row["subject_id"]),
            "hadm_id": int(row["hadm_id"]),
            "label": row["label"],
            "raw_output": raw,
            "parsed_pred": parsed,
            "parse_kategori": kat,
            "in_tokens": int(in_n),
            "sure_s": round(sure, 3),
        })
        sayac_yeni += 1
        if len(kayitlar) % 50 == 0:
            pd.DataFrame(kayitlar).to_csv(cp_yol, index=False)
            gecen = time.time() - t_baslangic
            ort = gecen / sayac_yeni if sayac_yeni > 0 else 0
            kalan = (hedef_n - len(kayitlar)) * ort
            print(f"   [{len(kayitlar):>4}/{hedef_n}] gecen {gecen/60:.1f}dk ort {ort:.2f}s/ornek kalan ~{kalan/60:.1f}dk")

    df_son = pd.DataFrame(kayitlar)
    df_son.to_csv(final_yol, index=False)
    df_son.to_csv(cp_yol, index=False)
    toplam = time.time() - t_baslangic
    print(f"\n   BITTI. {len(kayitlar)} kayit, {toplam/60:.1f}dk, ort {toplam/max(sayac_yeni,1):.2f}s/yeni ornek")

    # AUDIT
    n = len(df_son)
    ks = df_son["parse_kategori"].value_counts().to_dict()
    direct = ks.get("direct", 0)
    fuzzy = ks.get("fuzzy", 0)
    invalid = ks.get("invalid", 0) + ks.get("empty", 0)
    print(f"\nParser: direct={direct} ({100*direct/n:.1f}%), fuzzy={fuzzy} ({100*fuzzy/n:.1f}%), invalid={invalid} ({100*invalid/n:.1f}%)")

    y_true = df_son["label"].values
    y_pred = df_son["parsed_pred"].values
    valid_mask = y_pred != "INVALID"

    acc_full = accuracy_score(y_true, y_pred)
    mf1_full = f1_score(y_true, y_pred, average="macro", zero_division=0)
    wf1_full = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    acc_v = accuracy_score(y_true[valid_mask], y_pred[valid_mask]) if valid_mask.sum() else 0.0
    mf1_v = f1_score(y_true[valid_mask], y_pred[valid_mask], average="macro", zero_division=0) if valid_mask.sum() else 0.0
    wf1_v = f1_score(y_true[valid_mask], y_pred[valid_mask], average="weighted", zero_division=0) if valid_mask.sum() else 0.0

    print(f"\nQLoRA-S3 {split_name} metrikleri:")
    print(f"   {'Subset':<22s} {'N':>5s} {'Acc':>8s} {'MacroF1':>10s} {'WF1':>8s}")
    print("-" * 60)
    print(f"   {'Full':<22s} {n:>5} {acc_full:>8.4f} {mf1_full:>10.4f} {wf1_full:>8.4f}")
    print(f"   {'Valid only':<22s} {valid_mask.sum():>5} {acc_v:>8.4f} {mf1_v:>10.4f} {wf1_v:>8.4f}")

    labels = sorted(set(y_true) | (set(y_pred) - {"INVALID"}))
    p, r, fpc, supp = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    print(f"\nPer-class:")
    for cls, pi, ri, fi, si in zip(labels, p, r, fpc, supp):
        print(f"   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}")

    print(f"\nTahmin dagilimi (bias):")
    for cls, sayi in df_son["parsed_pred"].value_counts().items():
        print(f"   {cls:<25s} {sayi:>4} ({100*sayi/n:>5.1f}%)")

    # ZS karsilastirma
    zs_summary = ZS_DIR / f"zeroshot_v4_{split_name}_results_summary.csv"
    zs_full_acc, zs_full_mf1 = None, None
    if zs_summary.exists():
        zs = pd.read_csv(zs_summary)
        zs_full_acc = float(zs[zs["subset"]=="full"]["accuracy"].iloc[0])
        zs_full_mf1 = float(zs[zs["subset"]=="full"]["macro_f1"].iloc[0])
        print(f"\nQLoRA-S3 vs Zero-shot V4 ({split_name}):")
        print(f"   {'':<22s} {'ZS-V4':>8s} {'QL-S3':>8s} {'Delta':>10s}")
        print(f"   {'Full Acc':<22s} {zs_full_acc:>8.4f} {acc_full:>8.4f} {acc_full-zs_full_acc:>+10.4f}")
        print(f"   {'Full MacroF1':<22s} {zs_full_mf1:>8.4f} {mf1_full:>8.4f} {mf1_full-zs_full_mf1:>+10.4f}")

    # Summary kaydet
    ozet = pd.DataFrame({
        "subset": ["full","valid_only"],
        "n": [n, valid_mask.sum()],
        "accuracy": [acc_full, acc_v],
        "macro_f1": [mf1_full, mf1_v],
        "weighted_f1": [wf1_full, wf1_v],
    })
    ozet.to_csv(OUT_DIR / f"qlora_v4_s3_{split_name}_results_summary.csv", index=False)

    # Audit
    audit_yol = OUT_DIR / f"qlora_v4_s3_{split_name}_audit.txt"
    with open(audit_yol, "w", encoding="utf-8") as fh:
        fh.write(f"BLOK 15D-EVAL ({split_name}) AUDIT\n")
        fh.write("=" * 70 + "\n")
        fh.write(f"Adapter: v4_s3_full_2epoch_lr1e4/adapter\n")
        fh.write(f"Split: {split_name}, n={n}\n")
        fh.write(f"Parser: direct={direct}, fuzzy={fuzzy}, invalid={invalid}\n\n")
        fh.write(f"Full       : acc={acc_full:.4f}, macroF1={mf1_full:.4f}, wF1={wf1_full:.4f}\n")
        fh.write(f"Valid only : acc={acc_v:.4f}, macroF1={mf1_v:.4f}, wF1={wf1_v:.4f}\n\n")
        if zs_full_acc is not None:
            fh.write(f"Zero-shot V4 ({split_name}): acc={zs_full_acc:.4f}, MF1={zs_full_mf1:.4f}\n")
            fh.write(f"Delta: acc{acc_full-zs_full_acc:+.4f}, MF1{mf1_full-zs_full_mf1:+.4f}\n\n")
        fh.write("Per-class:\n")
        for cls, pi, ri, fi, si in zip(labels, p, r, fpc, supp):
            fh.write(f"   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}\n")
        fh.write("\nTahmin dagilimi:\n")
        for cls, sayi in df_son["parsed_pred"].value_counts().items():
            fh.write(f"   {cls:<25s} {sayi:>4} ({100*sayi/n:>5.1f}%)\n")

    # Confusion
    cm = confusion_matrix(y_true, y_pred, labels=ALLOWED + ["INVALID"])
    cm_df = pd.DataFrame(cm, index=ALLOWED + ["INVALID"], columns=ALLOWED + ["INVALID"])
    cm_df.to_csv(OUT_DIR / f"qlora_v4_s3_{split_name}_confusion.csv")

    return {
        "split": split_name, "n": n,
        "accuracy": acc_full, "macro_f1": mf1_full, "weighted_f1": wf1_full,
    }


# RUN VAL + TEST
print("\n[1/2] VAL (550 ornek, ~30 dk)")
val_sonuc = run_split("val", 550)

print("\n[2/2] TEST (1650 ornek, ~90 dk)")
test_sonuc = run_split("test", 1650)

print("\n" + "=" * 80)
print("BLOK 15D-EVAL TAMAMLANDI.")
print("=" * 80)
print(f"VAL : acc={val_sonuc['accuracy']:.4f}, MF1={val_sonuc['macro_f1']:.4f}, wF1={val_sonuc['weighted_f1']:.4f}")
print(f"TEST: acc={test_sonuc['accuracy']:.4f}, MF1={test_sonuc['macro_f1']:.4f}, wF1={test_sonuc['weighted_f1']:.4f}")
print(f"\nCikti klasoru: {OUT_DIR}")
print(f"Peak GPU: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")


# BLOK 16A-V4-VAL-GEMMA4 — Gemma 4 E2B-it V4 Time-aware Zero-shot Val

In [ ]:
"""Blok 16A-V4-VAL-GEMMA4: V4 (TIME-AWARE TRIAGE) zero-shot val on Gemma 4 E2B-it.
Ayni V4 prompt + parser + FUZZY_MAP (Blok 13B-V4-VAL ile bire bir).
Tek fark: model = local Gemma 4 E2B-it, 4-bit nf4.
TEST'e otomatik gecmez. Sonuc dosyalari ayri (gemma4 suffix)."""
import pandas as pd
import numpy as np
import torch
import time
import gc
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

OUTPUT_DIR = Path(r"./workdir/output")
SPLIT_YOL  = OUTPUT_DIR / "discharge_split.parquet"
ZS_DIR     = OUTPUT_DIR / "zeroshot"
ZS_DIR.mkdir(parents=True, exist_ok=True)

GEMMA_DIR = r"./data/models/google_gemma-4-E2B-it"

print("=" * 70)
print("BLOK 16A-V4-VAL-GEMMA4 — Gemma 4 E2B-it V4 Time-aware Zero-shot Val")
print("=" * 70)
print(f"   Model: {GEMMA_DIR}")
print("   Input: triage_input_v4_timeaware_masked")
print("   KURAL: Sadece val. Test'e otomatik gecmez.")

ALLOWED_CLASSES = [
    "SEPSIS","STROKE","ATRIAL_FIBRILLATION","AKI","PNEUMONIA","UTI",
    "HEART_FAILURE","GI_BLEEDING","DIABETES_COMPLICATION","COPD_EXACERBATION","OTHER"
]

PROMPT_TEMPLATE = """You are a clinical NLP system. Given a patient's triage information collected at first contact (chief complaint, symptoms, brief history, medications, allergies, vital signs), classify the most likely PRIMARY reason for admission into ONE of these 11 categories:

SEPSIS - Systemic infection, septic shock, bacteremia
STROKE - Ischemic or hemorrhagic stroke, CVA
ATRIAL_FIBRILLATION - Atrial fibrillation/flutter (acute)
AKI - Acute kidney injury, acute renal failure
PNEUMONIA - Lung infection (bacterial, viral, aspiration)
UTI - Urinary tract infection, pyelonephritis, cystitis
HEART_FAILURE - Congestive heart failure, decompensated CHF
GI_BLEEDING - Upper or lower GI hemorrhage
DIABETES_COMPLICATION - DKA, HHS, severe hyper/hypoglycemia
COPD_EXACERBATION - Acute exacerbation of COPD
OTHER - None of the above

Rules:
- Output exactly one class label from the list, using the exact uppercase label string.
- Choose the SINGLE most likely PRIMARY reason for admission.

TRIAGE NOTE:
{input}

PRIMARY DIAGNOSIS:"""

FUZZY_MAP = {
    "SEPS":"SEPSIS","SEPT":"SEPSIS","PNEUM":"PNEUMONIA",
    "STROK":"STROKE","CVA":"STROKE",
    "ATRIAL":"ATRIAL_FIBRILLATION","AFIB":"ATRIAL_FIBRILLATION","FIBRIL":"ATRIAL_FIBRILLATION",
    "AKI":"AKI","KIDNEY":"AKI","RENAL":"AKI",
    "UTI":"UTI","URIN":"UTI",
    "HEART":"HEART_FAILURE","CHF":"HEART_FAILURE",
    "GI":"GI_BLEEDING","BLEED":"GI_BLEEDING",
    "DIAB":"DIABETES_COMPLICATION","DKA":"DIABETES_COMPLICATION",
    "COPD":"COPD_EXACERBATION","OTHER":"OTHER","NONE":"OTHER",
}

def parse_class(raw):
    if not raw: return "INVALID","empty"
    text = raw.strip().upper()
    for cls in sorted(ALLOWED_CLASSES, key=len, reverse=True):
        if cls in text[:80]: return cls,"direct"
    for k,v in FUZZY_MAP.items():
        if k in text[:80]: return v,"fuzzy"
    return "INVALID","invalid"

# RAM/GPU temizle (Qwen yuklenmis olabilir)
try:
    del model, tok
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Model
print("\n[1/4] Gemma 4 E2B-it yukleme (4-bit nf4)...")
t0 = time.time()
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(GEMMA_DIR, local_files_only=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    GEMMA_DIR, quantization_config=bnb,
    device_map="auto", torch_dtype=torch.float16, local_files_only=True,
)
model.eval()
print(f"   Yuklendi {time.time()-t0:.1f}s, GPU {torch.cuda.memory_allocated()/1024**2:.0f}MB")

def inference(input_text):
    prompt = PROMPT_TEMPLATE.format(input=input_text)
    msgs = [{"role":"user","content":prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return raw, inputs.input_ids.shape[1]

# Veri
print("\n[2/4] V4 val seti...")
df = pd.read_parquet(SPLIT_YOL)
val_df = df[df["split"]=="val"].reset_index(drop=True)
print(f"   Val: {len(val_df):,}")

# Inference
print("\n[3/4] V4 val inference Gemma4 (checkpoint/resume)...")
cp_yol    = ZS_DIR / "zeroshot_v4_val_gemma4_checkpoint.csv"
final_yol = ZS_DIR / "zeroshot_v4_val_gemma4.csv"

if cp_yol.exists():
    cp = pd.read_csv(cp_yol)
    islenen = set(cp["note_id"].astype(str))
    kayitlar = cp.to_dict("records")
    print(f"   Resume: {len(islenen)} kayit zaten var.")
else:
    islenen = set()
    kayitlar = []

t_baslangic = time.time()
sayac_yeni = 0

for i, row in val_df.iterrows():
    nid = str(row["note_id"])
    if nid in islenen: continue
    inp = row["triage_input_v4_timeaware_masked"] if pd.notna(row["triage_input_v4_timeaware_masked"]) else ""
    t0 = time.time()
    raw, in_n = inference(inp)
    sure = time.time() - t0
    parsed, kat = parse_class(raw)
    kayitlar.append({
        "note_id": nid, "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]), "label": row["label"],
        "raw_output": raw, "parsed_pred": parsed,
        "parse_kategori": kat, "in_tokens": int(in_n),
        "sure_s": round(sure, 3),
        "leakage_severity_final": str(row.get("strict_current_leakage_severity_final","UNK")),
    })
    sayac_yeni += 1
    if len(kayitlar) % 50 == 0:
        pd.DataFrame(kayitlar).to_csv(cp_yol, index=False)
        gecen = time.time() - t_baslangic
        ort = gecen / sayac_yeni if sayac_yeni > 0 else 0
        kalan = (len(val_df) - len(kayitlar)) * ort
        print(f"   [{len(kayitlar):>3}/{len(val_df)}] gecen {gecen/60:.1f}dk ort {ort:.2f}s kalan ~{kalan/60:.1f}dk")

df_son = pd.DataFrame(kayitlar)
df_son.to_csv(final_yol, index=False)
df_son.to_csv(cp_yol, index=False)
toplam = time.time() - t_baslangic
print(f"   BITTI. {len(kayitlar)} kayit, {toplam/60:.1f}dk")

# AUDIT
print("\n[4/4] V4 VAL AUDIT (Gemma4)")
print("=" * 70)
n = len(df_son)
ks = df_son["parse_kategori"].value_counts().to_dict()
direct = ks.get("direct", 0); fuzzy = ks.get("fuzzy", 0)
invalid = ks.get("invalid", 0) + ks.get("empty", 0)
print(f"\nParser: direct={direct} ({100*direct/n:.1f}%) fuzzy={fuzzy} invalid={invalid} ({100*invalid/n:.1f}%)")

y_true = df_son["label"].values
y_pred = df_son["parsed_pred"].values
valid = y_pred != "INVALID"

acc_full = accuracy_score(y_true, y_pred)
mf1_full = f1_score(y_true, y_pred, average="macro", zero_division=0)
wf1_full = f1_score(y_true, y_pred, average="weighted", zero_division=0)
acc_v = accuracy_score(y_true[valid], y_pred[valid]) if valid.sum() else 0.0
mf1_v = f1_score(y_true[valid], y_pred[valid], average="macro", zero_division=0) if valid.sum() else 0.0

print(f"\nV4 Val metrikleri (Gemma4):")
print(f"   {'Subset':<22s} {'N':>5s} {'Acc':>8s} {'MacroF1':>10s} {'WF1':>8s}")
print("-" * 60)
print(f"   {'Full':<22s} {n:>5} {acc_full:>8.4f} {mf1_full:>10.4f} {wf1_full:>8.4f}")
print(f"   {'Valid only':<22s} {valid.sum():>5} {acc_v:>8.4f} {mf1_v:>10.4f}")

# Per-class
labels = sorted(set(y_true) | (set(y_pred) - {"INVALID"}))
p, r, f, supp = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
print(f"\nPer-class:")
for cls, pi, ri, fi, si in zip(labels, p, r, f, supp):
    print(f"   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}")

# Tahmin dagilimi
print(f"\nTahmin dagilimi (bias):")
for cls, sayi in df_son["parsed_pred"].value_counts().items():
    print(f"   {cls:<25s} {sayi:>4} ({100*sayi/n:>5.1f}%)")

# Qwen V4 val ile karsilastirma
print(f"\nQwen V4 vs Gemma4 V4 (val):")
qwen_summary_yol = ZS_DIR / "zeroshot_v4_val_results_summary.csv"
if qwen_summary_yol.exists():
    q = pd.read_csv(qwen_summary_yol)
    q_full = float(q[q["subset"]=="full"]["accuracy"].iloc[0])
    q_mf1  = float(q[q["subset"]=="full"]["macro_f1"].iloc[0])
    print(f"   {'':<22s} {'Qwen':>8s} {'Gemma4':>8s} {'Delta':>8s}")
    print(f"   {'Full Acc':<22s} {q_full:>8.4f} {acc_full:>8.4f} {acc_full-q_full:>+8.4f}")
    print(f"   {'Full MacroF1':<22s} {q_mf1:>8.4f} {mf1_full:>8.4f} {mf1_full-q_mf1:>+8.4f}")
else:
    print("   (Qwen V4 val summary bulunamadi)")

# Summary kaydet
ozet = pd.DataFrame({
    "subset": ["full","valid_only"],
    "n": [n, valid.sum()],
    "accuracy": [acc_full, acc_v],
    "macro_f1": [mf1_full, mf1_v],
    "weighted_f1": [wf1_full, 0.0],
})
ozet.to_csv(ZS_DIR / "zeroshot_v4_val_gemma4_results_summary.csv", index=False)

with open(ZS_DIR / "zeroshot_v4_val_gemma4_audit.txt", "w", encoding="utf-8") as fh:
    fh.write("BLOK 16A-V4-VAL-GEMMA4 AUDIT\n")
    fh.write("=" * 70 + "\n")
    fh.write(f"Model: {GEMMA_DIR}\n")
    fh.write(f"Val n: {n}\n")
    fh.write(f"Parser: direct={direct}, fuzzy={fuzzy}, invalid={invalid}\n\n")
    fh.write(f"Full      : acc={acc_full:.4f}, macroF1={mf1_full:.4f}, wF1={wf1_full:.4f}\n")
    fh.write(f"Valid only: acc={acc_v:.4f}, macroF1={mf1_v:.4f}\n\n")
    fh.write("Per-class:\n")
    for cls, pi, ri, fi, si in zip(labels, p, r, f, supp):
        fh.write(f"   {cls:<25s} P={pi:.3f} R={ri:.3f} F1={fi:.3f} N={si}\n")

# Karar
print("\n" + "=" * 70)
print("KARAR")
print("=" * 70)
print(f"   Gemma4 V4 val full acc: {acc_full:.4f}")
if acc_full < 0.20:
    print(f"   Acc < %20: TEST BASLATILMASIN. Promp/parser revize gerekebilir.")
elif acc_full <= 0.40:
    print(f"   Acc %20-40: Beklenen aralik. Test icin kullanici onayi iste.")
else:
    print(f"   Acc > %40: Iyi sonuc. Test icin kullanici onayi iste.")
print(f"   TEST'e otomatik gecis YOK.")
print("=" * 70)


In [ ]:
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# =========================
# PATH / MODEL AYARLARI
# =========================
MODEL_ID = "Qwen/Qwen3.5-4B"

ADAPTER_PATH = Path(
    r"./workdir/output/qlora_adapters/v4_s3_full_2epoch_lr1e4/adapter"
)

assert ADAPTER_PATH.exists(), f"Adapter bulunamadi: {ADAPTER_PATH}"

ALLOWED = [
    "SEPSIS",
    "STROKE",
    "ATRIAL_FIBRILLATION",
    "AKI",
    "PNEUMONIA",
    "UTI",
    "HEART_FAILURE",
    "GI_BLEEDING",
    "DIABETES_COMPLICATION",
    "COPD_EXACERBATION",
    "OTHER",
]

INSTRUCTION = (
    "Classify the following admission note into exactly one of these 11 classes: "
    "SEPSIS, STROKE, ATRIAL_FIBRILLATION, AKI, PNEUMONIA, UTI, HEART_FAILURE, "
    "GI_BLEEDING, DIABETES_COMPLICATION, COPD_EXACERBATION, OTHER. "
    "Output only the class name."
)

MAX_SEQ_LEN = 512
INPUT_TOKEN_BUDGET = 320
MAX_NEW_TOKENS = 12

# =========================
# MODEL YUKLEME
# =========================
print("Tokenizer yukleniyor...")
tokenizer = AutoTokenizer.from_pretrained(
    str(ADAPTER_PATH),
    trust_remote_code=True,
    local_files_only=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

print("Base model 4-bit yukleniyor...")
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    local_files_only=True,
)

print("LoRA adapter yukleniyor...")
model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_PATH),
)

model.eval()

print("Hazir. Adapter ile soru sorabilirsin.")

# =========================
# YARDIMCI FONKSIYONLAR
# =========================
def trim_input_to_budget(text, budget=INPUT_TOKEN_BUDGET):
    ids = tokenizer(text or "", add_special_tokens=False)["input_ids"][:budget]
    return tokenizer.decode(ids, skip_special_tokens=True).strip()


def apply_template(user_text):
    try:
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": user_text}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": user_text}],
            tokenize=False,
            add_generation_prompt=True,
        )


def parse_class(raw):
    if raw is None:
        return "INVALID"

    text = raw.strip().upper()

    for cls in sorted(ALLOWED, key=len, reverse=True):
        if cls in text[:80]:
            return cls

    return "INVALID"


def qwen_sor(klinik_metin):
    klinik_metin = trim_input_to_budget(klinik_metin)
    user_text = f"{INSTRUCTION}\n\n{klinik_metin}"

    prompt = apply_template(user_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    raw = tokenizer.decode(
        output[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    ).strip()

    parsed = parse_class(raw)

    return {
        "raw_output": raw,
        "prediction": parsed,
        "input_tokens": int(inputs.input_ids.shape[1]),
    }

In [ ]:
%pip install --upgrade pip
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install -U transformers accelerate peft bitsandbytes safetensors sentencepiece protobuf